In [ ]:
import pickle
import sys
# from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import pandas as pd
from matplotlib.ticker import MaxNLocator
import numpy as np
import os
from scipy.spatial import procrustes
from sklearn.decomposition import PCA
from scipy.stats import pearsonr

import matplotlib.pyplot as plt
import scipy
from scipy.interpolate import UnivariateSpline
from scipy.spatial import procrustes
from scipy.stats import wasserstein_distance, iqr
import matplotlib.pylab as pl
import ot
from scipy.special import iv
try:
    import pycircular as pc
    has_pycircular = True
except ImportError:
    has_pycircular = False
    
# how to refer to these functions in a different folder? Need to compile as a package?
# from cytosim_plot_functions import *
# from cytosim_output_functions import *
import matplotlib.cm as cm
save_figures = True

# working_dir = '/Users/makamats/Google Drive/drubin_lab/Modeling/cytosim_current/cytosim/parameter_sweeps_maxReporting/simulations_cortex/'

# default working directory is now the cytosim/folder. From when you open this notebook.
# working_dir = os.path.abspath(os.path.join(os.curdir, '..'))
# this sets the working directory as the folder above the folder where this notebook appears.
# working_dir = os.path.abspath('./..')

# or:
# working_dir =  '/Users/makamats/Google Drive/drubin_lab/Modeling/cytosim_dmitrieff/cytosim/'
# output_dir = working_dir + '/simulations/endocytosis_mammalian_timestep_vary_sameseed_output'

working_dir = '/Volumes/homes/akamatsuadmin'
simulations_dir = '/_Cytosim/abhi_simulations/'
#output_name = 'baseline_sims'
output_name = 'endocytosis2_baseline_aws'

output_dir = working_dir + simulations_dir + output_name + '_output'
output_dir

In [ ]:
working_dir

In [ ]:
import matplotlib.pyplot as plt  # plotting
plt.style.use('seaborn-v0_8-colorblind') # set plot style
plt.cool()                          # heatmap color scheme
%matplotlib inline

import seaborn as sns  # nicer plotting
sns.set_style('whitegrid')  # set plot style

from scipy.stats import binned_statistic_2d

SMALL_SIZE = 12
MEDIUM_SIZE = 18
BIGGER_SIZE = 20

plt.rc('font', size=MEDIUM_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=MEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=MEDIUM_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=MEDIUM_SIZE)    # legend fontsize
plt.rc('figure', titlesize=MEDIUM_SIZE)  # fontsize of the figure title

from matplotlib.colors import LogNorm
from matplotlib.colors import SymLogNorm
import datetime

    
now = datetime.datetime.now()
date = now.strftime('%Y%m%d')
pref = date+'_'+output_name
pref

In [ ]:
from matplotlib.ticker import MaxNLocator

# Set style and font globally
sns.set(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'

def plot_multiple_errorbars_iqr(ax, means, std1, std2, c, label, name, yticks):
    # Plot mean line
    ax.plot(means.index, means, color=c, label=name, linewidth=2)

    # Fill between 90th and 100th percentiles (std1 = 90th, std2 = 100th)
    ax.fill_between(means.index, np.asarray(std1), np.asarray(std2),
                    alpha=0.25, edgecolor=c, facecolor=c, linewidth=1)

    #ax.scatter(means.index, means, color=c, alpha=1, s=20, zorder=3)

    # Labels
    ax.set_xlabel('Actin Model Points', fontsize=18)
    ax.set_ylabel(label, fontsize=18)

    # Title handled externally for flexibility
    ax.set_yticks(yticks)
    
    # Set ticks format to two digits using MaxNLocator
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))  # Format y ticks to 2 decimal places
    ax.tick_params(axis='both', labelsize=14)
    ax.grid(True)
    ax.legend(loc='upper right', fontsize=14)


# Load Data

In [ ]:
# Load the recalibrated points data

arp23 = True
tension = True

if arp23 is True:
    # List of runs
    runs = [
        #'endocytosis2_baseline_aws_output', 
        'endocytosis2_baseline_zero_diffusion_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
        'endocytosis2_baseline_zero_diffusion_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
        #'endocytosis2_baseline_asymmetric_Arp23_34donut_aws_output',
        #'endocytosis2_baseline_asymmetric_Arp23_12donut_aws_output',
        #'endocytosis2_baseline_asymmetric_Arp23_14donut_aws_output'
    ]

    # Directory containing the output data
    output_dir_base = working_dir + simulations_dir 

    # Dictionaries to store the loaded data
    branched_actin_bound_points = {}
    df_with_lengths = {}

    # Iterate through each run
    for i, run in enumerate(runs):
        try:
            # Construct the output directory for the current run
            output_dir = f"{output_dir_base}{run}"
            
            # Load the recalibrated points data
            branched_actin_bound_points[run] = pd.read_pickle(
                f"{output_dir}/dataframes/branched-actin-bound-to-hip1r_positions_recalibrated.pkl"
            )
            
            # Load and process the dataframe with lengths
            df_with_lengths[run] = pd.read_pickle(
                f"{output_dir}/dataframes/branched-actin-bound-to-hip1r-ends_recalibrated.pkl"
            ).set_index(['run', 'time']).sort_index()
            
            print(f"Arp23 Success, run {i} ({run})")
            
        except Exception as e:
            # Handle any errors encountered during loading
            print(f"Arp23 Failure run {i} ({run}): {e}")

if tension is True:
    # List of runs
    tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']

    # Dictionary to store the data for each run
    hip1r_assoc_positions = {}

    # Iterate through each run
    for i, run in enumerate(tension_runs):
        try:
            # Build the file path for the current run's data
            file_path = f'/Volumes/homes/akamatsuadmin/_Cytosim/abhi_simulations/Data_Ben/endocytosis2_baseline_noforcedepcap{run}_aws_output/dataframes/branched-actin-bound-to-hip1r_positions_recalibrated.pkl'
            
            # Load the data and store it in the dictionary
            hip1r_assoc_positions[run] = pd.read_pickle(file_path)
            
            print(f"Tension Success run {i} ({run})")
            
        except Exception as e:
            # Handle errors (e.g., file not found)
            print(f"Tension Failure run {i} ({run}): {e}")

In [ ]:
plot_order = ['_tension10', '', '_tension1000', '_tension5000', '_tension10000']
tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']

zero_trials = [
    'endocytosis2_baseline_zero_diffusion_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
]

zero_trials_tension = [
    'endocytosis2_baseline_zero_diffusion_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
]

labels=[10, 150, 1000, 5000, 10000]

# Figure One

## 1A

In [ ]:
# Load the recalibrated points data

arp23 = True
tension = True

if arp23 is True:
    # List of runs
    runs = [
        'endocytosis2_baseline_aws_output', 
        'endocytosis2_baseline_zero_diffusion_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output'
        #'endocytosis2_baseline_asymmetric_Arp23_34donut_aws_output',
        #'endocytosis2_baseline_asymmetric_Arp23_12donut_aws_output',
        #'endocytosis2_baseline_asymmetric_Arp23_14donut_aws_output'
    ]

    # Directory containing the output data
    output_dir_base = working_dir + simulations_dir 

    # Dictionaries to store the loaded data
   
    arp_dict = {}

    # Iterate through each run
    for i, run in enumerate(runs):
        try:
            # Construct the output directory for the current run
            output_dir = f"{output_dir_base}{run}"
            
            # Load the recalibrated points data
            arp_dict[run] = pd.read_pickle(
                f"{output_dir}/dataframes/all_couple_arp.pkl"
            )
   
            print(f"Arp23 Success, run {i} ({run})")
            
        except Exception as e:
            # Handle any errors encountered during loading
            print(f"Arp23 Failure run {i} ({run}): {e}")    



In [ ]:
# Setup
zero_trials = [
    'endocytosis2_baseline_zero_diffusion_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
]

condition_labels = ['Full', '3/4', '1/2', '1/4']
colors = sns.color_palette("viridis", n_colors=len(zero_trials))

# Set font properties
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 14})

fig = plt.figure(figsize=(12, 10))

# Fixed axis limits
xlim = (-0.1, 0.1)
ylim = (-0.1, 0.1)
zlim = (-0.42, -0.38)

# Plotting
for i, (cond, label) in enumerate(zip(zero_trials, condition_labels)):
    ax = fig.add_subplot(2, 2, i+1, projection='3d')
    
    data = arp_dict[cond].reset_index()
    data = data.sample(frac=1).reset_index(drop=True)
    
    selected_run = np.random.choice(data['run'].unique())
    data_run_t0 = data[(data['run'] == selected_run) & (data['time'] == 0.1)].copy()
    
    ax.scatter(data_run_t0['xpos'], data_run_t0['ypos'], data_run_t0['zpos'], 
               color=colors[i], s=20)
    
    ax.set_title(label, fontsize=20)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)
    
    # Set 5 ticks per axis using fixed limits
    ax.set_xticks(np.linspace(*xlim, 5))
    ax.set_yticks(np.linspace(*ylim, 5))
    ax.set_zticks(np.linspace(*zlim, 5))
    
    # Format tick labels with one digit
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1g}'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1g}'))
    ax.zaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1g}'))

plt.tight_layout()
#plt.savefig('all_donuts.png',dpi=300)
plt.show()

## 1B

## 1C

In [ ]:
def WD_quick(df):
    WD = []
    
    for time in df.index.get_level_values('time').unique():
        row = df.loc[(slice(None), time), 'direction']

        # Ensure row is a NumPy array
        row = np.concatenate(row.values) if isinstance(row.iloc[0], np.ndarray) else row.to_numpy()
        
        if row.size == 0 or np.any(np.isnan(row)):  # Skip if empty or contains NaN
            wd = np.nan
        else:
            row = (row + np.pi) / (2 * np.pi)  # Normalize to [0,1]
            rn = np.random.uniform(0, 1, 10000)
            wd = ot.wasserstein_circle(row, rn,p=1)


        WD.append((time, wd))

    return pd.DataFrame(WD, columns=['time', 'Wasserstein Distance']).set_index('time')

In [ ]:

# Initialize dictionaries
runs_dict_1 = {}
runs_dict_2 = {}
runs_dict_3 = {}

# DataFrames
df1 = branched_actin_bound_points['endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output']
df2 = branched_actin_bound_points['endocytosis2_baseline_zero_diffusion_output']
df3 = branched_actin_bound_points['endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output']


def process_run_data(df, runs_dict, prefix, suffix_range=50):
    """
    Process the run data for a given DataFrame and store the directions in a dictionary.

    Args:
    df: The DataFrame containing the data.
    runs_dict: The dictionary to store the processed data.
    prefix: The prefix for the run names (e.g., 'run-endocytosis2_baseline_zero_diffusion').
    suffix_range: The range of suffixes to process (default is 50).
    """
    for suffix in range(suffix_range):
        # Generate the run identifier with the correct suffix
        run_to_plot = f'{prefix}{suffix:04d}'  # Format as 0000, 0001, ..., 0049
        
        # Filter the data for the current run
        run_data = df[df['run'] == run_to_plot]
        
        # Check if the filtered run data is empty
        if run_data.empty:
            print(f'{run_to_plot} is empty, skipping.')
            continue  # Skip this run if it's empty

        # Store each direction separately for the current run
        direction_entries = []

        # Loop over unique times in the current run
        for time in run_data['time'].unique():
            time_data = run_data[run_data['time'] == time]
            
            ypos_time = time_data.ypos_recalibrated
            xpos_time = time_data.xpos_recalibrated
            
            # Compute directions (angles)
            directions = np.arctan2(ypos_time, xpos_time)
            
            # Append each direction individually for proper indexing
            for direction in directions:
                direction_entries.append({'run': run_to_plot, 'time': time, 'direction': direction})

        # Convert to DataFrame with MultiIndex for the current run and store in the dictionary
        all_directions = pd.DataFrame(direction_entries).set_index(['run', 'time'])
        
        # Store the DataFrame in the dictionary
        runs_dict[run_to_plot] = all_directions

# Process the two dictionaries
process_run_data(df1, runs_dict_1, 'run-endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut0000-')
process_run_data(df2, runs_dict_2, 'run-endocytosis2_baseline_zero_diffusion0000-')
process_run_data(df3, runs_dict_3, 'run-endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut0000-')


# Check if everything is stored correctly
print(f"Number of runs processed in dict 1: {len(runs_dict_1)}")
print(f"Number of runs processed in dict 2: {len(runs_dict_2)}")
print(f"Number of runs processed in dict 3: {len(runs_dict_3)}")

# Save the dictionaries to files (e.g., as pickles or CSVs)
runs_dict_1_df = pd.concat(runs_dict_1.values())
runs_dict_1_df.to_pickle('runs_dict_1.pkl')  # Save as a pickle file

runs_dict_2_df = pd.concat(runs_dict_2.values())
runs_dict_2_df.to_pickle('runs_dict_2.pkl')  # Save as a pickle file

runs_dict_3_df = pd.concat(runs_dict_3.values())
runs_dict_3_df.to_pickle('runs_dict_3.pkl')  # Save as a pickle file

print("Dictionaries have been saved successfully.")


In [ ]:

run_wd_dict_1 = {}
run_wd_dict_2 = {}
run_wd_dict_3 = {}

for run, df in runs_dict_1.items():
    run_wd_dict_1[run] = WD_quick(df)

for run, df in runs_dict_2.items():
    run_wd_dict_2[run] = WD_quick(df)

for run, df in runs_dict_3.items():
    run_wd_dict_3[run] = WD_quick(df)

In [ ]:

# Timepoints at which to plot polar filament density histograms
timepoints = [2.0, 5.0, 7.0, 13.0, 15.0]
run_to_plot = 'run-endocytosis2_baseline_zero_diffusion0000-0001'
df = branched_actin_bound_points['endocytosis2_baseline_zero_diffusion_output']
run_data = df[df['run'] == run_to_plot]
wd_dict_data = run_wd_dict_2[run_to_plot]

N = 5
bottom = 0
theta = np.linspace(-np.pi, np.pi, N, endpoint=True)
theta_mid = [(theta[i] + theta[i+1]) / 2 for i in range(len(theta) - 1)]
width = (2 * np.pi) / N

# Resize figure to be wider than tall for better two-row layout
fig, axs = plt.subplots(2, len(timepoints), figsize=(12, 8), subplot_kw={'polar': True})
fig.suptitle(f'Run {run_to_plot}: Polar Plots', fontsize=16, fontweight='bold')

# Add more space between rows
plt.subplots_adjust(hspace=0.3)

global_max_r = 0

for j, timept in enumerate(timepoints):
    if timept in run_data['time'].values:
        branched_actin_bound_points_recalibrated_time = run_data[run_data['time'] == timept]
        ypos_time = branched_actin_bound_points_recalibrated_time.ypos_recalibrated
        xpos_time = branched_actin_bound_points_recalibrated_time.xpos_recalibrated
        
        ypos_fil = ypos_time * 10**3
        xpos_fil = xpos_time * 10**3
        r_fil = np.sqrt(xpos_fil.values**2 + ypos_fil.values**2)
        theta_fil = np.arctan2(ypos_fil.values, xpos_fil.values)
        
        if len(r_fil) > 0:
            global_max_r = max(global_max_r, np.max(r_fil))
        
            direction = np.arctan2(ypos_time, xpos_time)
        else:
            direction = []

    # Row 1: Polar histograms of model points
    ax1 = axs[0, j]
    radii, _ = np.histogram(direction, bins=theta)
    ax1.bar(theta_mid, radii, width=width, bottom=bottom, edgecolor='black', alpha=0.9)
    ax1.set_theta_zero_location("N")
    ax1.set_theta_direction(-1)
    ax1.set_rticks(np.linspace(0, np.ceil(np.max(radii)), 5, dtype=int))
    ax1.set_title(f'Time {timept} s', fontsize=12)
    
    # Reduce margins within each polar plot
    ax1.set_rorigin(-2.5)
    
    # Row 2: Radial plots of filament points
    ax2 = axs[1, j]
    ax2.scatter(theta_fil, r_fil, c='blue', s=10, alpha=0.7, label='Filament Points')
    ax2.set_theta_zero_location("N")
    ax2.set_theta_direction(-1)
    ax2.set_title(f'Filament Points at {timept} s', fontsize=12)
    
    # Reduce margins within each polar plot
    ax2.set_rorigin(-2.5)

for j in range(len(timepoints)):
    ax2 = axs[1, j]
    ax2.set_ylim([0, global_max_r+10])
    ax2.set_rticks(np.linspace(0, global_max_r+10, 5, dtype=int))

# Third plot (Wasserstein Distance) - Non-polar plot
fig2, axes2 = plt.subplots(1, 1, figsize=(12, 6))
axes2.plot(wd_dict_data.index, wd_dict_data['Wasserstein Distance'], color='black', label='Wasserstein Distance')
axes2.set_xlabel('Time', fontsize=12, fontweight='bold')
axes2.set_ylabel('Wasserstein Distance', fontsize=12, fontweight='bold')
axes2.set_title('Wasserstein Distance Over Time', fontsize=14, fontweight='bold', pad=15)

# Highlight the data points corresponding to 'timepoints' with different color and marker
highlight_times = wd_dict_data[wd_dict_data.index.isin(timepoints)]
axes2.scatter(highlight_times.index, highlight_times['Wasserstein Distance'], color='red', s=80, label='Above times', edgecolor='black', zorder=5)

# Formatting the third plot
axes2.legend(loc='upper right', fontsize=12)
axes2.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
axes2.set_ylim([0, np.max(wd_dict_data['Wasserstein Distance']) + 0.1])

# Use tight_layout with adjusted parameters
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Leave room for the suptitle
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Settings
# -----------------------------
timepoints = [1.0, 4.0, 7.0, 10.0, 13.0]
run_to_plot = 'run-endocytosis2_baseline_zero_diffusion0000-0013'
df = branched_actin_bound_points['endocytosis2_baseline_zero_diffusion_output']
run_data = df[df['run'] == run_to_plot]
wd_dict_data = run_wd_dict_2[run_to_plot]

N = 10
bottom = 0
theta = np.linspace(-np.pi, np.pi, N+1, endpoint=True)  # N bins
theta_mid = [(theta[i] + theta[i+1]) / 2 for i in range(len(theta)-1)]
width = (2 * np.pi) / N

# -----------------------------
# Figure layout
# -----------------------------
fig = plt.figure(figsize=(16, 12))
import matplotlib.gridspec as gridspec
gs = gridspec.GridSpec(3, len(timepoints), height_ratios=[1, 1, 1.2], hspace=0.4, wspace=0.3)
fig.suptitle(f'Run {run_to_plot}: Polar & Radial Plots + Wasserstein Distance', fontsize=16, fontweight='bold')

global_max_r = 0

# -----------------------------
# Loop through timepoints
# -----------------------------
for j, timept in enumerate(timepoints):
    # Extract filament positions at this time
    if timept in run_data['time'].values:
        data_t = run_data[run_data['time'] == timept]
        ypos_time = data_t.ypos_recalibrated
        xpos_time = data_t.xpos_recalibrated
        
        ypos_fil = ypos_time * 1e3  # convert to nm
        xpos_fil = xpos_time * 1e3
        r_fil = np.sqrt(xpos_fil.values**2 + ypos_fil.values**2)
        theta_fil = np.arctan2(ypos_fil.values, xpos_fil.values)
        
        if len(r_fil) > 0:
            global_max_r = max(global_max_r, np.max(r_fil))
            direction = np.arctan2(ypos_time, xpos_time)
        else:
            direction = []

    # -----------------------------
    # Row 1: Polar histogram
    # -----------------------------
    ax1 = fig.add_subplot(gs[0, j], polar=True)
    radii, _ = np.histogram(direction, bins=theta)
    ax1.bar(theta_mid, radii, width=width, bottom=bottom, edgecolor='black', alpha=0.9)
    ax1.set_theta_zero_location("N")
    ax1.set_theta_direction(-1)
    ax1.set_rticks(np.linspace(0, np.ceil(np.max(radii)), 5, dtype=int))
    ax1.set_title(f'Time {timept} s', fontsize=12)
    ax1.set_rorigin(-2.5)

    # -----------------------------
    # Row 2: Radial scatter
    # -----------------------------
    ax2 = fig.add_subplot(gs[1, j], polar=True)
    ax2.scatter(theta_fil, r_fil, c='blue', s=10, alpha=0.7)
    ax2.set_theta_zero_location("N")
    ax2.set_theta_direction(-1)
    ax2.set_title(f'Filament Points {timept} s', fontsize=12)
    ax2.set_rorigin(-2.5)

# -----------------------------
# Row 3: Wasserstein Distance
# -----------------------------
ax3 = fig.add_subplot(gs[2, :])
ax3.plot(wd_dict_data.index, wd_dict_data['Wasserstein Distance'], color='black', label='Wasserstein Distance')
ax3.set_xlabel('Time', fontsize=12, fontweight='bold')
ax3.set_ylabel('Wasserstein Distance', fontsize=12, fontweight='bold')
ax3.set_title('Wasserstein Distance Over Time', fontsize=14, fontweight='bold', pad=15)

# Highlight selected timepoints
highlight_times = wd_dict_data[wd_dict_data.index.isin(timepoints)]
ax3.scatter(highlight_times.index, highlight_times['Wasserstein Distance'], color='red', s=80, edgecolor='black', zorder=5, label='Selected Timepoints')
ax3.legend(loc='upper right', fontsize=12)
ax3.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax3.set_ylim([0, np.max(wd_dict_data['Wasserstein Distance']) + 0.1])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Settings
# -----------------------------
timepoints = [1.0, 4.0, 7.0, 10.0, 15.0]
run_to_plot = 'run-endocytosis2_baseline_zero_diffusion0000-0001'
df = branched_actin_bound_points['endocytosis2_baseline_zero_diffusion_output']
run_data = df[df['run'] == run_to_plot]
wd_dict_data = run_wd_dict_2[run_to_plot]

N = 10
bottom = 0
theta = np.linspace(-np.pi, np.pi, N+1, endpoint=True)  # N bins
theta_mid = [(theta[i] + theta[i+1]) / 2 for i in range(len(theta)-1)]
width = (2 * np.pi) / N

# -----------------------------
# Determine global max radius first
# -----------------------------
global_max_r = 0
for timept in timepoints:
    if timept in run_data['time'].values:
        data_t = run_data[run_data['time'] == timept]
        ypos_fil = data_t.ypos_recalibrated * 1e3
        xpos_fil = data_t.xpos_recalibrated * 1e3
        r_fil = np.sqrt(xpos_fil.values**2 + ypos_fil.values**2)
        if len(r_fil) > 0:
            global_max_r = max(global_max_r, np.max(r_fil))

# -----------------------------
# Figure layout
# -----------------------------
fig = plt.figure(figsize=(16, 12))
import matplotlib.gridspec as gridspec
gs = gridspec.GridSpec(3, len(timepoints), height_ratios=[1, 1, 1.2], hspace=0.4, wspace=0.3)
fig.suptitle(f'Run {run_to_plot}: Polar & Radial Plots + Wasserstein Distance', fontsize=16, fontweight='bold')

# -----------------------------
# Loop through timepoints
# -----------------------------
for j, timept in enumerate(timepoints):
    if timept in run_data['time'].values:
        data_t = run_data[run_data['time'] == timept]
        ypos_fil = data_t.ypos_recalibrated * 1e3
        xpos_fil = data_t.xpos_recalibrated * 1e3
        r_fil = np.sqrt(xpos_fil.values**2 + ypos_fil.values**2)
        theta_fil = np.arctan2(ypos_fil.values, xpos_fil.values)
        direction = np.arctan2(data_t.ypos_recalibrated, data_t.xpos_recalibrated)

    else:
        r_fil = np.array([])
        theta_fil = np.array([])
        direction = np.array([])

    # -----------------------------
    # Row 1: Polar histogram
    # -----------------------------
    ax1 = fig.add_subplot(gs[0, j], polar=True)
    radii, _ = np.histogram(direction, bins=theta)
    ax1.bar(theta_mid, radii, width=width, bottom=bottom, edgecolor='black', alpha=0.9)
    ax1.set_theta_zero_location("N")
    ax1.set_theta_direction(-1)
    ax1.set_rticks(np.linspace(0, np.ceil(np.max(radii)), 5, dtype=int))
    ax1.set_title(f'Time {timept} s', fontsize=12)
    ax1.set_rorigin(-2.5)
    # -----------------------------
    # Row 2: Radial scatter
    # -----------------------------
    ax2 = fig.add_subplot(gs[1, j], polar=True)
    ax2.scatter(theta_fil, r_fil, c='blue', s=10, alpha=0.7)
    ax2.set_theta_zero_location("N")
    ax2.set_theta_direction(-1)
    ax2.set_title(f'Filament Points {timept} s', fontsize=12)
    ax2.set_rorigin(-2.5)
    ax2.set_ylim(0, global_max_r)  # same scale for all radial scatter plots

# -----------------------------
# Row 3: Wasserstein Distance
# -----------------------------
ax3 = fig.add_subplot(gs[2, :])
ax3.plot(wd_dict_data.index, wd_dict_data['Wasserstein Distance'], color='black', label='Wasserstein Distance')
ax3.set_xlabel('Time', fontsize=12, fontweight='bold')
ax3.set_ylabel('Wasserstein Distance', fontsize=12, fontweight='bold')
ax3.set_title('Wasserstein Distance Over Time', fontsize=14, fontweight='bold', pad=15)

highlight_times = wd_dict_data[wd_dict_data.index.isin(timepoints)]
ax3.scatter(highlight_times.index, highlight_times['Wasserstein Distance'], color='red', s=80, edgecolor='black', zorder=5, label='Selected Timepoints')
ax3.legend(loc='upper right', fontsize=12)
ax3.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax3.set_ylim([0, np.max(wd_dict_data['Wasserstein Distance']) + 0.1])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# -----------------------------
# Settings
# -----------------------------
timepoints = [1.0, 4.0, 7.0, 10.0, 15.0]
run_to_plot = 'run-endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut0000-0003'
df = branched_actin_bound_points['endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output']
run_data = df[df['run'] == run_to_plot]
wd_dict_data = run_wd_dict_1[run_to_plot]

N = 10  # number of angular bins
theta_edges = np.linspace(-np.pi, np.pi, N+1)
theta_mid = [(theta_edges[i] + theta_edges[i+1])/2 for i in range(N)]
colors = cm.rainbow(np.linspace(0, 1, N))

# -----------------------------
# Determine global max radius
# -----------------------------
global_max_r = 0
for timept in timepoints:
    if timept in run_data['time'].values:
        data_t = run_data[run_data['time'] == timept]
        ypos_fil = data_t.ypos_recalibrated * 1e3
        xpos_fil = data_t.xpos_recalibrated * 1e3
        r_fil = np.sqrt(xpos_fil.values**2 + ypos_fil.values**2)
        if len(r_fil) > 0:
            global_max_r = max(global_max_r, np.max(r_fil))

# -----------------------------
# Figure layout
# -----------------------------
import matplotlib.gridspec as gridspec
fig = plt.figure(figsize=(16, 8))
gs = gridspec.GridSpec(2, len(timepoints), height_ratios=[1, 1.2], hspace=0.4, wspace=0.3)
fig.suptitle("Network remains nonuniform", fontsize=16, fontweight='bold')

# -----------------------------
# Row 1: Radial scatter with shaded bins
# -----------------------------
for j, timept in enumerate(timepoints):
    ax = fig.add_subplot(gs[0, j], polar=True)
    
    if timept in run_data['time'].values:
        data_t = run_data[run_data['time'] == timept]
        ypos_fil = data_t.ypos_recalibrated * 1e3
        xpos_fil = data_t.xpos_recalibrated * 1e3
        r_fil = np.sqrt(xpos_fil.values**2 + ypos_fil.values**2)
        theta_fil = np.arctan2(ypos_fil.values, xpos_fil.values)
        
        # Scatter filament points
        ax.scatter(theta_fil, r_fil, c='blue', s=10, alpha=0.7)
        
        # Shade angular sectors
        for i in range(N):
            ax.fill_betweenx(
                [0, global_max_r], 
                theta_edges[i], theta_edges[i+1], 
                color=colors[i], alpha=0.2
            )
    
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(f'Time {timept} s', fontsize=12)
    ax.set_rorigin(-2.5)
    ax.set_ylim(0, global_max_r)  # same radial scale for all scatter plots
    ax.set_yticklabels([])

# -----------------------------
# Row 2: Wasserstein Distance
# -----------------------------
ax2 = fig.add_subplot(gs[1, :])
ax2.plot(wd_dict_data.index, wd_dict_data['Wasserstein Distance'], color='black', label='Wasserstein Distance')

highlight_times = wd_dict_data[wd_dict_data.index.isin(timepoints)]
ax2.scatter(highlight_times.index, highlight_times['Wasserstein Distance'], color='red', s=80, edgecolor='black', zorder=5, label='Selected Timepoints')

ax2.set_xlabel('Time', fontsize=12, fontweight='bold')
ax2.set_ylabel('Wasserstein Distance', fontsize=12, fontweight='bold')
ax2.set_title('Wasserstein Distance Over Time', fontsize=14, fontweight='bold', pad=15)
ax2.legend(loc='upper right', fontsize=12)
ax2.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax2.set_ylim([0, np.max(wd_dict_data['Wasserstein Distance']) + 0.1])


plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 1D

In [ ]:
# Convenience functions for median and IQR

def median_and_iqr_errors(data):
    if isinstance(data, pd.Series):
        data = data.tolist()  # Convert Series to list if needed

    arr = np.array(data)

    # Compute quartiles
    q1 = np.percentile(arr, 25)  # First quartile (Q1)
    q3 = np.percentile(arr, 75)  # Third quartile (Q3)
    median = np.median(arr)  # Median

    # Compute lower and upper error for error bars
    #lower_error = median - q1
    #upper_error = q3 - median

    return median, q1, q3

# Figure Two

## 2A

In [ ]:
# Convenience functions for median and IQR

def median_and_iqr_errors(data):
    if isinstance(data, pd.Series):
        data = data.tolist()  # Convert Series to list if needed

    arr = np.array(data)

    # Compute quartiles
    q1 = np.percentile(arr, 25)  # First quartile (Q1)
    q3 = np.percentile(arr, 75)  # Third quartile (Q3)
    median = np.median(arr)  # Median

    # Compute lower and upper error for error bars
    #lower_error = median - q1
    #upper_error = q3 - median

    return median, q1, q3

In [ ]:
def circle_WD(dir_df, loops=1):
    timepoints = dir_df.index.get_level_values('time').unique()
    runs = dir_df.index.get_level_values('run').unique()
    max_length_per_run = dir_df.groupby('run')['length'].max()
    max_length_array = max_length_per_run.values

    all_wass_distances = []

    for time in timepoints:
        for run in runs:
            if (run, time) in dir_df.index:
                row = dir_df.loc[(run, time)]
                length = row['length']
                if length == 0:
                    continue  # Skip short tracks

                directions = row['direction']  # Angular data
                directions_shift = directions

                wass_dists = []
                baseline_sizes = []

                for _ in range(loops):
                    random_network1 = np.random.uniform(-np.pi, np.pi, 10000)
                    random_network2 = np.random.uniform(-np.pi, np.pi, len(directions))
                    random_network_shift = (np.pi + random_network1) / (2 * np.pi)
                    random_network2_shift = (np.pi + random_network2) / (2 * np.pi)

                    wass_dist = ot.wasserstein_circle(directions_shift, random_network_shift, p=1)
                    baseline_size = ot.wasserstein_circle(random_network2_shift, random_network_shift, p=1)

                    wass_dists.append(wass_dist)
                    baseline_sizes.append(baseline_size)

                all_wass_distances.append({
                    'run': run,
                    'time': time,
                    'avg_WD': np.mean(wass_dists),
                    'avg_BL': np.mean(baseline_sizes)
                })

    # Create DataFrame
    wass_distances_df = pd.DataFrame(all_wass_distances)

    # Aggregate WD and baseline
    grouped = wass_distances_df.groupby('time').agg({
        'avg_WD': lambda x: median_and_iqr_errors(x.tolist()),
        'avg_BL': lambda x: median_and_iqr_errors(x.tolist())
    })

    # Extract results
    (median_WD, lower_WD, upper_WD) = zip(*grouped['avg_WD'])
    (median_BL, lower_BL, upper_BL) = zip(*grouped['avg_BL'])

    result = pd.DataFrame({
        'Median_WD': median_WD,
        'Lower_WD': lower_WD,
        'Upper_WD': upper_WD,
        'Median_BL': median_BL,
        'Lower_BL': lower_BL,
        'Upper_BL': upper_BL
    }, index=grouped.index)

    # --- Baseline max distribution ---
    last_wd_dfs = []
    for length in max_length_array:
        if length > 0:
            for p_time in range(150):
                rn1 = np.random.uniform(-np.pi, np.pi, length)
                rn2 = np.random.uniform(-np.pi, np.pi, 10000)
                rn1_shift = (np.pi + rn1) / (2 * np.pi)
                rn2_shift = (np.pi + rn2) / (2 * np.pi)
                last_wd = ot.wasserstein_circle(rn1_shift, rn2_shift, p=1)
                last_wd_dfs.append({'time': p_time, 'avg_WD': last_wd})

    baseline_max = pd.DataFrame(last_wd_dfs)

    grouped_baseline = baseline_max.groupby('time').agg({
        'avg_WD': lambda x: median_and_iqr_errors(x.tolist())
    })

    (median_baseline, lower_baseline, upper_baseline) = zip(*grouped_baseline['avg_WD'])

    baseline_max = pd.DataFrame({
        'time': grouped_baseline.index,
        'Median': median_baseline,
        'Lower Error': lower_baseline,
        'Upper Error': upper_baseline
    })

    return result, baseline_max


In [ ]:
zero_trials = [
    'endocytosis2_baseline_zero_diffusion_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
]

WD_zero_dict = {}
baseline_dict = {}

ypos_recalibrated = []
xpos_recalibrated = []
direction = []
lengths = []
index = np.round(np.arange(0, 15.0, 0.1), 1) 

# Recalibrate positions and calculate directions
for trial in zero_trials:
    if trial in branched_actin_bound_points:
        ypos_recalibrated.append(branched_actin_bound_points[trial].dropna().ypos_recalibrated)
        xpos_recalibrated.append(branched_actin_bound_points[trial].dropna().xpos_recalibrated)
        dir = np.arctan2(ypos_recalibrated[-1], xpos_recalibrated[-1])
        dir_circle = (np.pi + dir) / (2 * np.pi)  # Convert to [0, 1] for circular WD
        direction.append(dir_circle)
        
        # Ensure NaN values are considered as length zero
        lengths.append(np.sum(~np.isnan(dir)))  # Count non-NaN values for length
        branched_actin_bound_points[trial]['direction'] = dir_circle

# Group directions by 'run' and 'time' and add lengths as a column
directions_grouped_dict = {}

for trial in zero_trials:
    # Group the directions
    grouped_directions = branched_actin_bound_points[trial].groupby(['run', 'time']).direction.apply(np.array)
    
    # Create a DataFrame with 'direction' and 'length' columns
    grouped_lengths = grouped_directions.apply(lambda x: np.sum(~np.isnan(x)))  # Count non-NaN values
    directions_grouped_dict[trial] = pd.DataFrame({
        'direction': grouped_directions,
        'length': grouped_lengths
    })
    # Get DF for current trial
    dir_df = directions_grouped_dict[trial]

    # Calculate corrected WD
    result, baseline = circle_WD(dir_df)
    result.index = index[:len(result)]
    baseline.index = index[:len(result)]
 
    WD_zero_dict[trial] = result
    baseline_dict[trial] = baseline

    print(f"Successfully processed trial: {trial}")

In [ ]:
import seaborn as sns

viridis_colors = sns.color_palette("viridis", len(zero_trials))

# Define the names (if you have a mapping)
names = ['Full', '3/4', '1/2', '1/4']

# --- Find global max WD across all trials + baseline ---
all_max = []

for trial in zero_trials:
    if trial in branched_actin_bound_points:
        all_max.append(np.max(WD_zero_dict[trial]['Upper_WD']))

all_max.append(np.max(baseline_dict['endocytosis2_baseline_zero_diffusion_output']['Upper Error']))

max_wd = max(all_max)

# Dynamically set yticks (5 steps between 0 and max_wd)
yticks = np.linspace(0, max_wd, 5)

# --- Plotting ---
fig, ax = plt.subplots(figsize=(8,6))  # set figure size here

for i, trial in enumerate(zero_trials):
    if trial in branched_actin_bound_points:
        trial_name = names[i % len(names)]
        
        median_values = WD_zero_dict[trial]['Median_WD']
        lower_iqr = WD_zero_dict[trial]['Lower_WD']
        upper_iqr = WD_zero_dict[trial]['Upper_WD']

        plot_multiple_errorbars_iqr(
            ax, median_values, lower_iqr, upper_iqr,
            viridis_colors[i], 'Wasserstein Distance', trial_name, yticks=yticks
        )

# Add baseline comparison
plot_multiple_errorbars_iqr(
    ax,
    WD_zero_dict['endocytosis2_baseline_zero_diffusion_output']['Median_BL'],
    WD_zero_dict['endocytosis2_baseline_zero_diffusion_output']['Lower_BL'],
    WD_zero_dict['endocytosis2_baseline_zero_diffusion_output']['Upper_BL'],
    'black', 'Wasserstein Distance', 'Uniform', yticks=yticks
)

# Finalize the plot
plt.title('Wasserstein Distance Over Time for Different Arp2/3 Distributions', fontsize=14)
plt.xlabel("Time (s)")
plt.ylabel("Wasserstein Distance")
plt.yticks(yticks)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()



## 2B

## 2G

In [ ]:
# Convenience functions for median and IQR

def median_and_iqr_errors(data):
    if isinstance(data, pd.Series):
        data = data.tolist()  # Convert Series to list if needed

    arr = np.array(data)

    # Compute quartiles
    q1 = np.percentile(arr, 25)  # First quartile (Q1)
    q3 = np.percentile(arr, 75)  # Third quartile (Q3)
    median = np.median(arr)  # Median

    # Compute lower and upper error for error bars
    #lower_error = median - q1
    #upper_error = q3 - median

    return median, q1, q3

In [ ]:
def circle_size_WD(dir_df, shared_bins, loops=1, baseline_loops=100):
    all_wass_distances = []

    for i in range(len(shared_bins) - 1):
        bin_start, bin_end = shared_bins[i], shared_bins[i + 1]
        subset = dir_df[(dir_df['length'] >= bin_start) & (dir_df['length'] < bin_end)]
        bin_count = len(subset)  # Count of elements in the bin

        if subset.empty:
            continue  # Skip empty bins

        wass_dists = []
        bl_sizes = []

        for _, row in subset.iterrows():
            directions = row['direction']
            #print(row)

            for _ in range(loops):
                random_network1 = np.random.uniform(-np.pi, np.pi, 10000)
                random_network2 = np.random.uniform(-np.pi, np.pi, int(round(row['length'])))
                random_network_shift = (np.pi + random_network1) / (2 * np.pi)
                random_network2_shift = (np.pi + random_network2) / (2 * np.pi)

                wass_dist = ot.wasserstein_circle(directions, random_network_shift, p=1)
                bl_size = ot.wasserstein_circle(random_network2_shift, random_network_shift, p=1)

                wass_dists.append(wass_dist)
                bl_sizes.append(bl_size)

        # Compute median and IQR for WD and BL
        median_WD, lower_WD, upper_WD = median_and_iqr_errors(wass_dists)
        median_BL, lower_BL, upper_BL = median_and_iqr_errors(bl_sizes)

        # Baseline estimate (Wasserstein distance between two random uniform networks of size = bin center)
        baseline_wass_dists = []
        for _ in range(baseline_loops):
            baseline_network1 = np.random.uniform(-np.pi, np.pi, 10000)
            baseline_network2 = np.random.uniform(-np.pi, np.pi, int((bin_start + bin_end) / 2))

            baseline_1_shift = (np.pi + baseline_network1) / (2 * np.pi)
            baseline_2_shift = (np.pi + baseline_network2) / (2 * np.pi)

            baseline_wass_dist = ot.wasserstein_circle(baseline_1_shift, baseline_2_shift, p=1)
            baseline_wass_dists.append(baseline_wass_dist)

        baseline_median = np.median(baseline_wass_dists)
        baseline_lower_error = np.percentile(baseline_wass_dists, 25)
        baseline_upper_error = np.percentile(baseline_wass_dists, 75)

        # Append results
        all_wass_distances.append({
            'Bin Center': ((bin_start + bin_end) / 2) * 2.75,
            'Median': median_WD,
            'Lower Error': lower_WD,
            'Upper Error': upper_WD,
            'Count': bin_count,
            'Median BL': median_BL,
            'Lower BL': lower_BL,
            'Upper BL': upper_BL,
            'Baseline Median WD': baseline_median,
            'Baseline Lower Error': baseline_lower_error,
            'Baseline Upper Error': baseline_upper_error
        })

    # Convert to DataFrame and return
    result = pd.DataFrame(all_wass_distances)
    result = result.sort_values(by='Bin Center').set_index('Bin Center')

    return result


In [ ]:
zero_trials = [
    'endocytosis2_baseline_zero_diffusion_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
]

WD_zero_size_dict = {}
WD_zero_dict_95 = {}


# Recalibrate positions and compute direction
for trial in zero_trials:
    if trial in branched_actin_bound_points:
        trial_data = branched_actin_bound_points[trial].dropna()
        trial_data_2 = df_with_lengths[trial].dropna()
        
        ypos_recalibrated = trial_data.ypos_recalibrated
        xpos_recalibrated = trial_data.xpos_recalibrated

        # Compute direction
        dir_angles = np.arctan2(ypos_recalibrated, xpos_recalibrated)
        dir_circle = (np.pi + dir_angles) / (2 * np.pi)  # Convert to [0, 1] for circular WD
        trial_data['direction'] = dir_circle

        # Compute length
        trial_data['length'] = trial_data['direction'].apply(lambda x: np.sum(~np.isnan(x)))

        # Store processed data
        branched_actin_bound_points[trial] = trial_data

# Group directions by 'run' and 'time'
directions_grouped_dict = {}
for trial in zero_trials:
    if trial in branched_actin_bound_points:
        trial_data = branched_actin_bound_points[trial]

        grouped_internalization = trial_data.groupby(['run', 'time'])['bud_internalization'].unique()
        grouped_directions = trial_data.groupby(['run', 'time'])['direction'].apply(np.array)
        grouped_lengths = grouped_directions.apply(lambda x: np.sum(~np.isnan(x)))  # Count non-NaN values
        
        directions_grouped_dict[trial] = pd.DataFrame({
            'direction': grouped_directions,
            'length': grouped_lengths, #* 10/2.75,
            'internalization': grouped_internalization
        })

WD_zero_size_dict = {}

for trial in zero_trials:
    if trial not in directions_grouped_dict:
        continue

    dir_df = directions_grouped_dict[trial]

    # # Compute quantile-based bins for this trial's lengths
    # trial_lengths = dir_df['length'].dropna().values
    # n_bins = 30
    # quantile_edges = np.quantile(trial_lengths, np.linspace(0, 1, n_bins + 1))
    # quantile_edges = np.unique(quantile_edges)  # Ensure no duplicate bin edges
    
    # Get trial lengths
    trial_lengths = dir_df['length'].dropna().values

    # Number of bins
    n_bins = 70

    # Fixed-width bins from min to max
    trial_min = trial_lengths.min()
    trial_max = trial_lengths.max()

    bin_edges = np.linspace(trial_min, trial_max, n_bins + 1) 
    # Call your circle_size_WD function on the full data
    result = circle_size_WD(dir_df, bin_edges)

    WD_zero_size_dict[trial] = result
    print(f"✅ Processed trial (all run/time, no filtering): {trial}")



In [ ]:
# Set global font
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

# Get viridis palette from seaborn
viridis_colors = sns.color_palette("viridis", len(zero_trials))

# Trial names
names = ['Full', '3/4', '1/2', '1/4']

fig, ax = plt.subplots(figsize=(8,6))

# Plot the baseline once
baseline_plotted = False

for i, trial in enumerate(zero_trials):
    if trial in branched_actin_bound_points:
        # Extract data for each trial
        print(trial)
        median_values = WD_zero_size_dict[trial]['Median']
        lower_iqr = WD_zero_size_dict[trial]['Lower Error']  
        upper_iqr = WD_zero_size_dict[trial]['Upper Error']  

        base_median = WD_zero_size_dict[trial]['Median BL']
        base_lower = WD_zero_size_dict[trial]['Lower BL']
        base_upper = WD_zero_size_dict[trial]['Upper BL']

        color = sns.color_palette("viridis", len(zero_trials))[i]
        name = names[i]
        yticks = np.linspace(0, 0.16, 5)

        # Plot the trial data with error bars
        plot_multiple_errorbars_iqr(ax, median_values, lower_iqr, upper_iqr, color, 'Wasserstein Distance (rad)', name, yticks)

        # Plot baseline only once
        if i == 3:
            plot_multiple_errorbars_iqr(ax, base_median, base_lower, base_upper, 'k', 'Baseline', 'Uniform', yticks)
            baseline_plotted = True  # Set to True to prevent future baseline plotting

# Title with size
ax.set_xscale("log")
ax.set_yscale("log")
#ax.set_title('Wasserstein Distance vs Size', fontsize=20)
ax.set_xlabel('Amount of actin (nm)', fontsize=18)
ax.set_ylabel("Wasserstein distance (rad)")


# Display the plot
plt.tight_layout()
plt.legend(loc='lower right')
#plt.savefig('WD_vs_size.png', dpi=300)
plt.show()


In [ ]:
# Set global font
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

# Get viridis palette from seaborn
viridis_colors = sns.color_palette("viridis", len(zero_trials))

# Trial names
names = ['Full', '3/4', '1/2', '1/4']

fig, ax = plt.subplots(figsize=(8,6))

# Plot the baseline once
baseline_plotted = False

# Compute global min/max for WD to dynamically set yticks
all_medians = []
all_lowers = []
all_uppers = []

for trial in zero_trials:
    if trial in WD_zero_size_dict:
        all_medians.append(WD_zero_size_dict[trial]['Median'].values)
        all_lowers.append(WD_zero_size_dict[trial]['Lower Error'].values)
        all_uppers.append(WD_zero_size_dict[trial]['Upper Error'].values)

all_medians = np.concatenate(all_medians)
all_lowers = np.concatenate(all_lowers)
all_uppers = np.concatenate(all_uppers)

ymin = np.nanmin(all_lowers)
ymax = np.nanmax(all_uppers)
yticks = np.linspace(ymin, ymax, 5)

for i, trial in enumerate(zero_trials):
    if trial in branched_actin_bound_points:
        print(f"Plotting trial: {trial}")
        median_values = WD_zero_size_dict[trial]['Median']
        lower_iqr = WD_zero_size_dict[trial]['Lower Error']  
        upper_iqr = WD_zero_size_dict[trial]['Upper Error']  

        base_median = WD_zero_size_dict[trial]['Median BL']
        base_lower = WD_zero_size_dict[trial]['Lower BL']
        base_upper = WD_zero_size_dict[trial]['Upper BL']

        color = viridis_colors[i]
        name = names[i]

        # Plot the trial data with error bars
        plot_multiple_errorbars_iqr(
            ax, median_values, lower_iqr, upper_iqr, color, 'Wasserstein Distance (rad)', name, yticks
        )

        # Plot baseline only once
        if not baseline_plotted:
            plot_multiple_errorbars_iqr(
                ax, base_median, base_lower, base_upper, 'k', 'Baseline', 'Uniform', yticks
            )
            baseline_plotted = True

ax.set_xlabel('Amount of actin', fontsize=18)
ax.set_ylabel("Wasserstein distance (rad)")

plt.tight_layout()
plt.legend(loc='upper right')
#plt.savefig('WD_vs_size.png', dpi=300)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import ot

trial_results = {}

# --- Precompute uniform reference once
uniform_network = np.random.uniform(-np.pi, np.pi, 10000)
uniform_shift = (np.pi + uniform_network) / (2 * np.pi)

for trial in zero_trials:
    if trial not in branched_actin_bound_points:
        continue

    print(f"Processing trial: {trial}")
    trial_data = branched_actin_bound_points[trial].dropna(subset=['direction', 'bud_internalization'])
    grouped = trial_data.groupby(['run', 'time'])

    rows = []

    for (run, time), group in grouped:
        # --- Safely collect all direction values ---
        dirs_list = []
        for d in group['direction']:
            if d is None:
                continue
            d = np.asarray(d, dtype=float)
            if d.ndim == 0:  # scalar
                d = np.array([d])
            d = d[~np.isnan(d)]
            if d.size > 0:
                dirs_list.append(d)

        if not dirs_list:
            continue

        all_dirs = np.concatenate(dirs_list)
        if all_dirs.size == 0:
            continue

        # --- Internalization: take unique value
        internal_vals = group['bud_internalization'].unique()
        if len(internal_vals) > 1:
            print(f"⚠️ Multiple internalization values in ({trial}, run={run}, time={time}) — taking first.")
        internalization = internal_vals[0]

        # --- Normalize and compute WD
        bl_directions = np.random.uniform(-np.pi, np.pi, all_dirs.size)
        bl_shift = (np.pi + bl_directions) / (2 * np.pi)

        directions_shift = (np.pi + all_dirs) / (2 * np.pi)
        wd = ot.wasserstein_circle(directions_shift, uniform_shift, p=1)
        wd_bl = ot.wasserstein_circle(bl_shift, uniform_shift, p=1)

        # --- Save a row for this run/time
        rows.append({
            'Run': run,
            'Time': time,
            'Internalization': internalization,
            'WD': wd,
            'WD_BL': wd_bl,
            'ActinAmount': all_dirs.size
        })

    # --- Convert to DataFrame per trial
    df_trial = pd.DataFrame(rows)
    trial_results[trial] = df_trial

print("✅ Done")


In [ ]:
for trial, df_trial in trial_results.items():
    if df_trial.empty:
        continue

    # Normalize WD and WD_BL per run
    df_trial[['WD_percent', 'WD_BL_percent']] = df_trial.groupby('Run')[['WD', 'WD_BL']].transform(
        lambda x: x / x.max() * 100
    )

    # Save back to the dictionary (optional, since df_trial is modified in place)
    trial_results[trial] = df_trial

print("✅ Added percent-normalized columns for all trials")


In [ ]:
trial_results['endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output']

In [ ]:


# --- Setup ---
names = ['Full', '3/4', '1/2', '1/4']
viridis_colors = sns.color_palette("viridis", 10)  # more colors for multiple runs
trial_to_name = dict(zip(zero_trials, names))

# --- Prepare 2x2 subplots ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

# --- Loop over conditions ---
for i, condition in enumerate(names):
    ax = axes[i]
    
    # Find trials corresponding to this condition
    trials_in_condition = [t for t in zero_trials if trial_to_name[t] == condition]
    
    for j, trial in enumerate(trials_in_condition):
        if trial not in trial_results:
            continue
        df = trial_results[trial]
        if df.empty:
            continue

        # --- Flatten any list entries ---
        x = np.array([np.mean(v) if isinstance(v, (list, np.ndarray)) else v for v in df['ActinAmount']])
        y = np.array([np.mean(v) if isinstance(v, (list, np.ndarray)) else v for v in df['WD_percent']])

        # --- Use a distinct color per run (cycle through palette) ---
        run_colors = sns.color_palette("tab10", len(df['Run'].unique()))
        for k, run in enumerate(df['Run'].unique()):
            mask = df['Run'] == run
            ax.plot(x[mask], y[mask], color=run_colors[k], alpha=0.6, label=f'Run {run}')

    # --- Labels and title per subplot ---
    ax.set_xlabel('Actin amount', fontsize=12)
    ax.set_ylabel('Wasserstein distance (% max per run)', fontsize=12)
    ax.set_title(f'Condition: {condition}', fontsize=14)
    ax.grid(True, alpha=0.3)

    # --- Legend inside the plot ---
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    #ax.legend(by_label.values(), by_label.keys(), fontsize=10, loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Select the Full trial(s) ---
full_trials = [t for t in zero_trials if trial_to_name[t] == 'Full']

# Flatten all runs across the selected trials
all_runs = []
for trial in full_trials:
    if trial not in trial_results:
        continue
    df = trial_results[trial]
    if df.empty:
        continue
    all_runs.extend(df['Run'].unique())

all_runs = np.unique(all_runs)
runs_per_subplot = 10
n_subplots = 5

fig, axes = plt.subplots(n_subplots, 1, figsize=(10, 18))
axes = axes.flatten()

for i in range(n_subplots):
    ax = axes[i]
    
    # Select runs for this subplot
    start_idx = i * runs_per_subplot
    end_idx = start_idx + runs_per_subplot
    runs_to_plot = all_runs[start_idx:end_idx]
    
    for j, run in enumerate(runs_to_plot):
        # Collect data for this run across all Full trials
        for trial in full_trials:
            if trial not in trial_results:
                continue
            df = trial_results[trial]
            if df.empty:
                continue
            df_run = df[df['Run'] == run]
            if df_run.empty:
                continue

            # Flatten potential list entries
            x = np.array([np.mean(v) if isinstance(v, (list, np.ndarray)) else v for v in df_run['ActinAmount']])
            y = np.array([np.mean(v) if isinstance(v, (list, np.ndarray)) else v for v in df_run['WD']])
            
            # Color per run (cycle through palette)
            color = sns.color_palette("tab10", runs_per_subplot)[j % runs_per_subplot]
            ax.plot(x, y, color=color, alpha=0.7, label=f'Run {run}')
    
    ax.set_xlabel('Actin amount', fontsize=12)
    ax.set_ylabel('Wasserstein distance', fontsize=12)
    ax.set_title(f'Full condition: runs {start_idx + 1}–{min(end_idx, len(all_runs))}', fontsize=14)
    ax.grid(True, alpha=0.3)
    #ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

viridis_colors = sns.color_palette("viridis", len(zero_trials))
names = ['Full', '3/4', '1/2', '1/4']
trial_to_color = dict(zip(zero_trials, viridis_colors))
trial_to_name = dict(zip(zero_trials, names))

plt.figure(figsize=(10, 6))

for trial in zero_trials:
    if trial not in trial_results:
        continue
    df = trial_results[trial]
    if df.empty:
        continue

    color = trial_to_color[trial]
    label = trial_to_name[trial]

    # --- Ensure ActinAmount and WD_percent are numeric scalars ---
    actin = np.array([np.mean(x) if isinstance(x, (list, np.ndarray)) else x for x in df['ActinAmount']])
    wd = np.array([np.mean(x) if isinstance(x, (list, np.ndarray)) else x for x in df['WD_percent']])

    # --- Scatter plot (raw data) ---
    #plt.scatter(actin, wd, color=color, alpha=0.25, s=25)

    # --- Bin ActinAmount (fixed 50 bins per trial) ---
    bins = np.linspace(np.nanmin(actin), np.nanmax(actin), 30)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_indices = np.digitize(actin, bins) - 1

    medians, lower, upper = [], [], []
    for i in range(len(bins) - 1):
        wd_bin = wd[bin_indices == i]
        wd_bin = wd_bin[np.isfinite(wd_bin)]
        if wd_bin.size == 0:
            medians.append(np.nan)
            lower.append(np.nan)
            upper.append(np.nan)
        else:
            medians.append(np.median(wd_bin))
            lower.append(np.percentile(wd_bin, 25))
            upper.append(np.percentile(wd_bin, 75))

    # Convert to 1D NumPy arrays
    medians = np.asarray(medians, dtype=float).flatten()
    lower = np.asarray(lower, dtype=float).flatten()
    upper = np.asarray(upper, dtype=float).flatten()

    valid = np.isfinite(medians)
    if np.any(valid):
        plt.plot(bin_centers[valid], medians[valid], color=color, lw=2, label=label)
        plt.fill_between(bin_centers[valid], lower[valid], upper[valid], color=color, alpha=0.2)

# --- Labels and legend ---
plt.xlabel('Actin amount', fontsize=14)
plt.ylabel('Wasserstein distance (% max, per run)', fontsize=14)
plt.title('Binned WD Percent vs Actin Amount (Full → 1/4)', fontsize=16)
plt.grid(True, alpha=0.3)

plt.legend(title='Condition', loc='upper right', frameon=True, fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Define ordered conditions and color map ---
names = ['Full', '3/4', '1/2', '1/4']
palette = sns.color_palette("viridis", len(names))
condition_colors = dict(zip(names, palette))

# --- Mapping from trials to condition names (adjust if needed) ---
trial_conditions = {trial: name for trial, name in zip(trial_results.keys(), names)}

# --- Plot setup ---
plt.figure(figsize=(10,6))

for trial_name, df in trial_results.items():
    if df.empty:
        continue

    condition = trial_conditions.get(trial_name, trial_name)
    color = condition_colors[condition]

    # --- Flatten potential list/array entries to scalars ---
    x = np.array([np.mean(v) if isinstance(v, (list, np.ndarray)) else v for v in df['ActinAmount']])
    wd = np.array([np.mean(v) if isinstance(v, (list, np.ndarray)) else v for v in df['WD_percent']])
    wd_bl = np.array([np.mean(v) if isinstance(v, (list, np.ndarray)) else v for v in df['WD_BL_percent']])

    # --- Adaptive binning (equal-count bins) ---
    N_bins = 50
    percentiles = np.linspace(0, 100, N_bins + 1)
    bin_edges = np.nanpercentile(x, percentiles)
    bin_edges = np.unique(bin_edges)
    bin_indices = np.digitize(x, bin_edges) - 1
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    # --- Compute per-bin median and IQR ---
    def bin_stats(y):
        med, low, high = [], [], []
        for i in range(len(bin_edges) - 1):
            y_bin = y[bin_indices == i]
            y_bin = y_bin[np.isfinite(y_bin)]
            if y_bin.size == 0:
                med.append(np.nan)
                low.append(np.nan)
                high.append(np.nan)
            else:
                med.append(np.median(y_bin))
                low.append(np.percentile(y_bin, 25))
                high.append(np.percentile(y_bin, 75))
        return np.array(med), np.array(low), np.array(high)

    med_wd, low_wd, high_wd = bin_stats(wd)
    med_bl, low_bl, high_bl = bin_stats(wd_bl)

    # --- Plot WD ---
    valid = ~np.isnan(med_wd)
    if np.any(valid):
        plt.plot(bin_centers[valid], med_wd[valid], color=color, lw=2, label=f'{condition} WD')
        plt.fill_between(bin_centers[valid], low_wd[valid], high_wd[valid], color=color, alpha=0.2)

    # # --- Plot WD_BL ---
    # valid_bl = ~np.isnan(med_bl)
    # if np.any(valid_bl):
    #     plt.plot(bin_centers[valid_bl], med_bl[valid_bl], color=color, lw=2, ls='--', label=f'{condition} WD_BL')
    #     plt.fill_between(bin_centers[valid_bl], low_bl[valid_bl], high_bl[valid_bl], color=color, alpha=0.1)

# --- Labels and legend ---
#plt.xscale('log')
plt.xlabel('Actin amount', fontsize=14)
plt.ylabel('Wasserstein distance (% max)', fontsize=14)
plt.title('Percent-normalized WD and WD_BL vs ActinAmount (adaptive bins)', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(title='Condition', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mutual_info_score

# --- Helper functions ---
def mutual_info(x, y, bins=30):
    H, _, _ = np.histogram2d(x, y, bins=bins)
    return mutual_info_score(None, None, contingency=H)

def conditional_mutual_info(x, y, z, bins=30):
    """I(X; Y | Z) by binning Z and averaging conditional MI of X,Y."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    z = np.asarray(z, dtype=float)

    z_bins = np.digitize(z, np.histogram_bin_edges(z, bins=bins))
    mi = 0.0
    for z_val in np.unique(z_bins):
        mask = z_bins == z_val
        if mask.sum() < 5:
            continue
        mi += mask.mean() * mutual_info(x[mask], y[mask], bins=bins)
    return mi

# --- Number of random iterations per run/time ---
n_random = 50

cmi_results = []

for trial, df_trial in trial_results.items():
    for run, df_run in df_trial.groupby('Run'):
        x = df_run['Internalization'].values
        y = df_run['WD'].values
        z = df_run['ActinAmount'].values

        if len(x) < 10:
            print(f"⚠️ Skipping small run: {trial}, {run}")
            continue

        # --- CMI using actual WD ---
        I_cmi = conditional_mutual_info(x, y, z, bins=30)

        # --- Compute average CMI over multiple random baselines ---
        I_rand_list = []
        I_rand_cond_list = []
        for _ in range(n_random):
            y_rand1 = np.random.uniform(0, 1, size=len(y))
            y_rand2 = np.random.uniform(0, 1, size=len(y))
            
            # Baseline CMI of Internalization vs rand_1 given ActinAmount
            I_rand_list.append(conditional_mutual_info(x, y_rand1, z, bins=30))
            
            # Conditional MI of Internalization vs rand_1 given rand_2
            I_rand_cond_list.append(conditional_mutual_info(x, y_rand1, y_rand2, bins=30))
        
        I_cmi_rand_avg = np.mean(I_rand_list)
        I_cmi_rand_cond_avg = np.mean(I_rand_cond_list)
        corrected = I_cmi - I_cmi_rand_avg

        cmi_results.append({
            'Trial': trial,
            'Run': run,
            'Corrected_I_Internalization_WD_given_Actin': corrected,
            'I_Internalization_WD_given_Actin': I_cmi,
            'I_Internalization_Rand_given_Actin': I_cmi_rand_avg,
            'I_Internalization_Rand1_given_Rand2': I_cmi_rand_cond_avg
        })

# --- Convert to DataFrame ---
df_cmi = pd.DataFrame(cmi_results)

In [ ]:

# Custom trial names in the order of appearance in df_cmi
names = ['Full', '3/4', '1/2', '1/4']
trials = df_cmi['Trial'].unique()

# Map original trial IDs to names
trial_name_map = dict(zip(trials, names))
df_cmi['TrialName'] = df_cmi['Trial'].map(trial_name_map)

# Color palette (one per custom trial name)
palette = dict(zip(names, sns.color_palette("viridis", n_colors=len(names))))

plt.figure(figsize=(10, 6))

# --- Violin plot with outline ---
sns.violinplot(
    x='TrialName',
    y='Corrected_I_Internalization_WD_given_Actin',
    data=df_cmi,
    inner='quart',          # remove inner lines
    linewidth=1.5,       # outline thickness
    palette=palette,
    fill=False,          # no fill
)

# --- Swarm plot on top ---
sns.swarmplot(
    x='TrialName',
    y='Corrected_I_Internalization_WD_given_Actin',
    data=df_cmi,
    hue='TrialName',
    palette=palette,
    size=8,
    dodge=False
)

plt.title("Corrected Conditional MI per Run across Trials")
plt.ylabel("I(Internalization; WD | ActinAmount) [bits]")
plt.xlabel("Trial")
plt.legend(title="Trial", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

# --- Violin plot for baseline (random) ---
sns.violinplot(
    x='TrialName',
    y='I_Internalization_Rand_given_Actin',
    data=df_cmi,
    inner='quart',        # show quartiles
    linewidth=1.5,        # outline thickness
    palette=palette,
    fill=False            # no fill
)

# --- Swarm plot on top ---
sns.swarmplot(
    x='TrialName',
    y='I_Internalization_Rand_given_Actin',
    data=df_cmi,
    hue='TrialName',
    palette=palette,
    size=8,
    dodge=False
)

plt.title("Conditional MI per Run across Trials — Baseline (Random WD)")
plt.ylabel("I(Internalization; Random | ActinAmount) [bits]")
plt.xlabel("Trial")
plt.legend(title="Trial", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# --- Prepare DataFrame in long format ---
df_long = pd.melt(
    df_cmi,
    id_vars=['TrialName', 'Run'],
    value_vars=['I_Internalization_WD_given_Actin', 'I_Internalization_Rand_given_Actin'],
    var_name='Type',
    value_name='CMI'
)

# Map types to nicer labels
type_name_map = {
    'I_Internalization_WD_given_Actin': 'WD',
    'I_Internalization_Rand_given_Actin': 'Random'
}
df_long['TypeName'] = df_long['Type'].map(type_name_map)

# --- Define colors ---
palette = {
    'WD': 'tab:blue',     # or use a viridis color if you like
    'Random': 'black'
}

plt.figure(figsize=(10, 6))

# --- Swarm plot with dodge=True for side-by-side ---
sns.swarmplot(
    x='TrialName',
    y='CMI',
    hue='TypeName',
    data=df_long,
    palette=palette,
    size=8,
    dodge=True
)

plt.title("Conditional MI per Run across Trials — WD vs Random")
plt.ylabel("Conditional MI [bits]")
plt.xlabel("Trial")
plt.axhline(0, color='k', linestyle='--', lw=1)
plt.legend(title="Type", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# --- Define flat palette matching TrialName values ---
names = ['Full', '3/4', '1/2', '1/4']
palette_flat = dict(zip(names, sns.color_palette("viridis", n_colors=len(names))))

plt.figure(figsize=(10, 6))

# --- Violin plot of the difference ---
sns.violinplot(
    x='TrialName',
    y='Delta_I',
    data=df_cmi,
    inner='quart',      # show quartiles
    linewidth=1.5,
    palette=palette_flat,  # use flat palette
    fill=False
)

# --- Swarm plot on top ---
sns.swarmplot(
    x='TrialName',
    y='Delta_I',
    data=df_cmi,
    hue='TrialName',
    palette=palette_flat,  # same palette for swarms
    size=8,
    dodge=False
)

plt.title("Difference in Conditional MI: WD vs Random Baseline")
plt.ylabel(r"$\Delta I = I(\mathrm{WD}) - I(\mathrm{Random})$ [bits]")
plt.xlabel("Trial")
plt.axhline(0, color='k', linestyle='--', lw=1)  # zero line
plt.legend(title="Trial", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Count per trial
num_runs_above_1_per_trial = df_cmi[df_cmi['Delta_I'] > 0].groupby('TrialName').size()
print(num_runs_above_1_per_trial)


In [ ]:
import pingouin as pg
import pandas as pd
import numpy as np

partial_corr_results = []

for trial, trial_dict in trial_results.items():
    # Convert dict of arrays to DataFrame
    df_trial = pd.DataFrame(trial_dict)
    
    # Unwrap WD if it is stored as single-element lists
    df_trial['WD'] = df_trial['WD'].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)
    df_trial['Internalization'] = df_trial['Internalization'].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)
    df_trial['ActinAmount'] = df_trial['ActinAmount'].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)

    for run, df_run in df_trial.groupby('Run'):
        # Drop rows with NaN
        df_run = df_run.dropna(subset=['Internalization', 'WD', 'ActinAmount'])
        if len(df_run) < 3:
            continue

        # Compute partial correlation
        pcorr = pg.partial_corr(
            data=df_run,
            x='Internalization',
            y='WD',
            covar='ActinAmount',
            method='pearson'
        )

        partial_corr_results.append({
            'Trial': trial,
            'Run': run,
            'PartialCorr_Internalization_WD_given_Actin': pcorr['r'].values[0],
            'pval': pcorr['p-val'].values[0]
        })

# Convert to DataFrame
df_partial_corr = pd.DataFrame(partial_corr_results)


In [ ]:

# Define trial names and color palette
names = ['Full', '3/4', '1/2', '1/4']
trials = df_partial_corr['Trial'].unique()
trial_name_map = dict(zip(trials, names))
df_partial_corr['TrialName'] = df_partial_corr['Trial'].map(trial_name_map)

# Flat viridis palette
palette = dict(zip(names, sns.color_palette("viridis", n_colors=len(names))))

plt.figure(figsize=(10, 6))

# --- Violin plot with outline ---
sns.violinplot(
    x='TrialName',
    y='PartialCorr_Internalization_WD_given_Actin',
    data=df_partial_corr,
    inner='quart',    # show quartiles
    linewidth=1.5,
    palette=palette,
    fill=False
)

# --- Swarm plot on top ---
sns.swarmplot(
    x='TrialName',
    y='PartialCorr_Internalization_WD_given_Actin',
    data=df_partial_corr,
    hue='TrialName',
    palette=palette,
    size=8,
    dodge=False
)

plt.title("Partial Correlation: Internalization vs WD | ActinAmount")
plt.ylabel("Partial correlation r")
plt.xlabel("Trial")
plt.axhline(0, color='k', linestyle='--', lw=1)  # zero reference line
plt.legend(title="Trial", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Define trial order
trial_order = ['Full', '3/4', '1/2', '1/4']

# Map trial IDs to names
names = trial_order
trials = df_partial_corr['Trial'].unique()
trial_name_map = dict(zip(trials, names))
df_partial_corr['TrialName'] = df_partial_corr['Trial'].map(trial_name_map)

plt.figure(figsize=(8, 5))

# Compute median + quartiles
summary = df_partial_corr.groupby('TrialName')['PartialCorr_Internalization_WD_given_Actin'].agg(
    median='median',
    q1=lambda x: np.percentile(x, 25),
    q3=lambda x: np.percentile(x, 75)
).reindex(trial_order).reset_index()  # ensure full order

# Plot median with error bars (Q1-Q3)
plt.errorbar(
    x=summary['TrialName'],
    y=summary['median'],
    yerr=[summary['median'] - summary['q1'], summary['q3'] - summary['median']],
    fmt='o',
    markersize=10,
    capsize=5,
    color='tab:blue',
    lw=2
)

plt.ylabel("Partial correlation r")
plt.xlabel("Trial")
plt.title("Partial Correlation: Internalization vs WD | ActinAmount")
plt.axhline(0, color='k', linestyle='--', lw=1)
plt.tight_layout()
plt.show()


## 2F

In [ ]:
def angle_WD_grouped(dir_df, loops=1, n_random=1000):
    """
    Compute circular Wasserstein distance of points relative to:
      1. Centroid of points (per run/time)
      2. Origin (0,0)
    
    Groups with fewer than 2 points are skipped (avg_WD = NaN).
    Also returns number of points per group.
    """
    def compute_group_WD(group):
        n_points = len(group)
        if n_points < 2:
            return pd.Series({
                'avg_WD_centroid': np.nan,
                'avg_WD_origin': np.nan,
                'n_points': n_points
            })
        
        # Centroid
        x_mean = group['xpos_recalibrated'].mean()
        y_mean = group['ypos_recalibrated'].mean()
        dx_centroid = group['xpos_recalibrated'].values - x_mean
        dy_centroid = group['ypos_recalibrated'].values - y_mean
        angles_centroid = np.arctan2(dy_centroid, dx_centroid)
        
        # Relative to origin
        dx_origin = group['xpos_recalibrated'].values
        dy_origin = group['ypos_recalibrated'].values
        angles_origin = np.arctan2(dy_origin, dx_origin)
        
        wd_centroid_values = []
        wd_origin_values = []
        for _ in range(loops):
            random_uniform = np.random.uniform(-np.pi, np.pi, n_random)
            random_shift = (np.pi + random_uniform) / (2*np.pi)
            
            angles_centroid_shift = (np.pi + angles_centroid) / (2*np.pi)
            angles_origin_shift = (np.pi + angles_origin) / (2*np.pi)
            
            wd_centroid = ot.wasserstein_circle(angles_centroid_shift, random_shift, p=1)
            wd_origin = ot.wasserstein_circle(angles_origin_shift, random_shift, p=1)
            
            wd_centroid_values.append(wd_centroid)
            wd_origin_values.append(wd_origin)
        
        return pd.Series({
            'avg_WD_centroid': np.mean(wd_centroid_values),
            'avg_WD_origin': np.mean(wd_origin_values),
            'n_points': n_points
        })

    # Apply to each (run, time) group
    wd_df = dir_df.groupby(['run', 'time']).apply(compute_group_WD).reset_index()
    
    # Aggregate by time (median + IQR + median n_points)
    grouped = wd_df.groupby('time').agg(
        median_centroid=('avg_WD_centroid', lambda x: np.nanmedian(x)),
        q25_centroid=('avg_WD_centroid', lambda x: np.nanpercentile(x, 25)),
        q75_centroid=('avg_WD_centroid', lambda x: np.nanpercentile(x, 75)),
        median_origin=('avg_WD_origin', lambda x: np.nanmedian(x)),
        q25_origin=('avg_WD_origin', lambda x: np.nanpercentile(x, 25)),
        q75_origin=('avg_WD_origin', lambda x: np.nanpercentile(x, 75)),
        median_n_points=('n_points', lambda x: np.nanmedian(x))
    ).reset_index()

    return wd_df, grouped


# --- Loop over datasets ---
results_dict = {}
for key, df in branched_actin_bound_points.items():
    wd_df, wd_summary = angle_WD_grouped(df, loops=1, n_random=1000)
    results_dict[key] = {'all': wd_df, 'summary': wd_summary}
    print(f"Processed {key}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- Collect data from both sources ---
violin_data = []

for key, res in results_dict.items():
    wd_all = res['all'].dropna(subset=['avg_WD_origin', 'avg_WD_centroid'])
    last_time = wd_all['time'].max()
    wd_last = wd_all[wd_all['time'] == last_time].copy()
    
    wd_last_origin = wd_last[['run', 'avg_WD_origin']].copy()
    wd_last_origin['Type'] = 'Origin'
    wd_last_origin['Condition'] = key
    wd_last_origin.rename(columns={'avg_WD_origin': 'WD'}, inplace=True)
    
    wd_last_centroid = wd_last[['run', 'avg_WD_centroid']].copy()
    wd_last_centroid['Type'] = 'Centroid'
    wd_last_centroid['Condition'] = key
    wd_last_centroid.rename(columns={'avg_WD_centroid': 'WD'}, inplace=True)
    
    violin_data.extend([wd_last_origin, wd_last_centroid])

df_violin = pd.concat(violin_data, ignore_index=True)

# --- Map condition keys to readable labels ---
labels = ['full', '3/4', '1/2', '1/4']
condition_keys = list(results_dict.keys())
condition_map = {k: l for k, l in zip(condition_keys, labels)}
df_violin['ConditionLabel'] = df_violin['Condition'].map(condition_map)

# --- Viridis palette ---
palette = sns.color_palette('viridis', len(labels))
palette_map = {lab: color for lab, color in zip(labels, palette)}

# --- Plot ---
plt.figure(figsize=(8,6))

sns.violinplot(
    x='ConditionLabel',
    y='WD',
    hue='Type',
    data=df_violin,
    split=True,
    palette=[palette_map[l] for l in labels],
    inner='quart',
    fill=False,
    linewidth=2,
    saturation=1
)

plt.xlabel("Condition")
plt.ylabel("Wasserstein Distance")
plt.title("Origin vs Centroid WD Distribution (Final Time Point)")

# --- Legend inside plot (upper right corner) ---
plt.legend(
    title='WD Type',
    loc='upper right',
    frameon=True,
    framealpha=0.9,
    borderpad=0.6
)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Collect data from both sources ---
violin_data = []

for key, res in results_dict.items():
    wd_all = res['all'].dropna(subset=['avg_WD_origin', 'avg_WD_centroid'])
    last_time = wd_all['time'].max()
    wd_last = wd_all[wd_all['time'] == last_time].copy()
    
    wd_last_origin = wd_last[['run', 'avg_WD_origin']].copy()
    wd_last_origin['Type'] = 'Origin'
    wd_last_origin['Condition'] = key
    wd_last_origin.rename(columns={'avg_WD_origin': 'WD'}, inplace=True)
    
    wd_last_centroid = wd_last[['run', 'avg_WD_centroid']].copy()
    wd_last_centroid['Type'] = 'Centroid'
    wd_last_centroid['Condition'] = key
    wd_last_centroid.rename(columns={'avg_WD_centroid': 'WD'}, inplace=True)
    
    violin_data.extend([wd_last_origin, wd_last_centroid])

df_violin = pd.concat(violin_data, ignore_index=True)

# --- Map condition keys to readable labels ---
labels = ['full', '3/4', '1/2', '1/4']
condition_keys = list(results_dict.keys())
condition_map = {k: l for k, l in zip(condition_keys, labels)}
df_violin['ConditionLabel'] = df_violin['Condition'].map(condition_map)

# --- Viridis palette ---
palette = sns.color_palette('viridis', len(labels))
palette_map = {lab: color for lab, color in zip(labels, palette)}

# --- Plot setup ---
plt.figure(figsize=(8,6))

# Plot one split violin per condition, each with same viridis hue
for i, label in enumerate(labels):
    subset = df_violin[df_violin['ConditionLabel'] == label]
    color = palette_map[label]
    
    sns.violinplot(
        x='ConditionLabel',
        y='WD',
        hue='Type',
        data=subset,
        split=True,
        inner='quart',
        linewidth=1,
        saturation=1,
        palette={'Origin': color, 'Centroid': color},
        width=0.8
    )

# Clean up overlapping legends and unify axes
plt.xlabel("Condition")
plt.ylabel("Wasserstein Distance")
plt.title("Origin vs Centroid WD Distribution (Final Time Point)")

# Remove duplicate legends (caused by multiple sns calls)
handles, labels_leg = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels_leg, handles))
plt.legend(by_label.values(), by_label.keys(), title='Reference', loc='upper right', frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.colors import to_rgb

def adjust_lightness(color, factor=1.3):
    r, g, b = to_rgb(color)
    return (min(r*factor, 1), min(g*factor, 1), min(b*factor, 1))

# --- Collect data ---
violin_data = []
for key, res in results_dict.items():
    wd_all = res['all'].dropna(subset=['avg_WD_origin', 'avg_WD_centroid'])
    last_time = wd_all['time'].max()
    wd_last = wd_all[wd_all['time'] == last_time].copy()
    
    wd_last_origin = wd_last[['run', 'avg_WD_origin']].copy()
    wd_last_origin['Type'] = 'Origin'
    wd_last_origin['Condition'] = key
    wd_last_origin.rename(columns={'avg_WD_origin': 'WD'}, inplace=True)
    
    wd_last_centroid = wd_last[['run', 'avg_WD_centroid']].copy()
    wd_last_centroid['Type'] = 'Centroid'
    wd_last_centroid['Condition'] = key
    wd_last_centroid.rename(columns={'avg_WD_centroid': 'WD'}, inplace=True)
    
    violin_data.extend([wd_last_origin, wd_last_centroid])

df_violin = pd.concat(violin_data, ignore_index=True)

# --- Map condition keys to readable labels ---
labels = ['Full', '3/4', '1/2', '1/4']
condition_keys = list(results_dict.keys())
condition_map = {k: l for k, l in zip(condition_keys, labels)}
df_violin['ConditionLabel'] = df_violin['Condition'].map(condition_map)

# --- Viridis palette ---
palette = sns.color_palette('viridis', len(labels))
palette_map = {lab: color for lab, color in zip(labels, palette)}

# --- Plot ---
plt.figure(figsize=(8,6))
ax = plt.gca()

for i, label in enumerate(labels):
    subset = df_violin[df_violin['ConditionLabel'] == label]
    base_color = palette_map[label]
    lighter = adjust_lightness(base_color, 1.15)
    darker = adjust_lightness(base_color, 0.7)
    
    sns.violinplot(
        x='ConditionLabel',
        y='WD',
        hue='Type',
        data=subset,
        split=True,
        palette={'Origin': lighter, 'Centroid': darker},
        fill=False,
        inner='quart',
        linewidth=2,
        saturation=1,
        ax=ax
    )

plt.xlabel("Condition")
plt.ylabel("Wasserstein Distance")
plt.title("Origin vs Centroid WD Distribution (Final Time Point)")

# --- Custom legend: light gray for Origin, dark gray for Centroid ---
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightgray', edgecolor='k', label='Origin'),
    Patch(facecolor='dimgray', edgecolor='k', label='Centroid')
]
plt.legend(
    handles=legend_elements,
    title='Reference',
    loc='upper right',
    frameon=True,
    framealpha=0.9
)

plt.tight_layout()
plt.show()



## 2H

In [ ]:
def angle_WD_grouped(dir_df, loops=1, n_random=1000):
    """
    Compute circular Wasserstein distance of points relative to:
      1. Centroid of points (per run/time)
      2. Origin (0,0)
    
    Groups with fewer than 2 points are skipped (avg_WD = NaN).
    Also returns number of points per group.
    """
    def compute_group_WD(group):
        n_points = len(group)
        if n_points < 2:
            return pd.Series({
                'avg_WD_centroid': np.nan,
                'avg_WD_origin': np.nan,
                'n_points': n_points,
                'internalization': np.nan
                
            })
        
        # Internalization:
        internalization = group['bud_internalization'].unique()

        # Centroid
        x_mean = group['xpos_recalibrated'].mean()
        y_mean = group['ypos_recalibrated'].mean()
        dx_centroid = group['xpos_recalibrated'].values - x_mean
        dy_centroid = group['ypos_recalibrated'].values - y_mean
        angles_centroid = np.arctan2(dy_centroid, dx_centroid)
        
        # Relative to origin
        dx_origin = group['xpos_recalibrated'].values
        dy_origin = group['ypos_recalibrated'].values
        angles_origin = np.arctan2(dy_origin, dx_origin)
        
        wd_centroid_values = []
        wd_origin_values = []
        for _ in range(loops):
            random_uniform = np.random.uniform(-np.pi, np.pi, n_random)
            random_shift = (np.pi + random_uniform) / (2*np.pi)
            
            angles_centroid_shift = (np.pi + angles_centroid) / (2*np.pi)
            angles_origin_shift = (np.pi + angles_origin) / (2*np.pi)
            
            wd_centroid = ot.wasserstein_circle(angles_centroid_shift, random_shift, p=1)
            wd_origin = ot.wasserstein_circle(angles_origin_shift, random_shift, p=1)
            
            wd_centroid_values.append(wd_centroid)
            wd_origin_values.append(wd_origin)
        
        return pd.Series({
            'avg_WD_centroid': np.mean(wd_centroid_values),
            'avg_WD_origin': np.mean(wd_origin_values),
            'n_points': n_points,
            'internalization': internalization[0]
        })

    # Apply to each (run, time) group
    wd_df = dir_df.groupby(['run', 'time']).apply(compute_group_WD).reset_index()
    
    # Aggregate by time (median + IQR + median n_points)
    grouped = wd_df.groupby('time').agg(
        median_centroid=('avg_WD_centroid', lambda x: np.nanmedian(x)),
        q25_centroid=('avg_WD_centroid', lambda x: np.nanpercentile(x, 25)),
        q75_centroid=('avg_WD_centroid', lambda x: np.nanpercentile(x, 75)),
        median_origin=('avg_WD_origin', lambda x: np.nanmedian(x)),
        q25_origin=('avg_WD_origin', lambda x: np.nanpercentile(x, 25)),
        q75_origin=('avg_WD_origin', lambda x: np.nanpercentile(x, 75)),
        median_n_points=('n_points', lambda x: np.nanmedian(x))
    ).reset_index()

    return wd_df, grouped


# --- Loop over datasets ---
results_dict = {}
for key, df in branched_actin_bound_points.items():
    wd_df, wd_summary = angle_WD_grouped(df, loops=1, n_random=1000)
    results_dict[key] = {'all': wd_df, 'summary': wd_summary}
    print(f"Processed {key}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# assume: zero_trials = ['trial1', 'trial2', 'trial3', 'trial4']

# --- Compute global vmin/vmax across all wd_dfs ---
all_dfs = [results_dict[k]['all'] for k in zero_trials if 'all' in results_dict[k]]
vmin = min(df['internalization'].min() for df in all_dfs)
vmax = max(df['internalization'].max() for df in all_dfs)

# --- Setup subplots ---
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
axes = axes.flatten()

# --- Plot each trial ---
for ax, trial in zip(axes, zero_trials):
    wd_df = results_dict[trial]['all']

    sc = ax.scatter(
        wd_df['n_points'],
        wd_df['avg_WD_origin'],
        c=wd_df['internalization'],
        cmap='magma',
        vmin=vmin, vmax=vmax,
        s=40, alpha=0.8, edgecolor='none'
    )

    ax.set_title(trial)
    ax.set_xlabel('n_points')
    ax.set_ylabel('avg_WD_origin')

# --- Shared colorbar ---
cbar = fig.colorbar(sc, ax=axes, fraction=0.03, pad=0.04)
cbar.set_label("Internalization", rotation=270, labelpad=15)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# zero_trials = ['trial1', 'trial2', 'trial3', 'trial4']

# --- Compute global color scale (optional, here use internalization / n_points) ---
all_dfs = [results_dict[k]['all'] for k in zero_trials if 'all' in results_dict[k]]
all_norm_internal = np.concatenate([df['internalization'].values / df['n_points'].values for df in all_dfs])
vmin, vmax = np.min(all_norm_internal), np.max(all_norm_internal)

# --- Set up 2x2 subplot grid ---
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
axes = axes.flatten()

# --- Plot each trial ---
for ax, trial in zip(axes, zero_trials):
    wd_df = results_dict[trial]['all']
    
    # Normalize by n_points
    x = wd_df['internalization'] / wd_df['n_points']
    y = wd_df['avg_WD_origin'] / wd_df['n_points']
    
    sc = ax.scatter(
        x,
        y,
        c='k',
        vmin=vmin,
        vmax=vmax,
        s=40,
        alpha=0.8,
        edgecolor='none'
    )
    
    ax.set_title(trial)
    ax.set_xlabel('Internalization / n_points')
    ax.set_ylabel('avg_WD_origin / n_points')
    ax.grid(True, alpha=0.3)

# --- Shared colorbar ---
#cbar = fig.colorbar(sc, ax=axes, fraction=0.03, pad=0.04)
#cbar.set_label("Internalization / n_points", rotation=270, labelpad=15)

plt.tight_layout()
plt.show()


In [ ]:
wd_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Create a discrete color palette from viridis ---
num_trials = len(zero_trials)
colors = sns.color_palette("viridis", num_trials)  # discrete colors

plt.figure(figsize=(8,6))

for i, trial in enumerate(zero_trials):
    wd_df = results_dict[trial]['all']
    wd_df = wd_df.dropna(subset=['avg_WD_origin', 'internalization', 'n_points'])
    
    # Average per run
    run_avg = wd_df.groupby('run').agg({
        'internalization': 'mean',
        'avg_WD_origin': 'mean',
        'n_points': 'mean'
    })
    
    x = run_avg['internalization'] / run_avg['n_points']
    y = run_avg['avg_WD_origin'] / run_avg['n_points']
    
    plt.scatter(
        x, y,
        color=colors[i],
        s=60,
        alpha=0.8,
        label=trial,
        edgecolor='none'
    )

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Internalization / n_points')
plt.ylabel('avg_WD_origin / n_points')
plt.grid(True, alpha=0.3)
plt.legend(title='Trial')
plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Create a discrete color palette from viridis ---
num_trials = len(zero_trials)
colors = sns.color_palette("viridis", num_trials)  # discrete colors
labels = ['Full', '3/4', '1/2', '1/4']
plt.figure(figsize=(8,6))

for i, trial in enumerate(zero_trials):
    wd_df = results_dict[trial]['all']
    wd_df = wd_df.dropna(subset=['avg_WD_origin', 'internalization', 'n_points'])
    
    # Average per run
    run_avg = wd_df.groupby('run').agg({
        'internalization': 'mean',
        'avg_WD_origin': 'mean',
        'n_points': 'mean'
    })
    
    # Normalize
    x = run_avg['internalization'] / run_avg['n_points']
    y = run_avg['avg_WD_origin'] / run_avg['n_points']
    
    # Filter out non-positive values to avoid log(0)
    mask = (x > 0) & (y > 0)
    x = x[mask]
    y = y[mask]
    
    # Scatter points
    plt.scatter(
        x, y,
        color=colors[i],
        s=60,
        alpha=0.8,
        label=labels[i],
        edgecolor='none'
    )
    
    # --- Linear fit on log-log scale ---
    log_x = np.log10(x)
    log_y = np.log10(y)
    slope, intercept = np.polyfit(log_x, log_y, 1)
    
    # Fit line (back to original scale)
    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = 10**intercept * x_fit**slope
    plt.plot(x_fit, y_fit, color=colors[i], linestyle='--')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Internalization / n_points')
plt.ylabel('avg_WD_origin / n_points')
plt.grid(True, alpha=0.3, which='both', linestyle='--')
plt.legend(title='Trial')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

labels = ['Full', '3/4', '1/2', '1/4']
num_trials = len(zero_trials)
colors = sns.color_palette("viridis", num_trials)  # discrete colors per trial

fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharex=True, sharey=True)

for ax_idx, color_mode in enumerate(['trial', 'WD', 'internalization', 'n_points']):
    ax = axes[ax_idx]
    
    for i, trial in enumerate(zero_trials):
        wd_df = results_dict[trial]['all'].dropna(subset=['avg_WD_origin', 'internalization', 'n_points'])
        
        # Compute per (run, time) ratios
        ratios = wd_df.copy()
        ratios['y_ratio'] = ratios['avg_WD_origin'] / ratios['n_points']  # WD per point
        ratios['x_ratio'] = ratios['internalization'] / ratios['n_points']                      # raw internalization
        
        # Average per run
        run_avg = ratios.groupby('run').agg({
            'x_ratio': 'mean',                     # average raw internalization
            'y_ratio': 'mean',                   # average WD/n_points
            'internalization': lambda x: np.percentile(x, 95),  # 95th percentile internalization
            'avg_WD_origin': 'mean',
            'n_points': 'mean'
        })
        
        x = run_avg['x_ratio']
        y = run_avg['y_ratio']
        
        # Filter non-positive values
        mask = (x > 0) & (y > 0)
        x = x[mask]
        y = y[mask]
        run_avg = run_avg.loc[mask]
        
        # Determine color
        if color_mode == 'trial':
            point_colors = [colors[i]] * len(x)
        elif color_mode == 'WD':
            point_colors = run_avg['avg_WD_origin']
            vmin = 0
            vmax = max([results_dict[t]['all']['avg_WD_origin'].max() for t in zero_trials])
        elif color_mode == 'internalization':
            point_colors = run_avg['internalization']
            vmin = 0
            vmax = max([results_dict[t]['all']['internalization'].max() for t in zero_trials])
        elif color_mode == 'n_points':
            point_colors = run_avg['n_points']
            vmin = 0
            vmax = max([results_dict[t]['all']['n_points'].max() for t in zero_trials])
        
        # Scatter plot
        sc = ax.scatter(x, y, c=point_colors, cmap='viridis' if color_mode != 'trial' else None,
                        s=60, alpha=0.8, label=labels[i] if color_mode=='trial' else None, edgecolor='none',
                        vmin=vmin if color_mode != 'trial' else None, vmax=vmax if color_mode != 'trial' else None)
        
        # Optional linear fit for trial panel
        if color_mode == 'trial':
            log_x = np.log10(x)
            log_y = np.log10(y)
            slope, intercept = np.polyfit(log_x, log_y, 1)
            x_fit = np.linspace(x.min(), x.max(), 100)
            y_fit = 10**intercept * x_fit**slope
            #ax.plot(x_fit, y_fit, color=colors[i], linestyle='--')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Internalization / n_points')
    if ax_idx == 0:
        ax.set_ylabel('avg_WD_origin / n_points')
    ax.grid(True, alpha=0.3, which='both', linestyle='--')
    
    # Add colorbars for panels 2-4
    if color_mode != 'trial':
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(color_mode.replace('_',' ').title())

# Legend for first panel
axes[0].legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

labels = ['Full', '3/4', '1/2', '1/4']
num_trials = len(zero_trials)
colors = sns.color_palette("viridis", num_trials)  # discrete colors per trial

# Define color modes for 3 panels
color_modes = ['trial', 'WD', 'n_points']

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

for ax_idx, color_mode in enumerate(color_modes):
    ax = axes[ax_idx]
    
    for i, trial in enumerate(zero_trials):
        wd_df = results_dict[trial]['all'].dropna(subset=['avg_WD_origin', 'internalization', 'n_points'])
        
        # Average per run
        run_avg = wd_df.groupby('run').agg({
            'internalization': lambda x: np.percentile(x, 95),
            'avg_WD_origin': 'mean',
            'n_points': 'mean'
        })
        
        # x = raw internalization, y = avg_WD_origin / n_points
        x = run_avg['internalization']
        y = run_avg['avg_WD_origin'] / run_avg['n_points']
        
        # Filter out non-positive values
        mask = (x > 0) & (y > 0)
        x = x[mask]
        y = y[mask]
        run_avg = run_avg.loc[mask]
        
        # Determine color
        if color_mode == 'trial':
            point_colors = [colors[i]] * len(x)
        elif color_mode == 'WD':
            point_colors = run_avg['avg_WD_origin']
            vmin = 0
            vmax = max([results_dict[t]['all']['avg_WD_origin'].max() for t in zero_trials])
        elif color_mode == 'n_points':
            point_colors = run_avg['n_points']
            vmin = 0
            vmax = max([results_dict[t]['all']['n_points'].max() for t in zero_trials])
        
        # Scatter plot
        sc = ax.scatter(x, y, c=point_colors, cmap='viridis' if color_mode != 'trial' else None,
                        s=60, alpha=0.8, label=labels[i] if color_mode=='trial' else None, edgecolor='none',
                        vmin=vmin if color_mode != 'trial' else None, vmax=vmax if color_mode != 'trial' else None)
    
    #ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Internalization (95th Percentile)')
    if ax_idx == 0:
        ax.set_ylabel('Avg WD / n_points')
    ax.grid(True, alpha=0.3, which='both', linestyle='--')
    
    # Add colorbar for panels 2-3
    if color_mode != 'trial':
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(color_mode.replace('_',' ').title())

# Legend for first panel
axes[0].legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

labels = ['Full', '3/4', '1/2', '1/4']
num_trials = len(zero_trials)
trial_colors = sns.color_palette("viridis", num_trials)  # one color per trial

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

for ax_idx, color_mode in enumerate(['trial', 'internalization']):
    ax = axes[ax_idx]
    
    for i, trial in enumerate(zero_trials):
        wd_df = results_dict[trial]['all'].dropna(subset=['avg_WD_origin', 'internalization', 'n_points'])
        
        # Average per run
        run_avg = wd_df.groupby('run').agg({
            'avg_WD_origin': 'mean',
            'n_points': 'mean',
            'internalization': 'mean'
        })
        
        x = run_avg['n_points']
        y = run_avg['avg_WD_origin']
        colors_run = run_avg['internalization']
        
        if color_mode == 'trial':
            point_colors = [trial_colors[i]] * len(x)
        else:  # internalization
            point_colors = colors_run
            vmin = 0
            vmax = max([results_dict[t]['all']['internalization'].max() for t in zero_trials])
        
        sc = ax.scatter(
            x, y, c=point_colors, cmap='viridis' if color_mode=='internalization' else None,
            s=80, alpha=0.8, edgecolor='none', label=labels[i] if color_mode=='trial' else None,
            vmin=vmin if color_mode=='internalization' else None,
            vmax=vmax if color_mode=='internalization' else None
        )
    
    ax.set_xlabel('N_points (raw, per run)')
    if ax_idx == 0:
        ax.set_ylabel('Avg WD (raw, per run)')
    ax.grid(True, alpha=0.3)
    
    # Add colorbar for internalization panel
    if color_mode == 'internalization':
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Internalization (raw, per run)')

# Legend for trial-colored panel
axes[0].legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

labels = ['Full', '3/4', '1/2', '1/4']
num_trials = len(zero_trials)
trial_colors = sns.color_palette("viridis", num_trials)  # one color per trial

plt.figure(figsize=(8,6))

for i, trial in enumerate(zero_trials):
    wd_df = results_dict[trial]['all'].dropna(subset=['avg_WD_origin', 'internalization', 'n_points'])
    
    # Compute per-run average WD and std of internalization
    run_stats = wd_df.groupby('run').agg({
        'avg_WD_origin': 'mean',
        'internalization': 'std'
    }).reset_index()
    
    x = run_stats['internalization']  # STD of internalization per run
    y = run_stats['avg_WD_origin']    # Avg WD per run
    
    plt.scatter(
        x, y,
        color=trial_colors[i],
        s=80,
        alpha=0.8,
        label=labels[i],
        edgecolor='none'
    )

plt.xlabel('Internalization STD (per run)')
plt.ylabel('Avg WD (per run)')
plt.grid(True, alpha=0.3)
plt.legend(title='Trial')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

labels = ['Full', '3/4', '1/2', '1/4']

# Prepare dataframe for plotting
cv_data = []

for i, trial in enumerate(zero_trials):
    wd_df = results_dict[trial]['all'].dropna(subset=['internalization', 'n_points'])
    
    # Compute per-run mean and std
    run_stats = wd_df.groupby('run').agg({
        'internalization': ['mean', 'std']
    }).reset_index()
    run_stats.columns = ['run', 'internalization_mean', 'internalization_std']
    
    # Only keep runs with mean > 0
    run_stats = run_stats[run_stats['internalization_mean'] > 0]
    
    # Compute CV
    run_stats['CV'] = run_stats['internalization_std'] / run_stats['internalization_mean']
    
    # Add trial label
    run_stats['trial'] = labels[i]
    cv_data.append(run_stats[['CV', 'trial']])

cv_df = pd.concat(cv_data, ignore_index=True)

# --- Compute median and IQR for each trial ---
summary_stats = cv_df.groupby('trial')['CV'].agg(['median', lambda x: np.percentile(x, 75), lambda x: np.percentile(x, 25)])
summary_stats.columns = ['median', 'q75', 'q25']

# --- Plot medians with IQR error bars ---
plt.figure(figsize=(8,6))
x_pos = np.arange(len(labels))
plt.errorbar(
    x=x_pos,
    y=summary_stats['median'],
    yerr=[summary_stats['median'] - summary_stats['q25'], summary_stats['q75'] - summary_stats['median']],
    fmt='o',
    color='darkblue',
    capsize=6,
    markersize=8,
    label='Median ± IQR'
)

plt.xticks(x_pos, labels)
#plt.yscale('log')
plt.xlabel('Trial')
plt.ylabel('Internalization CV (per run)')
plt.title('Median Internalization CV per Run with IQR')
plt.grid(True, which='both', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare dataframe for plotting
cv_data = []

for trial in zero_trials:
    wd_df = results_dict[trial]['all'].dropna(subset=['internalization', 'n_points'])
    
    # Compute per-run mean and std
    run_stats = wd_df.groupby('run').agg({
        'internalization': ['mean', 'std']
    }).reset_index()
    run_stats.columns = ['run', 'internalization_mean', 'internalization_std']
    
    # Only keep runs with mean > 0
    run_stats = run_stats[run_stats['internalization_mean'] > 0]
    
    # Compute CV
    run_stats['CV'] = run_stats['internalization_std'] / run_stats['internalization_mean']
    
    # Add trial label
    run_stats['trial'] = trial
    cv_data.append(run_stats[['CV', 'trial']])

cv_df = pd.concat(cv_data, ignore_index=True)

# --- Plot distribution ---
plt.figure(figsize=(8,6))
sns.violinplot(x='trial', y='CV', data=cv_df, palette='viridis')
plt.yscale('log')
plt.ylabel('Internalization Coefficient of Variation (per run)')
plt.xlabel('Trial')
plt.title('Distribution of Internalization CV per Run')
plt.grid(True, alpha=0.3, which='both', linestyle='--')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

labels = ['Full', '3/4', '1/2', '1/4']
num_trials = len(zero_trials)
trial_colors = sns.color_palette("viridis", num_trials)  # discrete trial colors

fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

for ax_idx, color_mode in enumerate(['trial', 'internalization_mean', 'internalization_std']):
    ax = axes[ax_idx]
    
    for i, trial in enumerate(zero_trials):
        wd_df = results_dict[trial]['all'].dropna(subset=['avg_WD_origin', 'internalization', 'n_points'])
        
        # Compute per-run average WD and internalization mean/std
        run_stats = wd_df.groupby('run').agg({
            'avg_WD_origin': 'mean',
            'internalization': ['mean', 'std']
        }).reset_index()
        
        # Flatten MultiIndex columns
        run_stats.columns = ['run', 'avg_WD_origin', 'internalization_mean', 'internalization_std']
        
        # Filter out runs with mean <= 0
        run_stats = run_stats[run_stats['internalization_mean'] > 0]
        
        x = run_stats['internalization_std'] / run_stats['internalization_mean']  # CV
        y = run_stats['avg_WD_origin']
        
        # Determine color
        if color_mode == 'trial':
            point_colors = [trial_colors[i]] * len(x)
        else:
            point_colors = run_stats[color_mode]
            vmin = 0
            vmax = max([results_dict[t]['all']['internalization'].max() for t in zero_trials]) if color_mode=='internalization_mean' else max([results_dict[t]['all']['internalization'].std() for t in zero_trials])
        
        sc = ax.scatter(
            x, y,
            c=point_colors,
            cmap='viridis' if color_mode != 'trial' else None,
            s=80, alpha=0.8, edgecolor='none',
            label=labels[i] if color_mode=='trial' else None,
            vmin=vmin if color_mode != 'trial' else None,
            vmax=vmax if color_mode != 'trial' else None
        )
    
    ax.set_xscale('log')
    ax.set_xlabel('Internalization CV (per run)')
    if ax_idx == 0:
        ax.set_ylabel('Avg WD (per run)')
    ax.grid(True, alpha=0.3)
    
    # Add colorbar for mean/internalization std
    if color_mode != 'trial':
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(color_mode.replace('_',' ').title())
        
# Legend for trial-colored panel
axes[0].legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

labels = ['Full', '3/4', '1/2', '1/4']
num_trials = len(zero_trials)
trial_colors = sns.color_palette("viridis", num_trials)  # discrete trial colors

plt.figure(figsize=(8,6))

for i, trial in enumerate(zero_trials):
    wd_df = results_dict[trial]['all'].dropna(subset=['avg_WD_origin', 'internalization', 'n_points'])
    
    # Compute per-run average WD and internalization mean/std
    run_stats = wd_df.groupby('run').agg({
        'avg_WD_origin': 'mean',
        'internalization': ['mean', 'std']
    }).reset_index()
    
    # Flatten MultiIndex columns
    run_stats.columns = ['run', 'avg_WD_origin', 'internalization_mean', 'internalization_std']
    
    # Filter out runs with mean <= 0
    run_stats = run_stats[run_stats['internalization_mean'] > 0]
    
    # Compute CV
    cv = run_stats['internalization_std'] / run_stats['internalization_mean']
    std = run_stats['internalization_std']
    
    plt.scatter(
        std, cv,
        color=trial_colors[i],
        s=80,
        alpha=0.8,
        label=labels[i],
        edgecolor='none'
    )

#plt.xscale('log')
plt.yscale('log')
plt.xlabel('Internalization STD (per run)')
plt.ylabel('Internalization CV (per run)')
plt.grid(True, alpha=0.3, which='both', linestyle='--')
plt.legend(title='Trial')
plt.tight_layout()
plt.show()


In [ ]:
branched_actin_bound_points['endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output']

In [ ]:

# Only use the first four trials (no tension)
no_tension_trials = zero_trials[:4]

# Prepare DataFrame for plotting
plot_data = []
for trial in no_tension_trials:
    trial_df = branched_actin_bound_points[trial].dropna(subset=['bud_zpos'])
    trial_df = trial_df.sort_values(['run', 'time'])
    
    for run_id, run_df in trial_df.groupby('run'):
        z_series = run_df['bud_zpos'].values.astype(float)
        displacements = np.diff(z_series)
        n_pos = np.sum(displacements > 0)
        n_neg = np.sum(displacements < 0)
        ratio = n_neg / n_pos if n_pos > 0 else np.nan
        if not np.isnan(ratio):
            plot_data.append({'condition_label': trial, 'z_bias_sum_ratio': ratio})

z_bias_sum_df_branched = pd.DataFrame(plot_data)

# Mapping of trial names to short labels
label_map = {
    'endocytosis2_baseline_zero_diffusion_output': 'Full',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output': '3/4',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output': '1/2',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output': '1/4'
}

# Replace trial names with short labels
z_bias_sum_df_branched['condition_label'] = z_bias_sum_df_branched['condition_label'].map(label_map)

# --- Plot ---
plt.figure(figsize=(8,6))
ax = sns.violinplot(
    data=z_bias_sum_df_branched,
    x='condition_label',
    y='z_bias_sum_ratio',
    palette='viridis',
    inner="box",  # shows median and IQR
    alpha=0.6
)

sns.stripplot(
    data=z_bias_sum_df_branched,
    x='condition_label',
    y='z_bias_sum_ratio',
    color='black',
    alpha=0.7,
    jitter=True,
    size=4
)

plt.ylabel('Sum(Δbud_zpos<0) / Sum(Δbud_zpos>0)')
plt.xlabel('Trial')
plt.title('No Tension Trials: Distribution of bud_zpos Ratios')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Only use the first four trials (no tension)
no_tension_trials = zero_trials[:4]
plot_labels = ['Full', '3/4', '1/2', '1/4']

# Prepare DataFrame for plotting
plot_data = []
for trial, label in zip(no_tension_trials, plot_labels):
    trial_df = branched_actin_bound_points[trial].dropna(subset=['bud_zpos'])
    trial_df = trial_df.sort_values(['run', 'time'])
    
    for run_id, run_df in trial_df.groupby('run'):
        z_series = run_df['bud_zpos'].values.astype(float)
        displacements = np.diff(z_series)
        n_pos = np.sum(displacements > 0)
        n_neg = np.sum(displacements < 0)
        ratio = n_neg / n_pos if n_pos > 0 else np.nan
        if not np.isnan(ratio):
            plot_data.append({'run_id': run_id, 'trial_label': label, 'ratio': ratio})

z_bias_sum_df_branched = pd.DataFrame(plot_data)

# Compute median and quartiles per trial
summary_stats = z_bias_sum_df_branched.groupby('trial_label').ratio.agg([
    ('median', 'median'),
    ('q25', lambda x: np.percentile(x, 25)),
    ('q75', lambda x: np.percentile(x, 75))
]).reset_index()

# --- Plot ---
plt.figure(figsize=(8,6))
palette = sns.color_palette("viridis", n_colors=len(plot_labels))

# Ensure x-axis order
summary_stats = summary_stats.set_index('trial_label').reindex(plot_labels).reset_index()

plt.errorbar(
    x=summary_stats['trial_label'],
    y=summary_stats['median'],
    yerr=[summary_stats['median'] - summary_stats['q25'], summary_stats['q75'] - summary_stats['median']],
    fmt='-o',
    color='purple',
    capsize=5
)

plt.xlabel('Trial')
plt.ylabel('Sum(Δbud_zpos<0) / Sum(Δbud_zpos>0)')
plt.title('No Tension Trials: bud_zpos Ratio (Median ± IQR)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def angle_WD_grouped(dir_df, loops=50):
    """
    Compute circular Wasserstein distance of point angles (relative to origin)
    compared to a uniform circular distribution.

    Groups with fewer than 2 points are skipped (avg_WD = NaN).
    Also returns number of points per group and internalization state.
    """
    def compute_group_WD(group):
        n_points = len(group)
        if n_points < 2:
            return pd.Series({
                'avg_WD_origin': np.nan,
                'avg_WD_origin_corrected': np.nan,
                'n_points': n_points,
                'internalization': np.nan
            })
        
        # Internalization
        internalization = group['bud_internalization'].unique()

        # Angles relative to origin
        dx_origin = group['xpos_recalibrated'].values
        dy_origin = group['ypos_recalibrated'].values
        angles_origin = np.arctan2(dy_origin, dx_origin)
        angles_origin_shift = (np.pi + angles_origin) / (2 * np.pi)
        
        wd_origin_values = []
        wd_origin_corrected_values = []
        for _ in range(loops):
            random_uniform = np.random.uniform(-np.pi, np.pi, 5000)
            rand_uniform_2 = np.random.uniform(-np.pi, np.pi, n_points)
            random_shift = (np.pi + random_uniform) / (2 * np.pi)
            random_shift_2 = (np.pi + rand_uniform_2) / (2 * np.pi)

            wd_origin = ot.wasserstein_circle(angles_origin_shift, random_shift, p=1)
            wd_uniform = ot.wasserstein_circle(random_shift, random_shift_2, p=1)

            wd_origin_corrected = wd_origin - wd_uniform
            wd_origin_values.append(wd_origin)
            wd_origin_corrected_values.append(wd_origin_corrected)
        
        return pd.Series({
            'avg_WD_origin': np.mean(wd_origin_values),
            'avg_WD_origin_corrected': np.mean(wd_origin_corrected_values),
            'n_points': n_points,
            'internalization': internalization[0]
        })

    # Apply to each (run, time) group
    wd_df = dir_df.groupby(['run', 'time']).apply(compute_group_WD).reset_index()
    
    # Aggregate by time (median + IQR + median n_points)
    grouped = wd_df.groupby('time').agg(
        median_origin=('avg_WD_origin', np.nanmedian),
        q25_origin=('avg_WD_origin', lambda x: np.nanpercentile(x, 25)),
        q75_origin=('avg_WD_origin', lambda x: np.nanpercentile(x, 75)),
        median_corrected=('avg_WD_origin_corrected', np.nanmedian),
        q25_corrected=('avg_WD_origin_corrected', lambda x: np.nanpercentile(x, 25)),
        q75_corrected=('avg_WD_origin_corrected', lambda x: np.nanpercentile(x, 75)),
        median_n_points=('n_points', np.nanmedian)
    ).reset_index()

    return wd_df, grouped


# # --- Loop over datasets ---
# results_dict = {}
# for key, df in branched_actin_bound_points.items():
#     wd_df, wd_summary = angle_WD_grouped(df, loops=50)
#     results_dict[key] = {'all': wd_df, 'summary': wd_summary}
#     print(f"Processed {key}")

# --- Loop over only zero_trials ---
results_dict = {}

for key in zero_trials:
    if key in branched_actin_bound_points:
        df = branched_actin_bound_points[key]
        wd_df, wd_summary = angle_WD_grouped(df, loops=50)
        results_dict[key] = {'all': wd_df, 'summary': wd_summary}
        print(f"Processed {key}")
    else:
        print(f"⚠️ Skipped {key}: not found in branched_actin_bound_points")



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set seaborn style
sns.set(style="whitegrid")

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(12,10))
axes = axes.flatten()

# Generate a viridis palette for the 4 datasets
palette = sns.color_palette("viridis", n_colors=len(zero_trials))

for i, key in enumerate(zero_trials):
    df = results_dict[key]['all']
    ax = axes[i]
    
    sns.scatterplot(
        x='avg_WD_origin_corrected', 
        y='internalization', 
        data=df, 
        ax=ax,
        color=palette[i],
        s=50,
        alpha=0.7
    )
    
    ax.set_title(key, fontsize=10)
    ax.set_xlabel("avg_WD_origin_corrected")
    ax.set_ylabel("Internalization")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

labels = ['Full', '3/4', '1/2', '1/4']

# Set seaborn style
sns.set(style="whitegrid")

# Create 2x2 subplot with shared axes
fig, axes = plt.subplots(2, 2, figsize=(12,10), sharex=True, sharey=True)
axes = axes.flatten()

# Viridis palette for the 4 datasets
palette = sns.color_palette("viridis", n_colors=len(zero_trials))

for i, key in enumerate(zero_trials):
    df = results_dict[key]['all'].copy()
    ax = axes[i]
    
    # Group by 'run' and compute mean per run
    df_grouped = df.groupby('run')[['avg_WD_origin_corrected', 'internalization']].mean().reset_index()
    df_grouped['internalization'] *= 1000  # convert µm → nm
    
    # Compute Pearson correlation
    corr = df_grouped[['avg_WD_origin_corrected', 'internalization']].corr().iloc[0, 1]
    
    # Scatter plot of averaged values
    sns.scatterplot(
        x='avg_WD_origin_corrected',
        y='internalization',
        data=df_grouped,
        ax=ax,
        color=palette[i],
        s=80,
        alpha=0.8,
        label=f"r = {corr:.2f}"
    )
    
    ax.legend(frameon=False, loc='upper left', fontsize=10)
    ax.set_title(labels[i], fontsize=20)
    ax.set_xlabel("Corrected WD")
    ax.set_ylabel("Internalization (nm)")

plt.suptitle("Internalization vs Corrected WD , Tension: 150 pN/um", fontsize=25)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

labels = ['Full', '3/4', '1/2', '1/4']

# Set seaborn style
sns.set(style="whitegrid")

# Create 2x2 subplot with shared axes
fig, axes = plt.subplots(2, 2, figsize=(12,10), sharex=True, sharey=True)
axes = axes.flatten()

# Determine global n_points range for consistent color scale
all_npoints = []
for key in zero_trials:
    df = results_dict[key]['all'].copy()
    df_grouped = df.groupby('run')[['avg_WD_origin_corrected', 'internalization', 'n_points']].mean().reset_index()
    all_npoints.extend(df_grouped['n_points'].values)
vmin = min(all_npoints)
vmax = max(all_npoints)

# Viridis colormap
cmap = plt.cm.magma

for i, key in enumerate(zero_trials):
    df = results_dict[key]['all'].copy()
    ax = axes[i]
    
    # Group by 'run' and compute mean per run
    df_grouped = df.groupby('run')[['avg_WD_origin_corrected', 'internalization', 'n_points']].mean().reset_index()
    df_grouped['internalization'] = df_grouped['internalization'] * 1000  # scale to nm
    
    # Scatter plot colored by n_points
    scatter = ax.scatter(
        df_grouped['avg_WD_origin_corrected'],
        df_grouped['internalization'],
        c=df_grouped['n_points'],
        cmap=cmap,
        s=80,
        alpha=0.8,
        vmin=vmin,
        vmax=vmax
    )
    
    ax.set_title(labels[i], fontsize=20)
    ax.set_xlabel("Corrected WD")
    ax.set_ylabel("Internalization (nm)")
    ax.grid(True, alpha=0.3)

# Add a single horizontal colorbar below the subplots
cbar_ax = fig.add_axes([0.15, -0.05, 0.7, 0.03])  # [left, bottom, width, height]
cbar = fig.colorbar(scatter, cax=cbar_ax, orientation='horizontal')
cbar.set_label("Avg Model Points", fontsize=14)

plt.suptitle("Internalization vs Corrected WD, Tension: 150 pN/um", fontsize=25)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

labels = ['Full', '3/4', '1/2', '1/4']

sns.set(style="whitegrid")

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(12,10), sharex=True, sharey=True)
axes = axes.flatten()

palette = sns.color_palette("viridis", n_colors=len(zero_trials))

for i, key in enumerate(zero_trials):
    df = results_dict[key]['all'].copy()
    ax = axes[i]
    
    WD_last_list = []
    int_95_list = []
    
    # Loop over runs
    for run, run_df in df.groupby('run'):
        last_time_idx = run_df['time'].idxmax()
        WD_last = run_df.loc[last_time_idx, 'avg_WD_origin_corrected']
        int_95 = np.nanpercentile(run_df['internalization'], 95)
        
        WD_last_list.append(WD_last)
        int_95_list.append(int_95 * 1000)  # scale to nm
    
    # Scatter plot
    ax.scatter(WD_last_list, int_95_list, s=80, alpha=0.8, color=palette[i])
    
    ax.set_title(labels[i], fontsize=20)
    ax.set_xlabel("Corrected WD (last time point)")
    ax.set_ylabel("95th percentile Internalization (nm)")
    ax.grid(True, alpha=0.3)

plt.suptitle("95th percentile Internalization vs Last Timepoint WD, Tension: 150 pN/um", fontsize=25)
plt.tight_layout()
plt.show()


In [ ]:
new_dict = {}

for key in zero_trials_tension:
    df = results_dict[key]['all'].copy()
    
    # Extract tension_first as before
    run_strs = df['run'].str.replace('run-', '', regex=False)
    tension_str = run_strs.str.split('_tension').str[1]
    tension_first = tension_str.str.split('-').str[0]
    
    # Add column for convenience (optional)
    df['tension_first'] = tension_first
    
    # Initialize nested dict
    new_dict[key] = {}
    
    # Group by unique tension_first
    for t in df['tension_first'].unique():
        new_dict[key][t] = df[df['tension_first'] == t].copy()
        
# Example usage:
# new_dict['endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output']['0000']


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set(style="whitegrid")

# Collect all unique tension_first across trials
all_tensions = set()
for trial in new_dict:
    all_tensions.update(new_dict[trial].keys())
all_tensions = sorted(all_tensions)  # sort numerically if possible

# Viridis palette for trials
trial_keys = list(new_dict.keys())
palette = sns.color_palette("viridis", n_colors=len(trial_keys))

# Loop through tensions and make 2x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(14,12))
axes = axes.flatten()

for i, tension in enumerate(all_tensions):
    ax = axes[i]
    
    for j, trial in enumerate(trial_keys):
        df_trial = new_dict[trial].get(tension)
        if df_trial is not None and len(df_trial) > 0:
            # Group by 'run' and take mean of WD and internalization
            grouped = df_trial.groupby('run')[['avg_WD_origin_corrected', 'internalization']].mean().reset_index()
            
            ax.scatter(
                grouped['avg_WD_origin_corrected'],
                grouped['internalization'],
                s=60,
                alpha=0.7,
                color=palette[j],
                label=trial
            )
    
    ax.set_xlabel("avg_WD_origin_corrected")
    ax.set_ylabel("Internalization")
    ax.set_title(f"Tension first = {tension}")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Focus on tension_first = '0003'
tension_target = '0003'

# Viridis palette for trials
trial_keys = list(new_dict.keys())
palette = sns.color_palette("viridis", n_colors=len(trial_keys))

# 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(14,12))
axes = axes.flatten()

for i, trial in enumerate(trial_keys):
    ax = axes[i]
    
    df_trial = new_dict[trial].get(tension_target)
    if df_trial is not None and len(df_trial) > 0:
        # Group by 'run' and take mean of WD and internalization
        grouped = df_trial.groupby('run')[['avg_WD_origin_corrected', 'internalization']].mean().reset_index()
        
        # Scatter plot
        ax.scatter(
            grouped['avg_WD_origin_corrected'],
            grouped['internalization'],
            s=60,
            alpha=0.8,
            color=palette[i]
        )
    
    ax.set_title(trial, fontsize=10)
    ax.set_xlabel("avg_WD_origin_corrected")
    ax.set_ylabel("Internalization")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Focus on tension_first = '0003'
tension_target = '0003'

# Viridis palette for trials (for consistency in other uses, optional here)
trial_keys = list(new_dict.keys())

# 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(14,12))
axes = axes.flatten()

for i, trial in enumerate(trial_keys):
    ax = axes[i]
    
    df_trial = new_dict[trial].get(tension_target)
    if df_trial is not None and len(df_trial) > 0:
        # Group by 'run' and take mean per run, including n_points
        grouped = df_trial.groupby('run')[['avg_WD_origin_corrected', 'internalization', 'n_points']].mean().reset_index()
        
        # Scatter plot colored by n_points
        scatter = ax.scatter(
            grouped['avg_WD_origin_corrected'],
            grouped['internalization'],
            c=grouped['n_points'],
            cmap='viridis',
            s=60,
            alpha=0.8
        )
        
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label("n_points")
    
    ax.set_title(trial, fontsize=10)
    ax.set_xlabel("avg_WD_origin_corrected")
    ax.set_ylabel("Internalization")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

zero_trials = [
    'endocytosis2_baseline_zero_diffusion_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
]

sns.set(style="whitegrid")
palette = sns.color_palette("viridis", n_colors=len(zero_trials))

plt.figure(figsize=(8,6))
n_bins = 10  # adjust as needed

for i, key in enumerate(zero_trials):
    df = results_dict[key]['all'].copy()
    # Drop NaNs
    df = df[['avg_WD_origin_corrected', 'internalization']].dropna()
    x = df['avg_WD_origin_corrected'].values
    y = df['internalization'].values
    
    # Bin edges
    bins = np.linspace(np.min(x), np.max(x), n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    medians = []
    q25 = []
    q75 = []
    
    for j in range(n_bins):
        if j < n_bins - 1:
            mask = (x >= bins[j]) & (x < bins[j+1])
        else:
            # Include rightmost edge in the last bin
            mask = (x >= bins[j]) & (x <= bins[j+1])
        
        y_bin = y[mask]
        if len(y_bin) > 0:
            medians.append(np.median(y_bin))
            q25.append(np.percentile(y_bin, 25))
            q75.append(np.percentile(y_bin, 75))
        else:
            medians.append(np.nan)
            q25.append(np.nan)
            q75.append(np.nan)
    
    # Convert lists to arrays and remove NaNs for plotting
    medians = np.array(medians)
    q25 = np.array(q25)
    q75 = np.array(q75)
    valid = ~np.isnan(medians)
    
    plt.errorbar(bin_centers[valid], medians[valid],
                 yerr=[medians[valid]-q25[valid], q75[valid]-medians[valid]],
                 fmt='o-', color=palette[i], label=key, capsize=5, markersize=6, alpha=0.8)

plt.xlabel("avg_WD_origin_corrected")
plt.ylabel("Internalization")
plt.title("Binned median ± IQR for all zero_trials")
plt.legend(frameon=False)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Figure Three

## 3A

## 3B

In [ ]:
summary_dict = {}
summary_growth_dict = {}

for key in plot_order:
    df = hip1r_assoc_positions[key]
    data = df.reset_index()
    positions_df = data[['run', 'time', 'fiber_id', 'zpos_recalibrated', 'bud_internalization']]

    positions_df = positions_df.sort_values(['run', 'time'])

    # Compute dz and sum_dz per run
    positions_df['dz'] = positions_df.groupby('run')['bud_internalization'].diff()
    positions_df['sum_dz'] = positions_df.groupby('run')['dz'].cumsum()

    fibers_count = positions_df.groupby(['run', 'time'])['fiber_id'].nunique().reset_index(name='num_fibers')
    median_fibers = fibers_count.groupby('time')['num_fibers'].median().reset_index(name='median_num_fibers')
    q25_fibers = fibers_count.groupby('time')['num_fibers'].quantile(0.25).reset_index(name='q25_num_fibers')
    q75_fibers = fibers_count.groupby('time')['num_fibers'].quantile(0.75).reset_index(name='q75_num_fibers')

    median_bud_internalization = positions_df.groupby('time')['bud_internalization'].median().reset_index(name='median_bud_internalization')
    q25_bud_internalization = positions_df.groupby('time')['bud_internalization'].quantile(0.25).reset_index(name='q25_bud_internalization')
    q75_bud_internalization = positions_df.groupby('time')['bud_internalization'].quantile(0.75).reset_index(name='q75_bud_internalization')

    # Optional: summarize dz and sum_dz as well
    median_dz = positions_df.groupby('time')['dz'].median().reset_index(name='median_dz')
    q25_dz = positions_df.groupby('time')['dz'].quantile(0.25).reset_index(name='q25_dz')
    q75_dz = positions_df.groupby('time')['dz'].quantile(0.75).reset_index(name='q75_dz')

    median_sum_dz = positions_df.groupby('time')['sum_dz'].median().reset_index(name='median_sum_dz')
    q25_sum_dz = positions_df.groupby('time')['sum_dz'].quantile(0.25).reset_index(name='q25_sum_dz')
    q75_sum_dz = positions_df.groupby('time')['sum_dz'].quantile(0.75).reset_index(name='q75_sum_dz')


    

    summary = pd.merge(median_fibers, q25_fibers, on='time')
    summary = pd.merge(summary, q75_fibers, on='time')
    summary = pd.merge(summary, median_bud_internalization, on='time')
    summary = pd.merge(summary, q25_bud_internalization, on='time')
    summary = pd.merge(summary, q75_bud_internalization, on='time')

    # Merge optional dz summaries if you want
    summary = pd.merge(summary, median_dz, on='time')
    summary = pd.merge(summary, q25_dz, on='time')
    summary = pd.merge(summary, q75_dz, on='time')

    summary = pd.merge(summary, median_sum_dz, on='time')
    summary = pd.merge(summary, q25_sum_dz, on='time')
    summary = pd.merge(summary, q75_sum_dz, on='time')

    #summary['condition'] = key
    summary_dict[key] = summary


In [ ]:

plt.figure(figsize=(10, 6))

# Get colors from Seaborn's viridis palette
colors = sns.color_palette("magma", n_colors=len(plot_order))

for i, cond in enumerate(plot_order):
    df = summary_dict[cond]

    time = df['time']
    median = df['median_num_fibers']
    q25 = df['q25_num_fibers']
    q75 = df['q75_num_fibers']

    color = colors[i]

    plt.plot(time, median, label=labels[i], color=color)
    plt.fill_between(time, q25, q75, alpha=0.2, color=color)

plt.xlabel('Time')
plt.ylabel('Filaments')
plt.title('Filaments vs Time for Tensions')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()

## 3C

In [ ]:

plt.figure(figsize=(10, 6))

# Get colors from Seaborn's viridis palette
colors = sns.color_palette("magma", n_colors=len(plot_order))

for i, cond in enumerate(plot_order):
    df = summary_dict[cond]

    time = df['time']
    median = df['median_bud_internalization'] * 1000
    q25 = df['q25_bud_internalization'] * 1000
    q75 = df['q75_bud_internalization'] * 1000

    color = colors[i]

    plt.plot(time, median, label=labels[i], color=color)
    plt.fill_between(time, q25, q75, alpha=0.2, color=color)

#plt.xscale('log')
#plt.yscale('log')
plt.xlabel('Time')
plt.ylabel('Bud Internalization (nm)')
plt.title('Bud Internalization vs Time Across Tensions')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()

## 3D

### Size

In [ ]:
# Convenience functions for median and IQR

def median_and_iqr_errors(data):
    if isinstance(data, pd.Series):
        data = data.tolist()  # Convert Series to list if needed

    arr = np.array(data)

    # Compute quartiles
    q1 = np.percentile(arr, 25)  # First quartile (Q1)
    q3 = np.percentile(arr, 75)  # Third quartile (Q3)
    median = np.median(arr)  # Median

    # Compute lower and upper error for error bars
    #lower_error = median - q1
    #upper_error = q3 - median

    return median, q1, q3

In [ ]:
def circle_size_WD(dir_df, shared_bins, loops=1, baseline_loops=100):
    all_wass_distances = []

    for i in range(len(shared_bins) - 1):
        bin_start, bin_end = shared_bins[i], shared_bins[i + 1]
        subset = dir_df[(dir_df['length'] >= bin_start) & (dir_df['length'] < bin_end)]
        bin_count = len(subset)  # Count of elements in the bin

        if subset.empty:
            continue  # Skip empty bins

        wass_dists = []
        for _, row in subset.iterrows():
            directions = row['direction']

            for _ in range(loops):
                random_network1 = np.random.uniform(-np.pi, np.pi, 10000)
                random_network_shift = (np.pi + random_network1) / (2 * np.pi)
                wass_dist = ot.wasserstein_circle(directions, random_network_shift,p=1)
                wass_dists.append(wass_dist)

        # Compute median and IQR errors
        median_WD, lower_WD, upper_WD = median_and_iqr_errors(wass_dists)

        # Baseline estimate (Wasserstein distance between two random uniform networks of size = bin center)
        baseline_wass_dists = []
        for _ in range(baseline_loops):
            baseline_network1 = np.random.uniform(-np.pi, np.pi, 10000)
            baseline_network2 = np.random.uniform(-np.pi, np.pi, int((bin_start + bin_end) / 2))

            baseline_1_shift = (np.pi + baseline_network1) / (2 * np.pi)
            baseline_2_shift = (np.pi + baseline_network2) / (2 * np.pi)

            baseline_wass_dist = ot.wasserstein_circle(baseline_1_shift, baseline_2_shift,p=1)
            baseline_wass_dists.append(baseline_wass_dist)

        # Compute the baseline median, lower error (25th percentile), and upper error (75th percentile)
        baseline_median = np.median(baseline_wass_dists)
        baseline_lower_error = np.percentile(baseline_wass_dists, 25)
        baseline_upper_error = np.percentile(baseline_wass_dists, 75)

        # Append results
        all_wass_distances.append({
            'Bin Center': ((bin_start + bin_end) / 2 ),
            'Median': median_WD,
            'Lower Error': lower_WD,
            'Upper Error': upper_WD,
            'Count': bin_count,  # Store count in each bin
            'Baseline Median WD': baseline_median,  # Store baseline median WD
            'Baseline Lower Error': baseline_lower_error,  # Store baseline lower error (25th percentile)
            'Baseline Upper Error': baseline_upper_error  # Store baseline upper error (75th percentile)
        })

    # Convert to DataFrame and return
    result = pd.DataFrame(all_wass_distances)
    result = result.sort_values(by='Bin Center').set_index('Bin Center')

    return result

In [ ]:
tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']

WD_tension_dict = {}
dir_df_dict =  {}
ypos_recalibrated = []
xpos_recalibrated = []
direction = []
lengths = []
index = np.round(np.arange(0, 15.0, 0.1), 1) 

# Recalibrate positions and calculate directions
for trial in tension_runs:
    if trial in hip1r_assoc_positions:
        ypos_recalibrated.append(hip1r_assoc_positions[trial].dropna().ypos_recalibrated)
        xpos_recalibrated.append(hip1r_assoc_positions[trial].dropna().xpos_recalibrated)
        dir = np.arctan2(ypos_recalibrated[-1], xpos_recalibrated[-1])
        dir_circle = (np.pi + dir) / (2 * np.pi)  # Convert to [0, 1] for circular WD
        direction.append(dir_circle)
        
        # Ensure NaN values are considered as length zero
        lengths.append(np.sum(~np.isnan(dir)))  # Count non-NaN values for length
        hip1r_assoc_positions[trial]['direction'] = dir_circle

directions_grouped_dict = {}

for trial in tension_runs:
    # Group the directions
    grouped_directions = hip1r_assoc_positions[trial].groupby(['run', 'time']).direction.apply(np.array)
    
    # Create a DataFrame with 'direction' and 'length' columns
    grouped_lengths = grouped_directions.apply(lambda x: np.sum(~np.isnan(x)))  # Count non-NaN values
    directions_grouped_dict[trial] = pd.DataFrame({
        'direction': grouped_directions,
        'length': grouped_lengths
    })
for trial in tension_runs:
    if trial in directions_grouped_dict:
        dir_df = directions_grouped_dict[trial]
        
        # Calculate histogram bins for each trial's length data individually
        trial_lengths = dir_df['length'].values
        #trial_bins = np.histogram_bin_edges(trial_lengths, bins=30)  # Adjust bin count as needed
        # number of bins you want
        n_bins = 30  

        # compute quantile-based bin edges
        quantile_edges = np.quantile(trial_lengths, np.linspace(0, 1, n_bins + 1))

        # ensure uniqueness (pd.qcut handles ties but np.quantile may repeat values if many are equal)
        quantile_edges = np.unique(quantile_edges)

        # pass into your function
        result = circle_size_WD(dir_df, quantile_edges)
            
        # Compute WD using individual bins for this trial
        #result = circle_size_WD(dir_df, trial_bins)
        
        # Store the result for the trial
        WD_tension_dict[trial] = result
        print(f"Successfully processed trial: {trial}")

In [ ]:
quantile_edges

In [ ]:
result

In [ ]:


# Explicit dict keys in desired order
plot_order = ['_tension10', '', '_tension1000', '_tension5000', '_tension10000']
# Matching clean labels for legend
labels = [10, 150, 1000, 5000, 10000]

# Color map
# cmap = plt.cm.magma
# colors = [cmap(i / (len(plot_order) - 1)) for i in range(len(plot_order))]
colors = sns.color_palette("magma", n_colors=len(plot_order))

fig, ax = plt.subplots(figsize=(8, 6))

# Loop over trials in explicit order
for i, trial in enumerate(plot_order):
    if trial not in WD_tension_dict or trial not in hip1r_assoc_positions:
        continue
    
    c = colors[i]

    median_values = WD_tension_dict[trial]['Median']
    lower_iqr = WD_tension_dict[trial]['Lower Error']
    upper_iqr = WD_tension_dict[trial]['Upper Error']

    yticks = np.linspace(0, 0.2, 5)

    plot_multiple_errorbars_iqr(
        ax,
        median_values,
        lower_iqr,
        upper_iqr,
        c,
        'Wasserstein Distance',
        str(labels[i]),
        yticks=yticks
    )

 #plot_multiple_errorbars_iqr(ax, median_values, lower_iqr, upper_iqr, colors[i], 'Kappa Distance', trial_name, yticks=yticks)

ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
plt.xticks(rotation=0)
plt.title("WD vs Size Tensions", fontsize=22)
plt.xlabel("# Model Points", fontsize=20)
plt.ylabel("Wasserstein Distance", fontsize=20)
plt.legend(fontsize=12, loc='upper right')
plt.grid(True)
plt.show()


### Time

In [ ]:
# Convenience functions for median and IQR

def median_and_iqr_errors(data):
    if isinstance(data, pd.Series):
        data = data.tolist()  # Convert Series to list if needed

    arr = np.array(data)

    # Compute quartiles
    q1 = np.percentile(arr, 25)  # First quartile (Q1)
    q3 = np.percentile(arr, 75)  # Third quartile (Q3)
    median = np.median(arr)  # Median

    # Compute lower and upper error for error bars
    #lower_error = median - q1
    #upper_error = q3 - median

    return median, q1, q3

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
import ot  # POT: Python Optimal Transport

def circle_WD(dir_df, loops=1):
    timepoints = dir_df.index.get_level_values('time').unique()
    runs = dir_df.index.get_level_values('run').unique()
    max_length_per_run = dir_df.groupby('run')['length'].max()
    max_length_array = max_length_per_run.values

    all_circ_wds = []
    all_lin_wds = []
    all_circ_noise = []  # <-- Store circ_noise

    for time in timepoints:
        for run in runs:
            if (run, time) in dir_df.index:
                row = dir_df.loc[(run, time)]
                length = row['length']
                if length == 0:
                    continue  # Skip short tracks

                directions = row['direction']  # Angular data
                circ_dists = []
                lin_dists = []
                circ_noise_dists = []

                for _ in range(loops):
                    # random uniform comparison distributions
                    random_network1 = np.random.uniform(-np.pi, np.pi, 10000)
                    random_network2 = np.random.uniform(-np.pi, np.pi, len(directions))

                    # --- Circular WD ---
                    rn1_shift = (np.pi + random_network1) / (2 * np.pi)
                    rn2_shift = (np.pi + random_network2) / (2 * np.pi)

                    circ_noise = ot.wasserstein_circle(rn1_shift, rn2_shift, p=1)
                    circ_dist = ot.wasserstein_circle(directions, rn1_shift, p=1)

                    circ_dists.append(circ_dist)
                    circ_noise_dists.append(circ_noise)

                    # --- Linear WD ---
                    lin_dist = wasserstein_distance(directions, random_network1)
                    lin_dists.append(lin_dist)


                # Store averages
                all_circ_wds.append({'run': run, 'time': time, 'avg_WD': np.mean(circ_dists)})
                all_lin_wds.append({'run': run, 'time': time, 'avg_WD': np.mean(lin_dists)})
                all_circ_noise.append({'run': run, 'time': time, 'avg_WD': np.mean(circ_noise_dists)})

    # --- Aggregate circular ---
    circ_df = pd.DataFrame(all_circ_wds)
    grouped_circ = circ_df.groupby('time').agg({'avg_WD': lambda x: median_and_iqr_errors(x.tolist())})
    (median_circ, lower_circ, upper_circ) = zip(*grouped_circ['avg_WD'])
    circ_result = pd.DataFrame({
        'Median': median_circ,
        'Lower Error': lower_circ,
        'Upper Error': upper_circ
    }, index=grouped_circ.index)

    # --- Aggregate linear ---
    lin_df = pd.DataFrame(all_lin_wds)
    grouped_lin = lin_df.groupby('time').agg({'avg_WD': lambda x: median_and_iqr_errors(x.tolist())})
    (median_lin, lower_lin, upper_lin) = zip(*grouped_lin['avg_WD'])
    lin_result = pd.DataFrame({
        'Median': median_lin,
        'Lower Error': lower_lin,
        'Upper Error': upper_lin
    }, index=grouped_lin.index)

    # --- Aggregate circ_noise ---
    noise_df = pd.DataFrame(all_circ_noise)
    grouped_noise = noise_df.groupby('time').agg({'avg_WD': lambda x: median_and_iqr_errors(x.tolist())})
    (median_noise, lower_noise, upper_noise) = zip(*grouped_noise['avg_WD'])
    circ_noise_df = pd.DataFrame({
        'Median': median_noise,
        'Lower Error': lower_noise,
        'Upper Error': upper_noise
    }, index=grouped_noise.index)

    # --- Baseline (circular only for now) ---
    last_wd_dfs = []
    for length in max_length_array:
        if length > 0:
            for p_time in range(150):
                rn1 = np.random.uniform(-np.pi, np.pi, length)
                rn2 = np.random.uniform(-np.pi, np.pi, 10000)
                rn1_shift = (np.pi + rn1) / (2 * np.pi)
                rn2_shift = (np.pi + rn2) / (2 * np.pi)
                last_wd = ot.wasserstein_circle(rn1_shift, rn2_shift, p=1)
                last_wd_dfs.append({'time': p_time, 'avg_WD': last_wd})

    last_wd_df = pd.DataFrame(last_wd_dfs)
    grouped_baseline = last_wd_df.groupby('time').agg({'avg_WD': lambda x: median_and_iqr_errors(x.tolist())})
    (median_baseline, lower_baseline, upper_baseline) = zip(*grouped_baseline['avg_WD'])
    last_wd_df = pd.DataFrame({
        'time': grouped_baseline.index,
        'Median': median_baseline,
        'Lower Error': lower_baseline,
        'Upper Error': upper_baseline
    })

    return circ_result, lin_result, circ_noise_df, last_wd_df


In [ ]:
tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']

WD_tension_dict = {}  # Store all WD results per trial
dir_df_dict = {}
ypos_recalibrated = []
xpos_recalibrated = []
direction = []
lengths = []
index = np.round(np.arange(0, 15.0, 0.1), 1) 

# Recalibrate positions and calculate directions
for trial in tension_runs:
    if trial in hip1r_assoc_positions:
        ypos_recalibrated.append(hip1r_assoc_positions[trial].dropna().ypos_recalibrated)
        xpos_recalibrated.append(hip1r_assoc_positions[trial].dropna().xpos_recalibrated)
        dir_vals = np.arctan2(ypos_recalibrated[-1], xpos_recalibrated[-1])
        dir_circle = (np.pi + dir_vals) / (2 * np.pi)  # Convert to [0, 1] for circular WD
        direction.append(dir_circle)
        
        # Ensure NaN values are considered as length zero
        lengths.append(np.sum(~np.isnan(dir_vals)))  # Count non-NaN values for length
        hip1r_assoc_positions[trial]['direction'] = dir_circle

directions_grouped_dict = {}

for trial in tension_runs:
    if trial not in hip1r_assoc_positions:
        continue

    # Group the directions
    grouped_directions = hip1r_assoc_positions[trial].groupby(['run', 'time']).direction.apply(np.array)
    
    # Create a DataFrame with 'direction' and 'length' columns
    grouped_lengths = grouped_directions.apply(lambda x: np.sum(~np.isnan(x)))  # Count non-NaN values
    dir_df = pd.DataFrame({
        'direction': grouped_directions,
        'length': grouped_lengths
    })
    directions_grouped_dict[trial] = dir_df
    dir_df_dict[trial] = dir_df
    
    # Compute circular, linear, and circular noise WDs
    circ_result, lin_result, circ_noise_df, last_wd_df = circle_WD(dir_df)

    # Align indices to your desired time index
    circ_result.index = index[:len(circ_result)]
    lin_result.index = index[:len(lin_result)]
    circ_noise_df.index = index[:len(circ_noise_df)]
    last_wd_df.index = index[:len(last_wd_df)]
 
    # Store all results in a nested dict
    WD_tension_dict[trial] = {
        'circular': circ_result,
        'linear': lin_result,
        'circular_noise': circ_noise_df,
        'baseline': last_wd_df
    }

    print(f"Successfully processed trial: {trial}")


In [ ]:

# --- Explicit dict keys in desired order ---
plot_order = ['_tension10', '', '_tension1000', '_tension5000', '_tension10000']
labels = [10, 150, 1000, 5000, 10000]

# cmap = plt.cm.magma
# colors = [cmap(i / (len(plot_order) - 1)) for i in range(len(plot_order))]
colors = sns.color_palette("magma", n_colors=len(plot_order))


fig, ax = plt.subplots(figsize=(8,6))

# Loop over the trials in explicit order
for i, trial_key in enumerate(plot_order):
    if trial_key in WD_tension_dict and trial_key in hip1r_assoc_positions:
        trial_name = str(labels[i])  # Clean label for legend

        # Get median and IQR
        median_values = WD_tension_dict[trial_key]['circular']['Median']
        lower_iqr = WD_tension_dict[trial_key]['circular']['Lower Error']
        upper_iqr = WD_tension_dict[trial_key]['circular']['Upper Error']
        yticks = np.linspace(0, 0.2, 5)

        # Plot using your helper function
        plot_multiple_errorbars_iqr(
            ax,
            median_values,
            lower_iqr,
            upper_iqr,
            colors[i],
            'Kappa Distance',
            trial_name,
            yticks=yticks
        )

# Finalize plot
plt.title("Wasserstein Distance over Time by Tension")
plt.xlabel("Time")
plt.xlim(0,15)
plt.ylabel("Wasserstein Distance")
plt.legend(title="Tension")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## 3E

In [ ]:
tension = True
if tension is True:
    # List of runs
    tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']

    # Dictionary to store the data for each run
    ends = {}

    # Iterate through each run
    for i, run in enumerate(tension_runs):
        try:
            # Build the file path for the current run's data
            file_path = f'/Users/benjaminbrown/Desktop/cytosim-graph/Data_Ben/endocytosis2_baseline_noforcedepcap{run}_aws_output/dataframes/branched-actin-bound-to-hip1r-ends_recalibrated.pkl'
            
            # Load the data and store it in the dictionary
            ends[run] = pd.read_pickle(file_path)
            
            print(f"Tension Success run {i} ({run})")
            
        except Exception as e:
            # Handle errors (e.g., file not found)
            print(f"Tension Failure run {i} ({run}): {e}")

In [ ]:
grouped_data_byrun = {}

for tension in tension_runs:
    tension_data = ends[tension]
    
    # Ensure 'time' is in the index (reset if necessary)
    if 'time' in tension_data.index.names:
        tension_data = tension_data.reset_index(level='time')
    
    # Group by 'run' and aggregate 'plus_zdir' into lists
    grouped_by_run = tension_data.groupby('run').agg({
        'plus_zdir': lambda x: list(x),  # Group plus_zdir by run
    })
    
    # Store the result in the dictionary for this tension
    grouped_data_byrun[tension] = grouped_by_run


In [ ]:


runmeans = {}
runstds = {}

# Loop through each tension
for tension in tension_runs:
    means_by_run = []
    stds_by_run = []
    
    # Get the data for this tension
    data = grouped_data_byrun[tension]
    
    # Get the unique runs for this tension
    runs = data.index.get_level_values('run').unique()
    
    # Loop through each run
    for run in runs:
        run_data = data.loc[data.index.get_level_values('run') == run]
        
        # Flatten the 'plus_zdir' column (which contains lists of angles)
        plus_zdir_values = np.concatenate(run_data['plus_zdir'].values)
        
        # Drop NaN values from the flattened 'plus_zdir' array
        plus_zdir_values = plus_zdir_values[~np.isnan(plus_zdir_values)]
        
        if len(plus_zdir_values) > 0:
            # Calculate the mean and standard deviation of plus_zdir for this run
            mean_plus_zdir = np.mean(plus_zdir_values)
            std_plus_zdir = np.std(plus_zdir_values)
            
            # Append the results to the respective lists
            means_by_run.append(mean_plus_zdir)
            stds_by_run.append(std_plus_zdir)
    
    # Convert lists to pandas DataFrame for each tension
    runmeans[tension] = pd.DataFrame(means_by_run, columns=['mean_plus_zdir'])
    runstds[tension] = pd.DataFrame(stds_by_run, columns=['std_plus_zdir'])


In [ ]:
grouped_data_by_run_time = {}

for tension in tension_runs:
    tension_data = ends[tension]

    # If 'run' and 'time' are indices, reset them
    for col in ['run', 'time']:
        if col in tension_data.index.names:
            tension_data = tension_data.reset_index(level=col)

    # Group by both 'run' and 'time'
    grouped = tension_data.groupby(['run', 'time']).agg({
        'plus_zdir': lambda x: list(x),
        'minus_zdir': lambda x: list(x),
    })

    # Store in dictionary
    grouped_data_by_run_time[tension] = grouped


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# User settings
# -----------------------------
tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']
label_map = {'': 150, '_tension10': 10, '_tension1000': 1000,
             '_tension5000': 5000, '_tension10000': 10000}
cmap = plt.get_cmap('viridis')
colors = [cmap(i / len(tension_runs)) for i in range(len(tension_runs))]

# -----------------------------
# Compute per-(run,time) ratio
# -----------------------------
all_ratios = []

for condition in tension_runs:
    grouped = grouped_data_by_run_time.get(condition)
    if grouped is None:
        continue

    # Loop over all (run, time) for this tension
    for (run, time), row in grouped.iterrows():
        plus_raw = row['plus_zdir']

        # Ensure it’s a NumPy array
        if isinstance(plus_raw, (list, pd.Series)):
            angles = np.array(plus_raw)
        elif isinstance(plus_raw, np.ndarray):
            angles = plus_raw
        else:  # single number
            angles = np.array([plus_raw])

        # Remove NaNs safely
        angles = angles[~np.isnan(angles)]

        if len(angles) == 0:
            continue

        n_pos = np.sum(angles > 0)
        n_neg = np.sum(angles < 0)
        total = len(angles)

        ratio = (n_pos - n_neg) / total

        all_ratios.append({
            "Condition": condition,
            "Run": run,
            "Time": time,
            "Ratio": ratio
        })

# Convert to DataFrame
ratio_df = pd.DataFrame(all_ratios)
ratio_df['Condition Label'] = ratio_df['Condition'].map(label_map)

# -----------------------------
# Example: Violin plot for final time point per run
# -----------------------------
final_time_df_list = []
for condition in tension_runs:
    df_cond = ratio_df[ratio_df['Condition'] == condition]
    if df_cond.empty:
        continue
    final_times = df_cond.groupby('Run')['Time'].max()
    for run, tmax in final_times.items():
        df_final = df_cond[(df_cond['Run'] == run) & (df_cond['Time'] == tmax)]
        final_time_df_list.append(df_final)

final_time_df = pd.concat(final_time_df_list, ignore_index=True)


### Option 1, order parameter

In [ ]:
plt.figure(figsize=(8,6))

# Violin outline only
sns.violinplot(
    data=final_time_df,
    x="Condition Label",
    y="Ratio",
    order=[10, 150, 1000, 5000, 10000],
    palette='magma',
    inner='quart',    # show quartiles
    fill=False,       # outline only
    saturation=1
)

# Overlay swarm plot
sns.swarmplot(
    data=final_time_df,
    x="Condition Label",
    y="Ratio",
    order=[10, 150, 1000, 5000, 10000],
    palette='magma',
    size=5,
    edgecolor='k',
    linewidth=0.5
)

#plt.axhline(0, color='black', linewidth=1)
plt.xlabel("Tension")
plt.ylabel("Order Parameter (#>0 - #<0 / total)")
plt.title("Filament Plus End Orientation Bias at Final Time Point")
plt.ylim(-1, 1)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Plot: Violin only (all times)
# -----------------------------
plt.figure(figsize=(7,6))

sns.violinplot(
    x='Condition Label',               # numeric labels
    y='Ratio',
    data=ratio_df,
    order=[10, 150, 1000, 5000, 10000],  # enforce desired order
    palette='magma',
    inner='quart',    # show quartiles
    fill=False,       # outline only
    saturation=1
)

#plt.axhline(0, color='black', linewidth=1)
plt.xlabel("Tension")
plt.ylabel("Order Parameter (#>0 - #<0 / total)")
plt.title("Filament Plus End Orientation Bias (All Runs & Times)")
plt.ylim(-1, 1)
plt.tight_layout()
plt.show()


### Option 2, Sum of angles

In [ ]:

# -----------------------------
# Compute sum of plus_zdir angles at final time point
# -----------------------------
angle_sums = []

for condition in tension_runs:
    df_cond = grouped_data_by_run_time.get(condition)
    if df_cond is None or df_cond.empty:
        continue

    # Find final time per run
    final_times = df_cond.groupby('run').apply(lambda g: g.index.get_level_values('time').max())

    for run, tmax in final_times.items():
        row = df_cond.loc[(run, tmax)]
        angles = row['plus_zdir']

        # Ensure array
        if isinstance(angles, (list, pd.Series)):
            angles = np.array(angles)
        elif isinstance(angles, np.ndarray):
            angles = angles
        else:
            angles = np.array([angles])

        # Remove NaNs
        angles = angles[~np.isnan(angles)]
        if len(angles) == 0:
            continue

        angle_sum = np.sum(angles)
        angle_sums.append({
            "Condition": condition,
            "Run": run,
            "Time": tmax,
            "AngleSum": angle_sum
        })

angle_sum_df = pd.DataFrame(angle_sums)
angle_sum_df['ConditionLabel'] = angle_sum_df['Condition'].map(label_map)

# -----------------------------
# Violin + Swarm plot for sum of angles
# -----------------------------
plt.figure(figsize=(8,6))

sns.violinplot(
    data=angle_sum_df,
    x="ConditionLabel",
    y="AngleSum",
    order=[10, 150, 1000, 5000, 10000],
    palette='magma',
    inner='quart',
    fill=False,
    saturation=1
)

sns.swarmplot(
    data=angle_sum_df,
    x="ConditionLabel",
    y="AngleSum",
    order=[10, 150, 1000, 5000, 10000],
    palette='magma',
    size=5,
    edgecolor='k',
    linewidth=0.5
)

plt.xlabel("Tension")
plt.ylabel("Sum of plus_zdir angles")
plt.title("Sum of Plus End Angles at Final Time Point")
plt.tight_layout()
plt.show()


### Option 3 Mean angle

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Compute mean of plus_zdir angles at final time point
# -----------------------------
angle_means = []

for condition in tension_runs:
    df_cond = grouped_data_by_run_time.get(condition)
    if df_cond is None or df_cond.empty:
        continue

    # Find final time per run
    final_times = df_cond.groupby('run').apply(lambda g: g.index.get_level_values('time').max())

    for run, tmax in final_times.items():
        row = df_cond.loc[(run, tmax)]
        angles = row['plus_zdir']

        # Ensure array
        if isinstance(angles, (list, pd.Series)):
            angles = np.array(angles)
        elif isinstance(angles, np.ndarray):
            angles = angles
        else:
            angles = np.array([angles])

        # Remove NaNs
        angles = angles[~np.isnan(angles)]
        if len(angles) == 0:
            continue

        angle_mean = np.mean(angles)
        angle_means.append({
            "Condition": condition,
            "Run": run,
            "Time": tmax,
            "AngleMean": angle_mean
        })

angle_mean_df = pd.DataFrame(angle_means)
angle_mean_df['ConditionLabel'] = angle_mean_df['Condition'].map(label_map)

# -----------------------------
# Violin + Swarm plot for mean angles
# -----------------------------
plt.figure(figsize=(8,6))

sns.violinplot(
    data=angle_mean_df,
    x="ConditionLabel",
    y="AngleMean",
    order=[10, 150, 1000, 5000, 10000],
    palette='magma',
    inner='quart',
    fill=False,
    saturation=1
)

sns.swarmplot(
    data=angle_mean_df,
    x="ConditionLabel",
    y="AngleMean",
    order=[10, 150, 1000, 5000, 10000],
    palette='magma',
    size=5,
    edgecolor='k',
    linewidth=0.5
)

plt.xlabel("Tension")
plt.ylabel("Mean of plus_zdir angles")
plt.title("Mean Plus End Angles at Final Time Point")
plt.tight_layout()
plt.show()


### Option 4 Angle Distributions

In [ ]:
grouped_data = {}

for tension in tension_runs:
    tension_data = ends[tension]
    
    # Ensure 'time' is in the index (reset if necessary)
    if 'time' in tension_data.index.names:
        tension_data = tension_data.reset_index(level='time')
    
    # Group by 'time', aggregate 'plus_zdir' and 'minus_zdir' into lists, and sum 'force_end'
    grouped = tension_data.groupby('time').agg({
        'plus_zdir': lambda x: list(x),
        'minus_zdir': lambda x: list(x),
        'force_end': 'sum'
    })
    
    # Store the result in the new dictionary
    grouped_data[tension] = grouped

labels = [10 ,150 , 1000, 5000, 10000]


In [ ]:


# Colormap
colors = sns.color_palette("magma", n_colors=len(plot_order))

# Number of bins
num_bins = 50

plt.figure(figsize=(10, 6))

ratios = []
diff_ratios = []  # (positive - negative)/total

for i, tension in enumerate(plot_order):
    tension_data = grouped_data.get(tension)
    if tension_data is None:
        ratios.append("N/A")
        diff_ratios.append("N/A")
        continue

    # Flatten plus_zdir values
    plus_zdir_values = [angle for sublist in tension_data['plus_zdir'] for angle in sublist]
    total_plus = len(plus_zdir_values)

    if total_plus == 0:
        ratios.append("N/A")
        diff_ratios.append("N/A")
        continue

    # Counts
    positive_angles = sum(1 for angle in plus_zdir_values if angle > 0)
    negative_angles = sum(1 for angle in plus_zdir_values if angle < 0)

    # Ratios
    ratio = positive_angles / negative_angles if negative_angles != 0 else "N/A"
    diff_ratio = (positive_angles - negative_angles) / total_plus

    ratios.append(ratio)
    diff_ratios.append(diff_ratio)

    # Histogram
    counts, bin_edges = np.histogram(
        plus_zdir_values, bins=num_bins, range=(-1, 1),
        weights=[1 / total_plus] * total_plus
    )
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Plot line histogram
    plt.plot(bin_centers, counts, '-', color=colors[i], label=f'{labels[i]} pN/um')

# Main labels
plt.xlabel('Angle')
plt.ylabel('Fraction of Total Angles')
plt.title('Arp2/3 Plus End Angle Distributions (all tensions)')
plt.legend()

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Create inset
ax_inset = inset_axes(plt.gca(), width="25%", height="25%", loc='lower right')

# Prepare numeric x-values and corresponding diff_ratios
xvals = []
yvals = []
point_colors = []

for lab, val, col in zip(labels, diff_ratios, colors):
    if isinstance(val, str):  # skip "N/A"
        continue
    xvals.append(float(lab))
    yvals.append(val)
    point_colors.append(col)  # use corresponding color

# Sort by tension to connect the line properly
xvals, yvals, point_colors = zip(*sorted(zip(xvals, yvals, point_colors)))

# Plot line connecting points
ax_inset.plot(xvals, yvals, '-', color='black', linewidth=1)

# Plot colored markers for each tension
for x, y, c in zip(xvals, yvals, point_colors):
    ax_inset.plot(x, y, 'o', color=c, markersize=4)

# Set log scale for x-axis
ax_inset.set_xscale("log")

# Move x-axis to top
ax_inset.xaxis.set_label_position('top')
ax_inset.xaxis.tick_top()

# Labels and title
ax_inset.set_xlabel('Tension (pN/um)', fontsize=7)
ax_inset.set_ylabel('% Positive', fontsize=7)
#ax_inset.set_title('Order Parameter', fontsize=9)

# Smaller tick labels
ax_inset.tick_params(axis='both', which='major', labelsize=7)


plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Colormap
colors = sns.color_palette("magma", n_colors=len(plot_order))

# Number of bins
num_bins = 50
bin_edges = np.linspace(-1, 1, num_bins+1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

plt.figure(figsize=(10, 6))

ratios = []
diff_ratios_median = []
diff_ratios_q25 = []
diff_ratios_q75 = []

for i, tension in enumerate(plot_order):
    runs_df = grouped_data_byrun.get(tension)
    if runs_df is None or runs_df.empty:
        ratios.append("N/A")
        diff_ratios_median.append("N/A")
        diff_ratios_q25.append("N/A")
        diff_ratios_q75.append("N/A")
        continue

    run_histograms = []
    run_diff_ratios = []
    run_ratios = []

    for run_id, row in runs_df.iterrows():
        plus_zdir_values = row['plus_zdir']
        total_plus = len(plus_zdir_values)
        if total_plus == 0:
            continue

        # Counts
        positive_angles = sum(1 for angle in plus_zdir_values if angle > 0)
        negative_angles = sum(1 for angle in plus_zdir_values if angle < 0)

        # Ratios per run
        ratio = positive_angles / negative_angles if negative_angles != 0 else np.nan
        diff_ratio = (positive_angles) / total_plus

        run_ratios.append(ratio)
        run_diff_ratios.append(diff_ratio)

        # Histogram normalized by run total
        counts, _ = np.histogram(
            plus_zdir_values, bins=bin_edges,
            weights=[1 / total_plus] * total_plus
        )
        run_histograms.append(counts)

    if len(run_histograms) == 0:
        ratios.append("N/A")
        diff_ratios_median.append("N/A")
        diff_ratios_q25.append("N/A")
        diff_ratios_q75.append("N/A")
        continue

    # Convert to array for quantiles
    run_histograms = np.array(run_histograms)

    # Take median + IQR for histograms
    median_hist = np.median(run_histograms, axis=0)
    q25 = np.percentile(run_histograms, 25, axis=0)
    q75 = np.percentile(run_histograms, 75, axis=0)

    # Ratios
    median_ratio = np.nanmedian(run_ratios)
    ratios.append(median_ratio)

    # Order parameter distribution
    diff_ratios_median.append(np.median(run_diff_ratios))
    diff_ratios_q25.append(np.percentile(run_diff_ratios, 25))
    diff_ratios_q75.append(np.percentile(run_diff_ratios, 75))

    # Plot median histogram with IQR shading
    plt.plot(bin_centers, median_hist, '-', color=colors[i], label=f'{labels[i]} pN/um')
    plt.fill_between(bin_centers, q25, q75, color=colors[i], alpha=0.25)

# Main labels
plt.xlabel('Angle')
plt.ylabel('Fraction')
plt.title('Filament Plus End Angle Distributions')
plt.legend()

# Inset plot with error bars
ax_inset = inset_axes(plt.gca(), width="25%", height="25%", loc='lower right')

xvals = []
yvals = []
yerr_low = []
yerr_high = []
point_colors = []

for lab, med, q25, q75, col in zip(labels, diff_ratios_median, diff_ratios_q25, diff_ratios_q75, colors):
    if isinstance(med, str):  # skip "N/A"
        continue
    xvals.append(float(lab))
    yvals.append(med)
    yerr_low.append(med - q25)
    yerr_high.append(q75 - med)
    point_colors.append(col)

if xvals:
    # Sort everything together by tension
    xvals, yvals, yerr_low, yerr_high, point_colors = zip(
        *sorted(zip(xvals, yvals, yerr_low, yerr_high, point_colors))
    )

    # Plot with error bars (convert y-values to percentages)
    ax_inset.errorbar(
        xvals, [100*y for y in yvals],
        yerr=[[100*yl for yl in yerr_low], [100*yh for yh in yerr_high]],
        fmt='-', color='black', ecolor='gray',
        elinewidth=1, capsize=3, markersize=4,
        zorder=1  # send line+bars to back
    )

    # Overlay colored points on top
    for x, y, c in zip(xvals, yvals, point_colors):
        ax_inset.plot(x, 100*y, 'o', color=c, markersize=5, zorder=3)


    # Set log scale for x-axis
    ax_inset.set_xscale("log")

    # Move x-axis to top
    ax_inset.xaxis.set_label_position('top')
    ax_inset.xaxis.tick_top()

    # Labels
    ax_inset.set_xlabel('Tension (pN/um)', fontsize=7)
    ax_inset.set_ylabel('% Positive', fontsize=7)
    #ax_inset.set_title('Order Parameter', fontsize=8)

    ax_inset.tick_params(axis='both', which='major', labelsize=7)


plt.tight_layout()
plt.savefig("filament_angle_distributions.pdf", format='pdf')
plt.show()


## 3F

### Forces

In [ ]:
tension=True

if tension is True:
    # List of runs
    tension_runs = ['', '_tension10', '_tension1000', '_tension5000', '_tension10000']

    # Dictionary to store the data for each run
    forces = {}

    # Iterate through each run
    for i, run in enumerate(tension_runs):
        try:
            # Build the file path for the current run's data
            file_path = f'/Users/benjaminbrown/Desktop/cytosim-graph/Data_Ben/endocytosis2_baseline_noforcedepcap{run}_aws_output/dataframes/confinement_forces.pkl'
            
            # Load the data and store it in the dictionary
            forces[run] = pd.read_pickle(file_path)
            
            print(f"Tension Success run {i} ({run})")
            
        except Exception as e:
            # Handle errors (e.g., file not found)
            print(f"Tension Failure run {i} ({run}): {e}")

k = labels

#### Confinement by z displacement x k

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  # for magma colormap

runs = plot_order
time_points = np.linspace(0.1, 15, 150).round(1)  # uniform time points
cmap = sns.color_palette("magma", len(runs))       # magma colors

median_force_per_fil = {}
q25_force_per_fil = {}
q75_force_per_fil = {}

for tension in runs:
    df = hip1r_assoc_positions[tension]
    
    # Select confined filaments
    df_force = df[df['zpos'] < -0.399].copy()
    
    # Compute confinement force based on displacement
    df_force['confinement_force'] = (-0.399 - df_force['zpos']) * 1e4
    
    # Sum confinement_force per run/time
    force_sum = df_force.groupby(['run', 'time'])['confinement_force'].sum()
    
    # Number of unique fiber_ids per run/time
    fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
        
    # Average confinement force per filament
    avg_force = force_sum / fiber_count.replace(0, np.nan)
    
    # Reindex each run to uniform time points and build DataFrame
    avg_force_runs = []
    for run in avg_force.index.get_level_values('run').unique():
        run_series = avg_force.xs(run, level='run').reindex(time_points).fillna(method='ffill')
        avg_force_runs.append(run_series)
    
    avg_force_df = pd.concat(avg_force_runs, axis=1)
    
    # Median and IQR across runs
    median_force_per_fil[tension] = avg_force_df.median(axis=1)
    q25_force_per_fil[tension] = avg_force_df.quantile(0.25, axis=1)
    q75_force_per_fil[tension] = avg_force_df.quantile(0.75, axis=1)

# -----------------------------
# Plotting with magma colors
# -----------------------------
plt.figure(figsize=(8, 6))
for i, tension in enumerate(runs):
    plt.plot(time_points, median_force_per_fil[tension], color=cmap[i], label=tension)
    plt.fill_between(
        time_points,
        q25_force_per_fil[tension],
        q75_force_per_fil[tension],
        color=cmap[i],
        alpha=0.3
    )

plt.xlabel('Time')
plt.ylabel('pN/filament')
plt.title('Confinement Force per Confined Filament Over Time')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

runs = plot_order
cmap = sns.color_palette("magma", len(runs))
n_bins = 10  # number of quantile bins per condition

plot_data = []

for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()

    # Select only confined filaments
    df_confined = df[df['zpos'] < -0.399].copy()

    # Compute confinement force based on displacement (μm → nm factor 1e4)
    df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

    # Sum of confinement force and count of confined fibers per run/time
    force_sum = df_confined.groupby(['run', 'time'])['confinement_force'].sum()
    fiber_count = df_confined.groupby(['run', 'time'])['fiber_id'].nunique()

    # Merge safely on index
    df_plot = pd.DataFrame({'TotalForce': force_sum}).merge(
        pd.DataFrame({'NumFilaments': fiber_count}),
        left_index=True, right_index=True, how='inner'
    ).reset_index()

    if df_plot.empty:
        print(f"⚠️ Skipping {tension}: no confined data.")
        continue

    # --- Quantile binning per condition (equal number of points per bin) ---
    if len(df_plot) < n_bins:
        print('true')
        # fewer points than bins: fallback to single bin
        df_plot['FilBin'] = pd.Categorical(["all"]*len(df_plot))
    else:
        df_plot['FilBin'] = pd.qcut(df_plot['NumFilaments'], q=n_bins, duplicates='drop')

    # Compute median and IQR of total confinement force per bin
    binned = df_plot.groupby('FilBin', observed=True)['TotalForce'].agg(
        median='median',
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).dropna().reset_index()

    # Use bin midpoints for plotting
    if binned['FilBin'].dtype.name != 'category':
        # single-bin fallback
        binned['BinMid'] = df_plot['NumFilaments'].median()
    else:
        binned['BinMid'] = binned['FilBin'].apply(lambda x: x.mid)

    binned['Tension'] = labels[i]
    plot_data.append(binned)

# Combine all conditions
summary = pd.concat(plot_data, axis=0)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 6))

for i, tension in enumerate(labels):
    df_t = summary[summary['Tension'] == tension]
    if df_t.empty:
        continue
    plt.plot(df_t['BinMid'], df_t['median'], color=cmap[i], lw=2, label=labels[i])
    plt.fill_between(df_t['BinMid'], df_t['q25'], df_t['q75'], color=cmap[i], alpha=0.3)
    plt.scatter(df_t['BinMid'], df_t['median'], color=cmap[i], s=40)

plt.xlabel('Number of Confined Filaments (per-condition quantile bins)')
plt.ylabel('Total Confinement Force (pN)')
plt.title(f'Total Confinement Force vs Number of Confined Filaments\n(10 equal-size bins per condition)')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Settings
# -----------------------------
runs = plot_order  # list of tensions
n_bins = 10        # target number of bins per tension
cmap = sns.color_palette("magma", len(runs))  # colors

plot_data = []

# -----------------------------
# Loop over each tension
# -----------------------------
for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()

    # Select only confined filaments
    df_confined = df[df['zpos'] < -0.399].copy()

    # Compute confinement force (μm → nm factor 1e4)
    df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

    # Sum of confinement force and count of confined fibers per run/time
    force_sum = df_confined.groupby(['run','time'])['confinement_force'].sum()
    fiber_count = df_confined.groupby(['run','time'])['fiber_id'].nunique()

    # Merge into single DataFrame
    df_plot = pd.DataFrame({'TotalForce': force_sum}).merge(
        pd.DataFrame({'NumFilaments': fiber_count}),
        left_index=True, right_index=True, how='inner'
    ).reset_index()

    if df_plot.empty:
        print(f"⚠️ Skipping {tension}: no confined data.")
        continue

    # -----------------------------
    # Quantile binning per tension
    # -----------------------------
    max_bins = min(n_bins, df_plot['NumFilaments'].nunique())
    if max_bins < 1:
        print(f"⚠️ Skipping {tension}: not enough unique points.")
        continue

    df_plot['FilBin'] = pd.qcut(df_plot['NumFilaments'], q=max_bins, duplicates='drop')

    # -----------------------------
    # Compute median and IQR per bin
    # -----------------------------
    binned = df_plot.groupby('FilBin')['TotalForce'].agg(
        median='median',
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()

    # Use median number of filaments in each bin as x-axis
    binned['BinMid'] = df_plot.groupby('FilBin')['NumFilaments'].median().values
    binned['Tension'] = labels[i]

    plot_data.append(binned)

# -----------------------------
# Combine all tensions
# -----------------------------
summary = pd.concat(plot_data, axis=0)

# -----------------------------
# Plot all tensions on one figure
# -----------------------------
plt.figure(figsize=(9,6))

for i, tension in enumerate(labels):
    df_t = summary[summary['Tension'] == tension]
    if df_t.empty:
        continue
    plt.plot(df_t['BinMid'], df_t['median'], color=cmap[i], lw=2, label=tension)
    plt.fill_between(df_t['BinMid'], df_t['q25'], df_t['q75'], color=cmap[i], alpha=0.3)
    plt.scatter(df_t['BinMid'], df_t['median'], color=cmap[i], s=40)

plt.xlabel('Number of Confined Filaments (per-condition quantile bins)')
plt.ylabel('Total Confinement Force (pN)')
plt.title(f'Total Confinement Force vs Number of Confined Filaments\n(10 quantile bins per tension)')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Settings
# -----------------------------
runs = plot_order  # list of tensions
cmap = sns.color_palette("magma", len(runs))  # colors

n_cols = 5
n_rows = 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5), sharey=True, sharex=False)

# Ensure axes is always iterable
if n_cols == 1:
    axes = [axes]

# -----------------------------
# Loop over each tension
# -----------------------------
for i, tension in enumerate(runs):
    ax = axes[i]
    df = hip1r_assoc_positions[tension].copy()

    # Select only confined filaments
    df_confined = df[df['zpos'] < -0.399].copy()

    # Compute confinement force (μm → nm factor 1e4)
    df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

    # Sum of confinement force and count of confined fibers per run/time
    force_sum = df_confined.groupby(['run','time'])['confinement_force'].sum()
    fiber_count = df_confined.groupby(['run','time'])['fiber_id'].nunique()

    # Merge into single DataFrame for scatter plot
    df_plot = pd.DataFrame({'TotalForce': force_sum, 'NumFilaments': fiber_count}).reset_index()

    if df_plot.empty:
        print(f"⚠️ Skipping {tension}: no confined data.")
        continue

    # -----------------------------
    # Scatter raw data
    # -----------------------------
    ax.scatter(df_plot['NumFilaments'], df_plot['TotalForce'], color=cmap[i], s=20, alpha=0.6, edgecolor='k')
    ax.set_title(f"Tension: {labels[i]}")
    ax.set_xlabel('Number of Confined Filaments')
    if i == 0:
        ax.set_ylabel('Total Confinement Force (pN)')

# -----------------------------
# Layout adjustments
# -----------------------------
plt.suptitle("Raw Total Confinement Force vs Number of Confined Filaments (per tension)", fontsize=16)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

runs = plot_order
cmap = sns.color_palette("magma", len(runs))
N = 10  # number of integers per bin

plt.figure(figsize=(8,6))

for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()
    df_confined = df[df['zpos'] < -0.399].copy()
    df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

    force_sum = df_confined.groupby(['run','time'])['confinement_force'].sum()
    fiber_count = df_confined.groupby(['run','time'])['fiber_id'].nunique()
    df_plot = pd.DataFrame({'TotalForce': force_sum, 'NumFilaments': fiber_count}).reset_index()
    if df_plot.empty:
        print(f"⚠️ Skipping {tension}: no confined data.")
        continue

    # -----------------------------
    # Integer binning (N=1)
    # -----------------------------
    if N == 1:
        df_plot['FilBin'] = df_plot['NumFilaments']  # each integer is its own bin
        binned = df_plot.groupby('FilBin')['TotalForce'].agg(
            median='median',
            q25=lambda x: np.percentile(x,25),
            q75=lambda x: np.percentile(x,75)
        ).reset_index()
        binned['BinMid'] = binned['FilBin']  # mid = integer itself
    else:
        min_f, max_f = df_plot['NumFilaments'].min(), df_plot['NumFilaments'].max()
        bins = np.arange(min_f-0.5, max_f+N, N)
        df_plot['FilBin'] = pd.cut(df_plot['NumFilaments'], bins, include_lowest=True)
        binned = df_plot.groupby('FilBin')['TotalForce'].agg(
            median='median',
            q25=lambda x: np.percentile(x,25),
            q75=lambda x: np.percentile(x,75)
        ).reset_index()
        binned['BinMid'] = binned['FilBin'].apply(lambda x: x.mid)

    # -----------------------------
    # Plot median + IQR
    # -----------------------------
    plt.plot(binned['BinMid'], binned['median'], color=cmap[i], lw=2, label=f"{labels[i]} ")
    plt.fill_between(binned['BinMid'], binned['q25'], binned['q75'], color=cmap[i], alpha=0.2)

    # -----------------------------
    # Scatter median values
    # -----------------------------
    plt.scatter(binned['BinMid'], binned['median'], color=cmap[i], edgecolor='k', s=50, zorder=5)
#plt.xscale('log')
#plt.yscale('log')
plt.xlabel('Number of Confined Filaments')
plt.ylabel('Total Confinement Force (pN)')
plt.title('Total Confinement Force vs Number of Confined Filaments')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Compute number of filaments experiencing force
# -----------------------------
filament_count_per_time = {}

for tension in tension_runs:
    df = hip1r_assoc_positions[tension]
    df_force = df[df['zpos'] < -0.399]
    
    # Count unique fibers per run/time
    fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
    
    # Reindex each run to uniform time points and build DataFrame
    fiber_count_runs = []
    for run in fiber_count.index.get_level_values('run').unique():
        run_series = fiber_count.xs(run, level='run').reindex(time_points).fillna(method='ffill')
        fiber_count_runs.append(run_series)
    
    fiber_count_df = pd.concat(fiber_count_runs, axis=1)
    
    # Median and IQR across runs
    filament_count_per_time[tension] = {
        'median': fiber_count_df.median(axis=1),
        'q25': fiber_count_df.quantile(0.25, axis=1),
        'q75': fiber_count_df.quantile(0.75, axis=1)
    }

# -----------------------------
# Plotting number of filaments over time
# -----------------------------
plt.figure(figsize=(8, 5))
for i, tension in enumerate(runs):
    median = filament_count_per_time[tension]['median']
    q25 = filament_count_per_time[tension]['q25']
    q75 = filament_count_per_time[tension]['q75']
    
    plt.plot(time_points, median, color=cmap[i], label=labels[i])
    plt.fill_between(time_points, q25, q75, color=cmap[i], alpha=0.3)

plt.xlabel('Time')
plt.ylabel('# Confined Filaments')
plt.title('Confined Filament Count Over Time')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Compute percent of filaments experiencing force
# -----------------------------
percent_filaments_per_time = {}

for tension in tension_runs:
    df = hip1r_assoc_positions[tension]
    
    # Total fibers per run/time
    total_fibers = df.groupby(['run', 'time'])['fiber_id'].nunique()
    
    # Fibers experiencing force
    df_force = df[df['zpos'] < -0.399]
    fibers_with_force = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
    
    # Percent per run/time
    percent = (fibers_with_force / total_fibers * 100).fillna(0)
    
    # Reindex each run to uniform time points
    percent_runs = []
    for run in percent.index.get_level_values('run').unique():
        run_series = percent.xs(run, level='run').reindex(time_points).fillna(method='ffill')
        percent_runs.append(run_series)
    
    percent_df = pd.concat(percent_runs, axis=1)
    
    # Median and IQR across runs
    percent_filaments_per_time[tension] = {
        'median': percent_df.median(axis=1),
        'q25': percent_df.quantile(0.25, axis=1),
        'q75': percent_df.quantile(0.75, axis=1)
    }

# -----------------------------
# Plotting percent of filaments over time
# -----------------------------
plt.figure(figsize=(8, 6))
for i, tension in enumerate(runs):
    median = percent_filaments_per_time[tension]['median']
    q25 = percent_filaments_per_time[tension]['q25']
    q75 = percent_filaments_per_time[tension]['q75']
    
    plt.plot(time_points, median, color=cmap[i], label=labels[i])
    plt.fill_between(time_points, q25, q75, color=cmap[i], alpha=0.3)

plt.xlabel('Time')
plt.ylabel('Percent of Filaments Experiencing Force (%)')
plt.title('Percent of Filaments Experiencing Force Over Time')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:


plt.figure(figsize=(8, 6))
cmap = sns.color_palette("magma", n_colors=len(runs))
n_bins = 20  # number of quantile bins
runs = plot_order

for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()

    # --- Select confined filaments ---
    df_confined = df[df['zpos'] < -0.399].copy()

    # --- Compute confinement force based on displacement ---
    df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4  # arbitrary units

    # --- Sum confinement force and count fibers per run/time ---
    force_sum = df_confined.groupby(['run', 'time'])['confinement_force'].sum()
    fiber_count = df_confined.groupby(['run', 'time'])['fiber_id'].nunique()

    confinement_df = pd.DataFrame({
        'run': force_sum.index.get_level_values('run'),
        'time': force_sum.index.get_level_values('time'),
        'confinement_force': force_sum.values
    })

    # --- Bud internalization data ---
    df_bud = df[['run', 'time', 'bud_internalization']].copy()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # --- Compute per-run normalized tension force ---
    all_points = []
    k_value = float(k[i])  # your spring constant
    for run_id in confinement_df['run'].unique():
        df_run_conf = confinement_df[confinement_df['run'] == run_id]
        df_run_bud = df_bud[df_bud['run'] == run_id].copy()

        # Normalize internalization so min = 0
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()

        # Compute effective tension force = k × internalization
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value

        # Merge confinement and tension forces by time
        df_merged = pd.merge(
            df_run_conf[['time', 'confinement_force']],
            df_run_bud_unique[['time', 'tension_force']],
            on='time'
        )

        # Sort by tension force and compute cumulative confinement
        df_merged = df_merged.sort_values('tension_force')
        df_merged['cum_confinement'] = df_merged['confinement_force'].cumsum()

        all_points.append(df_merged)

    # Combine all runs for this tension condition
    df_all = pd.concat(all_points, ignore_index=True)

    # --- Quantile-based bins of tension force ---
    df_all['bin'] = pd.qcut(df_all['tension_force'], q=n_bins, duplicates='drop')

    # --- Compute median and IQR of cumulative confinement per bin ---
    binned_stats = df_all.groupby('bin')['cum_confinement'].agg(
        median='median',
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()
    binned_stats['bin_center'] = binned_stats['bin'].apply(lambda x: x.mid)

    # --- Plot median + IQR ---
    plt.plot(
        binned_stats['bin_center'],
        binned_stats['median'],
        color=cmap[i],
        lw=2,
        label=labels[i]
    )
    plt.fill_between(
        binned_stats['bin_center'],
        binned_stats['q25'],
        binned_stats['q75'],
        color=cmap[i],
        alpha=0.3
    )
    plt.scatter(
        binned_stats['bin_center'],
        binned_stats['median'],
        color=cmap[i],
        edgecolor='k',
        s=60
    )

# -----------------------------
# Final plot settings
# -----------------------------
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Tension Force")
plt.ylabel("Sum confinement force (pN)")
plt.title("Cumulative Confinement vs Tension Force")
plt.legend(title="Tension")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

violin_data = []

for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()

    # --- Select filaments below the surface ---
    df_force = df[df['zpos'] < -0.399].copy()

    df_force['displacement'] = df_force['zpos'] + 0.399
    df_force['pred_force'] = -1e4 * df_force['displacement']  

    # --- Sum predicted confinement force per run/time ---
    force_sum = df_force.groupby(['run', 'time'])['pred_force'].sum()
    fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()

    # --- Average confinement force per filament ---
    avg_force = (force_sum / fiber_count.replace(0, np.nan)).reset_index(name='AvgForce')
    avg_force['Tension'] = labels[i]

    # --- Average over time for each run ---
    avg_force_per_run = avg_force.groupby('run')['AvgForce'].mean().reset_index()
    avg_force_per_run['Tension'] = labels[i]
    violin_data.append(avg_force_per_run[['Tension', 'AvgForce']])

# Combine all tensions
violin_df = pd.concat(violin_data, axis=0)

# -----------------------------
# Plot violin outlines + quartiles
plt.figure(figsize=(8, 6))

sns.violinplot(
    data=violin_df,
    x='Tension',
    y='AvgForce',
    palette=cmap,
    inner='quartile',  # median + IQR
    linewidth=1.5,
    cut=0,
    fill=False,
    order=labels
)

# Scatter individual run points
sns.swarmplot(
    data=violin_df,
    x='Tension',
    y='AvgForce',
    palette=cmap,
    size=6,
    edgecolor='k',
    alpha=0.9,
    order=labels
)

plt.xlabel('Tension')
plt.ylabel('Predicted Confinement Force per Filament (pN)')
plt.title('Distribution of Predicted Confinement Force per Filament Across Runs')
plt.tight_layout()
plt.show()


#### Other

In [ ]:


cmap = sns.color_palette("magma", n_colors=len(plot_order))
plt.figure(figsize=(10, 6))

for i, cond in enumerate(plot_order):
    # Confinement force
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')

    # Bud internalization
    df_bud = hip1r_assoc_positions[cond].reset_index()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # Spring constant
    k_value = float(k[i])
    
    # Loop over runs
    for run in df_confinement['run'].unique():
        df_run_confinement = df_confinement[df_confinement['run'] == run]
        df_run_bud = df_bud[df_bud['run'] == run].copy()

        # Normalize internalization per run (start at 0)
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()

        # Ensure one internalization per time
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()

        # Compute tension force
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value 

        # Merge with confinement force
        df_merged = pd.merge(
            df_run_confinement[['time', 'zforce']],
            df_run_bud_unique[['time', 'tension_force']],
            on='time'
        )

        # Compute excess force
        df_merged['force_diff'] = df_merged['zforce'] - df_merged['tension_force']

        # Plot each run as a separate line
        plt.plot(df_merged['time'], df_merged['force_diff'], color=cmap[i], alpha=0.5)

plt.xlabel('Time')
plt.ylabel('Excess Force (zforce - tension_force)')
plt.title('Excess Force per Run over Time')
plt.legend(plot_order, title='Tension')
plt.tight_layout()
plt.show()


In [ ]:


cmap = sns.color_palette("magma", n_colors=len(plot_order))
plt.figure(figsize=(10, 6))

for i, cond in enumerate(plot_order):
    # Confinement force
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')

    # Bud internalization
    df_bud = hip1r_assoc_positions[cond].reset_index()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # Spring constant
    k_value = float(k[i])

    # Shared time points
    time_points = np.sort(df_confinement['time'].unique())

    # Collect force_diff traces for all runs
    run_force_diff_traces = []

    for run in df_confinement['run'].unique():
        df_run_confinement = df_confinement[df_confinement['run'] == run]
        df_run_bud = df_bud[df_bud['run'] == run].copy()

        # Normalize internalization per run
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()

        # Ensure one internalization per time
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()

        # Compute tension force
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value

        # Merge with confinement force
        df_merged = pd.merge(
            df_run_confinement[['time', 'zforce']],
            df_run_bud_unique[['time', 'tension_force']],
            on='time'
        )

        # Compute excess force
        df_merged['force_diff'] = df_merged['zforce'] - df_merged['tension_force']

        # Interpolate onto the shared time grid
        interp = np.interp(time_points, df_merged['time'], df_merged['force_diff'])
        run_force_diff_traces.append(interp)

    # Convert to 2D array: rows=runs, columns=times
    run_force_diff_traces = np.vstack(run_force_diff_traces)

    # Compute median and IQR across runs
    median = np.median(run_force_diff_traces, axis=0)
    q25 = np.percentile(run_force_diff_traces, 25, axis=0)
    q75 = np.percentile(run_force_diff_traces, 75, axis=0)

    # Plot median and IQR
    plt.plot(time_points, median, label=cond, color=cmap[i])
    plt.fill_between(time_points, q25, q75, color=cmap[i], alpha=0.3)

plt.xlabel('Time')
plt.ylabel('Excess Force (zforce - tension_force)')
plt.title('Median Excess Force ± IQR Across Runs')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
cmap = sns.color_palette("magma", n_colors=len(plot_order))
n_bins = 50  # Number of equal-count bins

for i, cond in enumerate(plot_order):
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')
    df_confinement['n_filaments'] = pd.to_numeric(df_confinement['n_filaments'], errors='coerce')

    # Drop NaNs
    df_confinement = df_confinement.dropna(subset=['n_filaments', 'zforce'])

    # Equal-count bins
    df_confinement['bin'] = pd.qcut(df_confinement['n_filaments'], q=n_bins, duplicates='drop')

    # Compute median and IQR
    median = df_confinement.groupby('bin')['zforce'].median()
    q25 = df_confinement.groupby('bin')['zforce'].quantile(0.25)
    q75 = df_confinement.groupby('bin')['zforce'].quantile(0.75)

    # X-axis: mean n_filaments per bin
    bin_centers = df_confinement.groupby('bin')['n_filaments'].mean()

    # Plot line connecting median points
    plt.plot(bin_centers, median, color=cmap[i], linewidth=2)

    # Scatter median points
    plt.scatter(bin_centers, median, label=labels[i], color=cmap[i], s=60)

    # Shaded IQR
    plt.fill_between(bin_centers, q25, q75, color=cmap[i], alpha=0.3)

plt.xlabel('Number of Confined Filaments')
plt.ylabel('Confinement Force')
plt.title('Confinement Force vs Number of # Confined Filaments')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
cmap = sns.color_palette("magma", n_colors=len(plot_order))
n_bins = 100  # Number of equal-count bins

for i, cond in enumerate(plot_order):
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')
    df_confinement['n_filaments'] = pd.to_numeric(df_confinement['n_filaments'], errors='coerce')

    # Drop NaNs
    df_confinement = df_confinement.dropna(subset=['n_filaments', 'zforce'])

    # Equal-count bins
    df_confinement['bin'] = pd.qcut(df_confinement['n_filaments'], q=n_bins, duplicates='drop')

    # Compute median and IQR
    median = df_confinement.groupby('bin')['zforce'].median()
    q25 = df_confinement.groupby('bin')['zforce'].quantile(0.25)
    q75 = df_confinement.groupby('bin')['zforce'].quantile(0.75)

    # X-axis: mean n_filaments per bin
    bin_centers = df_confinement.groupby('bin')['n_filaments'].mean()

    # Plot line connecting median points
    plt.plot(bin_centers, median, color=cmap[i], linewidth=2)

    # Scatter median points
    plt.scatter(bin_centers, median, label=labels[i], color=cmap[i], s=60)

    # Shaded IQR
    plt.fill_between(bin_centers, q25, q75, color=cmap[i], alpha=0.3)

plt.xlabel('Number of Confined Filaments')
plt.ylabel('Confinement Force')
plt.title('Confinement Force vs Number of # Confined Filaments')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

plt.figure(figsize=(10, 6))
cmap = sns.color_palette("magma", n_colors=len(plot_order))

for i, cond in enumerate(plot_order):
    df_confinement = forces[cond].reset_index()
    
    # Ensure numeric types
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')
    df_confinement['n_filaments'] = pd.to_numeric(df_confinement['n_filaments'], errors='coerce')
    df_confinement['time'] = pd.to_numeric(df_confinement['time'], errors='coerce')
    
    # Drop NaNs
    df_confinement = df_confinement.dropna(subset=['n_filaments', 'zforce', 'time'])
    
    # Compute force per filament
    df_confinement['force_per_filament'] = df_confinement['zforce'] / df_confinement['n_filaments']
    
    # Group by time: median and IQR
    median = df_confinement.groupby('time')['force_per_filament'].median()
    q25 = df_confinement.groupby('time')['force_per_filament'].quantile(0.25)
    q75 = df_confinement.groupby('time')['force_per_filament'].quantile(0.75)
    
    # Plot median line
    plt.plot(median.index, median.values, color=cmap[i], linewidth=2)
    
    # Scatter median points
    plt.scatter(median.index, median.values, color=cmap[i], s=50, label=labels[i])
    
    # Shaded IQR
    plt.fill_between(median.index, q25, q75, color=cmap[i], alpha=0.3)

plt.xlabel('Time')
plt.ylabel('Confinement Force per Filament')
plt.title('Confinement Force per Filament vs Time (All Conditions)')
plt.legend(title='Condition')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- Prepare data for boxplot ---
boxplot_data = []

for cond, label in zip(plot_order, labels):
    df = forces[cond].reset_index()
    
    # Ensure numeric types
    df['zforce'] = -pd.to_numeric(df['zforce'], errors='coerce')
    df['n_filaments'] = pd.to_numeric(df['n_filaments'], errors='coerce')
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    
    # Drop NaNs
    df = df.dropna(subset=['zforce', 'n_filaments', 'time'])
    
    # Compute force per filament
    df['force_per_filament'] = df['zforce'] 
    
    # Take only the last time point per run
    last_time = df['time'].max()
    df_last = df[df['time'] == last_time]
    
    # Store with condition label
    df_last = df_last[['run', 'force_per_filament']].copy()
    df_last['condition'] = label
    boxplot_data.append(df_last)

boxplot_df = pd.concat(boxplot_data, ignore_index=True)

# --- Plot ---
plt.figure(figsize=(8, 6))
sns.boxplot(
    x='condition',
    y='force_per_filament',
    data=boxplot_df,
    palette=sns.color_palette("magma", n_colors=len(plot_order))
)

# Optional: overlay individual points
sns.stripplot(
    x='condition',
    y='force_per_filament',
    data=boxplot_df,
    color='k',
    size=5,
    jitter=True,
    alpha=0.6
)

plt.xlabel('Condition', fontsize=12)
plt.ylabel('Confinement Force per Filament (last time point)', fontsize=12)
plt.title('Distribution of Force per Filament at Last Time Point', fontsize=14)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# --- Prepare data ---
all_data = []

for i, cond in enumerate(plot_order):
    label_name = labels[i]  # custom label

    df_bud = hip1r_assoc_positions[cond].reset_index()

    for run in df_bud['run'].unique():
        df_run = df_bud[df_bud['run'] == run]
        if df_run.empty:
            continue

        # Sum force_z per time
        df_sum = df_run.groupby('time', as_index=False)['force_z'].sum()
        if df_sum.empty:
            continue

        # Take the last time point
        last_time = df_sum['time'].max()
        force_last = df_sum[df_sum['time'] == last_time]['force_z'].values[0]

        all_data.append({
            'Condition': label_name,
            'Force (last time point)': force_last
        })

# Convert to DataFrame
df_plot = pd.DataFrame(all_data)

In [ ]:
# Plot violin + swarm
plt.figure(figsize=(8,6))
sns.violinplot(
    x='Condition',
    y='Force (last time point)',
    data=df_plot,
    fill=False,
    palette='magma',
    saturation=1,
    inner='quart'
)
sns.swarmplot(
    x='Condition',
    y='Force (last time point)',
    data=df_plot,
    palette='magma',
    size=6,
    edgecolor='k',
    linewidth=0.5
)

plt.xlabel("Tension")
plt.ylabel("Final Bud Force (pN)")
plt.title("Distribution of Last Time Point Bud Force")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  # for magma colormap

runs = plot_order
time_points = np.linspace(0.1, 15, 150).round(1)  # uniform time points
cmap = sns.color_palette("magma", len(runs))       # magma colors

median_force_per_fil = {}
q25_force_per_fil = {}
q75_force_per_fil = {}

for tension in runs:
    df = hip1r_assoc_positions[tension]
    df_force = df[df['zpos'] < -0.399]
    df_force['force_z'] = -df_force['force_z']  # make positive

    # Sum of force_z per run/time
    force_sum = df_force.groupby(['run', 'time'])['force_z'].sum()

    # Number of unique fiber_ids per run/time
    fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
        
    # Average force per filament
    avg_force = force_sum / fiber_count.replace(0, np.nan)
    
    # Reindex each run to uniform time points and build DataFrame
    avg_force_runs = []
    for run in avg_force.index.get_level_values('run').unique():
        run_series = avg_force.xs(run, level='run').reindex(time_points).fillna(method='ffill')
        avg_force_runs.append(run_series)
    
    avg_force_df = pd.concat(avg_force_runs, axis=1)
    
    # Median and IQR across runs
    median_force_per_fil[tension] = avg_force_df.median(axis=1)
    q25_force_per_fil[tension] = avg_force_df.quantile(0.25, axis=1)
    q75_force_per_fil[tension] = avg_force_df.quantile(0.75, axis=1)

# -----------------------------
# Plotting with magma colors
# -----------------------------
plt.figure(figsize=(8, 6))
for i, tension in enumerate(runs):
    plt.plot(time_points, median_force_per_fil[tension], color=cmap[i], label=labels[i])
    plt.fill_between(
        time_points,
        q25_force_per_fil[tension],
        q75_force_per_fil[tension],
        color=cmap[i],
        alpha=0.3
    )

plt.xlabel('Time')
plt.ylabel('Confinement force / confined filament')
plt.title(' Force per Confined Filament Over Time')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  # for magma colormap

runs = plot_order
time_points = np.linspace(0.1, 15, 150).round(1)  # uniform time points
cmap = sns.color_palette("magma", len(runs))       # magma colors

median_force_per_fil = {}
q25_force_per_fil = {}
q75_force_per_fil = {}

for tension in runs:
    df = hip1r_assoc_positions[tension]
    
    # Select confined filaments
    df_force = df[df['zpos'] < -0.399].copy()
    
    # Compute confinement force based on displacement
    df_force['confinement_force'] = (-0.399 - df_force['zpos']) * 1e4
    
    # Sum confinement_force per run/time
    force_sum = df_force.groupby(['run', 'time'])['confinement_force'].sum()
    
    # Number of unique fiber_ids per run/time
    fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
        
    # Average confinement force per filament
    avg_force = force_sum / fiber_count.replace(0, np.nan)
    
    # Reindex each run to uniform time points and build DataFrame
    avg_force_runs = []
    for run in avg_force.index.get_level_values('run').unique():
        run_series = avg_force.xs(run, level='run').reindex(time_points).fillna(method='ffill')
        avg_force_runs.append(run_series)
    
    avg_force_df = pd.concat(avg_force_runs, axis=1)
    
    # Median and IQR across runs
    median_force_per_fil[tension] = avg_force_df.median(axis=1)
    q25_force_per_fil[tension] = avg_force_df.quantile(0.25, axis=1)
    q75_force_per_fil[tension] = avg_force_df.quantile(0.75, axis=1)

# -----------------------------
# Plotting with magma colors
# -----------------------------
plt.figure(figsize=(8, 6))
for i, tension in enumerate(runs):
    plt.plot(time_points, median_force_per_fil[tension], color=cmap[i], label=tension)
    plt.fill_between(
        time_points,
        q25_force_per_fil[tension],
        q75_force_per_fil[tension],
        color=cmap[i],
        alpha=0.3
    )

plt.xlabel('Time')
plt.ylabel('pN/filament')
plt.title('Confinement Force per Confined Filament Over Time')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

runs = plot_order
cmap = sns.color_palette("magma", len(runs))
n_bins = 10 # number of bins per condition

plot_data = []

for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()
    df_force = df[df['zpos'] < -0.399].copy()
    df_force['force_z'] = -df_force['force_z']  # make positive

    # Sum of force and count fibers per run/time
    force_sum = df_force.groupby(['run','time'])['force_z'].sum()
    fiber_count = df_force.groupby(['run','time'])['fiber_id'].nunique()

    df_plot = pd.DataFrame({
        'TotalForce': force_sum.values,
        'NumFilaments': fiber_count.values
    })

    # Determine bin edges manually (linear spacing)
    min_f, max_f = df_plot['NumFilaments'].min(), df_plot['NumFilaments'].max()
    bins = np.linspace(min_f, max_f, n_bins+1)
    df_plot['FilBin'] = pd.cut(df_plot['NumFilaments'], bins, include_lowest=True)

    # Compute median + IQR per bin
    binned = df_plot.groupby('FilBin')['TotalForce'].agg(
        median='median',
        q25=lambda x: np.percentile(x,25),
        q75=lambda x: np.percentile(x,75)
    ).reset_index()
    binned['BinMid'] = binned['FilBin'].apply(lambda x: x.mid)
    binned['Tension'] = labels[i]

    plot_data.append(binned)

# Combine all conditions
summary = pd.concat(plot_data, axis=0)

# -----------------------------
# Plot
plt.figure(figsize=(8,6))

for i, tension in enumerate(labels):
    df_t = summary[summary['Tension']==tension]
    plt.plot(df_t['BinMid'], df_t['median'], color=cmap[i], lw=2, label=labels[i])
    plt.fill_between(df_t['BinMid'], df_t['q25'], df_t['q75'], color=cmap[i], alpha=0.3)
    plt.scatter(df_t['BinMid'], df_t['median'], color=cmap[i], s=40)


plt.xlabel('Number of Confined Filaments')
plt.ylabel('Total Confinement Force')
plt.title(f'Total Confinement Force vs Number of Confined Filaments\n(All Run × Time Points, {n_bins} bins)')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import linregress

plt.figure(figsize=(8,6))

slopes = []

for i, tension in enumerate(labels):
    df_t = summary[summary['Tension']==tension]
    
    # Linear fit on median vs BinMid
    slope, intercept, r_value, p_value, std_err = linregress(df_t['BinMid'], df_t['median'])
    slopes.append(slope)
    
    # Plot median line
    plt.plot(df_t['BinMid'], df_t['median'], color=cmap[i], lw=2, label=f"{labels[i]} (slope={slope:.2f})")
    # Shaded IQR
    plt.fill_between(df_t['BinMid'], df_t['q25'], df_t['q75'], color=cmap[i], alpha=0.3)
    # Scatter median points
    plt.scatter(df_t['BinMid'], df_t['median'], color=cmap[i], s=40)

plt.xlabel('Number of Confined Filaments')
plt.ylabel('Total Confinement Force')
plt.title(f'Total Confinement Force vs Number of Confined Filaments')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()

# Print slopes
for label, slope in zip(labels, slopes):
    print(f"{label}: slope = {slope:.2f}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

runs = plot_order
cmap = sns.color_palette("magma", len(runs))
n_bins = 10  # number of bins per condition

plot_data = []

for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()

    # Select only confined filaments
    df_confined = df[df['zpos'] < -0.399].copy()

    # Compute confinement force based on displacement
    df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

    # Sum of confinement force and count of confined fibers per run/time
    force_sum = df_confined.groupby(['run', 'time'])['confinement_force'].sum()
    fiber_count = df_confined.groupby(['run', 'time'])['fiber_id'].nunique()

    # Combine into one DataFrame
    df_plot = pd.DataFrame({
        'TotalForce': force_sum.values,
        'NumFilaments': fiber_count.values
    })

    # Determine linear bins for filament counts
    min_f, max_f = df_plot['NumFilaments'].min(), df_plot['NumFilaments'].max()
    bins = np.linspace(min_f, max_f, n_bins + 1)
    df_plot['FilBin'] = pd.cut(df_plot['NumFilaments'], bins, include_lowest=True)

    # Compute median and IQR of total confinement force per bin
    binned = df_plot.groupby('FilBin')['TotalForce'].agg(
        median='median',
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()

    # Add bin midpoints and label
    binned['BinMid'] = binned['FilBin'].apply(lambda x: x.mid)
    binned['Tension'] = labels[i]

    plot_data.append(binned)

# Combine all conditions
summary = pd.concat(plot_data, axis=0)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 6))

for i, tension in enumerate(labels):
    df_t = summary[summary['Tension'] == tension]
    plt.plot(df_t['BinMid'], df_t['median'], color=cmap[i], lw=2, label=labels[i])
    plt.fill_between(df_t['BinMid'], df_t['q25'], df_t['q75'], color=cmap[i], alpha=0.3)
    plt.scatter(df_t['BinMid'], df_t['median'], color=cmap[i], s=40)

plt.xlabel('Number of Confined Filaments')
plt.ylabel('Total Confinement Force (×10⁶ units)')
plt.title(f'Total Confinement Force vs Number of Confined Filaments\n(All Run × Time Points, {n_bins} bins)')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
runs

In [ ]:
labels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

violin_data = []

for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension]
    df_force = df[df['zpos'] < -0.399].copy()
    df_force['force_z'] = -df_force['force_z']  # make positive

    # Sum force and count fibers per run/time
    force_sum = df_force.groupby(['run', 'time'])['force_z'].sum()
    fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
    
    # Average force per filament per run/time
    avg_force = (force_sum / fiber_count.replace(0, np.nan)).reset_index(name='AvgForce')
    avg_force['Tension'] = labels[i]  # use custom label
    
    # Average over time for each run
    avg_force_per_run = avg_force.groupby('run')['AvgForce'].mean().reset_index()
    avg_force_per_run['Tension'] = labels[i]
    violin_data.append(avg_force_per_run[['Tension', 'AvgForce']])

# Combine all tensions
violin_df = pd.concat(violin_data, axis=0)

# -----------------------------
# Plot violin outlines + quartiles
plt.figure(figsize=(8,6))

sns.violinplot(
    data=violin_df,
    x='Tension',
    y='AvgForce',
    palette=cmap,
    inner='quartile',  # median + 25/75
    linewidth=1.5,
    cut=0,
    fill=False,
    order=labels
)

# Scatter individual run points
sns.swarmplot(
    data=violin_df,
    x='Tension',
    y='AvgForce',
    palette=cmap,
    size=6,
    edgecolor='k',
    alpha=0.9,
    order=labels
)

plt.xlabel('Tension')
plt.ylabel('Average Force per Filament (per Run)')
plt.title('Distribution of Average Force per Filament Across Runs')
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Compute number of filaments experiencing force
# -----------------------------
filament_count_per_time = {}

for tension in tension_runs:
    df = hip1r_assoc_positions[tension]
    df_force = df[df['zpos'] < -0.399]
    
    # Count unique fibers per run/time
    fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
    
    # Reindex each run to uniform time points and build DataFrame
    fiber_count_runs = []
    for run in fiber_count.index.get_level_values('run').unique():
        run_series = fiber_count.xs(run, level='run').reindex(time_points).fillna(method='ffill')
        fiber_count_runs.append(run_series)
    
    fiber_count_df = pd.concat(fiber_count_runs, axis=1)
    
    # Median and IQR across runs
    filament_count_per_time[tension] = {
        'median': fiber_count_df.median(axis=1),
        'q25': fiber_count_df.quantile(0.25, axis=1),
        'q75': fiber_count_df.quantile(0.75, axis=1)
    }

# -----------------------------
# Plotting number of filaments over time
# -----------------------------
plt.figure(figsize=(8, 5))
for i, tension in enumerate(runs):
    median = filament_count_per_time[tension]['median']
    q25 = filament_count_per_time[tension]['q25']
    q75 = filament_count_per_time[tension]['q75']
    
    plt.plot(time_points, median, color=cmap[i], label=labels[i])
    plt.fill_between(time_points, q25, q75, color=cmap[i], alpha=0.3)

plt.xlabel('Time')
plt.ylabel('# Confined Filaments')
plt.title('Confined Filament Count Over Time')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Compute percent of filaments experiencing force
# -----------------------------
percent_filaments_per_time = {}

for tension in tension_runs:
    df = hip1r_assoc_positions[tension]
    
    # Total fibers per run/time
    total_fibers = df.groupby(['run', 'time'])['fiber_id'].nunique()
    
    # Fibers experiencing force
    df_force = df[df['zpos'] < -0.399]
    fibers_with_force = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
    
    # Percent per run/time
    percent = (fibers_with_force / total_fibers * 100).fillna(0)
    
    # Reindex each run to uniform time points
    percent_runs = []
    for run in percent.index.get_level_values('run').unique():
        run_series = percent.xs(run, level='run').reindex(time_points).fillna(method='ffill')
        percent_runs.append(run_series)
    
    percent_df = pd.concat(percent_runs, axis=1)
    
    # Median and IQR across runs
    percent_filaments_per_time[tension] = {
        'median': percent_df.median(axis=1),
        'q25': percent_df.quantile(0.25, axis=1),
        'q75': percent_df.quantile(0.75, axis=1)
    }

# -----------------------------
# Plotting percent of filaments over time
# -----------------------------
plt.figure(figsize=(8, 6))
for i, tension in enumerate(runs):
    median = percent_filaments_per_time[tension]['median']
    q25 = percent_filaments_per_time[tension]['q25']
    q75 = percent_filaments_per_time[tension]['q75']
    
    plt.plot(time_points, median, color=cmap[i], label=labels[i])
    plt.fill_between(time_points, q25, q75, color=cmap[i], alpha=0.3)

plt.xlabel('Time')
plt.ylabel('Percent of Filaments Experiencing Force (%)')
plt.title('Percent of Filaments Experiencing Force Over Time')
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
df_force

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
cmap = sns.color_palette("magma", n_colors=len(runs))
n_bins = 50  # number of quantile bins

for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()
    df_force = df[df['zpos'] < -0.399].copy()
    df_force['force_z'] = -df_force['force_z']  # make positive

    # Confinement force per run/time
    force_sum = df_force.groupby(['run','time'])['force_z'].sum()
    fiber_count = df_force.groupby(['run','time'])['fiber_id'].nunique()

    confinement_df = pd.DataFrame({
        'run': force_sum.index.get_level_values('run'),
        'time': force_sum.index.get_level_values('time'),
        'confinement_force': force_sum.values
    })

    # Bud internalization
    df_bud = df[['run','time','bud_internalization']].copy()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # Compute per-run normalized tension force
    all_points = []
    k_value = float(k[i])  # your spring constant
    for run_id in confinement_df['run'].unique():
        df_run_conf = confinement_df[confinement_df['run']==run_id]
        df_run_bud = df_bud[df_bud['run']==run_id].copy()

        # Normalize internalization
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value

        # Merge confinement force
        df_merged = pd.merge(
            df_run_conf[['time','confinement_force']],
            df_run_bud_unique[['time','tension_force']],
            on='time'
        )
        all_points.append(df_merged)

    df_all = pd.concat(all_points, ignore_index=True)

    # Quantile-based bins of tension force
    df_all['bin'] = pd.qcut(df_all['tension_force'], q=n_bins, duplicates='drop')

    # Compute median + IQR per bin
    binned_stats = df_all.groupby('bin')['confinement_force'].agg(
        median='median',
        q25=lambda x: np.percentile(x,25),
        q75=lambda x: np.percentile(x,75)
    ).reset_index()
    binned_stats['bin_center'] = binned_stats['bin'].apply(lambda x: x.mid)

    # Plot median + IQR
    plt.plot(binned_stats['bin_center'], binned_stats['median'], color=cmap[i], lw=2, label=labels[i])
    plt.fill_between(binned_stats['bin_center'], binned_stats['q25'], binned_stats['q75'], color=cmap[i], alpha=0.3)
    # Scatter median points
    plt.scatter(binned_stats['bin_center'], binned_stats['median'], color=cmap[i], edgecolor='k', s=60)

plt.xlabel("Tension Force")
plt.ylabel("Confinement Force (sum of fiber forces)")
plt.title("Median ± IQR of Confinement vs Tension Force (quantile bins)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
cmap = sns.color_palette("magma", n_colors=len(runs))
n_bins = 20  # number of quantile bins
runs = plot_order
for i, tension in enumerate(runs):
    df = hip1r_assoc_positions[tension].copy()
    df_force = df[df['zpos'] < -0.399].copy()
    df_force['force_z'] = -df_force['force_z']  # make positive

    # Confinement force per run/time
    force_sum = df_force.groupby(['run','time'])['force_z'].sum()
    fiber_count = df_force.groupby(['run','time'])['fiber_id'].nunique()

    confinement_df = pd.DataFrame({
        'run': force_sum.index.get_level_values('run'),
        'time': force_sum.index.get_level_values('time'),
        'confinement_force': force_sum.values
    })

    # Bud internalization
    df_bud = df[['run','time','bud_internalization']].copy()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # Compute per-run normalized tension force
    all_points = []
    k_value = float(k[i])  # your spring constant
    for run_id in confinement_df['run'].unique():
        df_run_conf = confinement_df[confinement_df['run']==run_id]
        df_run_bud = df_bud[df_bud['run']==run_id].copy()

        # Normalize internalization
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value

        # Merge confinement force
        df_merged = pd.merge(
            df_run_conf[['time','confinement_force']],
            df_run_bud_unique[['time','tension_force']],
            on='time'
        )

        # Sort by tension_force to compute cumulative confinement
        df_merged = df_merged.sort_values('tension_force')
        df_merged['cum_confinement'] = df_merged['confinement_force'].cumsum()

        all_points.append(df_merged)

    df_all = pd.concat(all_points, ignore_index=True)

    # Quantile-based bins of tension force
    df_all['bin'] = pd.qcut(df_all['tension_force'], q=n_bins, duplicates='drop')

    # Compute median + IQR per bin of cumulative confinement
    binned_stats = df_all.groupby('bin')['cum_confinement'].agg(
        median='median',
        q25=lambda x: np.percentile(x,25),
        q75=lambda x: np.percentile(x,75)
    ).reset_index()
    binned_stats['bin_center'] = binned_stats['bin'].apply(lambda x: x.mid)

    # Plot median + IQR
    plt.plot(binned_stats['bin_center'], binned_stats['median'], color=cmap[i], lw=2, label=labels[i])
    plt.fill_between(binned_stats['bin_center'], binned_stats['q25'], binned_stats['q75'], color=cmap[i], alpha=0.3)
    plt.scatter(binned_stats['bin_center'], binned_stats['median'], color=cmap[i], edgecolor='k', s=60)

plt.xscale('log')
plt.yscale('log')
plt.xlabel("Tension Force")
plt.ylabel("Cumulative Confinement Force (Σ fiber forces)")
plt.title("Cumulative Confinement vs Tension Force")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

runs = tension_runs
cmap = sns.color_palette("magma", len(runs))  # magma colors

plt.figure(figsize=(8,6))

for i, tension in enumerate(runs):
    # Filament force
    df = hip1r_assoc_positions[tension]
    df_force = df[df['force_z'] < 0]
    
    # Sum of filament forces per run and time
    filament_force_sum = df_force.groupby(['run', 'time'])['force_z'].sum()
    
    # Confinement force
    df_confinement = forces[tension].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')
    df_confinement = df_confinement.set_index(['run', 'time'])
    
    for run in filament_force_sum.index.get_level_values('run').unique():
        # Get filament force for this run
        f_force = filament_force_sum.xs(run, level='run')
        
        # Get confinement force for this run
        if run in df_confinement.index.get_level_values('run'):
            c_force = df_confinement.xs(run, level='run')['zforce']
        else:
            continue
        
        # Align time points
        common_times = f_force.index.intersection(c_force.index)
        f_force = f_force.loc[common_times]
        c_force = c_force.loc[common_times]
        
        plt.scatter(f_force, c_force, color=cmap[i], alpha=0.3, s=15)

plt.xlabel('Sum of filament force_z')
plt.ylabel('Confinement z-force')
plt.title('Filament force vs. Confinement force')
plt.tight_layout()
plt.show()


In [ ]:
# Prepare data
all_data = []

for i, cond in enumerate(plot_order):
    label_name = labels[i]  # use the custom label for plotting
    
    # Confinement force
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')

    # Bud internalization
    df_bud = hip1r_assoc_positions[cond].reset_index()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # Spring constant
    k_value = float(k[i])

    # Loop over runs
    for run in df_confinement['run'].unique():
        df_run_confinement = df_confinement[df_confinement['run'] == run]
        df_run_bud = df_bud[df_bud['run'] == run].copy()

        # Normalize internalization per run
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()

        # Ensure one internalization per time
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()

        # Compute tension force
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value

        # Merge with confinement force
        df_merged = pd.merge(
            df_run_confinement[['time', 'zforce']],
            df_run_bud_unique[['time', 'tension_force']],
            on='time'
        )

        # Compute excess force
        df_merged['force_diff'] = df_merged['zforce'] - df_merged['tension_force']

        # Sum over all times for this run
        total_excess_force = df_merged['force_diff'].sum()

        # Add to dataset using label
        all_data.append({
            'Tension': label_name,
            'Total Excess Force': total_excess_force
        })

# Convert to DataFrame
df_plot = pd.DataFrame(all_data)

# Plot violin + swarm
plt.figure(figsize=(8,5))
sns.violinplot(
    x='Tension',
    y='Total Excess Force',
    data=df_plot,
    fill=False,
    palette='magma',
    saturation=1,
    inner='quart'
)
sns.swarmplot(
    x='Tension',
    y='Total Excess Force',
    data=df_plot,
    palette='magma',
    size=6,
    edgecolor='k',
    linewidth=0.5
)

plt.xlabel("Tension")
plt.ylabel("Total Excess Force (sum over time)")
plt.title("Distribution of Total Excess Force per Run")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data
all_data = []

for i, cond in enumerate(plot_order):
    # Confinement force
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')

    # Bud internalization
    df_bud = hip1r_assoc_positions[cond].reset_index()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # Spring constant
    k_value = float(k[i])

    # Loop over runs
    for run in df_confinement['run'].unique():
        df_run_confinement = df_confinement[df_confinement['run'] == run]
        df_run_bud = df_bud[df_bud['run'] == run].copy()

        # Normalize internalization per run
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()

        # Ensure one internalization per time
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()

        # Compute tension force
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value

        # Merge with confinement force
        df_merged = pd.merge(
            df_run_confinement[['time', 'zforce']],
            df_run_bud_unique[['time', 'tension_force', 'internalization']],
            on='time'
        )

        # Compute total excess force
        df_merged['force_diff'] = df_merged['zforce'] - df_merged['tension_force']
        total_excess_force = df_merged['force_diff'].sum()

        # Get final internalization for this run
        final_internalization = df_run_bud_unique['internalization'].iloc[-1]

        # Add to dataset
        all_data.append({
            'Tension': cond,
            'Total Excess Force': total_excess_force,
            'Final Internalization': final_internalization
        })

# Convert to DataFrame
df_plot = pd.DataFrame(all_data)

# Scatter plot: Total Excess Force vs Final Internalization
plt.figure(figsize=(8,6))
sns.scatterplot(
    x='Final Internalization',
    y='Total Excess Force',
    hue='Tension',
    palette='magma',
    data=df_plot,
    s=80
)

plt.xlabel("Final Internalization")
plt.ylabel("Total Excess Force (sum over time)")
plt.title("Total Excess Force vs Final Internalization per Run")
plt.legend(title='Tension')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
cmap = sns.color_palette("magma", n_colors=len(plot_order))

for i, cond in enumerate(plot_order):
    # Confinement force
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')
    
    # Bud internalization
    df_bud = hip1r_assoc_positions[cond].reset_index()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
    
    k_value = float(k[i])
    
    # Loop over runs
    for run in df_confinement['run'].unique():
        df_run_confinement = df_confinement[df_confinement['run'] == run]
        df_run_bud = df_bud[df_bud['run'] == run].copy()
        
        # Normalize internalization per run
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()
        
        # One internalization per time
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()
        
        # Compute tension force
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value
        
        # Merge with confinement force
        df_merged = pd.merge(
            df_run_confinement[['time', 'zforce']],
            df_run_bud_unique[['time', 'tension_force']],
            on='time'
        )
        
        # Scatter each run
        plt.scatter(df_merged['tension_force'], df_merged['zforce'], color=cmap[i], alpha=0.5, label=labels[i] if run==0 else "")

plt.xlabel("Tension Force")
plt.ylabel("Confinement Force (zforce)")
plt.title("Confinement vs Tension Force")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:


plt.figure(figsize=(8,6))
cmap = sns.color_palette("magma", n_colors=len(plot_order))

n_bins = 10  # Number of quantile bins

for i, cond in enumerate(plot_order):
    # Confinement force
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')
    
    # Bud internalization
    df_bud = hip1r_assoc_positions[cond].reset_index()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
    
    k_value = float(k[i])
    
    all_points = []
    for run in df_confinement['run'].unique():
        df_run_confinement = df_confinement[df_confinement['run'] == run]
        df_run_bud = df_bud[df_bud['run'] == run].copy()
        
        # Normalize internalization
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value
        
        # Merge confinement
        df_merged = pd.merge(
            df_run_confinement[['time','zforce']],
            df_run_bud_unique[['time','tension_force']],
            on='time'
        )
        all_points.append(df_merged[['tension_force','zforce']])
    
    df_all = pd.concat(all_points, ignore_index=True)
    
    # Quantile-based bins
    df_all['bin'] = pd.qcut(df_all['tension_force'], q=n_bins, duplicates='drop')
    
    # Compute median and IQR per bin
    binned_stats = df_all.groupby('bin')['zforce'].agg(
        median='median',
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()
    binned_stats['bin_center'] = binned_stats['bin'].apply(lambda x: x.mid)
    
    # Plot median line
    plt.plot(binned_stats['bin_center'], binned_stats['median'], color=cmap[i], lw=2, label=labels[i])
    # Plot IQR
    plt.fill_between(binned_stats['bin_center'], binned_stats['q25'], binned_stats['q75'], color=cmap[i], alpha=0.3)
    # Scatter points for each bin
    for _, row in binned_stats.iterrows():
        plt.scatter(row['bin_center'], row['median'], color=cmap[i], edgecolor='k', s=60)


plt.xlabel("Tension Force")
plt.ylabel("Confinement Force (zforce)")
plt.title("Median ± IQR of Confinement vs Tension Force (quantile bins)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Prepare data
all_data = []

for i, cond in enumerate(plot_order):
    # Confinement force
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')

    # Bud internalization
    df_bud = hip1r_assoc_positions[cond].reset_index()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # Spring constant
    k_value = float(k[i])

    # Loop over runs
    for run in df_confinement['run'].unique():
        df_run_confinement = df_confinement[df_confinement['run'] == run]
        df_run_bud = df_bud[df_bud['run'] == run].copy()

        # Normalize internalization per run
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()

        # Ensure one internalization per time
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()

        # Compute tension force
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value

        # Merge with confinement force
        df_merged = pd.merge(
            df_run_confinement[['time', 'zforce']],
            df_run_bud_unique[['time', 'tension_force']],
            on='time'
        )

        # Compute excess force
        df_merged['force_diff'] = df_merged['zforce'] - df_merged['tension_force']

        # Get excess force at last time point
        last_time_force = df_merged.iloc[-1]['force_diff']

        # Add to dataset
        all_data.append({
            'Tension': cond,
            'Excess Force (last time)': last_time_force
        })

# Convert to DataFrame
df_plot = pd.DataFrame(all_data)

# Plot violin + swarm
plt.figure(figsize=(8,5))
sns.violinplot(
    x='Tension',
    y='Excess Force (last time)',
    data=df_plot,
    fill=False,
    palette='magma',
    saturation=1,
    inner='quart'
)
sns.swarmplot(
    x='Tension',
    y='Excess Force (last time)',
    data=df_plot,
    palette='magma',
    size=6,
    edgecolor='k',
    linewidth=0.5
)

plt.xlabel("Tension")
plt.ylabel("Excess Force at Last Time Point")
plt.title("Distribution of Excess Force per Run (Last Time)")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data
all_data = []

for i, cond in enumerate(plot_order):
    # Confinement force
    df_confinement = forces[cond].reset_index()
    df_confinement['zforce'] = -pd.to_numeric(df_confinement['zforce'], errors='coerce')

    # Bud internalization
    df_bud = hip1r_assoc_positions[cond].reset_index()
    df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')

    # Spring constant
    k_value = float(k[i])

    # Loop over runs
    for run in df_confinement['run'].unique():
        df_run_confinement = df_confinement[df_confinement['run'] == run]
        df_run_bud = df_bud[df_bud['run'] == run].copy()

        # Normalize internalization per run
        df_run_bud['internalization'] -= df_run_bud['internalization'].min()

        # Ensure one internalization per time
        df_run_bud_unique = df_run_bud.groupby('time', as_index=False)['internalization'].mean()

        # Compute tension force
        df_run_bud_unique['tension_force'] = df_run_bud_unique['internalization'] * k_value

        # Merge with confinement force
        df_merged = pd.merge(
            df_run_confinement[['time', 'zforce']],
            df_run_bud_unique[['time', 'tension_force', 'internalization']],
            on='time'
        )

        # Find 95% internalization threshold
        threshold_95 = 0.95 * df_run_bud_unique['internalization'].max()

        # Select times where internalization ≥ 95% max
        df_high_internal = df_merged[df_merged['internalization'] >= threshold_95].copy()

        # Compute excess force
        df_high_internal['force_diff'] = df_high_internal['zforce'] - df_high_internal['tension_force']

        # Mean excess force for this run
        mean_force = df_high_internal['force_diff'].mean()

        # Add one value per run
        all_data.append({
            'Tension': cond,
            'Excess Force (mean ≥95% internalization)': mean_force
        })

# Convert to DataFrame
df_plot = pd.DataFrame(all_data)

# Plot violin + swarm
plt.figure(figsize=(8,5))
sns.violinplot(
    x='Tension',
    y='Excess Force (mean ≥95% internalization)',
    data=df_plot,
    fill=False,
    palette='magma',
    saturation=1,
    inner='quart'
)
sns.swarmplot(
    x='Tension',
    y='Excess Force (mean ≥95% internalization)',
    data=df_plot,
    palette='magma',
    size=6,
    edgecolor='k',
    linewidth=0.5
)

plt.xlabel("Tension")
plt.ylabel("Mean Excess Force at ≥95% Internalization")
plt.title("Distribution of Mean Excess Force per Run (High Internalization)")
plt.tight_layout()
plt.show()


### WD / Actin

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(8,6))
cmap = sns.color_palette("magma", n_colors=len(tension_runs))

scatter_data = []

for i, trial in enumerate(tension_runs):
    if trial not in hip1r_assoc_positions:
        continue

    # Last time point
    df_bud = hip1r_assoc_positions[trial].reset_index()
    last_time = df_bud['time'].max()
    df_bud_last = df_bud[df_bud['time'] == last_time]

    dir_df = directions_grouped_dict[trial]

    for run in df_bud_last['run'].unique():
        df_run_bud = df_bud_last[df_bud_last['run'] == run]
        df_run_dir = dir_df.loc[dir_df.index.get_level_values('run') == run]

        if df_run_dir.empty:
            continue

        # WD from your precomputed WD_tension_dict for this trial
        wd_result = WD_tension_dict[trial]

        # Mean WD across bins
        median_WD = wd_result['Median'].mean()
        n_dirs = df_run_dir['length'].sum()  # total # of directions in this run

        wd_per_dir = median_WD / n_dirs
        internalization = df_run_bud['bud_internalization'].values[0]

        scatter_data.append({
            'Tension': trial,
            'WD_per_dir': wd_per_dir,
            'Internalization': internalization
        })

df_scatter = pd.DataFrame(scatter_data)

# Scatter plot
sns.scatterplot(
    data=df_scatter,
    x='Internalization',
    y='WD_per_dir',
    hue='Tension',
    palette=cmap,
    s=80
)

plt.xscale('log')
plt.yscale('log')
plt.xlabel("Bud Internalization (last time)")
plt.ylabel("WD / #directions")
plt.title("WD per Actin vs Internalization (last time point)")
plt.legend(title="Tension")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import ot  # wasserstein_circle

n_uniform = 10000
cmap = sns.color_palette("magma", n_colors=len(tension_runs))

scatter_data = []

for i, trial in enumerate(tension_runs):
    if trial not in hip1r_assoc_positions:
        continue

    # Last time point per run
    df_bud = hip1r_assoc_positions[trial].reset_index()
    last_time = df_bud['time'].max()
    df_bud_last = df_bud[df_bud['time'] == last_time]

    dir_df = directions_grouped_dict[trial]

    for run in df_bud_last['run'].unique():
        df_run_bud = df_bud_last[df_bud_last['run'] == run]
        df_run_dir = dir_df.loc[dir_df.index.get_level_values('run') == run]

        if df_run_dir.empty:
            continue

        # Flatten directions and remove NaNs
        directions = np.concatenate(df_run_dir['direction'].values)
        directions = directions[~np.isnan(directions)]

        if len(directions) == 0:
            continue

        # Wasserstein distance against uniform [0,1]
        uniform_points = np.random.uniform(0, 1, n_uniform)
        wd_value = ot.wasserstein_1d(directions, uniform_points)

        # Raw internalization
        raw_internalization = df_run_bud['bud_internalization'].values[0]

        scatter_data.append({
            'Tension': trial,
            'WD': wd_value,
            'Internalization': raw_internalization
        })

# Convert to DataFrame
df_scatter = pd.DataFrame(scatter_data)

# Plot one figure per condition
for i, trial in enumerate(tension_runs):
    df_plot = df_scatter[df_scatter['Tension'] == trial]
    if df_plot.empty:
        continue

    plt.figure(figsize=(6,5))
    sns.scatterplot(
        data=df_plot,
        x='Internalization',
        y='WD',
        s=80,
        color=cmap[i]
    )
    plt.xlabel("Bud Internalization (raw)")
    plt.ylabel("Wasserstein Distance (WD)")
    plt.title(f"WD vs Internalization - {trial}")
    plt.tight_layout()
    plt.show()


In [ ]:
df_scatter

In [ ]:
scatter_data

# Figure 3 V2

In [ ]:
# Convenience functions for median and IQR

def median_and_iqr_errors(data):
    if isinstance(data, pd.Series):
        data = data.tolist()  # Convert Series to list if needed

    arr = np.array(data)

    # Compute quartiles
    q1 = np.percentile(arr, 25)  # First quartile (Q1)
    q3 = np.percentile(arr, 75)  # Third quartile (Q3)
    median = np.median(arr)  # Median

    # Compute lower and upper error for error bars
    #lower_error = median - q1
    #upper_error = q3 - median

    return median, q1, q3

In [ ]:
def circle_size_WD(dir_df, shared_bins, loops=1, baseline_loops=100):
    all_wass_distances = []

    for i in range(len(shared_bins) - 1):
        bin_start, bin_end = shared_bins[i], shared_bins[i + 1]
        subset = dir_df[(dir_df['length'] >= bin_start) & (dir_df['length'] < bin_end)]
        bin_count = len(subset)

        if subset.empty:
            continue  # Skip empty bins entirely

        wass_dists = []
        bl_sizes = []

        for _, row in subset.iterrows():
            directions = row['direction']

            # Skip invalid or empty direction arrays
            if directions is None or len(directions) == 0:
                continue

            # Ensure row['length'] > 0 and integer
            n2 = int(round(row['length']))
            if n2 <= 0:
                continue  # nothing to compare

            for _ in range(loops):
                random_network1 = np.random.uniform(-np.pi, np.pi, 10000)
                random_network2 = np.random.uniform(-np.pi, np.pi, n2)

                # Convert to [0, 1) domain
                random_network_shift = (np.pi + random_network1) / (2 * np.pi)
                random_network2_shift = (np.pi + random_network2) / (2 * np.pi)

                # Double-check before calling POT
                if len(random_network_shift) == 0 or len(random_network2_shift) == 0:
                    continue

                try:
                    wass_dist = ot.wasserstein_circle(directions, random_network_shift, p=1)
                    bl_size = ot.wasserstein_circle(random_network2_shift, random_network_shift, p=1)
                except ZeroDivisionError:
                    # Safety fallback: skip any edge cases POT doesn't handle
                    continue

                wass_dists.append(wass_dist)
                bl_sizes.append(bl_size)

        # Skip bins where no valid distances were computed
        if len(wass_dists) == 0 or len(bl_sizes) == 0:
            continue

        # Compute median and IQR for WD and BL
        median_WD, lower_WD, upper_WD = median_and_iqr_errors(wass_dists)
        median_BL, lower_BL, upper_BL = median_and_iqr_errors(bl_sizes)

        # --- Baseline estimate ---
        baseline_wass_dists = []
        bin_center = (bin_start + bin_end) / 2
        n_baseline = max(1, int(round(bin_center)))  # ensure at least one element

        for _ in range(baseline_loops):
            baseline_network1 = np.random.uniform(-np.pi, np.pi, 10000)
            baseline_network2 = np.random.uniform(-np.pi, np.pi, n_baseline)

            baseline_1_shift = (np.pi + baseline_network1) / (2 * np.pi)
            baseline_2_shift = (np.pi + baseline_network2) / (2 * np.pi)

            # Guard again
            if len(baseline_1_shift) == 0 or len(baseline_2_shift) == 0:
                continue

            baseline_wass_dist = ot.wasserstein_circle(baseline_1_shift, baseline_2_shift, p=1)
            baseline_wass_dists.append(baseline_wass_dist)

        if len(baseline_wass_dists) == 0:
            continue

        baseline_median = np.median(baseline_wass_dists)
        baseline_lower_error = np.percentile(baseline_wass_dists, 25)
        baseline_upper_error = np.percentile(baseline_wass_dists, 75)

        # Append results
        all_wass_distances.append({
            'Bin Center': bin_center,
            'Median': median_WD,
            'Lower Error': lower_WD,
            'Upper Error': upper_WD,
            'Count': bin_count,
            'Median BL': median_BL,
            'Lower BL': lower_BL,
            'Upper BL': upper_BL,
            'Baseline Median WD': baseline_median,
            'Baseline Lower Error': baseline_lower_error,
            'Baseline Upper Error': baseline_upper_error
        })

    if not all_wass_distances:
        return pd.DataFrame(columns=[
            'Median', 'Lower Error', 'Upper Error',
            'Median BL', 'Lower BL', 'Upper BL',
            'Baseline Median WD', 'Baseline Lower Error', 'Baseline Upper Error'
        ])

    result = pd.DataFrame(all_wass_distances)
    result = result.sort_values(by='Bin Center').set_index('Bin Center')
    return result


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
import ot  # POT: Python Optimal Transport

def circle_WD(dir_df, loops=1):
    timepoints = dir_df.index.get_level_values('time').unique()
    runs = dir_df.index.get_level_values('run').unique()
    max_length_per_run = dir_df.groupby('run')['length'].max()
    max_length_array = max_length_per_run.values

    all_circ_wds = []
    all_lin_wds = []
    all_circ_noise = []  # <-- Store circ_noise

    for time in timepoints:
        for run in runs:
            if (run, time) in dir_df.index:
                row = dir_df.loc[(run, time)]
                length = row['length']
                if length == 0:
                    continue  # Skip short tracks

                directions = row['direction']  # Angular data
                circ_dists = []
                lin_dists = []
                circ_noise_dists = []

                for _ in range(loops):
                    # random uniform comparison distributions
                    random_network1 = np.random.uniform(-np.pi, np.pi, 10000)
                    random_network2 = np.random.uniform(-np.pi, np.pi, len(directions))

                    # --- Circular WD ---
                    rn1_shift = (np.pi + random_network1) / (2 * np.pi)
                    rn2_shift = (np.pi + random_network2) / (2 * np.pi)

                    circ_noise = ot.wasserstein_circle(rn1_shift, rn2_shift, p=1)
                    circ_dist = ot.wasserstein_circle(directions, rn1_shift, p=1)

                    circ_dists.append(circ_dist)
                    circ_noise_dists.append(circ_noise)

                    # --- Linear WD ---
                    lin_dist = wasserstein_distance(directions, random_network1)
                    lin_dists.append(lin_dist)


                # Store averages
                all_circ_wds.append({'run': run, 'time': time, 'avg_WD': np.mean(circ_dists)})
                all_lin_wds.append({'run': run, 'time': time, 'avg_WD': np.mean(lin_dists)})
                all_circ_noise.append({'run': run, 'time': time, 'avg_WD': np.mean(circ_noise_dists)})

    # --- Aggregate circular ---
    circ_df = pd.DataFrame(all_circ_wds)
    grouped_circ = circ_df.groupby('time').agg({'avg_WD': lambda x: median_and_iqr_errors(x.tolist())})
    (median_circ, lower_circ, upper_circ) = zip(*grouped_circ['avg_WD'])
    circ_result = pd.DataFrame({
        'Median': median_circ,
        'Lower Error': lower_circ,
        'Upper Error': upper_circ
    }, index=grouped_circ.index)

    # --- Aggregate linear ---
    lin_df = pd.DataFrame(all_lin_wds)
    grouped_lin = lin_df.groupby('time').agg({'avg_WD': lambda x: median_and_iqr_errors(x.tolist())})
    (median_lin, lower_lin, upper_lin) = zip(*grouped_lin['avg_WD'])
    lin_result = pd.DataFrame({
        'Median': median_lin,
        'Lower Error': lower_lin,
        'Upper Error': upper_lin
    }, index=grouped_lin.index)

    # --- Aggregate circ_noise ---
    noise_df = pd.DataFrame(all_circ_noise)
    grouped_noise = noise_df.groupby('time').agg({'avg_WD': lambda x: median_and_iqr_errors(x.tolist())})
    (median_noise, lower_noise, upper_noise) = zip(*grouped_noise['avg_WD'])
    circ_noise_df = pd.DataFrame({
        'Median': median_noise,
        'Lower Error': lower_noise,
        'Upper Error': upper_noise
    }, index=grouped_noise.index)

    # --- Baseline (circular only for now) ---
    last_wd_dfs = []
    for length in max_length_array:
        if length > 0:
            for p_time in range(150):
                rn1 = np.random.uniform(-np.pi, np.pi, length)
                rn2 = np.random.uniform(-np.pi, np.pi, 10000)
                rn1_shift = (np.pi + rn1) / (2 * np.pi)
                rn2_shift = (np.pi + rn2) / (2 * np.pi)
                last_wd = ot.wasserstein_circle(rn1_shift, rn2_shift, p=1)
                last_wd_dfs.append({'time': p_time, 'avg_WD': last_wd})

    last_wd_df = pd.DataFrame(last_wd_dfs)
    grouped_baseline = last_wd_df.groupby('time').agg({'avg_WD': lambda x: median_and_iqr_errors(x.tolist())})
    (median_baseline, lower_baseline, upper_baseline) = zip(*grouped_baseline['avg_WD'])
    last_wd_df = pd.DataFrame({
        'time': grouped_baseline.index,
        'Median': median_baseline,
        'Lower Error': lower_baseline,
        'Upper Error': upper_baseline
    })

    return circ_result, lin_result, circ_noise_df, last_wd_df


In [ ]:
zero_trials_tension = [
    'endocytosis2_baseline_zero_diffusion_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
]

In [ ]:
import re

zero_trials_tension = [
    'endocytosis2_baseline_zero_diffusion_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
]

unique_runs = {}

for trial in zero_trials_tension:
    if trial not in branched_actin_bound_points:
        print(f"⚠️ Missing trial: {trial}")
        unique_runs[trial] = []
        continue
    
    subdict = branched_actin_bound_points[trial]
    
    # If subdict is a DataFrame
    run_column = subdict['run']  # adjust if column has a different name
    
    run_ids = set()
    for run_str in run_column:
        match = re.search(r'tension(\d+)-\d+', run_str)
        if match:
            run_ids.add(match.group(1))
    
    unique_runs[trial] = sorted(run_ids, key=int)

# Show results
for trial, runs in unique_runs.items():
    print(f"{trial}: {runs}")


In [ ]:
import pandas as pd

# unique_runs already contains lists of middle IDs per trial
# Example: unique_runs[trial] = ['0000', '0001', '0002', '0003']

# Dictionary to store the resulting DataFrames per trial and per run
trial_run_dfs = {}

for trial in zero_trials_tension:
    if trial not in branched_actin_bound_points:
        trial_run_dfs[trial] = {}
        continue
    
    subdict = branched_actin_bound_points[trial]
    run_column = subdict['run']  # column with run strings

    # Initialize nested dict for this trial
    trial_run_dfs[trial] = {}
    
    for run_id in unique_runs[trial]:
        # Select rows where the middle tension ID matches run_id
        mask = run_column.apply(lambda x: run_id == x.split("tension")[1].split("-")[0])
        df_run = subdict[mask].copy()  # get DataFrame subset
        trial_run_dfs[trial][run_id] = df_run

In [ ]:
import re

# List of trials
zero_trials_tension = [
    'endocytosis2_baseline_zero_diffusion_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
]

unique_runs_lengths = {}

for trial in zero_trials_tension:
    if trial not in df_with_lengths:
        print(f"⚠️ Missing trial: {trial}")
        unique_runs_lengths[trial] = []
        continue
    
    subdict = df_with_lengths[trial].reset_index()
    
    # If subdict is a DataFrame
    run_column = subdict['run']  # adjust if column has a different name
    
    run_ids = set()
    for run_str in run_column:
        match = re.search(r'tension(\d+)-\d+', str(run_str))
        if match:
            run_ids.add(match.group(1))
    
    unique_runs_lengths[trial] = sorted(run_ids, key=int)

# Show results
for trial, runs in unique_runs_lengths.items():
    print(f"{trial}: {runs}")


In [ ]:
import pandas as pd

# unique_runs_lengths already contains lists of middle IDs per trial
# Example: unique_runs_lengths[trial] = ['0000', '0001', '0002', '0003']

# Dictionary to store resulting DataFrames per trial and per tension/run
trial_run_dfs_lengths = {}

for trial in zero_trials_tension:
    if trial not in df_with_lengths:
        trial_run_dfs_lengths[trial] = {}
        continue
    
    df_trial = df_with_lengths[trial].reset_index()  # ensure proper indexing
    run_column = df_trial['run']  # column with run strings

    # Initialize nested dict for this trial
    trial_run_dfs_lengths[trial] = {}
    
    for tension_id in unique_runs_lengths[trial]:
        # Select rows where the middle tension ID matches tension_id
        mask = run_column.apply(lambda x: tension_id == str(x).split("tension")[1].split("-")[0])
        df_run = df_trial[mask].copy()  # subset DataFrame
        trial_run_dfs_lengths[trial][tension_id] = df_run

# Example: access df for trial 1, tension '0000'
# trial_run_dfs_lengths[zero_trials_tension[0]]['0000']


## Prelim (actin amount)

In [ ]:
summary_dict = {}  # nested dict: trial -> run_id -> summary DataFrame

for trial, run_dfs in trial_run_dfs.items():
    summary_dict[trial] = {}
    
    for run_id, df in run_dfs.items():
        if df.empty:
            continue  # skip empty DataFrames
        
        positions_df = df.reset_index()
        positions_df = positions_df[['run', 'time', 'fiber_id', 'zpos_recalibrated', 'bud_internalization']]
        positions_df = positions_df.sort_values(['run', 'time'])
        
        # Compute dz and cumulative sum per run
        positions_df['dz'] = positions_df.groupby('run')['bud_internalization'].diff()
        positions_df['sum_dz'] = positions_df.groupby('run')['dz'].cumsum()
        
        # Fiber counts per run and median/quartiles across runs
        fibers_count = positions_df.groupby(['run', 'time'])['fiber_id'].nunique().reset_index(name='num_fibers')
        median_fibers = fibers_count.groupby('time')['num_fibers'].median().reset_index(name='median_num_fibers')
        q25_fibers = fibers_count.groupby('time')['num_fibers'].quantile(0.25).reset_index(name='q25_num_fibers')
        q75_fibers = fibers_count.groupby('time')['num_fibers'].quantile(0.75).reset_index(name='q75_num_fibers')
        
        # Bud internalization summaries
        median_bud = positions_df.groupby('time')['bud_internalization'].median().reset_index(name='median_bud_internalization')
        q25_bud = positions_df.groupby('time')['bud_internalization'].quantile(0.25).reset_index(name='q25_bud_internalization')
        q75_bud = positions_df.groupby('time')['bud_internalization'].quantile(0.75).reset_index(name='q75_bud_internalization')
        
        # Optional dz and sum_dz summaries
        median_dz = positions_df.groupby('time')['dz'].median().reset_index(name='median_dz')
        q25_dz = positions_df.groupby('time')['dz'].quantile(0.25).reset_index(name='q25_dz')
        q75_dz = positions_df.groupby('time')['dz'].quantile(0.75).reset_index(name='q75_dz')
        
        median_sum_dz = positions_df.groupby('time')['sum_dz'].median().reset_index(name='median_sum_dz')
        q25_sum_dz = positions_df.groupby('time')['sum_dz'].quantile(0.25).reset_index(name='q25_sum_dz')
        q75_sum_dz = positions_df.groupby('time')['sum_dz'].quantile(0.75).reset_index(name='q75_sum_dz')
        
        # Merge all summaries
        summary = median_fibers.merge(q25_fibers, on='time').merge(q75_fibers, on='time')
        summary = summary.merge(median_bud, on='time').merge(q25_bud, on='time').merge(q75_bud, on='time')
        summary = summary.merge(median_dz, on='time').merge(q25_dz, on='time').merge(q75_dz, on='time')
        summary = summary.merge(median_sum_dz, on='time').merge(q25_sum_dz, on='time').merge(q75_sum_dz, on='time')
        
        summary_dict[trial][run_id] = summary

# Access example:
# summary_dict['endocytosis2_baseline_zero_diffusion_output']['0000']


In [ ]:


# Define run_ids (assuming all trials have the same run_ids)
run_ids = ['0000', '0001', '0002', '0003']
tension_names = ['10', '150', '1000', '10000']
tensions = ['10', '150', '1000', '10000']

# Desired plot labels for trials
plot_labels = ['Full', '3/4', '1/2', '1/4']

# Color palette from Seaborn Viridis
palette = sns.color_palette("viridis", n_colors=len(zero_trials))

# Set up 2x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False, sharey=False)
axes = axes.flatten()

for i, run_id in enumerate(run_ids):
    ax = axes[i]
    
    for j, trial in enumerate(zero_trials_tension):
        # Check if this run exists for the trial
        if run_id in summary_dict[trial]:
            df = summary_dict[trial][run_id]
            
            # Plot median
            ax.plot(df['time'], df['median_bud_internalization'], label=plot_labels[j], color=palette[j])
            
            # Shade between 25th and 75th percentiles
            ax.fill_between(df['time'], df['q25_bud_internalization'], df['q75_bud_internalization'], 
                            color=palette[j], alpha=0.3)
    
    ax.set_title(f"{tension_names[i]} pN/um")
    ax.set_xlabel("Time")
    ax.set_ylabel("Internalization (um)")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define run_ids (assuming all trials have the same run_ids)
run_ids = ['0000', '0001', '0002', '0003']
tension_names = ['10', '150', '1000', '10000']
tensions = ['10', '150', '1000', '10000']

# Desired plot labels for trials
plot_labels = ['Full', '3/4', '1/2', '1/4']

# Color palette from Seaborn Viridis
palette = sns.color_palette("viridis", n_colors=len(zero_trials))

# Set up 2x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False, sharey=False)
axes = axes.flatten()

for i, run_id in enumerate(run_ids):
    ax = axes[i]
    
    for j, trial in enumerate(zero_trials_tension):
        # Check if this run exists for the trial
        if run_id in summary_dict[trial]:
            df = summary_dict[trial][run_id]
            
            # Plot median
            ax.plot(df['time'], df['median_num_fibers'], label=plot_labels[j], color=palette[j])
            
            # Shade between 25th and 75th percentiles
            ax.fill_between(df['time'], df['q25_num_fibers'], df['q75_num_fibers'], 
                            color=palette[j], alpha=0.3)
    
    ax.set_title(f"{tension_names[i]} pN/um")
    ax.set_xlabel("Time")
    ax.set_ylabel("Internalization (um)")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define run_ids and trials
run_ids = ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels

# Color palette for Run IDs
palette = sns.color_palette("viridis", n_colors=len(run_ids))

# Set up subplots: one per trial
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False, sharey=False)
axes = axes.flatten()

for i, trial in enumerate(zero_trials_tension):
    ax = axes[i]
    
    for j, run_id in enumerate(run_ids):
        if run_id in summary_dict[trial]:
            df = summary_dict[trial][run_id]
            
            # Plot median
            ax.plot(df['time'], df['median_num_fibers'], label=f"Run {run_id}", color=palette[j])
            
            # Shade between 25th and 75th percentiles
            ax.fill_between(df['time'], df['q25_num_fibers'], df['q75_num_fibers'],
                            color=palette[j], alpha=0.3)
    
    ax.set_title(f"Trial: {plot_labels[i]}")
    ax.set_xlabel("Time")
    ax.set_ylabel("Median Fiber Count")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- Setup ---
run_ids = ['0000', '0001', '0002', '0003']
tensions = ['10', '150', '1000', '10000']  # or tension mapping if needed
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable labels
palette = sns.color_palette("viridis", n_colors=len(zero_trials_tension))  # one color per trial

# --- Collect final internalization values ---
final_data = []

for j, trial in enumerate(zero_trials_tension):
    for run_id in run_ids:
        if run_id not in summary_dict[trial]:
            print(f"⚠️ Missing data for trial {trial}, run {run_id}")
            continue
        df = summary_dict[trial][run_id]
        final_value = df['median_bud_internalization'].iloc[-1]  # last timepoint
        q25 = df['q25_bud_internalization'].iloc[-1]
        q75 = df['q75_bud_internalization'].iloc[-1]
        final_data.append({
            'Tension': trial,        # keep original trial name
            'Trial Label': plot_labels[j],  # readable label for legend
            'Run ID': run_id,
            'Median': final_value,
            'Q25': q25,
            'Q75': q75
        })

final_df = pd.DataFrame(final_data)

In [ ]:
import pandas as pd

final_data = []

for j, trial in enumerate(zero_trials_tension):
    for run_id in run_ids:
        if run_id not in trial_run_dfs[trial]:
            print(f"⚠️ Missing data for trial {trial}, run {run_id}")
            continue
        
        run_data = trial_run_dfs[trial][run_id]  # assumed to be a DataFrame
        run_data['bud_internalization'] = run_data['bud_internalization']
        
        # Group by 'run' (if there are multiple runs inside the DF), take last bud_internalization
        grouped = run_data.groupby('run')['bud_internalization']
        last_values = grouped.last()  # Series: index = run, value = last bud_internalization
        
        # Optional: if you want Q25/Q75, compute across all entries per run
        q25 = grouped.quantile(0.25)
        q75 = grouped.quantile(0.75)
        
        # Store one row per run in final_data
        for run_sub_id in last_values.index:
            final_data.append({
                'Condition': trial,
                'Trial Label': plot_labels[j],
                'Run ID': run_id,
                'Last': last_values[run_sub_id]
            })

final_df = pd.DataFrame(final_data)

median_df = final_df.groupby(['Condition', 'Run ID']).agg(
    Trial_Label=('Trial Label', 'first'),   # preserve trial label
    Median=('Last', 'median'),
    Q25=('Last', lambda x: x.quantile(0.25)),
    Q75=('Last', lambda x: x.quantile(0.75))
).reset_index()

In [ ]:
# --- Plot: one subplot per tension ---
run_ids = ['0000', '0001', '0002', '0003']
tensions = ['10', '150', '1000', '10000']  # subplot order
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable labels
palette = sns.color_palette("viridis", n_colors=len(plot_labels))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, tension in enumerate(run_ids):
    ax = axes[i]
    sub = median_df[median_df['Run ID'] == tension]

    for j, trial in enumerate(plot_labels):
        trial_sub = sub[sub['Trial_Label'] == trial]
        if trial_sub.empty:
            continue
        ax.errorbar(trial_sub['Trial_Label'], trial_sub['Median'],
                    yerr=[trial_sub['Median'] - trial_sub['Q25'],
                          trial_sub['Q75'] - trial_sub['Median']],
                    fmt='-o', color=palette[j], capsize=3, label=plot_labels[j])

    ax.set_title(f"Tension: {tension}")
    ax.set_xlabel("Run ID")
    ax.set_ylabel("Final Bud Internalization")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:

# --- Setup ---
tensions = ['10', '150', '1000', '10000']  # different tension values
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
palette = sns.color_palette("magma", n_colors=len(tensions))  # color per tension

fig, ax = plt.subplots(figsize=(8, 6))

# --- Loop over tensions ---
for i, tens in enumerate(run_ids):
    sub = median_df[median_df['Run ID'] == tens]

    # Plot median
    ax.errorbar(sub['Trial_Label'], sub['Median'],
                yerr=[sub['Median'] - sub['Q25'], sub['Q75'] - sub['Median']],
                fmt='-o', color=palette[i], capsize=3, label=f"{tensions[i]} pN/um")

ax.set_xlabel("Trial")
ax.set_ylabel("Final Bud Internalization")
ax.set_title("Final Bud Internalization Across Trials (All Tensions)")
ax.grid(True)
ax.legend(title="Tension")
plt.tight_layout()
plt.show()


In [ ]:


run_ids = ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']
palette = sns.color_palette("viridis", n_colors=len(zero_trials_tension))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, run_id in enumerate(run_ids):
    ax = axes[i]
    
    plot_data = []
    
    for j, trial in enumerate(zero_trials_tension):
        if run_id in trial_run_dfs[trial]:
            df = trial_run_dfs[trial][run_id]
            if df.empty:
                continue
            
            # Find the last timepoint per run
            last_time_df = df.groupby('run')['time'].max().reset_index(name='last_time')
            
            # Merge to get rows at last timepoint
            merged = pd.merge(df, last_time_df, on=['run'])
            final_df = merged[merged['time'] == merged['last_time']]
            
            # Count unique fiber_id per run
            final_counts_df = final_df.groupby('run')['fiber_id'].nunique().reset_index(name='final_fiber_count')
            final_counts_df['trial'] = plot_labels[j]
            
            plot_data.append(final_counts_df)
    
    if plot_data:
        plot_df = pd.concat(plot_data, ignore_index=True)
        sns.violinplot(x='trial', y='final_fiber_count', data=plot_df, ax=ax, palette=palette, fill=False, cut=0, inner='quart')
        sns.swarmplot(x='trial', y='final_fiber_count', data=plot_df, ax=ax, palette=palette, size=6, edgecolor='k', linewidth=0.5)
    
    ax.set_title(f"Run ID: {run_id}")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Final Fiber Count")
    ax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

run_ids = ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # for subplot titles
palette = sns.color_palette("viridis", n_colors=len(run_ids))

fig, axes = plt.subplots(1, len(zero_trials_tension), figsize=(16, 5), sharey=True)

for i, trial in enumerate(zero_trials_tension):
    ax = axes[i]
    plot_data = []
    
    for run_id_idx, run_id in enumerate(run_ids):
        if run_id in trial_run_dfs[trial]:
            df = trial_run_dfs[trial][run_id]
            if df.empty:
                continue
            
            # Find last timepoint per run
            last_time_df = df.groupby('run')['time'].max().reset_index(name='last_time')
            merged = pd.merge(df, last_time_df, on='run')
            final_df = merged[merged['time'] == merged['last_time']]
            
            # Count unique fibers per run
            final_counts_df = final_df.groupby('run')['fiber_id'].nunique().reset_index(name='final_fiber_count')
            final_counts_df['run_id'] = run_id  # X-axis is run_id (0000-0003)
            plot_data.append(final_counts_df)
    
    if plot_data:
        plot_df = pd.concat(plot_data, ignore_index=True)
        sns.violinplot(x='run_id', y='final_fiber_count', data=plot_df, ax=ax, palette=palette, fill=False, cut=0, inner='quart')
        sns.swarmplot(x='run_id', y='final_fiber_count', data=plot_df, ax=ax, palette=palette, size=6, edgecolor='k', linewidth=0.5)
    
    ax.set_title(plot_labels[i])  # subplot title corresponds to trial
    ax.set_xlabel("Run ID")
    ax.set_ylabel("Final Fiber Count")
    ax.grid(True)

plt.tight_layout()
plt.show()


## 3A Relative internalization

In [ ]:
import pandas as pd

final_data = []

for j, trial in enumerate(zero_trials_tension):
    for run_id in run_ids:
        if run_id not in trial_run_dfs[trial]:
            print(f"⚠️ Missing data for trial {trial}, run {run_id}")
            continue
        
        run_data = trial_run_dfs[trial][run_id]  # assumed to be a DataFrame
        run_data['bud_internalization'] = run_data['bud_internalization'] / 100
        
        # Group by 'run' (if there are multiple runs inside the DF), take last bud_internalization
        grouped = run_data.groupby('run')['bud_internalization']
        last_values = grouped.last()  # Series: index = run, value = last bud_internalization
        
        # Optional: if you want Q25/Q75, compute across all entries per run
        q25 = grouped.quantile(0.25)
        q75 = grouped.quantile(0.75)
        
        # Store one row per run in final_data
        for run_sub_id in last_values.index:
            final_data.append({
                'Condition': trial,
                'Trial Label': plot_labels[j],
                'Run ID': run_id,
                'Last': last_values[run_sub_id]
            })

final_df = pd.DataFrame(final_data)

median_df = final_df.groupby(['Condition', 'Run ID']).agg(
    Trial_Label=('Trial Label', 'first'),   # preserve trial label
    Median=('Last', 'median'),
    Q25=('Last', lambda x: x.quantile(0.25)),
    Q75=('Last', lambda x: x.quantile(0.75))
).reset_index()

In [ ]:
# --- Setup ---
tensions = ['10', '150', '1000', '10000']
plot_labels = ['Full', '3/4', '1/2', '1/4']
palette = sns.color_palette("magma", n_colors=len(tensions))

# Enforce categorical order
median_df['Trial_Label'] = pd.Categorical(
    median_df['Trial_Label'],
    categories=plot_labels,
    ordered=True
)
median_df = median_df.sort_values('Trial_Label')

fig, ax = plt.subplots(figsize=(8, 6))

# --- Loop over tensions ---
for i, tens in enumerate(run_ids):
    sub = median_df[median_df['Run ID'] == tens]

    ax.errorbar(
        sub['Trial_Label'], sub['Median'],
        yerr=[sub['Median'] - sub['Q25'], sub['Q75'] - sub['Median']],
        fmt='-o', color=palette[i], capsize=3,
        label=f"{tensions[i]} pN/um"
    )

ax.set_xlabel("Trial")
ax.set_ylabel("Final Bud Internalization")
ax.set_title("Final Bud Internalization Across Trials (All Tensions)")
ax.grid(True)
ax.legend(title="Tension", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# --- Setup ---
tensions = ['150', '10000']           # only the two tensions
plot_labels = ['Full', '3/4', '1/2', '1/4']
palette = sns.color_palette("magma", n_colors=len(tensions))

# --- Filter median_df for run IDs 0001 and 0003 ---
filtered_df = median_df[median_df['Run ID'].isin(['0001', '0003'])]

# Enforce categorical order for x-axis
filtered_df['Trial_Label'] = pd.Categorical(
    filtered_df['Trial_Label'],
    categories=plot_labels,
    ordered=True
)
filtered_df = filtered_df.sort_values('Trial_Label')

fig, ax = plt.subplots(figsize=(8, 6))

# --- Loop over filtered run IDs ---
run_ids_filtered = ['0001', '0003']
for i, run_id in enumerate(run_ids_filtered):
    sub = filtered_df[filtered_df['Run ID'] == run_id]
    ax.errorbar(
        sub['Trial_Label'], sub['Median'],
        yerr=[sub['Median'] - sub['Q25'], sub['Q75'] - sub['Median']],
        fmt='-o', color=palette[i], capsize=3,
        label=f"{tensions[i]} pN/um"
    )

ax.set_xlabel("Arp2/3 Distribution", fontsize=16)
ax.set_ylabel("Final Bud Internalizatio (nm)", fontsize=16)
#ax.set_title("Final Bud Internalization Across Selected Trials")
ax.grid(True)
ax.legend(title="Tension", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

final_data = []

for j, trial in enumerate(zero_trials_tension):
    for run_id in run_ids:
        if run_id not in trial_run_dfs[trial]:
            print(f"⚠️ Missing data for trial {trial}, run {run_id}")
            continue
        
        run_data = trial_run_dfs[trial][run_id].copy()  # assume DataFrame with 'time' and 'bud_internalization'
        run_data['bud_internalization'] = run_data['bud_internalization']*1000
        
        # If multiple sub-runs in the DF
        for run_sub_id, df_sub in run_data.groupby('run'):
            times = df_sub['time'].values
            values = df_sub['bud_internalization'].values
            
            max_val = np.max(values)
            t_max = times[np.argmax(values)]
            
            half_max = max_val / 2
            # Find first time that internalization reaches or exceeds half-max
            t_half_candidates = times[values >= half_max]
            t_half = t_half_candidates[0] if len(t_half_candidates) > 0 else np.nan
            
            ratio = t_half / t_max if t_max != 0 else np.nan
            
            final_data.append({
                'Condition': trial,
                'Trial Label': plot_labels[j],
                'Run ID': run_id,
                'Last': values[-1],
                'Time_to_Half_Max': t_half,
                'Time_to_Max': t_max,
                'Half_Max_Ratio': ratio
            })

final_df = pd.DataFrame(final_data)

median_df = final_df.groupby(['Condition', 'Run ID']).agg(
    Trial_Label=('Trial Label', 'first'),
    Median=('Last', 'median'),
    Q25=('Last', lambda x: x.quantile(0.25)),
    Q75=('Last', lambda x: x.quantile(0.75)),
    Median_t_half=('Time_to_Half_Max', 'median'),
    Q25_t_half=('Time_to_Half_Max', lambda x: x.quantile(0.25)),
    Q75_t_half=('Time_to_Half_Max', lambda x: x.quantile(0.75)),
    Median_t_max=('Time_to_Max', 'median'),
    Q25_t_max=('Time_to_Max', lambda x: x.quantile(0.25)),
    Q75_t_max=('Time_to_Max', lambda x: x.quantile(0.75)),
    Median_ratio=('Half_Max_Ratio', 'median'),
    Q25_ratio=('Half_Max_Ratio', lambda x: x.quantile(0.25)),
    Q75_ratio=('Half_Max_Ratio', lambda x: x.quantile(0.75))
).reset_index()


In [ ]:

# --- Setup ---
tensions = ['10', '150', '1000', '10000']
plot_labels = ['Full', '3/4', '1/2', '1/4']
palette = sns.color_palette("magma", n_colors=len(tensions))

# Enforce categorical order
median_df['Trial_Label'] = pd.Categorical(
    median_df['Trial_Label'],
    categories=plot_labels,
    ordered=True
)
median_df = median_df.sort_values('Trial_Label')

# --- Create side-by-side plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

# Metrics order: first is ratio, then t_half, then t_max
metrics = [
    ('Median_ratio', 'Q25_ratio', 'Q75_ratio', 'Half-Max / Max Time Ratio'),
    ('Median_t_half', 'Q25_t_half', 'Q75_t_half', 'Time to Half-Max'),
    ('Median_t_max', 'Q25_t_max', 'Q75_t_max', 'Time to Max')
]

for ax, (median_col, q25_col, q75_col, ylabel) in zip(axes, metrics):
    for i, tens in enumerate(run_ids):
        sub = median_df[median_df['Run ID'] == tens]
        
        if q25_col and q75_col:
            yerr = [sub[median_col] - sub[q25_col], sub[q75_col] - sub[median_col]]
        else:
            yerr = None
        
        ax.errorbar(
            sub['Trial_Label'], sub[median_col],
            yerr=yerr,
            fmt='-o', color=palette[i], capsize=3,
            label=f"{tensions[i]} pN/um"
        )
    ax.set_xlabel("Trial")
    ax.set_ylabel(ylabel)
    ax.grid(True)
    ax.set_title(ylabel)

# --- Horizontal legend above the plots ---
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center', bbox_to_anchor=(0.5, 1.05),
    ncol=len(tensions), frameon=False, fontsize=10
)

plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space on top for legend
plt.show()


In [ ]:
median_df

In [ ]:
# --- Setup ---
plot_run_ids = ['0001', '0003']          # actual run IDs to use
plot_labels_for_legend = ['150', '10000']  # labels in legend
plot_labels = ['Full', '3/4', '1/2', '1/4']
palette = sns.color_palette("magma", n_colors=len(plot_run_ids))

# Enforce categorical order
median_df['Trial_Label'] = pd.Categorical(
    median_df['Trial_Label'],
    categories=plot_labels,
    ordered=True
)
median_df = median_df.sort_values('Trial_Label')

# --- Create side-by-side plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

# Metrics order: first is ratio, then t_half, then t_max
metrics = [
    ('Median_ratio', 'Q25_ratio', 'Q75_ratio', 'Half-Max / Max Time Ratio'),
    ('Median_t_half', 'Q25_t_half', 'Q75_t_half', 'Time to Half-Max'),
    ('Median_t_max', 'Q25_t_max', 'Q75_t_max', 'Time to Max')
]

for ax, (median_col, q25_col, q75_col, ylabel) in zip(axes, metrics):
    for i, run_id in enumerate(plot_run_ids):
        sub = median_df[median_df['Run ID'] == run_id]
        
        if q25_col and q75_col:
            yerr = [sub[median_col] - sub[q25_col], sub[q75_col] - sub[median_col]]
        else:
            yerr = None
        
        ax.errorbar(
            sub['Trial_Label'], sub[median_col],
            yerr=yerr,
            fmt='-o', color=palette[i], capsize=3,
            label=f"{plot_labels_for_legend[i]} pN/um"
        )
    ax.set_xlabel("Trial")
    ax.set_ylabel(ylabel)
    ax.grid(True)
    ax.set_title(ylabel)

# --- Horizontal legend above the plots ---
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center', bbox_to_anchor=(0.5, 1.05),
    ncol=len(plot_run_ids), frameon=False, fontsize=10
)

plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space on top for legend
plt.show()


In [ ]:
# Define which runs to plot
plot_run_ids = ['0001', '0003']
tension_labels = ['150', '10000']  # labels for the legend
plot_labels = ['Full', '3/4', '1/2', '1/4']

# Color palette for trials
palette = sns.color_palette("viridis", n_colors=len(plot_labels))

fig, ax = plt.subplots(figsize=(10, 6))

for i, run_id in enumerate(plot_run_ids):
    for j, trial in enumerate(zero_trials_tension):
        if run_id in summary_dict[trial]:
            df = summary_dict[trial][run_id]
            
            # Plot median
            ax.plot(df['time'], df['median_bud_internalization'], 
                    label=f"{tension_labels[i]} pN/um - {plot_labels[j]}", 
                    color=palette[j])
            
            # Shade between 25th and 75th percentiles
            ax.fill_between(df['time'], df['q25_bud_internalization'], df['q75_bud_internalization'], 
                            color=palette[j], alpha=0.2)

ax.set_xlabel("Time")
ax.set_ylabel("Bud Internalization (um)")
ax.set_title("Bud Internalization Over Time for Runs 0001 and 0003")
ax.legend(fontsize=9, loc='upper left', bbox_to_anchor=(1.05, 1))
ax.grid(True)
plt.tight_layout()
plt.show()


## 3B

In [ ]:
WD_trial_run_dict = {}  # Store WD results per trial/run
dir_df_dict = {}
index = np.round(np.arange(0, 15.0, 0.1), 1)

for trial in zero_trials_tension:
    WD_trial_run_dict[trial] = {}
    
    for run_id in trial_run_dfs[trial]:
        df_run = trial_run_dfs[trial][run_id]
        if df_run.empty:
            continue
        
        # Recalibrate positions
        ypos = df_run['ypos_recalibrated']
        xpos = df_run['xpos_recalibrated']
        dir_vals = np.arctan2(ypos, xpos)
        dir_circle = (np.pi + dir_vals) / (2 * np.pi)  # normalize to [0,1]
        df_run = df_run.copy()
        df_run['direction'] = dir_circle
        
        # Group by run and time to prepare input for circle_WD
        grouped_directions = df_run.groupby(['run', 'time'])['direction'].apply(np.array)
        grouped_lengths = grouped_directions.apply(lambda x: np.sum(~np.isnan(x)))
        
        dir_df = pd.DataFrame({
            'direction': grouped_directions,
            'length': grouped_lengths
        })
        
        # Set MultiIndex names to match circle_WD expectations
        dir_df.index.names = ['run', 'time']
        
        # Store for reference
        dir_df_dict[(trial, run_id)] = dir_df
        
        # Compute WD
        circ_result, lin_result, circ_noise_df, last_wd_df = circle_WD(dir_df)
        
        # Align indices to your desired time index
        circ_result.index = index[:len(circ_result)]
        lin_result.index = index[:len(lin_result)]
        circ_noise_df.index = index[:len(circ_noise_df)]
        last_wd_df.index = index[:len(last_wd_df)]
        
        # Store in nested dict
        WD_trial_run_dict[trial][run_id] = {
            'circular': circ_result,
            'linear': lin_result,
            'circular_noise': circ_noise_df,
            'baseline': last_wd_df
        }
    
    print(f"✅ Successfully processed trial: {trial}")


In [ ]:
WD_trial_run_dict['endocytosis2_baseline_zero_diffusion_tension_output']['0000']['circular']

In [ ]:
tensions = ['10', '150', '1000', '10000']
run_ids = ['0000', '0001', '0002', '0003']
trial_labels = ['Full', '3/4', '1/2', '1/4']  # friendly names
palette = sns.color_palette("viridis", n_colors=len(WD_trial_run_dict))

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False, sharey=True)
axes = axes.flatten()

for i, run_id in enumerate(run_ids):
    ax = axes[i]
    
    for j, trial in enumerate(WD_trial_run_dict.keys()):
        if run_id not in WD_trial_run_dict[trial]:
            continue
        
        circ_wd = WD_trial_run_dict[trial][run_id]['circular']
        median = circ_wd['Median']
        lower = circ_wd['Lower Error']
        upper = circ_wd['Upper Error']
        
        # Plot median line
        ax.plot(median.index, median.values, color=palette[j], label=trial_labels[j])
        
        # Shade between Lower and Upper Error
        ax.fill_between(median.index, lower, upper, color=palette[j], alpha=0.3)
    
    ax.set_title(f"{tensions[i]} pN/um")
    ax.set_xlabel("Time")
    ax.set_ylabel("Circular WD")
    ax.grid(True)
    ax.legend(title="Trial")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Setup ---
tensions = run_ids  # ['0000', '0001', '0002', '0003']
tension_labels = ['10', '150', '1000', '10000']
trial_labels = ['Full', '3/4', '1/2', '1/4']  # friendly names
palette = sns.color_palette("magma", n_colors=len(tensions))
final_time = max(next(iter(WD_trial_run_dict.values()))[tensions[0]]['circular']['Median'].index)

# --- Prepare figure ---
fig, ax = plt.subplots(figsize=(8, 6))

for i, tension in enumerate(tensions):
    median_vals = []
    q25_vals = []
    q75_vals = []
    
    for j, trial in enumerate(WD_trial_run_dict.keys()):
        if tension not in WD_trial_run_dict[trial]:
            median_vals.append(np.nan)
            q25_vals.append(np.nan)
            q75_vals.append(np.nan)
            continue
        
        circ_wd = WD_trial_run_dict[trial][tension]['circular']
        median_series = circ_wd['Median'].reindex([final_time]).fillna(method='ffill')
        lower_series = circ_wd['Lower Error'].reindex([final_time]).fillna(method='ffill')
        upper_series = circ_wd['Upper Error'].reindex([final_time]).fillna(method='ffill')
        
        median_vals.append(median_series.values[0])
        q25_vals.append(lower_series.values[0])
        q75_vals.append(upper_series.values[0])
    
    # Compute asymmetric error bars
    yerr_lower = np.array(median_vals) - np.array(q25_vals)
    yerr_upper = np.array(q75_vals) - np.array(median_vals)
    yerr = [yerr_lower, yerr_upper]

    # Plot median with error bars
    ax.errorbar(trial_labels, median_vals, yerr=yerr, fmt='o-', color=palette[i],
                elinewidth=2, capsize=5, markersize=8, label=f"{tension_labels[i]} pN/μm")

# --- Labels and formatting ---
ax.set_xlabel("Arp2/3 Distribution")
ax.set_ylabel("Final Circular WD")
ax.set_title("Final Circular WD Across Arp2/3 Distributions")
ax.grid(True)
ax.legend(title="Tension")
plt.tight_layout()
plt.show()


In [ ]:

# --- Setup ---
run_ids_filtered = ['0001', '0003']             # only the two run IDs
tension_labels_filtered = ['150', '10000']      # corresponding tension labels
trial_labels = ['Full', '3/4', '1/2', '1/4']   # friendly names
palette = sns.color_palette("magma", n_colors=len(run_ids_filtered))
final_time = max(next(iter(WD_trial_run_dict.values()))[run_ids_filtered[0]]['circular']['Median'].index)

# --- Prepare figure ---
fig, ax = plt.subplots(figsize=(8, 6))

for i, tension in enumerate(run_ids_filtered):
    median_vals = []
    q25_vals = []
    q75_vals = []
    
    for trial in WD_trial_run_dict.keys():
        if tension not in WD_trial_run_dict[trial]:
            median_vals.append(np.nan)
            q25_vals.append(np.nan)
            q75_vals.append(np.nan)
            continue
        
        circ_wd = WD_trial_run_dict[trial][tension]['circular']
        median_series = circ_wd['Median'].reindex([final_time]).fillna(method='ffill')
        lower_series = circ_wd['Lower Error'].reindex([final_time]).fillna(method='ffill')
        upper_series = circ_wd['Upper Error'].reindex([final_time]).fillna(method='ffill')
        
        median_vals.append(median_series.values[0])
        q25_vals.append(lower_series.values[0])
        q75_vals.append(upper_series.values[0])
    
    # Compute asymmetric error bars
    yerr_lower = np.array(median_vals) - np.array(q25_vals)
    yerr_upper = np.array(q75_vals) - np.array(median_vals)
    yerr = [yerr_lower, yerr_upper]

    # Plot median with error bars
    ax.errorbar(trial_labels, median_vals, yerr=yerr, fmt='o-', color=palette[i],
                elinewidth=2, capsize=5, markersize=8, label=f"{tension_labels_filtered[i]}")

# --- Labels and formatting ---
ax.set_xlabel("Arp2/3 Distribution", fontsize=16)
ax.set_ylabel("Final Circular WD", fontsize=16)
#ax.set_title("Final Circular WD Across Selected Arp2/3 Distributions")
ax.grid(True)
ax.legend(title="Tension (pN/μm)")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

run_ids = ['0000', '0001', '0002', '0003']
trial_labels = ['Full', '3/4', '1/2', '1/4']  # friendly names
palette = sns.color_palette("magma", n_colors=len(run_ids))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, trial in enumerate(WD_trial_run_dict.keys()):
    ax = axes[i]
    
    for j, run_id in enumerate(run_ids):
        if run_id not in WD_trial_run_dict[trial]:
            continue
        
        circ_wd = WD_trial_run_dict[trial][run_id]['circular']
        median = circ_wd['Median']
        lower = circ_wd['Lower Error']
        upper = circ_wd['Upper Error']
        
        # Plot median line
        ax.plot(median.index, median.values, color=palette[j], label=run_ids[j])
        
        # Shade between Lower and Upper Error
        ax.fill_between(median.index, lower, upper, color=palette[j], alpha=0.3)
    
    # Set subplot titles and labels
    ax.set_title(trial_labels[i])  # or map to friendly names if desired
    ax.set_xlabel("Time")
    ax.set_ylabel("Circular WD")
    ax.grid(True)
    ax.legend(title="Run ID")

plt.tight_layout()
plt.show()


In [ ]:
WD_trial_run_dict_size = {}
index = np.round(np.arange(0, 15.0, 0.1), 1)  # desired time index

for trial in zero_trials_tension:
    # Skip if no branched_actin_bound_points entry
    if trial not in branched_actin_bound_points:
        WD_trial_run_dict_size[trial] = {}
        continue

    subdict = branched_actin_bound_points[trial]
    run_column = subdict['run']

    # Initialize nested dict for this trial
    WD_trial_run_dict_size[trial] = {}

    for run_id in unique_runs[trial]:
        # Select rows for this run_id
        mask = run_column.apply(lambda x: run_id == x.split("tension")[1].split("-")[0])
        df_run = subdict[mask].copy()
        if df_run.empty:
            continue

        # --- Compute circular directions ---
        ypos = df_run['ypos_recalibrated'].dropna()
        xpos = df_run['xpos_recalibrated'].dropna()
        dir_vals = np.arctan2(ypos, xpos)
        dir_circle = (np.pi + dir_vals) / (2 * np.pi)
        df_run['direction'] = dir_circle
        length = np.sum(~np.isnan(dir_vals))

        # --- Group by run/time ---
        grouped_directions = df_run.groupby(['run', 'time']).direction.apply(np.array)
        grouped_lengths = grouped_directions.apply(lambda x: np.sum(~np.isnan(x)))
        dir_df = pd.DataFrame({'direction': grouped_directions, 'length': grouped_lengths})

        # --- Quantile-based bin edges ---
        n_bins = 30
        lengths = dir_df['length'].values
        quantile_edges = np.unique(np.quantile(lengths, np.linspace(0, 1, n_bins + 1)))
        if len(quantile_edges) <= 2:
            print(f"⚠️  Collapsed quantiles for trial {trial}, run {run_id} — all lengths nearly identical?")


        # --- Compute Wasserstein distances ---
        result = circle_size_WD(dir_df, quantile_edges)

        # Align to target index length if possible
        #result.index = index[:len(result)]

        # --- Store result ---
        WD_trial_run_dict_size[trial][run_id] = result

    print(f"Successfully processed trial: {trial}")


In [ ]:
WD_trial_run_dict_size['endocytosis2_baseline_zero_diffusion_tension_output']['0000']

In [ ]:

run_ids = ['0000', '0001', '0002', '0003']
run_labels = ['Full', '3/4', '1/2', '1/4']
palette = sns.color_palette("viridis", n_colors=len(zero_trials_tension))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, run_id in enumerate(run_ids):
    ax = axes[i]
    for j, trial in enumerate(zero_trials_tension):
        if run_id not in WD_trial_run_dict_size[trial]:
            continue

        circ_wd = WD_trial_run_dict_size[trial][run_id]
        median = circ_wd['Median']
        lower = circ_wd['Lower Error']
        upper = circ_wd['Upper Error']
        
        ax.plot(circ_wd.index, median, color=palette[j], label=run_labels[j])
        ax.fill_between(circ_wd.index, lower, upper, color=palette[j], alpha=0.2)

    ax.set_title(f"Run ID: {run_ids[i]}")
    ax.set_xlabel("Size")
    ax.set_ylabel("Circular WD")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# --- Setup ---
run_ids = ['0000', '0001', '0002', '0003']
run_labels = ['Full', '3/4', '1/2', '1/4']
palette = sns.color_palette("magma", n_colors=len(run_ids))

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False, sharey=True)
axes = axes.flatten()

# --- Loop over trials (one per subplot) ---
for i, trial in enumerate(zero_trials_tension):
    ax = axes[i]
    for j, run_id in enumerate(run_ids):
        if run_id not in WD_trial_run_dict_size[trial]:
            continue

        circ_wd = WD_trial_run_dict_size[trial][run_id]
        median = circ_wd['Median']
        lower = circ_wd['Lower Error']
        upper = circ_wd['Upper Error']

        ax.plot(circ_wd.index, median, color=palette[j], label=f"Run {run_id}")
        ax.fill_between(circ_wd.index, lower, upper, color=palette[j], alpha=0.2)

    ax.set_title(f"Trial: {run_labels[i]}")
    ax.set_xlabel("Size")
    ax.set_ylabel("Circular WD")
    ax.grid(True)
    ax.legend(title="Run ID", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import ot

# --- Dictionary to store WD time series per trial/run ---
WD_trial_run_timeseries = {}

for trial in zero_trials_tension:
    WD_trial_run_timeseries[trial] = {}

    if trial not in branched_actin_bound_points:
        print(f"⚠️ Skipping trial {trial}: not in branched_actin_bound_points")
        continue

    subdict = branched_actin_bound_points[trial]
    run_column = subdict['run'].astype(str)

    print(f"\n🧪 Processing trial: {trial}")
    print("  Unique tension IDs expected:", unique_runs[trial])

    for run_id in unique_runs[trial]:
        # --- Select rows for this tension ID (middle part of run string) ---
        mask = run_column.apply(lambda x: run_id in x.split("tension")[1].split("-")[0])
        df_run = subdict[mask].copy()

        if df_run.empty:
            print(f"  ⚠️ No rows for tension ID {run_id}")
            continue

        print(f"  ➡️ Run {run_id}: {len(df_run)} rows")

        # Compute circular direction (0–1 normalized)
        ypos = df_run['ypos_recalibrated']
        xpos = df_run['xpos_recalibrated']

        df_run['confinement_force'] = (-0.399 - df_run['zpos']) * 1e4
        df_run['direction'] = (np.pi + np.arctan2(ypos, xpos)) / (2 * np.pi)
 

        # --- Group by internal run and time ---
        grouped = df_run.groupby(['run', 'time'])
        dir_list = grouped['direction'].apply(lambda x: x.dropna().values).to_dict()
        lengths = grouped.size().to_dict()
        bud_zpos_series = grouped['bud_internalization'].mean().to_dict()

        if not dir_list:
            print(f"   ⚠️ No valid (run, time) groups for tension {run_id}")
            continue

        # --- Reference uniform direction set ---
        ref_large = np.random.uniform(-np.pi, np.pi, 10000)
        ref_shift_large = (np.pi + ref_large) / (2 * np.pi)

        wd_series, run_series, times = [], [], []

        # --- Identify last time point per *internal run* ---
        internal_runs = df_run['run'].unique()

        for internal_run in internal_runs:
            run_times = [t for (r, t) in dir_list.keys() if r == internal_run]
            if not run_times:
                continue
            last_time = max(run_times)

            for (r, t), directions in dir_list.items():
                if r != internal_run:
                    continue

                n = lengths[(r, t)]
                run_series.append(r)
                times.append(t)

                if len(directions) == 0 or n <= 0:
                    wd_series.append(np.nan)
                    continue

                # --- Compute Wasserstein distances ---
                if t == 16:
                    wd_corrected_repeats = []
                    for _ in range(1):
                        ref_mini = np.random.uniform(-np.pi, np.pi, n)
                        ref_mini_shift = (np.pi + ref_mini) / (2 * np.pi)
                        wd = ot.wasserstein_circle(directions, ref_shift_large, p=1)
                        wd_mini = ot.wasserstein_circle(ref_mini_shift, ref_shift_large, p=1)
                        wd_corrected_repeats.append(wd - wd_mini)
                    wd_series.append(np.mean(wd_corrected_repeats))
                else:
                    wd = ot.wasserstein_circle(directions, ref_shift_large, p=1)
                    wd_series.append(wd)

        if not run_series:
            print(f"   ⚠️ No valid WD data computed for tension {run_id}")
            continue

        # --- Store WD series as DataFrame ---
        WD_trial_run_timeseries[trial][run_id] = (
            pd.DataFrame({
                'Run': run_series,
                'Time': times,
                'WD': wd_series,
                'ClusterSize': [lengths[(r, t)] for (r, t) in dir_list.keys()],
                'Confinement_Force': [df_run.loc[(df_run['run']==r) & (df_run['time']==t), 'confinement_force'].sum() for (r,t) in dir_list.keys()],
                'Internalization': [bud_zpos_series[(r, t)] for (r, t) in dir_list.keys()],
            })
            .sort_values('Time')
            .set_index('Time')
        )

        print(f"   ✅ Stored WD time series for tension {run_id}: {len(WD_trial_run_timeseries[trial][run_id])} points")

    print(f"✅ Finished trial: {trial}")

print("✅ All trials done")


In [ ]:
tensions  = ['150', '10000']

# --- Map trial names to short labels ---
label_map = {
    'endocytosis2_baseline_zero_diffusion_output': 'Full',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output': '3/4',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output': '1/2',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output': '1/4',
    'endocytosis2_baseline_zero_diffusion_tension_output': 'Full',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output': '3/4',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output': '1/2',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output': '1/4'
}

run_ids = ['0001', '0003']

fig, axes = plt.subplots(1, len(run_ids), figsize=(12, 5), sharex=True, sharey=True)

for ax, run_id in zip(axes, run_ids):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        trial_label = label_map.get(trial, trial)

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        # Group by internal run
        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            # Last time point in this run
            last_time = df_run.index.max()

            # WD at last time point
            wd_last = df_run.loc[last_time, 'WD']
            if isinstance(wd_last, (np.ndarray, list)):
                wd_last = np.nanmean(wd_last)

            # Internalization at last time point
            internalization_last = df_run.loc[last_time, 'Internalization']
            if isinstance(internalization_last, (np.ndarray, list)):
                internalization_last = np.nanmean(internalization_last)

            plot_data.append({
                'trial_label': trial_label,
                'WD_last': wd_last,
                'bud_z_last': internalization_last * 1000  # scale if needed
            })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    sns.scatterplot(
        data=plot_df,
        x='WD_last',
        y='bud_z_last',
        hue='trial_label',
        palette='viridis',
        s=80,
        ax=ax
    )

    ax.set_xlabel("Final Corrected WD")
    ax.set_ylabel("Final Internalization" if ax == axes[0] else "")
    ax.set_title(f"{tensions[run_ids.index(run_id)]} pN/μm")
    ax.grid(True, alpha=0.3)
    ax.legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:

tensions = ['150', '10000']

# --- Map trial names to short labels ---
label_map = {
    'endocytosis2_baseline_zero_diffusion_output': 'Full',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output': '3/4',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output': '1/2',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output': '1/4',
    'endocytosis2_baseline_zero_diffusion_tension_output': 'Full',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output': '3/4',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output': '1/2',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output': '1/4'
}

run_ids = ['0001', '0003']

fig, axes = plt.subplots(1, len(run_ids), figsize=(12, 5), sharex=True, sharey=True)

for ax, run_id in zip(axes, run_ids):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        trial_label = label_map.get(trial, trial)

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        # Group by internal run
        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            # Ensure Internalization is scalar
            internalization_series = df_run['Internalization'].apply(
                lambda x: np.nanmean(x) if isinstance(x, (np.ndarray, list)) else x
            )

            # 95th percentile of internalization
            internalization_95 = np.nanpercentile(internalization_series, 95)

            # Time points where internalization >= 95th percentile
            high_times = internalization_series[internalization_series >= internalization_95].index
            if len(high_times) == 0:
                continue

            # Average WD over those time points
            wd_values = df_run.loc[high_times, 'WD'].apply(
                lambda x: np.nanmean(x) if isinstance(x, (np.ndarray, list)) else x
            )
            wd_avg = wd_values.mean()
            internalization_avg = internalization_series.loc[high_times].mean() * 1000  # scale to nm

            plot_data.append({
                'trial_label': trial_label,
                'WD_avg': wd_avg,
                'bud_z_avg': internalization_avg
            })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    sns.scatterplot(
        data=plot_df,
        x='WD_avg',
        y='bud_z_avg',
        hue='trial_label',
        palette='viridis',
        s=80,
        ax=ax
    )

    ax.set_xlabel("Average WD at >= 95th % Internalization")
    ax.set_ylabel("95th Percentile Internalization (nm)" if ax == axes[0] else "")
    ax.set_title(f"{tensions[run_ids.index(run_id)]} pN/μm")
    ax.grid(True, alpha=0.3)
    ax.legend(title='Arp2/3 Distribution')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(1, len(run_ids), figsize=(12, 5), sharex=True, sharey=True)

tensions  = ['150', '10000']

# --- Collect all cluster sizes for global color scale ---
all_cluster_sizes = []
for run_id in run_ids:
    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue
        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue
            internalization_series = df_run['Internalization'].apply(
                lambda x: np.nanmean(x) if isinstance(x, (list, np.ndarray)) else x
            )
            internalization_95 = np.nanpercentile(internalization_series, 95)
            high_times = internalization_series[internalization_series >= internalization_95].index
            if len(high_times) == 0:
                continue
            cluster_vals = df_run.loc[high_times, 'ClusterSize']
            all_cluster_sizes.extend(cluster_vals.dropna().values)

vmin = np.min(all_cluster_sizes)
vmax = np.max(all_cluster_sizes)

# --- Plot ---
sc = None  # store last scatter for colorbar
for ax, run_id in zip(axes, run_ids):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            internalization_series = df_run['Internalization'].apply(
                lambda x: np.nanmean(x) if isinstance(x, (list, np.ndarray)) else x
            )
            internalization_95 = np.nanpercentile(internalization_series, 95)
            high_times = internalization_series[internalization_series >= internalization_95].index
            if len(high_times) == 0:
                continue

            wd_values = df_run.loc[high_times, 'WD'].apply(
                lambda x: np.nanmean(x) if isinstance(x, (list, np.ndarray)) else x
            )
            cluster_values = df_run.loc[high_times, 'ClusterSize']

            wd_avg = wd_values.mean()
            cluster_avg = cluster_values.mean()
            internalization_avg = internalization_series.loc[high_times].mean() * 1000  # scale to nm

            plot_data.append({
                'WD_avg': wd_avg,
                'Internalization_avg': internalization_avg,
                'cluster_avg': cluster_avg
            })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    sc = ax.scatter(
        x=plot_df['WD_avg'],
        y=plot_df['Internalization_avg'],
        c=plot_df['cluster_avg'],
        cmap='magma',
        s=80,
        edgecolor='k',
        vmin=vmin,
        vmax=vmax
    )

    ax.set_xlabel("WD (avg ≥ 95th % Internalization)")
    ax.set_ylabel("Internalization (nm)" if ax == axes[0] else "")
    ax.set_title(f"{tensions[run_ids.index(run_id)]} pN/μm")
    ax.grid(True, alpha=0.3)

# --- Add small inset colorbar inside the last subplot ---
ax = axes[-1]
# inset_axes arguments: [x0, y0, width, height] in axes fraction coordinates
cax = inset_axes(ax, width="50%", height="5%", loc='upper right', borderpad=1)
cbar = fig.colorbar(sc, cax=cax, orientation='horizontal')
cbar.set_label('# Model Points', fontsize=8)
cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(run_ids), figsize=(12, 5), sharex=True, sharey=True)

for ax, run_id in zip(axes, run_ids):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        trial_label = label_map.get(trial, trial)

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        # Group by internal run
        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            # 95th percentile of internalization
            internalization_95 = np.nanpercentile(df_run['Internalization'], 95)

            # Time points where internalization >= 95th percentile
            high_times = df_run.index[df_run['Internalization'] >= internalization_95]
            if len(high_times) == 0:
                continue

            # Average WD over those time points
            wd_values = df_run.loc[high_times, 'WD']
            if isinstance(wd_values.iloc[0], (np.ndarray, list)):
                wd_values = wd_values.apply(np.nanmean)

            wd_avg = wd_values.mean()
            internalization_avg = df_run.loc[high_times, 'Internalization'].mean()

            plot_data.append({
                'trial_label': trial_label,
                'WD_last': wd_avg,
                'bud_z_last': internalization_avg
            })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    sns.scatterplot(
        data=plot_df,
        x='WD_last',
        y='bud_z_last',
        hue='trial_label',
        palette='viridis',
        s=80,
        ax=ax
    )

    ax.set_xlabel("WD", fontsize=15)
    if ax == axes[0]:
        ax.set_ylabel("95th Percentile Internalization (nm)", fontsize=15)
    ax.set_title(f"{tensions[run_ids.index(run_id)]} pN/μm", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:
run_ids = ['0000','0001','0002','0003']
tensions  = ['10', '150', '1000', '10000']

fig, axes = plt.subplots(1, len(run_ids), figsize=(18, 5), sharex=True, sharey=True)

for i, (ax, run_id) in enumerate(zip(axes, run_ids)):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        trial_label = label_map.get(trial, trial)

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        # Group by internal run
        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            # 95th percentile of internalization
            internalization_95 = np.nanpercentile(df_run['Internalization'], 95)

            # Time points where internalization >= 95th percentile
            high_times = df_run.index[df_run['Internalization'] >= internalization_95]
            if len(high_times) == 0:
                continue

            # Average WD over those time points
            wd_values = df_run.loc[high_times, 'WD']
            if isinstance(wd_values.iloc[0], (np.ndarray, list)):
                wd_values = wd_values.apply(np.nanmean)

            wd_avg = wd_values.mean()
            internalization_avg = internalization_95

            plot_data.append({
                'trial_label': trial_label,
                'WD_last': wd_avg,
                'bud_z_last': internalization_avg
            })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    sns.scatterplot(
        data=plot_df,
        x='WD_last',
        y='bud_z_last',
        hue='trial_label',
        palette='viridis',
        s=80,
        ax=ax
    )

    ax.set_xlabel("Corrected WD", fontsize=15)
    ax.set_ylabel("95th Percentile Internalization (nm)" if ax == axes[0] else "", fontsize=15)
    ax.set_title(f"{tensions[i]} pN/μm", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:
run_ids = ['0000','0001','0002','0003']
tensions = ['10', '150', '1000', '10000']

# --- First pass: collect all valid WD and Internalization values ---
all_wd = []
all_bud = []

for run_id in run_ids:
    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            internalization_95 = np.nanpercentile(df_run['Internalization'], 95)
            high_times = df_run.index[df_run['Internalization'] >= internalization_95]
            if len(high_times) == 0:
                continue

            wd_values = df_run.loc[high_times, 'WD']
            if isinstance(wd_values.iloc[0], (np.ndarray, list)):
                wd_values = wd_values.apply(np.nanmean)

            wd_avg = wd_values.mean()

            if not np.isnan(wd_avg) and not np.isnan(internalization_95):
                all_wd.append(wd_avg)
                all_bud.append(internalization_95 * 1000)  # scale for plotting

# --- Global axis limits ---
if all_wd and all_bud:
    wd_min, wd_max = np.min(all_wd), np.max(all_wd)
    bud_min, bud_max = np.min(all_bud), np.max(all_bud)
else:
    wd_min, wd_max = 0, 1
    bud_min, bud_max = 0, 1

# --- Plot one figure per tension/run_id ---
for i, run_id in enumerate(run_ids):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        trial_label = label_map.get(trial, trial)

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            # 95th percentile of internalization
            internalization_95 = np.nanpercentile(df_run['Internalization'], 95)
            high_times = df_run.index[df_run['Internalization'] >= internalization_95]
            if len(high_times) == 0:
                continue

            # Average WD over those time points
            wd_values = df_run.loc[high_times, 'WD']
            if isinstance(wd_values.iloc[0], (np.ndarray, list)):
                wd_values = wd_values.apply(np.nanmean)

            wd_avg = wd_values.mean()

            if not np.isnan(wd_avg) and not np.isnan(internalization_95):
                plot_data.append({
                    'trial_label': trial_label,
                    'WD_last': wd_avg,
                    'bud_z_last': internalization_95 * 1000  # use actual 95th percentile
                })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    plt.figure(figsize=(7, 6))
    sns.scatterplot(
        data=plot_df,
        x='WD_last',
        y='bud_z_last',
        hue='trial_label',
        palette='viridis',
        s=80
    )

    plt.xlabel("Corrected WD", fontsize=15)
    plt.ylabel("95th Percentile Internalization (nm)", fontsize=15)
    plt.title(f"Tension: {tensions[i]} pN/μm", fontsize=16)
    plt.xlim(-0.01, 0.20)
    plt.ylim(-10, 200)
    plt.grid(True, alpha=0.3)
    plt.legend(title='Arp2/3 Distribution')
    plt.tight_layout()
    plt.show()


In [ ]:
run_ids = ['0000','0001','0002','0003']
tensions = ['10', '150', '1000', '10000']

# --- Fixed axis limits ---
xlim = (-0.01, 0.20)
ylim = (-10, 200)

# Determine global cluster_size min/max for consistent color scale
all_cluster_sizes = []

for run_id in run_ids:
    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            internalization_95 = np.nanpercentile(df_run['Internalization'], 95)
            high_times = df_run.index[df_run['Internalization'] >= internalization_95]
            if len(high_times) == 0:
                continue

            # Assuming 'ClusterSize' column exists
            cluster_values = df_run.loc[high_times, 'ClusterSize']
            if isinstance(cluster_values.iloc[0], (np.ndarray, list)):
                cluster_values = cluster_values.apply(np.nanmean)
            cluster_avg = cluster_values.mean()
            if not np.isnan(cluster_avg):
                all_cluster_sizes.append(cluster_avg)

if all_cluster_sizes:
    cluster_min, cluster_max = np.min(all_cluster_sizes), np.max(all_cluster_sizes)
else:
    cluster_min, cluster_max = 0, 1

# --- Plotting ---
for i, run_id in enumerate(run_ids):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            internalization_95 = np.nanpercentile(df_run['Internalization'], 95)
            high_times = df_run.index[df_run['Internalization'] >= internalization_95]
            if len(high_times) == 0:
                continue

            wd_values = df_run.loc[high_times, 'WD']
            if isinstance(wd_values.iloc[0], (np.ndarray, list)):
                wd_values = wd_values.apply(np.nanmean)
            wd_avg = wd_values.mean()

            cluster_values = df_run.loc[high_times, 'ClusterSize']
            if isinstance(cluster_values.iloc[0], (np.ndarray, list)):
                cluster_values = cluster_values.apply(np.nanmean)
            cluster_avg = cluster_values.mean()

            if not np.isnan(wd_avg) and not np.isnan(internalization_95) and not np.isnan(cluster_avg):
                plot_data.append({
                    'WD_last': wd_avg,
                    'bud_z_last': internalization_95 * 1000,  # actual 95th percentile
                    'cluster_size': cluster_avg
                })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    plt.figure(figsize=(7, 6))
    scatter = plt.scatter(
        x=plot_df['WD_last'],
        y=plot_df['bud_z_last'],
        c=plot_df['cluster_size'],
        cmap='magma',  # or 'magma' for Magma global color scale
        s=80,
        vmin=cluster_min,
        vmax=cluster_max
    )
    cbar = plt.colorbar(scatter)
    cbar.set_label("Actin Amount", fontsize=12)

    plt.xlabel("Corrected WD", fontsize=15)
    plt.ylabel("95th Percentile Internalization (nm)", fontsize=15)
    plt.title(f"Tension: {tensions[i]} pN/μm", fontsize=16)
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.grid(True, alpha=0.3)

    filename = f"internalization_vs_WD_cluster_{tensions[i]}pN.pdf"
    plt.savefig(filename)
    print(f"Saved {filename}")
    plt.show()


In [ ]:


# Map trial names to short labels
label_map = {
    'endocytosis2_baseline_zero_diffusion_output': 'Full',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output': '3/4',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output': '1/2',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output': '1/4',
    'endocytosis2_baseline_zero_diffusion_tension_output': 'Full',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output': '3/4',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output': '1/2',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output': '1/4'
}

# Loop over RUN_IDs (tensions)
for run_id in ['0001', '0003']:  # example tensions
    plt.figure(figsize=(6,5))
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries or trial not in branched_actin_bound_points:
            continue
        trial_label = label_map.get(trial, trial)

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        # Loop over internal runs
        trial_df = branched_actin_bound_points[trial].dropna(subset=['bud_zpos']).sort_values(['run', 'time'])
        for internal_run_id, internal_run_df in trial_df.groupby('run'):
            if internal_run_id not in wd_df['Run'].values:
                continue

            # Flatten WD for this internal run
            wd_values = wd_df.loc[wd_df['Run'] == internal_run_id, 'WD'].values
            flattened_wd = np.hstack([w if isinstance(w, (list, np.ndarray)) else [w] for w in wd_values])
            avg_wd = np.nanmean(flattened_wd)

            # Compute bud_zpos displacement ratio
            z_series = internal_run_df['bud_zpos'].values.astype(float)
            displacements = np.diff(z_series)
            n_pos = np.sum(displacements > 0)
            n_neg = np.sum(displacements < 0)
            if n_pos == 0:
                continue
            ratio = n_neg / n_pos

            plot_data.append({
                'trial_label': trial_label,
                'avg_WD': avg_wd,
                'z_bias_ratio': ratio
            })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    # Scatter plot: now x = ratio, y = avg_WD
    sns.scatterplot(
        data=plot_df,
        x='z_bias_ratio',
        y='avg_WD',
        hue='trial_label',
        palette='viridis',
        s=80
    )

    plt.ylabel("Average WD (flattened)")
    plt.xlabel("Sum(Δbud_zpos < 0) / Sum(Δbud_zpos > 0)")
    plt.title(f"Run {run_id} (Tension) - WD vs bud_zpos ratio")
    plt.grid(True, alpha=0.3)
    plt.legend(title='Trial')
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Loop over RUN_IDs (tensions)
for run_id in ['0001', '0003']:
    plt.figure(figsize=(6,5))
    plot_data = []

    # Collect all internal runs across all trials
    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries or trial not in branched_actin_bound_points:
            continue

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        trial_df = branched_actin_bound_points[trial].dropna(subset=['bud_zpos']).sort_values(['run', 'time'])
        for internal_run_id, internal_run_df in trial_df.groupby('run'):
            if internal_run_id not in wd_df['Run'].values:
                continue

            # Flatten WD
            wd_values = wd_df.loc[wd_df['Run'] == internal_run_id, 'WD'].values
            flattened_wd = np.hstack([w if isinstance(w, (list, np.ndarray)) else [w] for w in wd_values])
            avg_wd = np.nanmean(flattened_wd)

            # Compute bud_zpos displacement ratio
            z_series = internal_run_df['bud_zpos'].values.astype(float)
            displacements = np.diff(z_series)
            n_pos = np.sum(displacements > 0)
            n_neg = np.sum(displacements < 0)
            if n_pos == 0:
                continue
            ratio = n_neg / n_pos

            plot_data.append({'avg_WD': avg_wd, 'z_bias_ratio': ratio})

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    # Scatter plot (all points in black)
    sns.scatterplot(
        data=plot_df,
        x='z_bias_ratio',
        y='avg_WD',
        color='black',
        s=80
    )

    # Linear fit
    X = plot_df['z_bias_ratio'].values.reshape(-1,1)
    Y = plot_df['avg_WD'].values
    linreg = LinearRegression()
    linreg.fit(X, Y)
    Y_pred = linreg.predict(X)
    r2 = r2_score(Y, Y_pred)

    # Plot fit line
    plt.plot(plot_df['z_bias_ratio'], Y_pred, color='red', linestyle='--',
             label=f'Linear fit, $R^2$={r2:.2f}')

    plt.xlabel("Sum(Δbud_zpos < 0) / Sum(Δbud_zpos > 0)")
    plt.ylabel("Average WD (flattened)")
    plt.title(f"Run {run_id} - WD vs bud_zpos ratio")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(run_ids), figsize=(12, 5), sharex=True, sharey=True)

for ax, run_id in zip(axes, run_ids):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        trial_label = label_map.get(trial, trial)

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            last_time = df_run.index.max()

            wd_last = df_run.loc[last_time, 'WD']
            if isinstance(wd_last, (list, np.ndarray)):
                wd_last = np.nanmean(wd_last)

            force_last = df_run.loc[last_time, 'Confinement_Force']
            if isinstance(force_last, (list, np.ndarray)):
                force_last = np.nanmean(force_last)

            plot_data.append({
                'trial_label': trial_label,
                'WD_last': wd_last,
                'Confinement_Force_last': np.abs(force_last)
            })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    sns.scatterplot(
        data=plot_df,
        x='WD_last',
        y='Confinement_Force_last',
        hue='trial_label',
        palette='viridis',
        s=80,
        ax=ax
    )

    ax.set_xlabel("Final WD", fontsize=15)
    ax.set_ylabel("Final Confinement Force" if ax == axes[0] else "", fontsize=15)
    ax.set_title(f"{tensions[run_ids.index(run_id)]} pN/μm")
    ax.grid(True, alpha=0.3)
    ax.legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(run_ids), figsize=(12, 5), sharex=True, sharey=True)

for ax, run_id in zip(axes, run_ids):
    plot_data = []

    for trial in zero_trials_tension:
        if trial not in WD_trial_run_timeseries:
            continue
        trial_label = label_map.get(trial, trial)

        wd_df = WD_trial_run_timeseries[trial].get(run_id, None)
        if wd_df is None or wd_df.empty:
            continue

        for internal_run_id in wd_df['Run'].unique():
            df_run = wd_df[wd_df['Run'] == internal_run_id]
            if df_run.empty:
                continue

            # 95th percentile of confinement force
            force_95 = np.nanpercentile(df_run['Confinement_Force'], 95)

            # Time points where confinement force >= 95th percentile
            high_times = df_run.index[df_run['Confinement_Force'] >= force_95]
            if len(high_times) == 0:
                continue

            # Average WD over those high-confinement times
            wd_values = df_run.loc[high_times, 'WD']
            if isinstance(wd_values.iloc[0], (list, np.ndarray)):
                wd_values = wd_values.apply(np.nanmean)
            wd_avg = wd_values.mean()

            plot_data.append({
                'trial_label': trial_label,
                'WD_avg_high_force': wd_avg,
                'Force_95': np.abs(force_95)
            })

    if not plot_data:
        continue

    plot_df = pd.DataFrame(plot_data)

    sns.scatterplot(
        data=plot_df,
        x='WD_avg_high_force',
        y='Force_95',
        hue='trial_label',
        palette='viridis',
        s=80,
        ax=ax
    )

    ax.set_yscale('log')
    ax.set_xlabel("Average WD at high Confinement Force", fontsize=15)
    ax.set_ylabel("95th Percentile Confinement Force", fontsize=15 if ax == axes[0] else 0)
    ax.set_title(f"{tensions[run_ids.index(run_id)]} pN/μm")
    ax.grid(True, alpha=0.3)
    ax.legend(title='Trial')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Labels for trials
plot_labels = ['Full', '3/4', '1/2', '1/4']

# Iterate over run_ids (tensions)
run_ids_sorted = sorted(list({rid for trial_dict in WD_trial_run_timeseries.values() 
                              for rid in trial_dict.keys()}))
colors = sns.color_palette("viridis", n_colors=len(WD_trial_run_timeseries))  # one color per trial

for run_id in run_ids_sorted:
    plt.figure(figsize=(8,5))
    
    for color, trial in zip(colors, zero_trials_tension):
        if run_id not in WD_trial_run_timeseries[trial]:
            continue
        
        df = WD_trial_run_timeseries[trial][run_id].copy()
        # Flatten WD if stored as lists
        wd_flat = [w if isinstance(w, (float, np.float64, int)) else (w[0] if len(w)>0 else np.nan) for w in df['WD']]
        cluster_size = df['ClusterSize'].values
        time = df.index.values
        
        plt.scatter(cluster_size, wd_flat, alpha=0.6, color=color, label=plot_labels[zero_trials_tension.index(trial)])
    
    plt.xlabel("Cluster Size")
    plt.ylabel("Wasserstein Distance (WD)")
    plt.title(f"Run_ID {run_id} – WD vs Cluster Size across Trials")
    plt.legend(title="Trial")
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
from scipy.stats import pearsonr
import pandas as pd

# Dictionary to store correlations per trial/tension/run
WD_trial_run_corr_summary = {}

for trial, tensions in WD_trial_run_timeseries.items():
    WD_trial_run_corr_summary[trial] = {}
    
    for tension, df in tensions.items():  # tension is run_id
        run_corrs = []
        per_col_run_corrs = {}  # store correlation per column run
        
        # Ensure WD and ClusterSize are numeric
        df['WD'] = pd.to_numeric(df['WD'], errors='coerce')
        df['ClusterSize'] = pd.to_numeric(df['ClusterSize'], errors='coerce')
        
        # Get unique column runs
        column_runs = df['Run'].unique() if 'Run' in df.columns else [tension]
        
        for col_run in column_runs:
            subdf = df[df['Run'] == col_run]
            wd = subdf['WD'].values
            size = subdf['ClusterSize'].values
            
            # Mask out NaNs
            mask = (~pd.isna(wd)) & (~pd.isna(size))
            wd = wd[mask]
            size = size[mask]
            
            if len(wd) >= 3:
                r, _ = pearsonr(size, wd)
                run_corrs.append(r)
                per_col_run_corrs[col_run] = r
            else:
                per_col_run_corrs[col_run] = np.nan
        
        # Summarize correlations for this run_id/tension
        if len(run_corrs) == 0:
            summary = {'median': np.nan, 'lower': np.nan, 'upper': np.nan}
        else:
            run_corrs = np.array(run_corrs)
            run_corrs = run_corrs[~np.isnan(run_corrs)]
            summary = {
                'median': np.median(run_corrs),
                'lower': np.percentile(run_corrs, 25),
                'upper': np.percentile(run_corrs, 75)
            }
        
        # Store both summary and per-column correlations
        WD_trial_run_corr_summary[trial][tension] = {
            'summary': summary,
            'per_run': per_col_run_corrs
        }

print("✅ Correlation summary computed and stored per column run for all trials/tensions")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Labels and tensions
plot_labels = ['Full', '3/4', '1/2', '1/4']
tensions = ['10', '150', '1000', '5000', '10000']  # match your run_ids

# --- Build DataFrame from summary dictionary ---
corr_rows = []
for trial, runs in WD_trial_run_corr_summary.items():
    trial_label = plot_labels[zero_trials_tension.index(trial)]
    for run_id, data in runs.items():
        summary = data['summary']
        corr_rows.append({
            'Trial': trial_label,
            'Run_ID': run_id,
            'Median': summary['median'],
            'Lower': summary['lower'],
            'Upper': summary['upper']
        })

corr_df = pd.DataFrame(corr_rows)

# Ensure correct x-axis order
corr_df['Trial'] = pd.Categorical(corr_df['Trial'], categories=plot_labels, ordered=True)

# Sort run_ids and assign colors
run_ids_sorted = sorted(corr_df['Run_ID'].unique())
colors = sns.color_palette("magma", n_colors=len(run_ids_sorted))

# --- Plot ---
plt.figure(figsize=(8,5))

for color, run_id in zip(colors, run_ids_sorted):
    subdf = corr_df[corr_df['Run_ID'] == run_id].sort_values('Trial')
    x = subdf['Trial']
    y = subdf['Median']
    y_lower = subdf['Lower']
    y_upper = subdf['Upper']

    plt.plot(x, y, marker='o', color=color, label=f'{tensions[int(run_id)]} pN/μm')
    plt.fill_between(x, y_lower, y_upper, color=color, alpha=0.3)

plt.axhline(0, color='gray', linestyle='--', lw=1)
plt.xlabel("Trial", fontsize=12)
plt.ylabel("Correlation (WD vs Cluster Size)", fontsize=12)
plt.title("WD vs Cluster Size Correlation per Run", fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Tension")
plt.tight_layout()
plt.show()


In [ ]:
corr_df

## 3C

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Setup ---
runs = run_ids  # e.g., ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
time_points = np.linspace(0.1, 15, 150).round(1)
cmap = sns.color_palette("viridis", n_colors=len(zero_trials_tension))  # one color per trial

# --- Prepare 2x2 subplots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# --- Loop over runs for subplots ---
for i, run_id in enumerate(runs):
    ax = axes[i]
    
    # Loop over trials (conditions)
    for j, trial in enumerate(zero_trials_tension):
        if run_id not in trial_run_dfs[trial]:
            continue
        df = trial_run_dfs[trial][run_id]
        
        # Select confined filaments
        df_force = df[df['zpos'] < -0.399].copy()
        df_force['confinement_force'] = (-0.399 - df_force['zpos']) * 1e4
        
        # Average per filament per run/time
        force_sum = df_force.groupby(['run', 'time'])['confinement_force'].sum()
        fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
        avg_force = force_sum / fiber_count.replace(0, np.nan)
        
        # Reindex to uniform time points per run
        avg_force_runs = []
        for run in avg_force.index.get_level_values('run').unique():
            run_series = avg_force.xs(run, level='run').reindex(time_points).fillna(method='ffill')
            avg_force_runs.append(run_series)
        avg_force_df = pd.concat(avg_force_runs, axis=1)
        
        # Median and IQR across runs
        median_force = avg_force_df.median(axis=1)
        q25_force = avg_force_df.quantile(0.25, axis=1)
        q75_force = avg_force_df.quantile(0.75, axis=1)
        
        # Plot median and shaded IQR
        ax.plot(time_points, median_force, color=cmap[j], label=plot_labels[j])
        ax.fill_between(time_points, q25_force, q75_force, color=cmap[j], alpha=0.3)
    
    ax.set_title(f"Run ID: {run_id}")
    ax.set_xlabel("Time")
    ax.set_ylabel("Confinement Force per Filament (pN/fil)")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
runs = run_ids  # e.g., ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']
cmap = sns.color_palette("viridis", n_colors=len(zero_trials_tension))  # one color per trial
n_bins = 15  # adjust number of bins

# --- Prepare 2x2 subplots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# --- Loop over runs for subplots ---
for i, run_id in enumerate(runs):
    ax = axes[i]

    for j, trial in enumerate(zero_trials_tension):
        if run_id not in trial_run_dfs[trial]:
            continue
        df = trial_run_dfs[trial][run_id]

        # Select confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

        # Number of confined filaments at each time
        df_confined['num_confined'] = df_confined.groupby('time')['fiber_id'].transform('nunique')

        # Force per filament
        df_confined['force_per_filament'] = df_confined['confinement_force']

        # --- Bin by number of confined filaments ---
        df_confined['bin'] = pd.qcut(df_confined['num_confined'], q=min(n_bins, len(df_confined)), duplicates='drop')

        # Compute median and IQR per bin
        binned_stats = df_confined.groupby('bin').agg(
            num_confined_median=('num_confined', 'median'),
            force_median=('force_per_filament', 'median'),
            force_q25=('force_per_filament', lambda x: np.percentile(x, 25)),
            force_q75=('force_per_filament', lambda x: np.percentile(x, 75))
        ).reset_index()

        if binned_stats.empty:
            continue

        # --- Plot median with IQR shading ---
        ax.plot(binned_stats['num_confined_median'], binned_stats['force_median'],
                color=cmap[j], lw=2, label=plot_labels[j])
        ax.fill_between(binned_stats['num_confined_median'],
                        binned_stats['force_q25'],
                        binned_stats['force_q75'],
                        color=cmap[j], alpha=0.3)

    ax.set_xlabel("Number of Confined Filaments")
    ax.set_ylabel("Confinement Force per Filament (pN/fil)")
    ax.set_title(f"Run ID: {run_id}")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:


# --- Setup ---
runs = run_ids  # e.g., ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
time_points = np.linspace(0.1, 15, 150).round(1)
cmap = sns.color_palette("viridis", n_colors=len(zero_trials_tension))  # one color per trial

# --- Prepare 2x2 subplots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# --- Loop over runs for subplots ---
for i, run_id in enumerate(runs):
    ax = axes[i]
    
    # Loop over trials (conditions)
    for j, trial in enumerate(zero_trials_tension):
        if run_id not in trial_run_dfs[trial]:
            continue
        df = trial_run_dfs[trial][run_id]
        
        # Select confined filaments
        df_force = df[df['zpos'] < -0.399].copy()
        
        # Count unique fibers per run/time
        fiber_count = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
        
        # Reindex to uniform time points per run
        fiber_count_runs = []
        for run in fiber_count.index.get_level_values('run').unique():
            run_series = fiber_count.xs(run, level='run').reindex(time_points).fillna(method='ffill')
            fiber_count_runs.append(run_series)
        fiber_count_df = pd.concat(fiber_count_runs, axis=1)
        
        # Median and IQR across runs
        median_count = fiber_count_df.median(axis=1)
        q25_count = fiber_count_df.quantile(0.25, axis=1)
        q75_count = fiber_count_df.quantile(0.75, axis=1)
        
        # Plot median and shaded IQR
        ax.plot(time_points, median_count, color=cmap[j], label=plot_labels[j])
        ax.fill_between(time_points, q25_count, q75_count, color=cmap[j], alpha=0.3)
    
    ax.set_title(f"Run ID: {run_id}")
    ax.set_xlabel("Time")
    ax.set_ylabel("Confined Filament Count")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Setup ---
runs = run_ids  # e.g., ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
time_points = np.linspace(0.1, 15, 150).round(1)
cmap = sns.color_palette("viridis", n_colors=len(zero_trials_tension))  # one color per trial

# --- Prepare 2x2 subplots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# --- Loop over runs for subplots ---
for i, run_id in enumerate(runs):
    ax = axes[i]
    
    # Loop over trials (conditions)
    for j, trial in enumerate(zero_trials_tension):
        if run_id not in trial_run_dfs[trial]:
            continue
        df = trial_run_dfs[trial][run_id]
        
        # Total fibers per run/time
        total_fibers = df.groupby(['run', 'time'])['fiber_id'].nunique()
        
        # Fibers experiencing force
        df_force = df[df['zpos'] < -0.399]
        fibers_with_force = df_force.groupby(['run', 'time'])['fiber_id'].nunique()
        
        # Percent per run/time
        percent = (fibers_with_force / total_fibers * 100).fillna(0)
        
        # Reindex to uniform time points per run
        percent_runs = []
        for run in percent.index.get_level_values('run').unique():
            run_series = percent.xs(run, level='run').reindex(time_points).fillna(method='ffill')
            percent_runs.append(run_series)
        percent_df = pd.concat(percent_runs, axis=1)
        
        # Median and IQR across runs
        median_percent = percent_df.median(axis=1)
        q25_percent = percent_df.quantile(0.25, axis=1)
        q75_percent = percent_df.quantile(0.75, axis=1)
        
        # Plot median and shaded IQR
        ax.plot(time_points, median_percent, color=cmap[j], label=plot_labels[j])
        ax.fill_between(time_points, q25_percent, q75_percent, color=cmap[j], alpha=0.3)
    
    ax.set_title(f"Run ID: {run_id}")
    ax.set_xlabel("Time")
    ax.set_ylabel("Percent of Filaments Experiencing Force (%)")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# -----------------------------
# Setup
# -----------------------------
runs = run_ids  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]  # spring constants per run_id, same order as runs
trial_labels = ['Full', '3/4', '1/2', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '3/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    '1/2': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 20
cmap = sns.color_palette("magma", n_colors=len(trial_labels))  # one color per trial

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# -----------------------------
# Loop over runs
# -----------------------------
for i, run_id in enumerate(runs):
    ax = axes[i]
    k_value = k_values[i]  # k applied per run

    for j, trial_label in enumerate(trial_labels):
        trial_key = trial_key_map[trial_label]

        if trial_key not in trial_run_dfs:
            print(f"⚠️ Trial key {trial_key} not found, skipping")
            continue
        if run_id not in trial_run_dfs[trial_key]:
            print(f"⚠️ Run ID {run_id} not found for trial {trial_label}, skipping")
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        # --- Confined filaments ---
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            print(f"⚠️ No confined filaments for trial {trial_label}, run {run_id}")
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

        # Sum confinement per time
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'confinement_force': force_sum.values
        })

        # --- Bud internalization ---
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        # Effective tension force (now per run)
        df_bud_unique['tension_force'] = df_bud_unique['internalization'] * k_value

        # Merge confinement and tension forces
        df_merged = pd.merge(df_conf, df_bud_unique[['time','tension_force']], on='time')
        df_merged = df_merged.sort_values('tension_force')
        df_merged['cum_confinement'] = df_merged['confinement_force'].cumsum()

        # --- Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_merged))
        if n_bins_actual < 2:
            print(f"⚠️ Not enough points for binning in trial {trial_label}, run {run_id}, skipping")
            continue
        df_merged['bin'] = pd.qcut(df_merged['tension_force'], q=n_bins_actual, duplicates='drop')

        # Median and IQR per bin
        binned_stats = df_merged.groupby('bin')['cum_confinement'].agg(
            median='median',
            q25=lambda x: np.percentile(x,25),
            q75=lambda x: np.percentile(x,75)
        ).reset_index()
        if binned_stats.empty:
            print(f"⚠️ No binned data for trial {trial_label}, run {run_id}, skipping")
            continue
        binned_stats['bin_center'] = binned_stats['bin'].apply(lambda x: x.mid)

        # --- Plot ---
        ax.plot(binned_stats['bin_center'], binned_stats['median'], color=cmap[j], lw=2, label=trial_label)
        ax.fill_between(binned_stats['bin_center'], binned_stats['q25'], binned_stats['q75'], color=cmap[j], alpha=0.3)
        ax.scatter(binned_stats['bin_center'], binned_stats['median'], color=cmap[j], edgecolor='k', s=40)

    #ax.set_xscale('log')
    #ax.set_yscale('log')
    ax.set_xlabel("Tension Force")
    ax.set_ylabel("Cumulative Confinement (pN)")
    ax.set_title(f"Run ID: {run_id}")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# -----------------------------
# Setup
# -----------------------------
runs = run_ids  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]  # spring constants per run_id, same order as runs
trial_labels = ['Full', '3/4', '1/2', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '3/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    '1/2': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 20
cmap = sns.color_palette("magma", n_colors=len(trial_labels))  # one color per trial

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# -----------------------------
# Loop over runs
# -----------------------------
for i, run_id in enumerate(runs):
    ax = axes[i]
    k_value = k_values[i]  # k applied per run

    # First, find the maximum cumulative confinement across all trials for this run
    max_cum_conf = 0
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id].copy()
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > max_cum_conf:
            max_cum_conf = cum_conf

    if max_cum_conf == 0:
        print(f"⚠️ No confinement data for run {run_id}, skipping plot")
        continue

    # Now plot each trial normalized to max_cum_conf
    for j, trial_label in enumerate(trial_labels):
        trial_key = trial_key_map[trial_label]

        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        # --- Confined filaments ---
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

        # Sum confinement per time
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'confinement_force': force_sum.values
        })

        # --- Bud internalization ---
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()
        df_bud_unique['tension_force'] = df_bud_unique['internalization'] * k_value

        # Merge confinement and tension forces
        df_merged = pd.merge(df_conf, df_bud_unique[['time','tension_force']], on='time')
        df_merged = df_merged.sort_values('tension_force')
        df_merged['cum_confinement'] = df_merged['confinement_force'].cumsum()

        # Normalize to max_cum_conf
        df_merged['cum_confinement_pct'] = df_merged['cum_confinement'] / max_cum_conf * 100

        # --- Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_merged))
        if n_bins_actual < 2:
            continue
        df_merged['bin'] = pd.qcut(df_merged['tension_force'], q=n_bins_actual, duplicates='drop')

        # Median and IQR per bin
        binned_stats = df_merged.groupby('bin')['cum_confinement_pct'].agg(
            median='median',
            q25=lambda x: np.percentile(x,25),
            q75=lambda x: np.percentile(x,75)
        ).reset_index()
        if binned_stats.empty:
            continue
        binned_stats['bin_center'] = binned_stats['bin'].apply(lambda x: x.mid)

        # --- Plot ---
        ax.plot(binned_stats['bin_center'], binned_stats['median'], color=cmap[j], lw=2, label=trial_label)
        ax.fill_between(binned_stats['bin_center'], binned_stats['q25'], binned_stats['q75'], color=cmap[j], alpha=0.3)
        ax.scatter(binned_stats['bin_center'], binned_stats['median'], color=cmap[j], edgecolor='k', s=40)

    ax.set_xscale('log')
    ax.set_xlabel("Tension Force")
    ax.set_ylabel("Cumulative Confinement (% of max)")
    ax.set_title(f"Run ID: {run_id}")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# -----------------------------
# Setup
# -----------------------------
runs = run_ids  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]  # spring constants per run_id, same order as runs
trial_labels = ['Full', '3/4', '1/2', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '3/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    '1/2': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 20
cmap = sns.color_palette("viridis", n_colors=len(trial_labels))  # one color per trial

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# -----------------------------
# Loop over runs
# -----------------------------
for i, run_id in enumerate(runs):
    ax = axes[i]
    k_value = k_values[i]  # k applied per run

    # First, find the maximum cumulative confinement across all trials for this run
    max_cum_conf = 0
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id].copy()
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > max_cum_conf:
            max_cum_conf = cum_conf

    if max_cum_conf == 0:
        print(f"⚠️ No confinement data for run {run_id}, skipping plot")
        continue

    # Now plot each trial normalized to max_cum_conf
    for j, trial_label in enumerate(trial_labels):
        trial_key = trial_key_map[trial_label]

        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        # --- Confined filaments ---
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

        # Sum confinement per time
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'confinement_force': force_sum.values
        })

        # --- Bud internalization ---
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()
        df_bud_unique['tension_force'] = df_bud_unique['internalization'] * 1000 #* k_value

        # Merge confinement and tension forces
        df_merged = pd.merge(df_conf, df_bud_unique[['time','tension_force']], on='time')
        df_merged = df_merged.sort_values('tension_force')
        df_merged['cum_confinement'] = df_merged['confinement_force'].cumsum()

        # Normalize to max_cum_conf
        df_merged['cum_confinement_pct'] = df_merged['cum_confinement'] / max_cum_conf * 100

        # --- Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_merged))
        if n_bins_actual < 2:
            continue
        df_merged['bin'] = pd.qcut(df_merged['tension_force'], q=n_bins_actual, duplicates='drop')

        # Median and IQR per bin
        binned_stats = df_merged.groupby('bin')['tension_force'].agg(
            median='median',
            q25=lambda x: np.percentile(x,25),
            q75=lambda x: np.percentile(x,75)
        ).reset_index()
        if binned_stats.empty:
            continue
        binned_stats['bin_center'] = binned_stats['bin'].apply(lambda x: x.mid)

        # --- Plot (flip axes) ---
        ax.plot(df_merged['cum_confinement_pct'], df_merged['tension_force'], color=cmap[j], lw=2, label=trial_label)

        #ax.fill_betweenx(df_merged['tension_force'], 0, df_merged['cum_confinement_pct'], color=cmap[j], alpha=0.1)

    #ax.set_yscale('log')
    ax.set_xlabel("Cumulative Confinement (% of max)")
    ax.set_ylabel("Internalization")
    ax.set_title(f"{k_values[i]} pN/um")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# -----------------------------
# Setup
# -----------------------------
runs = run_ids  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]  # spring constants per run_id, same order as runs
trial_labels = ['Full', '3/4', '1/2', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '3/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    '1/2': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 20
cmap = sns.color_palette("viridis", n_colors=len(trial_labels))  # one color per trial

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# -----------------------------
# Loop over runs
# -----------------------------
for i, run_id in enumerate(runs):
    ax = axes[i]

    # Find max cumulative confinement across all trials
    max_cum_conf = 0
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id].copy()
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > max_cum_conf:
            max_cum_conf = cum_conf

    if max_cum_conf == 0:
        print(f"⚠️ No confinement data for run {run_id}, skipping plot")
        continue

    # Plot each trial
    for j, trial_label in enumerate(trial_labels):
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        # --- Confined filaments ---
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'cum_confinement': force_sum.values
        })
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement'] / max_cum_conf * 100

        # --- Bud internalization (exactly as requested) ---
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()
        df_bud_unique['tension_force'] = df_bud_unique['internalization'] * 1000  # keep as is

        # Merge
        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('cum_confinement_pct')

        # Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['cum_confinement_pct'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('cum_confinement_pct','median'),
            x_q25=('cum_confinement_pct', lambda x: np.percentile(x,25)),
            x_q75=('cum_confinement_pct', lambda x: np.percentile(x,75)),
            y_median=('internalization','median'),
            y_q25=('internalization', lambda x: np.percentile(x,25)),
            y_q75=('internalization', lambda x: np.percentile(x,75))
        ).reset_index()

        # Scatter with x and y error bars
        ax.errorbar(
            binned_stats['x_median'], binned_stats['y_median'],
            xerr=[binned_stats['x_median'] - binned_stats['x_q25'], binned_stats['x_q75'] - binned_stats['x_median']],
            yerr=[binned_stats['y_median'] - binned_stats['y_q25'], binned_stats['y_q75'] - binned_stats['y_median']],
            fmt='o', color=cmap[j], ecolor=cmap[j], elinewidth=2, capsize=4, label=trial_label
        )

    ax.set_xlabel("Cumulative Confinement (% of max)")
    ax.set_ylabel("Internalization")
    ax.set_title(f"{k_values[i]} pN/um")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
runs = sorted(run_ids)
k_values = [10, 150, 1000, 10000]
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 20

# --- Viridis palette reversed: dark = low tension ---
palette = sns.color_palette("magma_r", n_colors=len(runs))
palette = palette[::-1]  # index 0 (10 pN) = dark

# --- Step 1: Compute global maximum cumulative confinement ---
global_max_cum_conf = 0
for run_id in runs:
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id]
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        global_max_cum_conf = max(global_max_cum_conf, cum_conf)

# --- Step 2: Plot all runs/trials normalized to global max ---
fig, ax = plt.subplots(figsize=(8, 6))

for i, run_id in enumerate(runs):
    print(run_id)
    color = palette[i]  # color by tension/run_id
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id].copy()

        # --- Confined filaments ---
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({'time': force_sum.index, 'cum_confinement': force_sum.values})
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement'] / global_max_cum_conf * 100

        # --- Bud internalization ---
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        # Merge
        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('cum_confinement_pct')

        # Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['internalization'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('internalization','median'),
            x_q25=('internalization', lambda x: np.percentile(x,25)),
            x_q75=('internalization', lambda x: np.percentile(x,75)),
            y_median=('cum_confinement_pct','median'),
            y_q25=('cum_confinement_pct', lambda x: np.percentile(x,25)),
            y_q75=('cum_confinement_pct', lambda x: np.percentile(x,75))
        ).reset_index()

        # Marker type
        marker = '.' if trial_label == 'Full' else '*'

        # Error bars (absolute values to prevent negative)
        yerr_lower = np.abs(binned_stats['y_median'] - binned_stats['y_q25'])
        yerr_upper = np.abs(binned_stats['y_q75'] - binned_stats['y_median'])

        ax.errorbar(
            binned_stats['x_median'], binned_stats['y_median'],
            fmt=marker, color=color, ecolor=color, elinewidth=2, capsize=4, markersize=10,
            label=f"{trial_label}, {k_values[i]} pN/um"
        )

#ax.set_xscale('log')
ax.set_xlabel("Internalization")
ax.set_ylabel("Cumulative Confinement (% of global max)")
ax.set_title("Full vs 1/4 Trials Across All Tensions")
ax.grid(True)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
runs = sorted(run_ids)  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 20

# --- Continuous color palette for tensions ---
palette = sns.color_palette("magma", n_colors=len(runs))

# --- Step 1: Compute global max for normalization ---
global_max_cum_conf = 0
for run_id in runs:
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id]
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > global_max_cum_conf:
            global_max_cum_conf = cum_conf

# --- Step 2: Create figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# Loop over runs/tensions
for i, run_id in enumerate(runs):
    color = palette[i]

    # --- Collect binned stats for Full and 1/4 ---
    stats_dict = {}
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        # Confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'cum_confinement': force_sum.values
        })
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement'] / global_max_cum_conf * 100

        # Bud internalization
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        # Merge and sort
        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('internalization')

        # Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['internalization'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('internalization','median'),
            x_q25=('internalization', lambda x: np.percentile(x,25)),
            x_q75=('internalization', lambda x: np.percentile(x,75)),
            y_median=('cum_confinement_pct','median'),
            y_q25=('cum_confinement_pct', lambda x: np.percentile(x,25)),
            y_q75=('cum_confinement_pct', lambda x: np.percentile(x,75))
        ).reset_index()

        stats_dict[trial_label] = binned_stats

    # --- Compute difference Full - 1/4 ---
    if 'Full' in stats_dict and '1/4' in stats_dict:
        df_diff = stats_dict['Full'].copy()
        df_diff['y_median'] = stats_dict['Full']['y_median'] - stats_dict['1/4']['y_median']
        df_diff['y_q25'] = stats_dict['Full']['y_q25'] - stats_dict['1/4']['y_q25']
        df_diff['y_q75'] = stats_dict['Full']['y_q75'] - stats_dict['1/4']['y_q75']

        # Ensure positive error bars
        yerr = [
            np.abs(df_diff['y_median'] - df_diff['y_q25']),
            np.abs(df_diff['y_q75'] - df_diff['y_median'])
        ]
        xerr = [
            np.abs(df_diff['x_median'] - df_diff['x_q25']),
            np.abs(df_diff['x_q75'] - df_diff['x_median'])
        ]

        # Plot with connecting line
        ax.errorbar(
            df_diff['x_median'], df_diff['y_median'],
            xerr=xerr, yerr=yerr,
            fmt='o-', color=color, ecolor=color,
            elinewidth=2, capsize=4, markersize=8,
            label=f"{k_values[i]} pN/um"
        )

#ax.set_xscale('log')
ax.set_xlabel("Internalization")
ax.set_ylabel("Δ Input Force [% of global max]")
ax.set_title("Difference Between Full and 1/4 Trials Across All Tensions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
runs = sorted(run_ids)  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 50

# --- Continuous color palette for tensions ---
palette = sns.color_palette("magma", n_colors=len(runs))

# --- Step 1: Compute global max for normalization ---
global_max_cum_conf = 0
for run_id in runs:
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id]
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > global_max_cum_conf:
            global_max_cum_conf = cum_conf

# --- Step 2: Create figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# Loop over runs/tensions
for i, run_id in enumerate(runs):
    color = palette[i]

    # --- Collect binned stats for Full and 1/4 ---
    stats_dict = {}
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        # Confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'cum_confinement': force_sum.values
        })
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement'] / global_max_cum_conf * 100

        # Bud internalization (scale to nm)
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud['internalization'] *= 1000  # convert to nm
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        # Merge and sort
        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('internalization')

        # Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['internalization'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('internalization','median'),
            x_q25=('internalization', lambda x: np.percentile(x,25)),
            x_q75=('internalization', lambda x: np.percentile(x,75)),
            y_median=('cum_confinement_pct','median'),
            y_q25=('cum_confinement_pct', lambda x: np.percentile(x,25)),
            y_q75=('cum_confinement_pct', lambda x: np.percentile(x,75))
        ).reset_index()

        stats_dict[trial_label] = binned_stats

    # --- Compute difference Full - 1/4 for x and y ---
    if 'Full' in stats_dict and '1/4' in stats_dict:
        df_diff = stats_dict['Full'].copy()
        df_diff['x_median'] = stats_dict['Full']['x_median'] - stats_dict['1/4']['x_median']
        df_diff['x_q25'] = stats_dict['Full']['x_q25'] - stats_dict['1/4']['x_q25']
        df_diff['x_q75'] = stats_dict['Full']['x_q75'] - stats_dict['1/4']['x_q75']

        df_diff['y_median'] = stats_dict['Full']['y_median'] - stats_dict['1/4']['y_median']
        df_diff['y_q25'] = stats_dict['Full']['y_q25'] - stats_dict['1/4']['y_q25']
        df_diff['y_q75'] = stats_dict['Full']['y_q75'] - stats_dict['1/4']['y_q75']

        # Ensure positive error bars
        xerr = [np.abs(df_diff['x_median'] - df_diff['x_q25']),
                np.abs(df_diff['x_q75'] - df_diff['x_median'])]
        yerr = [np.abs(df_diff['y_median'] - df_diff['y_q25']),
                np.abs(df_diff['y_q75'] - df_diff['y_median'])]

        # Plot with connecting line
        ax.errorbar(
            df_diff['x_median'], df_diff['y_median'],
            xerr=xerr, yerr=yerr,
            fmt='o-', color=color, ecolor=color,
            elinewidth=2, capsize=4, markersize=8,
            label=f"{k_values[i]} pN/um"
        )

ax.set_xlabel("Δ Internalization (Full - 1/4) [nm]")
ax.set_ylabel("Δ Force Input (Full - 1/4) [% global max]")
ax.set_title("Difference Between Full and 1/4 Trials Across All Tensions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
runs = sorted(run_ids)  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 50

# --- Continuous color palette for tensions ---
palette = sns.color_palette("magma", n_colors=len(runs))

# --- Step 1: Compute global max for normalization ---
global_max_cum_conf = 0
for run_id in runs:
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id]
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > global_max_cum_conf:
            global_max_cum_conf = cum_conf

# --- Step 2: Create figure ---
fig, ax = plt.subplots(figsize=(6, 6))

# Loop over runs/tensions
for i, run_id in enumerate(runs):
    color = palette[i]

    # --- Collect binned stats for Full and 1/4 ---
    stats_dict = {}
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        # Confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'cum_confinement': force_sum.values
        })
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement'] / global_max_cum_conf * 100

        # Bud internalization (scale to nm)
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud['internalization'] *= 1000  # convert to nm
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        # Merge and sort
        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('internalization')

        # Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['internalization'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('internalization','median'),
            x_q25=('internalization', lambda x: np.percentile(x,25)),
            x_q75=('internalization', lambda x: np.percentile(x,75)),
            y_median=('cum_confinement_pct','median'),
            y_q25=('cum_confinement_pct', lambda x: np.percentile(x,25)),
            y_q75=('cum_confinement_pct', lambda x: np.percentile(x,75))
        ).reset_index()

        stats_dict[trial_label] = binned_stats

    # --- Compute % difference (Full vs 1/4) ---
    if 'Full' in stats_dict and '1/4' in stats_dict:
        full = stats_dict['Full']
        qtr = stats_dict['1/4']

        # Avoid divide-by-zero
        qtr_safe = qtr.replace(0, np.nan)

        df_diff = full.copy()
        df_diff['x_median'] = (full['x_median'] - qtr['x_median']) / qtr_safe['x_median'] * 100
        df_diff['x_q25']    = (full['x_q25'] - qtr['x_q25']) / qtr_safe['x_q25'] * 100
        df_diff['x_q75']    = (full['x_q75'] - qtr['x_q75']) / qtr_safe['x_q75'] * 100

        df_diff['y_median'] = full['y_median'] - qtr['y_median']
        df_diff['y_q25']    = full['y_q25'] - qtr['y_q25']
        df_diff['y_q75']    = full['y_q75'] - qtr['y_q75']

        # Error bars
        xerr = [np.abs(df_diff['x_median'] - df_diff['x_q25']),
                np.abs(df_diff['x_q75'] - df_diff['x_median'])]
        yerr = [np.abs(df_diff['y_median'] - df_diff['y_q25']),
                np.abs(df_diff['y_q75'] - df_diff['y_median'])]

        # Plot
        ax.errorbar(
            df_diff['x_median'], df_diff['y_median'],
            xerr=xerr, yerr=yerr,
            fmt='o-', color=color, ecolor=color,
            elinewidth=2, capsize=4, markersize=8,
            label=f"{k_values[i]} pN/μm"
        )

# --- Labels and formatting ---
ax.set_xlabel("Δ Internalization (%)  [(Full - 1/4) / 1/4 × 100]")
ax.set_ylabel("Δ Force Input (Full - 1/4) [% global max]")
ax.set_title("Percent Difference Between Full and 1/4 Trials Across All Tensions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
runs = sorted(run_ids)  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 50

palette = sns.color_palette("magma", n_colors=len(runs))

# --- Step 1: Compute global max for normalization ---
global_max_cum_conf = 0
for run_id in runs:
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id]
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        global_max_cum_conf = max(global_max_cum_conf, cum_conf)

# --- Step 2: Create figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# Loop over runs/tensions
for i, run_id in enumerate(runs):
    color = palette[i]
    stats_dict = {}

    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'cum_confinement': force_sum.values
        })
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement'] / global_max_cum_conf * 100

        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud['internalization'] *= 1000
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('internalization')

        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['internalization'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('internalization','median'),
            x_q25=('internalization', lambda x: np.percentile(x,25)),
            x_q75=('internalization', lambda x: np.percentile(x,75)),
            y_median=('cum_confinement_pct','median'),
            y_q25=('cum_confinement_pct', lambda x: np.percentile(x,25)),
            y_q75=('cum_confinement_pct', lambda x: np.percentile(x,75))
        ).reset_index()

        stats_dict[trial_label] = binned_stats

    # --- Compute difference Full - 1/4 and swap x/y ---
    if 'Full' in stats_dict and '1/4' in stats_dict:
        df_diff = stats_dict['Full'].copy()
        df_diff['x_median'] = stats_dict['Full']['y_median'] - stats_dict['1/4']['y_median']  # swapped
        df_diff['x_q25']    = stats_dict['Full']['y_q25'] - stats_dict['1/4']['y_q25']
        df_diff['x_q75']    = stats_dict['Full']['y_q75'] - stats_dict['1/4']['y_q75']

        df_diff['y_median'] = stats_dict['Full']['x_median'] - stats_dict['1/4']['x_median']  # swapped
        df_diff['y_q25']    = stats_dict['Full']['x_q25'] - stats_dict['1/4']['x_q25']
        df_diff['y_q75']    = stats_dict['Full']['x_q75'] - stats_dict['1/4']['x_q75']

        xerr = [np.abs(df_diff['x_median'] - df_diff['x_q25']),
                np.abs(df_diff['x_q75'] - df_diff['x_median'])]
        yerr = [np.abs(df_diff['y_median'] - df_diff['y_q25']),
                np.abs(df_diff['y_q75'] - df_diff['y_median'])]

        ax.errorbar(
            df_diff['x_median'], df_diff['y_median'],
            xerr=xerr, yerr=yerr,
            fmt='o-', color=color, ecolor=color,
            elinewidth=2, capsize=4, markersize=8,
            label=f"{k_values[i]} pN/um"
        )

ax.set_xlabel("Δ Force Input (Full - 1/4) [% global max]")
ax.set_ylabel("Δ Internalization (Full - 1/4) [nm]")
ax.set_title("Difference Between Full and 1/4 Trials Across All Tensions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
import ot  # POT library
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Storage dict: trial -> run -> dataframe of force, WD, WD_per_filament, fraction confined, average internalization
WD_force_dict = {}

# --- Loop over trials ---
for trial in zero_trials_tension:
    WD_force_dict[trial] = {}

    # --- Loop over runs ---
    for run_id in trial_run_dfs[trial]:
        df = trial_run_dfs[trial][run_id]
        if df.empty:
            continue

        # --- Confined filaments ---
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

        # --- Count total and confined filaments per run/time ---
        df_counts_total = df.groupby(['run','time'])['fiber_id'].nunique().reset_index(name='num_total')
        df_counts_confined = df_confined.groupby(['run','time'])['fiber_id'].nunique().reset_index(name='num_confined')
        df_sum_force = df_confined.groupby(['run','time'])['confinement_force'].sum().reset_index(name='total_force')

        # --- Merge counts and forces ---
        df_plot = pd.merge(df_sum_force, df_counts_total, on=['run','time'])
        df_plot = pd.merge(df_plot, df_counts_confined, on=['run','time'], how='left')
        df_plot['num_confined'] = df_plot['num_confined'].fillna(0)
        df_plot['force_per_filament'] = df_plot['total_force'] / df_plot['num_total']
        df_plot['fraction_confined'] = df_plot['num_confined'] / df_plot['num_total']

        if df_plot.empty:
            continue

        # --- Compute circular WD per (run, time) for confined filaments ---
        wd_list = []
        for (r, t), group in df_confined.groupby(['run','time']):
            directions = np.arctan2(group['ypos_recalibrated'], group['xpos_recalibrated']).values
            directions = (np.pi + directions) / (2*np.pi)  # normalize to [0,1]
            uniform_sample = np.linspace(0, 1, 10000)
            wd = ot.wasserstein_circle(directions, uniform_sample)
            wd_list.append({'run': r, 'time': t, 'WD': wd})

        wd_df = pd.DataFrame(wd_list)

        # --- Merge WD with force and fraction confined ---
        merged_df = pd.merge(df_plot, wd_df, on=['run','time'])

        # --- WD per filament across all filaments ---
        merged_df['WD_per_filament'] = merged_df['WD'] / merged_df['num_total']
        merged_df['WD_per_filament'] = merged_df['WD_per_filament'].fillna(0)
        merged_df['WD'] = merged_df['WD'].apply(lambda x: float(x[0]) if isinstance(x, (list, np.ndarray)) and len(x) > 0 else np.nan)
        X = sm.add_constant(np.log(merged_df['num_total']))
        y = merged_df['WD']

        model = sm.OLS(y, X).fit()
        merged_df['WD_residual'] = model.resid


        # --- Average bud internalization per run/time ---
        if 'bud_internalization' in df.columns:
            avg_internalization = df.groupby(['run','time'])['bud_internalization'].mean().reset_index(name='avg_bud_internalization')
            merged_df = pd.merge(merged_df, avg_internalization, on=['run','time'], how='left')
        else:
            merged_df['avg_bud_internalization'] = np.nan
            
        # --- Store back in dict ---
        WD_force_dict[trial][run_id] = merged_df


In [ ]:
merged_df

In [ ]:
# --- Setup ---
plot_labels = ['Full', '3/4', '1/2', '1/4']
cmap = sns.color_palette("viridis", n_colors=len(zero_trials_tension))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# --- Loop over run_ids for subplots ---
for i, run_id in enumerate(runs):
    ax = axes[i]

    # Loop over trials
    for j, trial in enumerate(zero_trials_tension):
        if run_id not in WD_force_dict[trial]:
            continue
        df_merged = WD_force_dict[trial][run_id]
        df_merged_avg = df_merged.groupby('run').mean().reset_index()
        if df_merged.empty:
            continue

        ax.scatter(df_merged_avg['WD_residual'], df_merged_avg['avg_bud_internalization'],
                   alpha=0.7, s=50, color=cmap[j], edgecolor='k', label=plot_labels[j])

    #ax.set_xlabel("Fraction confined")
    #ax.set_ylabel("Total Filaments")
    ax.set_title(f"Run ID: {run_id}")
    ax.grid(True)
    #ax.set_xlim(0, 1)  # fraction ranges from 0 to 1
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import pearsonr

# --- Setup ---
plot_labels = ['Full', '3/4', '1/2', '1/4']
cmap = sns.color_palette("viridis", n_colors=len(zero_trials_tension))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# --- Loop over run_ids for subplots ---
for i, run_id in enumerate(runs):
    ax = axes[i]
    corr_vals = []  # store (r, p) for each trial

    for j, trial in enumerate(zero_trials_tension):
        if run_id not in WD_force_dict[trial]:
            continue
        df_merged = WD_force_dict[trial][run_id]
        if df_merged.empty:
            continue

        df_merged_avg = df_merged.groupby('run').mean().reset_index()

        # Scatter plot
        ax.scatter(
            df_merged_avg['WD_residual'],
            df_merged_avg['avg_bud_internalization'],
            alpha=0.7, s=50, color=cmap[j], edgecolor='k', label=plot_labels[j]
        )

        # --- Compute Pearson correlation ---
        x = df_merged_avg['WD_residual']
        y = df_merged_avg['avg_bud_internalization']
        if len(x) > 1 and np.all(np.isfinite(x)) and np.all(np.isfinite(y)):
            r, p = pearsonr(x, y)
            corr_vals.append((plot_labels[j], r, p))

    # --- Annotate correlations ---
    text_y = ax.get_ylim()[1] * 0.9
    for k, (label, r, p) in enumerate(corr_vals):
        ax.text(
            0.05, 0.9 - 0.07*k,
            f"{label}: r = {r:.2f}, p = {p:.3f}",
            transform=ax.transAxes,
            fontsize=9,
            color=cmap[k],
        )

    ax.set_title(f"Run ID: {run_id}", fontsize=12)
    ax.set_xlabel("Size-corrected WD (residual)")
    ax.set_ylabel("Avg. Bud Internalization")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Setup ---
plot_labels = ['Full', '3/4', '1/2', '1/4']
cmap = sns.color_palette("viridis", n_colors=len(zero_trials_tension))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# --- Loop over run_ids for subplots ---
for i, run_id in enumerate(runs):
    ax = axes[i]

    # Loop over trials
    for j, trial in enumerate(zero_trials_tension):
        if run_id not in WD_force_dict[trial]:
            continue
        df_merged = WD_force_dict[trial][run_id]
        if df_merged.empty:
            continue

        # Flatten WD list -> scalar
        df_merged['WD_scalar'] = df_merged['WD'].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)

        # Average per run (or other grouping if needed)
        df_avg = df_merged.groupby('run').agg(
            num_total=('num_total', 'mean'),
            num_confined=('num_confined', 'mean'),
            WD_avg=('WD_scalar', 'mean')
        ).reset_index()

        # Normalize WD for coloring
        norm = plt.Normalize(df_avg['WD_avg'].min(), df_avg['WD_avg'].max())
        colors = plt.cm.viridis(norm(df_avg['WD_avg']))

        # Plot scatter
        ax.scatter(df_avg['num_total'], df_avg['num_confined'],
                   c=colors, s=50, edgecolor='k', alpha=0.8, label=plot_labels[j])

    ax.set_xlabel("Total Filaments")
    ax.set_ylabel("Number Confined")
    ax.set_title(f"Run ID: {run_id}")
    ax.grid(True)
    ax.legend(title="Trial", fontsize=8)

# Colorbar for WD
sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
#cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.8)
cbar.set_label("Average Wasserstein Distance")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Setup ---
plot_labels = ['Full', '3/4', '1/2', '1/4']
cmap = sns.color_palette("viridis", n_colors=len(zero_trials_tension))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# --- Loop over run_ids for subplots ---
for i, run_id in enumerate(runs):
    ax = axes[i]

    # Loop over trials
    for j, trial in enumerate(zero_trials_tension):
        if run_id not in WD_force_dict[trial]:
            continue
        df_merged = WD_force_dict[trial][run_id]
        if df_merged.empty:
            continue

        # Flatten WD column
        df_merged['WD_scalar'] = df_merged['WD'].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)

        # Filter fraction_confined < 0.2
        df_filtered = df_merged[df_merged['fraction_confined'] < 0.2].copy()
        if df_filtered.empty:
            continue

        # Quantile-based bins (roughly equal points)
        n_bins = min(5, len(df_filtered))
        df_filtered['bin'] = pd.qcut(df_filtered['fraction_confined'], q=n_bins, duplicates='drop')

        # Compute median and quartiles per bin
        binned_stats = df_filtered.groupby('bin').agg(
            x_median=('fraction_confined', 'median'),
            y_median=('WD_scalar', 'median'),
            y_q25=('WD_scalar', lambda x: np.percentile(x, 25)),
            y_q75=('WD_scalar', lambda x: np.percentile(x, 75))
        ).reset_index()

        # Plot median line
        ax.plot(binned_stats['x_median'], binned_stats['y_median'], '-o', color=cmap[j], label=plot_labels[j])

        # Shade between 25th and 75th percentiles
        ax.fill_between(binned_stats['x_median'],
                        binned_stats['y_q25'],
                        binned_stats['y_q75'],
                        color=cmap[j], alpha=0.2)

    ax.set_xlabel("Fraction of Confined Filaments")
    ax.set_ylabel("Wasserstein Distance")
    ax.set_title(f"Run ID: {run_id}")
    ax.grid(True)
    ax.set_xlim(0, 0.2)
    ax.legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


## 3D 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Setup ---
tensions = sorted({tid for runs in unique_runs_lengths.values() for tid in runs})  # all tension IDs
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
cmap = sns.color_palette("magma", n_colors=len(tensions))  # one color per tension

# --- Prepare figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# --- Loop over tensions ---
for i, tension in enumerate(tensions):
    median_frac = []
    q25_frac = []
    q75_frac = []

    for trial in zero_trials_tension:
        df = trial_run_dfs_lengths.get(trial, {}).get(tension, None)
        if df is None or df.empty:
            median_frac.append(np.nan)
            q25_frac.append(np.nan)
            q75_frac.append(np.nan)
            continue

        # Compute fraction of plus_zdir > 0 per run
        runs_in_df = df['run'].unique()
        fractions = []
        for run in runs_in_df:
            df_run = df[df['run'] == run]
            total = len(df_run['plus_zdir'])
            if total == 0:
                fractions.append(np.nan)
            else:
                frac = (df_run['plus_zdir'] > 0).sum() / total
                fractions.append(frac)
        
        # Median and quartiles across runs
        fractions = np.array(fractions)
        fractions = fractions[~np.isnan(fractions)]
        if len(fractions) == 0:
            median_frac.append(np.nan)
            q25_frac.append(np.nan)
            q75_frac.append(np.nan)
        else:
            median_frac.append(np.median(fractions))
            q25_frac.append(np.percentile(fractions, 25))
            q75_frac.append(np.percentile(fractions, 75))

    # --- Plot line and shaded IQR ---
    ax.plot(plot_labels, median_frac, 'o-', color=cmap[i], label=f"{k_values[i]} pN/μm")
    ax.fill_between(plot_labels, q25_frac, q75_frac, color=cmap[i], alpha=0.3)

# --- Labels and formatting ---
ax.set_xlabel("Arp2/3 Distribution")
ax.set_ylabel("Fraction of Filaments with plus_zdir > 0")
ax.set_title("Fraction of Filament Tips Oriented Upwards Across Distributions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import kruskal

# --- Setup ---
tension = '0003'  # highest tension
plot_labels = ['Full', '3/4', '1/2', '1/4']
data_for_violin = []

# --- Collect data ---
for trial in zero_trials_tension:
    df = trial_run_dfs_lengths.get(trial, {}).get(tension, None)
    if df is None or df.empty:
        continue
    
    # Compute fraction per run
    for run in df['run'].unique():
        df_run = df[df['run'] == run]
        total = len(df_run['plus_zdir'])
        if total == 0:
            continue
        frac = (df_run['plus_zdir'] > 0).sum() / total
        data_for_violin.append({'Distribution': trial, 'FractionUp': frac})

# Convert to DataFrame
df_violin = pd.DataFrame(data_for_violin)

# Map trial names to readable labels
trial_map = dict(zip(zero_trials_tension, plot_labels))
df_violin['Distribution'] = df_violin['Distribution'].map(trial_map)

# --- Violin plot ---
plt.figure(figsize=(8,6))
sns.violinplot(x='Distribution', y='FractionUp', data=df_violin, inner='quartile', palette='magma')
plt.ylabel("Fraction of Filaments with plus_zdir > 0")
plt.title(f"Filament Growth Direction at Tension {tension}")
plt.grid(True, alpha=0.3)
plt.show()

# --- Statistical test: Kruskal-Wallis across distributions ---
groups = [group['FractionUp'].values for name, group in df_violin.groupby('Distribution')]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H-test across distributions: H={stat:.3f}, p={p:.4g}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Setup ---
tension = '0003'
trials_to_plot = ['Full', '1/4']
trial_mapping = dict(zip(zero_trials_tension, ['Full', '3/4', '1/2', '1/4']))
colors = ['blue', 'green']

# --- Prepare figure ---
plt.figure(figsize=(8,6))

for i, trial_label in enumerate(trials_to_plot):
    # Get the corresponding trial key
    trial_key = [k for k,v in trial_mapping.items() if v == trial_label][0]
    df = trial_run_dfs_lengths.get(trial_key, {}).get(tension, None)
    if df is None or df.empty:
        continue
    
    # Compute filament angles
    # Assuming you have 'plus_zdir' column; angle = arccos(z/|v|) if needed
    # Here we just use plus_zdir normalized to [-1,1] as proxy
    angles = df['plus_zdir'].values  # or compute actual angle if you have x,y,z
    
    sns.histplot(angles, bins=30, kde=False, stat='density',
                 label=trial_label, color=colors[i], alpha=0.5)

plt.xlabel("Filament Angle / plus_zdir")
plt.ylabel("Density")
plt.title(f"Histogram of Filament Angles (Tension {tension})")
plt.legend(title="Trial")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3E

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Setup ---
tensions = sorted(run_ids)  # e.g., ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
cmap = sns.color_palette("magma", n_colors=len(tensions))  # one color per tension
time_points = np.linspace(0.1, 15, 150).round(1)
final_time = max(time_points)  # final time point

# --- Prepare figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# --- Loop over tensions ---
for i, tension in enumerate(tensions):
    median_percents = []
    q25_percents = []
    q75_percents = []
    
    for trial in zero_trials_tension:
        # Collect final percent across all runs for this tension/trial
        final_values = []
        
        # Skip if this tension not in trial
        if tension not in trial_run_dfs.get(trial, {}):
            median_percents.append(np.nan)
            q25_percents.append(np.nan)
            q75_percents.append(np.nan)
            continue
        
        df = trial_run_dfs[trial][tension]

        # Total fibers per run/time
        total_fibers = df.groupby(['run','time'])['fiber_id'].nunique()
        
        # Fibers experiencing force per run/time
        df_force = df[df['zpos'] < -0.399]
        fibers_with_force = df_force.groupby(['run','time'])['fiber_id'].nunique()
        
        available_runs = total_fibers.index.get_level_values('run').unique()
        available_runs = available_runs.intersection(fibers_with_force.index.get_level_values('run').unique())

        for run in available_runs:
            percent_run = (fibers_with_force.xs(run, level='run') / 
                        total_fibers.xs(run, level='run') * 100)
            percent_run = percent_run.reindex(time_points).fillna(method='ffill')
            final_values.append(percent_run.loc[final_time])

        
        # Aggregate statistics across runs for this trial/tension
        if len(final_values) == 0:
            median_percents.append(np.nan)
            q25_percents.append(np.nan)
            q75_percents.append(np.nan)
        else:
            median_percents.append(np.median(final_values))
            q25_percents.append(np.percentile(final_values, 25))
            q75_percents.append(np.percentile(final_values, 75))
    
    # Compute symmetric error bars for IQR
    yerr_lower = np.array(q25_percents)
    yerr_upper = np.array(median_percents)
    yerr = [yerr_lower, yerr_upper]

    # Plot line with IQR error bars
    ax.errorbar(
        plot_labels, median_percents,
        yerr=yerr,
        fmt='o-', color=cmap[i], ecolor=cmap[i],
        elinewidth=2, capsize=4, markersize=8,
        label=f"{k_values[i]} pN/μm"
    )

# --- Labels and formatting ---
ax.set_xlabel("Arp2/3 Distribution")
ax.set_ylabel("Final % of Filaments Experiencing Force")
ax.set_title("Final Percent of Filaments Experiencing Force Across Distributions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Setup ---
tensions = sorted(run_ids)  # e.g., ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
cmap = sns.color_palette("magma", n_colors=len(tensions))  # one color per tension
time_points = np.linspace(0.1, 15, 150).round(1)
final_time = max(time_points)  # final time point

# --- Prepare figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# --- Loop over tensions ---
for i, tension in enumerate(tensions):
    median_percents = []
    q25_percents = []
    q75_percents = []
    
    for trial in zero_trials_tension:
        final_values = []
        
        if tension not in trial_run_dfs.get(trial, {}):
            median_percents.append(np.nan)
            q25_percents.append(np.nan)
            q75_percents.append(np.nan)
            continue
        
        df = trial_run_dfs[trial][tension]

        total_fibers = df.groupby(['run','time'])['fiber_id'].nunique()
        df_force = df[df['zpos'] < -0.399]
        fibers_with_force = df_force.groupby(['run','time'])['fiber_id'].nunique()
        
        # Only runs present in both Series
        available_runs = total_fibers.index.get_level_values('run').unique()
        available_runs = available_runs.intersection(fibers_with_force.index.get_level_values('run').unique())

        for run in available_runs:
            percent_run = (fibers_with_force.xs(run, level='run') / 
                           total_fibers.xs(run, level='run') * 100)
            percent_run = percent_run.reindex(time_points).fillna(method='ffill')
            final_values.append(percent_run.loc[final_time])

        if len(final_values) == 0:
            median_percents.append(np.nan)
            q25_percents.append(np.nan)
            q75_percents.append(np.nan)
        else:
            median_percents.append(np.median(final_values))
            q25_percents.append(np.percentile(final_values, 25))
            q75_percents.append(np.percentile(final_values, 75))
    
    # --- Plot median line ---
    ax.plot(plot_labels, median_percents, 'o-', color=cmap[i], label=f"{k_values[i]} pN/μm")
    # --- Shade between 25th and 75th percentile ---
    ax.fill_between(plot_labels, q25_percents, q75_percents, color=cmap[i], alpha=0.3)

# --- Labels and formatting ---
ax.set_xlabel("Arp2/3 Distribution")
ax.set_ylabel("Final % of Filaments Experiencing Force")
ax.set_title("Final Percent of Filaments Experiencing Force Across Distributions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Setup ---
tensions = sorted(run_ids)  # e.g., ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
cmap = sns.color_palette("magma", n_colors=len(tensions))  # one color per tension
time_points = np.linspace(0.1, 15, 150).round(1)
final_time = max(time_points)  # final time point

# --- Prepare figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# --- Loop over tensions ---
for i, tension in enumerate(tensions):
    final_counts = []

    for trial in zero_trials_tension:
        counts_across_runs = []
        
        if tension not in trial_run_dfs.get(trial, {}):
            final_counts.append(np.nan)
            continue
        
        df = trial_run_dfs[trial][tension]

        # Total fibers per run/time
        df_force = df[df['zpos'] < -0.399]
        fibers_with_force = df_force.groupby(['run','time'])['fiber_id'].nunique()

        # Only runs present in fibers_with_force
        available_runs = fibers_with_force.index.get_level_values('run').unique()

        for run in available_runs:
            run_counts = fibers_with_force.xs(run, level='run').reindex(time_points).fillna(method='ffill')
            counts_across_runs.append(run_counts.loc[final_time])
        
        # 95th percentile across runs for this trial/tension
        if len(counts_across_runs) == 0:
            final_counts.append(np.nan)
        else:
            final_counts.append(np.percentile(counts_across_runs, 95))
    
    # --- Plot line (no shading) ---
    ax.plot(plot_labels, final_counts, 'o-', color=cmap[i], label=f"{k_values[i]} pN/μm")

# --- Labels and formatting ---
ax.set_xlabel("Arp2/3 Distribution")
ax.set_ylabel("Number of Filaments Experiencing Force (95th percentile)")
ax.set_title("Final Filaments Experiencing Force Across Distributions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Setup ---
tensions = sorted(run_ids)  # e.g., ['0000', '0001', '0002', '0003']
plot_labels = ['Full', '3/4', '1/2', '1/4']  # readable trial labels
cmap = sns.color_palette("magma", n_colors=len(tensions))  # one color per tension
time_points = np.linspace(0.1, 15, 150).round(1)
final_time = max(time_points)  # final time point

# --- Prepare figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# --- Loop over tensions ---
for i, tension in enumerate(tensions):
    median_counts = []
    q25_counts = []
    q75_counts = []

    for trial in zero_trials_tension:
        counts_across_runs = []
        
        if tension not in trial_run_dfs.get(trial, {}):
            median_counts.append(np.nan)
            q25_counts.append(np.nan)
            q75_counts.append(np.nan)
            continue
        
        df = trial_run_dfs[trial][tension]

        # Fibers experiencing force per run/time
        df_force = df[df['zpos'] < -0.399]
        fibers_with_force = df_force.groupby(['run','time'])['fiber_id'].nunique()

        # Only runs present in fibers_with_force
        available_runs = fibers_with_force.index.get_level_values('run').unique()

        for run in available_runs:
            run_counts = fibers_with_force.xs(run, level='run').reindex(time_points).fillna(method='ffill')
            counts_across_runs.append(run_counts.loc[final_time])
        
        # Aggregate across runs: median and quartiles
        if len(counts_across_runs) == 0:
            median_counts.append(np.nan)
            q25_counts.append(np.nan)
            q75_counts.append(np.nan)
        else:
            median_counts.append(np.median(counts_across_runs))
            q25_counts.append(np.percentile(counts_across_runs, 25))
            q75_counts.append(np.percentile(counts_across_runs, 75))
    
    # --- Plot median line ---
    ax.plot(plot_labels, median_counts, 'o-', color=cmap[i], label=f"{k_values[i]} pN/μm")
    # --- Shade between 25th and 75th percentile ---
    ax.fill_between(plot_labels, q25_counts, q75_counts, color=cmap[i], alpha=0.3)

# --- Labels and formatting ---
ax.set_xlabel("Arp2/3 Distribution")
ax.set_ylabel("Number of Filaments Experiencing Force")
ax.set_title("Average # Filaments Experiencing Force Across Distributions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8)
plt.tight_layout()
plt.show()


## 3F

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
runs = sorted(run_ids)  # ['0000', '0001', '0002', '0003']
k_values = [10, 150, 1000, 10000]
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 50

# --- Continuous color palette for tensions ---
palette = sns.color_palette("magma", n_colors=len(runs))

# --- Step 1: Compute global max for normalization ---
global_max_cum_conf = 0
for run_id in runs:
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id]
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > global_max_cum_conf:
            global_max_cum_conf = cum_conf

# --- Step 2: Create figure ---
fig, ax = plt.subplots(figsize=(8, 6))

# Loop over runs/tensions
for i, run_id in enumerate(runs):
    color = palette[i]

    # --- Collect binned stats for Full and 1/4 ---
    stats_dict = {}
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()

        # Confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'time': force_sum.index,
            'cum_confinement': force_sum.values
        })
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement'] / global_max_cum_conf * 100

        # Bud internalization (scale to nm)
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud['internalization'] *= 1000  # convert to nm
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        # Merge and sort
        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('internalization')

        # Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['internalization'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('internalization','median'),
            x_q25=('internalization', lambda x: np.percentile(x,25)),
            x_q75=('internalization', lambda x: np.percentile(x,75)),
            y_median=('cum_confinement_pct','median'),
            y_q25=('cum_confinement_pct', lambda x: np.percentile(x,25)),
            y_q75=('cum_confinement_pct', lambda x: np.percentile(x,75))
        ).reset_index()

        stats_dict[trial_label] = binned_stats

    # --- Compute % difference (Full vs 1/4) ---
    if 'Full' in stats_dict and '1/4' in stats_dict:
        full = stats_dict['Full']
        qtr = stats_dict['1/4']

        # Avoid divide-by-zero
        qtr_safe = qtr.replace(0, np.nan)

        df_diff = full.copy()
        df_diff['x_median'] = (full['x_median'] - qtr['x_median']) / qtr_safe['x_median'] * 100
        df_diff['x_q25']    = (full['x_q25'] - qtr['x_q25']) / qtr_safe['x_q25'] * 100
        df_diff['x_q75']    = (full['x_q75'] - qtr['x_q75']) / qtr_safe['x_q75'] * 100

        df_diff['y_median'] = full['y_median'] - qtr['y_median']
        df_diff['y_q25']    = full['y_q25'] - qtr['y_q25']
        df_diff['y_q75']    = full['y_q75'] - qtr['y_q75']

        # Error bars
        xerr = [np.abs(df_diff['x_median'] - df_diff['x_q25']),
                np.abs(df_diff['x_q75'] - df_diff['x_median'])]
        yerr = [np.abs(df_diff['y_median'] - df_diff['y_q25']),
                np.abs(df_diff['y_q75'] - df_diff['y_median'])]

        # Plot
        ax.errorbar(
            df_diff['x_median'], df_diff['y_median'],
            xerr=xerr, yerr=yerr,
            fmt='o-', color=color, ecolor=color,
            elinewidth=2, capsize=4, markersize=8,
            label=f"{k_values[i]} pN/μm"
        )

# --- Labels and formatting ---
ax.set_xlabel("Δ Internalization (%)  [(Full - 1/4) / 1/4 × 100]")
ax.set_ylabel("Δ Force Input (Full - 1/4) [% global max]")
ax.set_title("Percent Difference Between Full and 1/4 Trials Across All Tensions")
ax.grid(True)
ax.legend(title="Tension", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
run_ids_filtered = ['0001', '0003']          # only the two runs
tension_labels_filtered = [150, 10000]       # corresponding tensions
trial_labels = ['Full', '1/4']              # trials to compare
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 10

# Continuous magma palette for just the two tensions
palette = sns.color_palette("magma", n_colors=len(run_ids_filtered))

# Step 1: compute global max for normalization
global_max_cum_conf = 0
for run_id in run_ids_filtered:
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id]
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > global_max_cum_conf:
            global_max_cum_conf = cum_conf

# Step 2: create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Loop over filtered run_ids/tensions
for i, run_id in enumerate(run_ids_filtered):
    color = palette[i]
    stats_dict = {}

    # Collect binned stats for Full and 1/4
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby(['run','time'])['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'run': force_sum.index.get_level_values('run'),
            'time': force_sum.index.get_level_values('time'),
            'cum_confinement': force_sum.values
        })
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement'] / global_max_cum_conf * 100

        # Bud internalization (nm)
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        # Merge and sort
        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('internalization')

        # Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['internalization'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('internalization','median'),
            x_q25=('internalization', lambda x: np.percentile(x,25)),
            x_q75=('internalization', lambda x: np.percentile(x,75)),
            y_median=('cum_confinement_pct','median'),
            y_q25=('cum_confinement_pct', lambda x: np.percentile(x,25)),
            y_q75=('cum_confinement_pct', lambda x: np.percentile(x,75))
        ).reset_index()

        stats_dict[trial_label] = binned_stats

    # Compute % difference Full vs 1/4
    if 'Full' in stats_dict and '1/4' in stats_dict:
        full = stats_dict['Full']
        qtr = stats_dict['1/4']

        qtr_safe = qtr.replace(0, np.nan)
        df_diff = full.copy()
        df_diff['x_median'] = (full['x_median'] - qtr['x_median']) / qtr_safe['x_median'] * 100
        df_diff['x_q25']    = (full['x_q25'] - qtr['x_q25']) / qtr_safe['x_q25'] * 100
        df_diff['x_q75']    = (full['x_q75'] - qtr['x_q75']) / qtr_safe['x_q75'] * 100

        df_diff['y_median'] = full['y_median'] - qtr['y_median']
        df_diff['y_q25']    = full['y_q25'] - qtr['y_q25']
        df_diff['y_q75']    = full['y_q75'] - qtr['y_q75']

        xerr = [np.abs(df_diff['x_median'] - df_diff['x_q25']),
                np.abs(df_diff['x_q75'] - df_diff['x_median'])]
        yerr = [np.abs(df_diff['y_median'] - df_diff['y_q25']),
                np.abs(df_diff['y_q75'] - df_diff['y_median'])]

        ax.errorbar(
            df_diff['x_median'], df_diff['y_median'],
            xerr=xerr, yerr=yerr,
            fmt='o-', color=color, ecolor=color,
            elinewidth=2, capsize=4, markersize=8,
            label=f"{tension_labels_filtered[i]} pN/μm"
        )

# --- Labels and formatting ---
ax.set_xlabel("Δ Internalization (%)  [(Full - 1/4) / 1/4 × 100]")
ax.set_ylabel("Δ Force Input (Full - 1/4) [% global max]")
#ax.set_title("Percent Difference Between Full and 1/4 Trials (Selected Tensions)")
ax.grid(True)
ax.legend(title="Tension", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Setup ---
run_ids_filtered = ['0001', '0003']          
tension_labels_filtered = [150, 10000]       
trial_labels = ['Full', '1/4']              
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}
tension_bins = 10

# Colormap for average time
cmap = plt.cm.rainbow
norm = plt.Normalize(vmin=0, vmax=15)  # map time_avg 0-15

# Step 1: compute global max for normalization
global_max_cum_conf = 0
for run_id in run_ids_filtered:
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id]
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        cum_conf = df_confined.groupby('time')['confinement_force'].sum().cumsum().max()
        if cum_conf > global_max_cum_conf:
            global_max_cum_conf = cum_conf

# Step 2: create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Loop over runs/tensions
for i, run_id in enumerate(run_ids_filtered):
    stats_dict = {}

    # Collect binned stats for Full and 1/4
    for trial_label in trial_labels:
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue

        df = trial_run_dfs[trial_key][run_id].copy()
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby(['run','time'])['confinement_force'].sum()
        df_conf = pd.DataFrame({
            'run': force_sum.index.get_level_values('run'),
            'time': force_sum.index.get_level_values('time'),
            'cum_confinement': force_sum.values
        })
        df_conf['cum_confinement'] = df_conf['cum_confinement'].cumsum()
        df_conf['cum_confinement_pct'] = df_conf['cum_confinement']# / global_max_cum_conf * 100

        # Bud internalization (nm)
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        df_bud_unique = df_bud.groupby('time', as_index=False)['internalization'].mean()

        # Merge and sort
        df_plot = pd.merge(df_conf[['time','cum_confinement_pct']], df_bud_unique[['time','internalization']], on='time')
        df_plot = df_plot.sort_values('internalization')

        # Quantile-based bins
        n_bins_actual = min(tension_bins, len(df_plot))
        if n_bins_actual < 2:
            continue
        df_plot['bin'] = pd.qcut(df_plot['internalization'], q=n_bins_actual, duplicates='drop')

        binned_stats = df_plot.groupby('bin').agg(
            x_median=('internalization','median'),
            y_median=('cum_confinement_pct','median'),
            time_avg=('time','mean')
        ).reset_index()

        stats_dict[trial_label] = binned_stats

    # Compute absolute differences Full vs 1/4
    if 'Full' in stats_dict and '1/4' in stats_dict:
        full = stats_dict['Full']
        qtr = stats_dict['1/4']

        df_diff = full.copy()
        df_diff['x_median'] = np.abs(full['x_median'] - qtr['x_median'])
        df_diff['y_median'] = np.abs(full['y_median'] - qtr['y_median'])
        df_diff['time_avg'] = full['time_avg']

        # Marker: filled for 10000, hollow for 150
        if tension_labels_filtered[i] == 10000:
            facecolors = cmap(norm(df_diff['time_avg']))
            edgecolors = 'k'
        else:
            facecolors = 'none'
            edgecolors = cmap(norm(df_diff['time_avg']))

        ax.scatter(df_diff['x_median'], df_diff['y_median'],
                   s=80, marker='o', facecolors=facecolors,
                   edgecolors=edgecolors, linewidths=1.5,
                   label=f"{tension_labels_filtered[i]} pN/μm")

# --- Labels and formatting ---
ax.set_yscale('log')
ax.set_ylim(1e3,1e6)
ax.set_xlabel("Δ Internalization (nm)")
ax.set_ylabel("Δ Force Input (pN")
ax.grid(True)
ax.legend(title="Tension", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

# --- Setup ---
runs = ['0001', '0003']  # left and right subplots
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}

# --- Color palette ---
palette = sns.color_palette("tab10", n_colors=len(trial_labels))

# --- Figure with 2 subplots ---
fig, axes = plt.subplots(1, len(runs), figsize=(12, 5), sharey=True)

for ax, run_id in zip(axes, runs):

    for color, trial_label in zip(palette, trial_labels):
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id].copy()

        # Confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        cum_conf = force_sum.cumsum()

        # Bud internalization
        df_bud = df[['time','bud_internalization']].copy()
        df_bud['internalization'] = pd.to_numeric(df_bud['bud_internalization'], errors='coerce')
        df_bud['internalization'] -= df_bud['internalization'].min()
        internalization_mean = df_bud.groupby('time')['internalization'].mean()

        # Align by time
        merged = pd.DataFrame({
            'cum_conf': cum_conf,
            'internalization': internalization_mean
        }).dropna()

        if merged.empty:
            continue

        # Plot: X = cumulative confinement, Y = internalization
        ax.plot(merged['cum_conf'].values, merged['internalization'].values,
                label=trial_label, color=color, alpha=0.8)

    # Axes labels and formatting
    ax.set_xlabel("Cumulative Confinement Force")
    ax.set_title(f"Run {run_id}")
    ax.grid(True)
    #ax.set_xscale('log')  # log scale on X-axis

# Y-axis label only on left plot
axes[0].set_ylabel("Internalization (nm)")
axes[1].set_ylabel("")

# Single legend for both subplots
axes[0].legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

# --- Setup ---
runs = ['0001', '0003']  # left and right subplots
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}

# --- Color palette ---
palette = sns.color_palette("tab10", n_colors=len(trial_labels))

# --- Figure with 2 subplots ---
fig, axes = plt.subplots(1, len(runs), figsize=(12, 5), sharey=True)

for ax, run_id in zip(axes, runs):

    for color, trial_label in zip(palette, trial_labels):
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id].copy()

        # Confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4
        force_sum = df_confined.groupby('time')['confinement_force'].sum()
        cum_conf = force_sum.cumsum()

        # Number of filaments per time
        n_filaments = df_confined.groupby('time')['fiber_id'].nunique()

        # Align by time
        merged = pd.DataFrame({
            'cum_conf': cum_conf,
            'n_filaments': n_filaments
        }).dropna()

        if merged.empty:
            continue

        # Plot: X = cumulative confinement, Y = number of filaments
        ax.plot(merged['cum_conf'].values, merged['n_filaments'].values,
                label=trial_label, color=color, alpha=0.8)

    # Axes labels and formatting
    ax.set_xlabel("Cumulative Confinement Force")
    ax.set_title(f"Run {run_id}")
    ax.grid(True)
    #ax.set_xscale('log')  # log scale on X-axis

# Y-axis label only on left plot
axes[0].set_ylabel("Number of Filaments")
axes[1].set_ylabel("")

# Single legend for both subplots
axes[0].legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# --- Setup ---
runs = ['0001', '0003']  # left and right subplots
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}

# --- Color palette ---
palette = sns.color_palette("tab10", n_colors=len(trial_labels))

# --- Number of bins ---
n_bins = 10

# --- Figure ---
fig, axes = plt.subplots(1, len(runs), figsize=(12, 5), sharey=True)

for ax, run_id in zip(axes, runs):

    for color, trial_label in zip(palette, trial_labels):
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id].copy()

        # Confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

        # Group by run and time
        grouped = df_confined.groupby(['run', 'time']).agg(
            total_force=('confinement_force', 'sum'),
            n_filaments=('fiber_id', 'nunique')
        ).reset_index()
        grouped['force_per_filament'] = grouped['total_force'] / grouped['n_filaments']

        # --- Bin by number of filaments ---
        min_fil = grouped['n_filaments'].min()
        max_fil = grouped['n_filaments'].max()
        bins = np.linspace(min_fil, max_fil, n_bins + 1)
        grouped['fil_bin'] = pd.cut(grouped['n_filaments'], bins=bins, labels=False, include_lowest=True)

        binned_stats = grouped.groupby('fil_bin').agg(
            avg_n_filaments=('n_filaments', 'mean'),
            avg_force_per_filament=('force_per_filament', 'mean'),
            q25=('force_per_filament', lambda x: np.percentile(x, 25)),
            q75=('force_per_filament', lambda x: np.percentile(x, 75))
        ).dropna()

        # Plot with shaded 25–75 percentile
        ax.plot(binned_stats['avg_n_filaments'], binned_stats['avg_force_per_filament'],
                label=trial_label, color=color)
        ax.fill_between(binned_stats['avg_n_filaments'],
                        binned_stats['q25'], binned_stats['q75'],
                        color=color, alpha=0.3)

    # Axes labels and formatting
    ax.set_xlabel("Number of Filaments")
    ax.set_title(f"Run {run_id}")
    ax.grid(True)

# Y-axis label only on left subplot
axes[0].set_ylabel("Force per Filament")
axes[1].set_ylabel("")

# Single legend on left
axes[0].legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# --- Setup ---
runs = ['0001', '0003']  # left and right subplots
trial_labels = ['Full', '1/4']
trial_key_map = {
    'Full': 'endocytosis2_baseline_zero_diffusion_tension_output',
    '1/4': 'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
}

# --- Color palette ---
palette = sns.color_palette("tab10", n_colors=len(trial_labels))

# --- Figure ---
fig, axes = plt.subplots(1, len(runs), figsize=(12, 5), sharey=True)

for ax, run_id in zip(axes, runs):

    for color, trial_label in zip(palette, trial_labels):
        trial_key = trial_key_map[trial_label]
        if trial_key not in trial_run_dfs or run_id not in trial_run_dfs[trial_key]:
            continue
        df = trial_run_dfs[trial_key][run_id].copy()

        # Confined filaments
        df_confined = df[df['zpos'] < -0.399].copy()
        if df_confined.empty:
            continue
        df_confined['confinement_force'] = (-0.399 - df_confined['zpos']) * 1e4

        # --- Group by run and time, compute force per filament ---
        grouped = df_confined.groupby(['run', 'time']).agg(
            total_force=('confinement_force', 'sum'),
            n_filaments=('fiber_id', 'nunique')
        ).reset_index()
        grouped['force_per_filament'] = grouped['total_force'] / grouped['n_filaments']

        # --- Average across runs at each time ---
        avg_per_time = grouped.groupby('time')['force_per_filament'].mean()
        time_points = avg_per_time.index.values

        # Plot: x = time, y = avg force per filament
        ax.plot(time_points, avg_per_time.values,
                label=trial_label, color=color, alpha=0.8)

    # Axes labels and formatting
    ax.set_xlabel("Time")
    ax.set_title(f"Run {run_id}")
    ax.grid(True)

# Y-axis label only on left subplot
axes[0].set_ylabel("Force per Filament")
axes[1].set_ylabel("")

# Single legend on left
axes[0].legend(title="Trial", fontsize=8)

plt.tight_layout()
plt.show()


## 3 Messy

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Trials and run info
zero_trials_tension = [
    'endocytosis2_baseline_zero_diffusion_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
]

plot_labels = ['Full', '3/4', '1/2', '1/4']
run_ids = ['0000', '0001', '0002', '0003']
tension_names = ['10', '150', '1000', '10000']

# Collect all displacement ratios
plot_data = []

for run_id, tension_name in zip(run_ids, tension_names):
    for trial, label in zip(zero_trials_tension, plot_labels):
        if run_id in trial_run_dfs_lengths[trial]:
            df_run = trial_run_dfs_lengths[trial][run_id].dropna(subset=['bud_zpos']).sort_values('time')
            
            # Group by internal 'run' column if multiple runs exist within this dataframe
            for internal_run_id, internal_run_df in df_run.groupby('run'):
                z_series = internal_run_df['bud_zpos'].values.astype(float)
                displacements = np.diff(z_series)
                n_pos = np.sum(displacements > 0)
                n_neg = np.sum(displacements < 0)
                
                if n_pos > 0:  # avoid division by zero
                    ratio = n_neg / n_pos
                    plot_data.append({
                        'run_id': run_id,
                        'tension_name': tension_name,
                        'condition_label': label,
                        'internal_run': internal_run_id,
                        'z_bias_sum_ratio': ratio
                    })

# Convert to DataFrame
z_bias_sum_df_tension = pd.DataFrame(plot_data)

# --- Plot 2x2 violin plots, one per run_id/tension ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharey=True)
axes = axes.flatten()

for i, run_id in enumerate(run_ids):
    ax = axes[i]
    data_run = z_bias_sum_df_tension[z_bias_sum_df_tension['run_id'] == run_id]
    
    sns.violinplot(
        data=data_run,
        x='condition_label',
        y='z_bias_sum_ratio',
        palette='viridis',
        inner='box',  # shows median and IQR
        alpha=0.6,
        ax=ax
    )
    sns.stripplot(
        data=data_run,
        x='condition_label',
        y='z_bias_sum_ratio',
        color='black',
        alpha=0.7,
        jitter=True,
        size=4,
        ax=ax
    )
    
    ax.set_title(f"Tension {tension_names[i]} pN/um (Run {run_id})")
    ax.set_xlabel('')
    ax.set_ylabel('Neg/Pos Δbud_zpos Ratio' if i % 2 == 0 else '')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Trials, labels, runs, tensions
zero_trials_tension = [
    'endocytosis2_baseline_zero_diffusion_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
]

plot_labels = ['Full', '3/4', '1/2', '1/4']  # desired order
run_ids = ['0001', '0003']  # only these runs
tension_names = {'0001': '150', '0003': '10000'}

# Collect median + quartiles for each trial/run/tension
summary_data = []

for run_id in run_ids:
    for trial, label in zip(zero_trials_tension, plot_labels):
        if run_id in trial_run_dfs_lengths[trial]:
            df_run = trial_run_dfs_lengths[trial][run_id].dropna(subset=['bud_zpos']).sort_values('time')
            
            for internal_run_id, internal_run_df in df_run.groupby('run'):
                z_series = internal_run_df['bud_zpos'].values.astype(float)
                displacements = np.diff(z_series)
                n_pos = np.sum(displacements > 0)
                n_neg = np.sum(displacements < 0)
                
                if n_pos > 0:
                    ratio = n_neg / n_pos
                    summary_data.append({
                        'run_id': run_id,
                        'tension_name': tension_names[run_id],
                        'trial_label': label,
                        'internal_run': internal_run_id,
                        'ratio': ratio
                    })

# Convert to DataFrame
summary_df = pd.DataFrame(summary_data)

# Compute median and IQR per trial/run_id
summary_stats = summary_df.groupby(['trial_label', 'run_id']).ratio.agg([
    ('median', 'median'),
    ('q25', lambda x: np.percentile(x, 25)),
    ('q75', lambda x: np.percentile(x, 75))
]).reset_index()

# --- Plot ---
plt.figure(figsize=(8,6))
palette = sns.color_palette("magma", n_colors=len(run_ids))

# Ensure x-axis order
x_order = plot_labels

for i, run_id in enumerate(run_ids):
    df_run = summary_stats[summary_stats['run_id'] == run_id]
    # Sort according to desired x_order
    df_run = df_run.set_index('trial_label').reindex(x_order).reset_index()
    
    plt.errorbar(
        x=df_run['trial_label'],
        y=df_run['median'],
        yerr=[df_run['median'] - df_run['q25'], df_run['q75'] - df_run['median']],
        fmt='-o',
        label=f"Run {run_id} ({tension_names[run_id]} pN/um)",
        color=palette[i],
        capsize=5
    )

plt.xlabel("Trial")
plt.ylabel("Distance Retracted / Distance Advanced")
plt.title("bud_zpos Bias for Selected Runs")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Prepare no tension data ---
no_tension_trials = zero_trials[:4]  # first four trials
plot_labels = ['Full', '3/4', '1/2', '1/4']  # desired order

no_tension_data = []
for trial, label in zip(no_tension_trials, plot_labels):
    trial_df = branched_actin_bound_points[trial].dropna(subset=['bud_zpos'])
    trial_df = trial_df.sort_values(['run', 'time'])
    
    for run_id, run_df in trial_df.groupby('run'):
        z_series = run_df['bud_zpos'].values.astype(float)
        displacements = np.diff(z_series)
        n_pos = np.sum(displacements > 0)
        n_neg = np.sum(displacements < 0)
        if n_pos > 0:
            ratio = n_neg / n_pos
            no_tension_data.append({
                'trial_label': label,
                'ratio': ratio,
                'tension_group': '150'  # no tension treated as 150 pN
            })

no_tension_df = pd.DataFrame(no_tension_data)

# --- Prepare tension data ---
tension_df_list = []
for trial, label in zip(zero_trials_tension, plot_labels):
    for run_id in ['0001', '0003']:  # your selected runs
        if run_id in trial_run_dfs_lengths[trial]:
            df_run = trial_run_dfs_lengths[trial][run_id].dropna(subset=['bud_zpos']).sort_values('time')
            
            for internal_run_id, internal_run_df in df_run.groupby('run'):
                z_series = internal_run_df['bud_zpos'].values.astype(float)
                displacements = np.diff(z_series)
                n_pos = np.sum(displacements > 0)
                n_neg = np.sum(displacements < 0)
                if n_pos > 0:
                    ratio = n_neg / n_pos
                    # Assign tension group based on run_id
                    tension_val = '150' if run_id == '0001' else '10000'
                    tension_df_list.append({
                        'trial_label': label,
                        'ratio': ratio,
                        'tension_group': tension_val
                    })

tension_df = pd.DataFrame(tension_df_list)

# --- Combine all data ---
combined_df = pd.concat([no_tension_df, tension_df], ignore_index=True)

# --- Compute median and IQR per trial and tension_group ---
summary_stats = combined_df.groupby(['trial_label', 'tension_group']).ratio.agg([
    ('median', 'median'),
    ('q25', lambda x: np.percentile(x, 25)),
    ('q75', lambda x: np.percentile(x, 75))
]).reset_index()

# --- Plot ---
plt.figure(figsize=(8,6))
palette = sns.color_palette("magma", n_colors=2)  # 2 tension lines: 150 and 10000
tension_order = ['150', '10000']  # consistent color order

x_order = plot_labels  # Full, 3/4, 1/2, 1/4

for i, tension_val in enumerate(tension_order):
    df_tension = summary_stats[summary_stats['tension_group'] == tension_val]
    df_tension = df_tension.set_index('trial_label').reindex(x_order).reset_index()
    
    plt.errorbar(
        x=df_tension['trial_label'],
        y=df_tension['median'],
        yerr=[df_tension['median'] - df_tension['q25'], df_tension['q75'] - df_tension['median']],
        fmt='-o',
        label=f"Tension {tension_val} pN",
        color=palette[i],
        capsize=5
    )

#plt.ylim(0,1.05)
plt.xlabel("Trial")
plt.ylabel("Distance Retracted / Distance Advanced")
plt.title("Internalization Bias")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Trials and tension IDs ---
zero_trials_tension = [
    'endocytosis2_baseline_zero_diffusion_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
    'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
]
plot_labels = ['Full', '3/4', '1/2', '1/4']
tension_ids = ['0000', '0001', '0002', '0003']
tension_names = ['10', '150', '1000', '10000']

In [ ]:
# --- Collect average internalization rates per internal run ---
plot_data = []

for trial, label in zip(zero_trials_tension, plot_labels):
    if trial not in trial_run_dfs:
        continue
    run_dfs = trial_run_dfs[trial]
    
    for tension_id, tension_name in zip(tension_ids, tension_names):
        if tension_id not in run_dfs:
            continue
        df_run = run_dfs[tension_id]
        if df_run.empty:
            continue
        
        # --- Loop over internal runs ---
        for internal_run_id, internal_run_df in df_run.groupby('run'):
            # Sort by time
            internal_run_df = internal_run_df.sort_values('time')
            
            z_internal = internal_run_df['bud_internalization'].values.astype(float)
            time = internal_run_df['time'].values.astype(float)
            
            if len(z_internal) < 2:
                continue  # cannot compute rates
            
            dz = np.diff(z_internal)
            dt = np.diff(time)
            
            # Only keep valid dt (avoid division by zero)
            valid_mask = (~np.isnan(dz)) & (~np.isnan(dt)) & (dt != 0)
            if np.sum(valid_mask) == 0:
                continue
            
            rates = dz[valid_mask] / dt[valid_mask]
            avg_rate = np.mean(rates)
            
            plot_data.append({
                'trial': trial,
                'label': label,
                'tension_id': tension_id,
                'tension_name': tension_name,
                'internal_run': internal_run_id,
                'average_rate': avg_rate
            })

# --- Convert to DataFrame ---
rate_df = pd.DataFrame(plot_data)

# --- Drop any remaining NaNs ---
rate_df = rate_df.dropna(subset=['average_rate'])


In [ ]:
# --- Plot: 1 row per tension, trials on x-axis ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, len(tension_ids), figsize=(26,6), sharey=False)

for i, tension_id in enumerate(tension_ids):
    ax = axes[i]
    df_plot = rate_df[rate_df['tension_id'] == tension_id]
    
    sns.violinplot(
        data=df_plot,
        x='label',
        y='average_rate',
        palette='viridis',
        inner='box',
        alpha=0.7,
        ax=ax
    )
    sns.stripplot(
        data=df_plot,
        x='label',
        y='average_rate',
        color='black',
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax
    )
    
    ax.set_title(f"Tension {tension_names[i]} pN/μm")
    ax.set_xlabel('')
    if i == 0:
        ax.set_ylabel('Average Internalization Rate (ΔI/Δt)')
    else:
        ax.set_ylabel('')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Parameters ---
bin_size = 25  # number of rows per rate interval
plot_labels = [label_map.get(trial, trial) for trial in zero_trials_tension]

# --- Collect time-binned internalization rates per internal run ---
plot_data = []

for trial, label in zip(zero_trials_tension, plot_labels):
    if trial not in trial_run_dfs:
        continue
    run_dfs = trial_run_dfs[trial]
    
    for tension_id, tension_name in zip(tension_ids, tension_names):
        if tension_id not in run_dfs:
            continue
        df_run = run_dfs[tension_id]
        if df_run.empty:
            continue
        
        # --- Loop over internal runs ---
        for internal_run_id, internal_run_df in df_run.groupby('run'):
            # Sort by time
            internal_run_df = internal_run_df.sort_values('time').drop_duplicates(subset='time')
            
            z_internal = internal_run_df['bud_internalization'].values.astype(float)
            time = internal_run_df['time'].values.astype(float)
            n_points = len(z_internal)
            
            if n_points <= bin_size:
                continue  # not enough points for a bin
            
            # Compute rate in non-overlapping bins of bin_size
            times_seen = set()
            for i in range(0, n_points - bin_size + 1, bin_size):
                dz = z_internal[i + bin_size - 1] - z_internal[i]
                dt = time[i + bin_size - 1] - time[i]
                if dt == 0:
                    continue
                rate = dz / dt
                time_mid = (time[i + bin_size - 1] + time[i]) / 2
                
                if time_mid in times_seen:
                    continue  # skip duplicate times
                times_seen.add(time_mid)
                
                plot_data.append({
                    'trial': trial,
                    'label': label,
                    'tension_id': tension_id,
                    'tension_name': tension_name,
                    'internal_run': internal_run_id,
                    'time': time_mid,
                    'internalization_rate': rate
                })

# --- Convert to DataFrame ---
rate_df = pd.DataFrame(plot_data)
rate_df = rate_df.dropna(subset=['internalization_rate'])


In [ ]:
# --- Plot median ± IQR time series, grouped by trial in 2x2 grid ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=False)
axes = axes.flatten()

colors = sns.color_palette("viridis", n_colors=len(zero_trials_tension))

for ax, tension_name in zip(axes, tension_names):
    df_tension = rate_df[rate_df['tension_name'] == tension_name]
    if df_tension.empty:
        continue
    
    # Loop over trials
    for i, (trial, label) in enumerate(zip(zero_trials_tension, plot_labels)):
        df_trial = df_tension[df_tension['trial'] == trial]
        if df_trial.empty:
            continue
        
        # Group by time across internal runs
        grouped = df_trial.groupby('time')['internalization_rate']
        times = grouped.mean().index.values
        medians = grouped.median().values
        q1 = grouped.quantile(0.25).values
        q3 = grouped.quantile(0.75).values
        
        ax.plot(times, medians, color=colors[i], label=f"{label} Median")
        ax.fill_between(times, q1, q3, color=colors[i], alpha=0.3)
    
    ax.set_title(f"{tension_name} pN/μm")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Internalization rate")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

# Remove empty subplots if any
for ax in axes[len(tension_names):]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- Prepare summary stats: median ± quartiles ---
summary_list = []

for tension_id, tension_name in zip(tension_ids, tension_names):
    df_tension = rate_df[rate_df['tension_id'] == tension_id]
    
    for label in plot_labels:
        df_label = df_tension[df_tension['label'] == label]
        if df_label.empty:
            continue
        
        median = df_label['average_rate'].median()
        q25 = df_label['average_rate'].quantile(0.25)
        q75 = df_label['average_rate'].quantile(0.75)
        
        summary_list.append({
            'label': label,
            'tension_name': tension_name,
            'median': median,
            'q25': q25,
            'q75': q75
        })

summary_df = pd.DataFrame(summary_list)

# --- Normalize by max median per tension ---
normalized_df_list = []
for tension_name in tension_names:
    df_t = summary_df[summary_df['tension_name'] == tension_name].copy()
    max_median = df_t['median'].max()
    df_t['median_norm'] = df_t['median'] / max_median
    df_t['q25_norm'] = df_t['q25'] / max_median
    df_t['q75_norm'] = df_t['q75'] / max_median
    normalized_df_list.append(df_t)

normalized_df = pd.concat(normalized_df_list, ignore_index=True)

# --- Plot ---
sns.set(style="whitegrid")
fig, ax = plt.subplots(figsize=(10,6))

# Colors from magma palette
palette = sns.color_palette("magma", len(tension_ids))
tension_colors = dict(zip(tension_names, palette))

# Plot one line per tension (normalized)
for tension_name in tension_names:
    df_t = normalized_df[normalized_df['tension_name'] == tension_name]
    x = np.arange(len(df_t))
    medians = df_t['median_norm'].values
    q25 = df_t['q25_norm'].values
    q75 = df_t['q75_norm'].values
    
    ax.plot(x, medians, marker='o', color=tension_colors[tension_name], label=f"{tension_name} pN/μm")
    ax.fill_between(x, q25, q75, color=tension_colors[tension_name], alpha=0.3)

# Formatting
ax.set_xticks(np.arange(len(plot_labels)))
ax.set_xticklabels(plot_labels)
ax.set_xlabel("Trial")
ax.set_ylabel("Normalized Average Internalization Rate")
ax.set_title("Normalized Internalization Rate Across Trials by Tension")
ax.legend(title="Tension")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import re

arp23 = True
tension = True

if arp23 is True:
    runs = [
        'endocytosis2_baseline_zero_diffusion_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_output', 
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_output',
        'endocytosis2_baseline_zero_diffusion_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_34donut_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_12donut_tension_output',
        'endocytosis2_baseline_asymmetric_zero_diffusion_Arp23_14donut_tension_output'
    ]

    output_dir_base = working_dir + simulations_dir
    df_clusters = {}

    for i, run in enumerate(runs):
        try:
            output_dir = f"{output_dir_base}{run}"
            df = pd.read_pickle(f"{output_dir}/dataframes/actin_clusters.pkl")

            # Flatten MultiIndex if needed
            if isinstance(df.index, pd.MultiIndex):
                df = df.reset_index(drop=False)
            else:
                df = df.reset_index()  # make sure the run names are accessible as a column

            # --- If it's a tension run, split by 000x pattern in 'run' column ---
            if tension and "tension" in run and 'run' in df.columns:
                df['run_id'] = df['run'].apply(
                    lambda x: re.search(r'000\d', x).group(0) if re.search(r'000\d', x) else None
                )

                unique_run_ids = df['run_id'].dropna().unique()

                for rid in unique_run_ids:
                    df_sub = df[df['run_id'] == rid].reset_index(drop=True)
                    key = f"{run}_{rid}"
                    df_clusters[key] = df_sub
                    print(f"Tension split: {key} ({len(df_sub)} rows)")

            else:
                df_clusters[run] = df
                print(f"Arp23 Success, run {i} ({run})")

        except Exception as e:
            print(f"Arp23 Failure run {i} ({run}): {e}")


In [ ]:
import re
import pandas as pd
import numpy as np

records = []

for key, df in df_clusters.items():
    # only analyze the tension-split runs (with 000x in the key)
    match = re.search(r'000\d', key)
    if not match:
        continue

    run_id = match.group(0)
    parent_run = key.replace(f"_{run_id}", "")

    # --- Get final time point per run ---
    last_time = df['time'].max()
    df_final = df[df['time'] == last_time]

    # --- Count unique clusters at that time point per run ---
    cluster_counts = df_final.groupby('run')['cluster_id'].nunique()

    # --- Compute median and quartiles for this trial ---
    median_val = cluster_counts.median()
    q25_val = cluster_counts.quantile(0.25)
    q75_val = cluster_counts.quantile(0.75)

    # --- Store record ---
    records.append({
        'parent_run': parent_run,
        'trial': run_id,
        'median_clusters': median_val,
        'q25_clusters': q25_val,
        'q75_clusters': q75_val
    })

    print(f"{key}: median={median_val:.2f}, Q1={q25_val:.2f}, Q3={q75_val:.2f} clusters")

# --- Convert to DataFrame ---
df_avg_final_clusters = pd.DataFrame(records)


In [ ]:
cluster_counts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Map trial IDs to tension labels
trial_to_tension = {'0001': '150', '0003': '10000'}

# Filter only trials 0001 and 0003
df_filtered = df_avg_final_clusters[df_avg_final_clusters['trial'].isin(['0001', '0003'])]

# Create discrete magma palette
trials = df_filtered['trial'].unique()
palette = sns.color_palette("magma", n_colors=len(trials))
trial_palette = dict(zip(trials, palette))

# Plot median with IQR error bars
plt.figure(figsize=(8,6))

for trial, group in df_filtered.groupby('trial'):
    x = np.arange(len(group['condition']))  # numerical x-axis
    y = group['median_clusters'].values
    yerr = np.vstack((
        y - group['q25_clusters'].values,  # lower
        group['q75_clusters'].values - y   # upper
    ))
    
    plt.errorbar(
        x, y,
        yerr=yerr,
        marker='o',
        markersize=6,
        markeredgecolor='black',
        markeredgewidth=0.8,
        linewidth=2,
        color=trial_palette[trial],
        label=trial_to_tension[trial],  # legend shows tension
        capsize=4
    )

# x-axis labels
condition_order = ['Full', '3/4', '1/2', '1/4']

plt.xticks(np.arange(len(condition_order)), condition_order)
plt.xlabel('Arp2/3 Distribution', fontsize=16)
plt.ylabel('Final number of actin clusters', fontsize=16)
plt.grid(True, linestyle='--', alpha=0.4)

# Legend inside the plot
plt.legend(title='Tension (pN/μm)', loc='upper right', frameon=True)
plt.tight_layout()
plt.show()


# Figure Four

# Procrustes Test

In [ ]:

def generate_sphere_points(n_points=10000, radius=1.0):
    """
    Generate n_points uniformly on the surface of a sphere of given radius.
    """
    phi = np.random.uniform(0, 2*np.pi, n_points)
    costheta = np.random.uniform(-1, 1, n_points)
    theta = np.arccos(costheta)
    
    x = radius * np.sin(theta) * np.cos(phi)
    y = radius * np.sin(theta) * np.sin(phi)
    z = radius * np.cos(theta)
    
    return np.vstack((x, y, z)).T  # shape: (n_points, 3)

# Example: generate reference sphere
ref_sphere = generate_sphere_points(10000, radius=1.0)


In [ ]:

def generate_uniform_sphere(n_points, radius=1.0):
    """Generate n_points uniformly on a sphere of given radius."""
    phi = np.random.uniform(0, 2*np.pi, n_points)
    costheta = np.random.uniform(-1, 1, n_points)
    theta = np.arccos(costheta)
    
    x = radius * np.sin(theta) * np.cos(phi)
    y = radius * np.sin(theta) * np.sin(phi)
    z = radius * np.cos(theta)
    
    return np.vstack((x, y, z)).T

# Store results per run
procrustes_stats_dict = {}  # keyed by tension, each value = list of dicts per run

for i, trial in enumerate(tension_runs):
    if trial not in hip1r_assoc_positions:
        continue
    
    df = hip1r_assoc_positions[trial].dropna()
    last_time = df['time'].max()
    df_last_time = df[df['time'] == last_time]
    
    procrustes_stats_dict[trial] = []
    
    for run_id in df_last_time['run'].unique():
        df_run = df_last_time[df_last_time['run'] == run_id]
        
        positions = df_run[['xpos_recalibrated','ypos_recalibrated','zpos_recalibrated']].to_numpy()
        n_points = positions.shape[0]
        
        # Run 100 iterations of random sphere comparisons
        disparities = []
        for _ in range(100):
            random_sphere = generate_uniform_sphere(n_points)
            _, _, disparity = procrustes(random_sphere, positions)
            disparities.append(disparity)
        
        disparities = np.array(disparities)
        stats = {
            'run_id': run_id,
            'n_points': n_points,        # <-- store number of points
            'mean': np.mean(disparities),
            'q25': np.percentile(disparities, 25),
            'q75': np.percentile(disparities, 75)
        }
        procrustes_stats_dict[trial].append(stats)


In [ ]:


# Choose a magma palette with enough colors for your tension conditions
palette = sns.color_palette("magma", n_colors=len(tension_runs))

plt.figure(figsize=(8,6))

for i, trial in enumerate(tension_runs):
    if trial not in procrustes_stats_dict:
        continue
    
    stats_list = procrustes_stats_dict[trial]
    n_points = [s['n_points'] for s in stats_list]
    mean_disparity = [s['mean'] for s in stats_list]
    
    plt.scatter(n_points, mean_disparity, 
                color=palette[i], s=50, alpha=0.7, label=labels[i])

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of Filaments (points) in Run')
plt.ylabel('Mean Procrustes Distance')
plt.title('Procrustes Distance vs Number of Filaments Across Tensions')
plt.legend(title='Tension Condition')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:

def generate_uniform_sphere(n_points, radius=1.0):
    """Generate n_points uniformly on a sphere of given radius."""
    phi = np.random.uniform(0, 2*np.pi, n_points)
    costheta = np.random.uniform(-1, 1, n_points)
    theta = np.arccos(costheta)
    
    x = radius * np.sin(theta) * np.cos(phi)
    y = radius * np.sin(theta) * np.sin(phi)
    z = radius * np.cos(theta)
    
    return np.vstack((x, y, z)).T

# Store results per run and time
procrustes_stats_dict = {}  # keyed by tension, each value = list of dicts per run-time

for i, trial in enumerate(tension_runs):
    print(f"Processing trial: {trial}")
    if trial not in hip1r_assoc_positions:
        continue
    
    df = hip1r_assoc_positions[trial].dropna()
    
    procrustes_stats_dict[trial] = []
    
    for run_id in df['run'].unique():  # loop over all runs
        df_run = df[df['run'] == run_id]
        
        for t in sorted(df_run['time'].unique()):  # loop over all time points
            df_time = df_run[df_run['time'] == t]
            
            positions = df_time[['xpos_recalibrated','ypos_recalibrated','zpos_recalibrated']].to_numpy()
            n_points = positions.shape[0]
            
            # Run 100 iterations of random sphere comparisons
            disparities = []
            for _ in range(10):
                random_sphere = generate_uniform_sphere(n_points)
                _, _, disparity = procrustes(random_sphere, positions)
                disparities.append(disparity)
            
            disparities = np.array(disparities)
            stats = {
                'run_id': run_id,
                'time': t,
                'n_points': n_points,
                'mean': np.mean(disparities),
                'q25': np.percentile(disparities, 25),
                'q75': np.percentile(disparities, 75)
            }
            procrustes_stats_dict[trial].append(stats)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Collect all n_points across all tensions
all_n_points = []
for trial in tension_runs:
    if trial not in procrustes_stats_dict:
        continue
    df_stats = pd.DataFrame(procrustes_stats_dict[trial])
    all_n_points.extend(df_stats['n_points'].values)

all_n_points = np.array(all_n_points)


n_bins = 50
bin_edges = np.linspace(all_n_points.min(), all_n_points.max(), n_bins+1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

plt.figure(figsize=(8,6))
palette = sns.color_palette("magma", n_colors=len(tension_runs))

for i, trial in enumerate(tension_runs):
    if trial not in procrustes_stats_dict:
        continue
    
    df_stats = pd.DataFrame(procrustes_stats_dict[trial])
    
    median_pd_per_bin = []
    q25_per_bin = []
    q75_per_bin = []
    
    for start, end in zip(bin_edges[:-1], bin_edges[1:]):
        df_bin = df_stats[(df_stats['n_points'] >= start) & (df_stats['n_points'] < end)]
        if len(df_bin) > 0:
            median_pd_per_bin.append(df_bin['mean'].median())
            q25_per_bin.append(df_bin['q25'].median())
            q75_per_bin.append(df_bin['q75'].median())
        else:
            median_pd_per_bin.append(np.nan)
            q25_per_bin.append(np.nan)
            q75_per_bin.append(np.nan)
    
    median_pd_per_bin = np.array(median_pd_per_bin)
    q25_per_bin = np.array(q25_per_bin)
    q75_per_bin = np.array(q75_per_bin)
    
    # Plot median line
    plt.plot(bin_centers, median_pd_per_bin, marker='o', color=palette[i], label=labels[i])
    # Add shaded region for 25-75th percentile
    plt.fill_between(bin_centers, q25_per_bin, q75_per_bin, color=palette[i], alpha=0.3)

plt.xscale('log')
#plt.yscale('log')
plt.xlabel('Number of Filaments (binned)')
plt.ylabel('Median Procrustes Distance')
plt.title('Median Procrustes Distance vs Filament Number (Dynamic Bins)')
plt.legend(title='Tension Condition')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import procrustes
import matplotlib.pyplot as plt
import seaborn as sns

def generate_uniform_sphere(n_points, radius=1.0):
    """Generate n_points uniformly on a sphere of given radius."""
    phi = np.random.uniform(0, 2*np.pi, n_points)
    costheta = np.random.uniform(-1, 1, n_points)
    theta = np.arccos(costheta)
    
    x = radius * np.sin(theta) * np.cos(phi)
    y = radius * np.sin(theta) * np.sin(phi)
    z = radius * np.cos(theta)
    
    return np.vstack((x, y, z)).T

# Dictionary to store results
procrustes_stats_dict = {}

for trial, df in branched_actin_bound_points.items():
    print(f"Processing trial: {trial}")
    df = df.dropna()
    procrustes_stats_dict[trial] = []
    
    for run_id in df['run'].unique():
        df_run = df[df['run'] == run_id]
        
        for t in sorted(df_run['time'].unique()):
            df_time = df_run[df_run['time'] == t]
            positions = df_time[['xpos_recalibrated', 'ypos_recalibrated', 'zpos_recalibrated']].to_numpy()
            n_points = positions.shape[0]
            if n_points < 2:
                continue
            
            disparities = []
            for _ in range(10):  # random sphere comparisons
                random_sphere = generate_uniform_sphere(n_points)
                _, _, disparity = procrustes(random_sphere, positions)
                disparities.append(disparity)
            
            stats = {
                'run_id': run_id,
                'time': t,
                'n_points': n_points,
                'mean': np.mean(disparities),
                'q25': np.percentile(disparities, 25),
                'q75': np.percentile(disparities, 75)
            }
            procrustes_stats_dict[trial].append(stats)

# Convert to a single DataFrame for plotting
plot_df_list = []
for trial, stats_list in procrustes_stats_dict.items():
    df_stats = pd.DataFrame(stats_list)
    df_stats['Condition'] = trial  # use trial name as condition
    plot_df_list.append(df_stats)

df_plot = pd.concat(plot_df_list, ignore_index=True)



In [ ]:
# Example plotting: median Procrustes distance vs number of filaments
all_n_points = df_plot['n_points'].values
n_bins = 50
bin_edges = np.linspace(all_n_points.min(), all_n_points.max(), n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

plt.figure(figsize=(10, 6))
# Use viridis palette
palette = sns.color_palette("viridis", n_colors=len(procrustes_stats_dict))

for i, cond in enumerate(df_plot['Condition'].unique()):
    df_cond = df_plot[df_plot['Condition'] == cond]
    median_pd_per_bin = []
    q25_per_bin = []
    q75_per_bin = []
    
    for start, end in zip(bin_edges[:-1], bin_edges[1:]):
        df_bin = df_cond[(df_cond['n_points'] >= start) & (df_cond['n_points'] < end)]
        if len(df_bin) > 0:
            median_pd_per_bin.append(df_bin['mean'].median())
            q25_per_bin.append(df_bin['q25'].median())
            q75_per_bin.append(df_bin['q75'].median())
        else:
            median_pd_per_bin.append(np.nan)
            q25_per_bin.append(np.nan)
            q75_per_bin.append(np.nan)
    
    plt.plot(bin_centers, median_pd_per_bin, marker='o', color=palette[i], label=cond, linewidth=2.5)
    plt.fill_between(bin_centers, q25_per_bin, q75_per_bin, color=palette[i], alpha=0.3)

plt.xscale('log')
plt.xlabel('Number of Filaments (binned)')
plt.ylabel('Median Procrustes Distance')
plt.title('Median Procrustes Distance vs Number of Filaments')
plt.legend(title='Condition')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:


def partial_procrustes(X, Y):
    """
    Perform Partial Procrustes analysis (translation + rotation, no scaling).
    Translates and rotates Y to best align with X.
    Returns the disparity (sum of squared distances).
    """
    # Ensure arrays are float
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)

    # Center both configurations
    Xc = X - X.mean(axis=0)
    Yc = Y - Y.mean(axis=0)

    # Optimal rotation via SVD
    M = Yc.T @ Xc
    U, _, Vt = np.linalg.svd(M)
    R = U @ Vt

    # Fix improper rotation (reflection)
    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    # Apply rotation
    Y_aligned = Yc @ R

    # Compute disparity
    disparity = np.sum((Xc - Y_aligned)**2)

    return disparity


In [ ]:
def generate_uniform_sphere(n_points, radius=1.0):
    """Generate n_points uniformly on a sphere of given radius."""
    phi = np.random.uniform(0, 2*np.pi, n_points)
    costheta = np.random.uniform(-1, 1, n_points)
    theta = np.arccos(costheta)
    
    x = radius * np.sin(theta) * np.cos(phi)
    y = radius * np.sin(theta) * np.sin(phi)
    z = radius * np.cos(theta)
    
    return np.vstack((x, y, z)).T

# Dictionary to store results
procrustes_stats_dict = {}

for trial, df in branched_actin_bound_points.items():
    print(f"Processing trial: {trial}")
    df = df.dropna()
    procrustes_stats_dict[trial] = []
    
    for run_id in df['run'].unique():
        df_run = df[df['run'] == run_id]
        
        for t in sorted(df_run['time'].unique()):
            df_time = df_run[df_run['time'] == t]
            positions = df_time[['xpos_recalibrated', 'ypos_recalibrated', 'zpos_recalibrated']].to_numpy()
            n_points = positions.shape[0]
            if n_points < 2:
                continue
            
            disparities = []
            for _ in range(10):  # random sphere comparisons
                random_sphere = generate_uniform_sphere(n_points)
                disparity = partial_procrustes(random_sphere, positions)
                disparities.append(disparity)
            
            stats = {
                'run_id': run_id,
                'time': t,
                'n_points': n_points,
                'mean': np.mean(disparities),
                'q25': np.percentile(disparities, 25),
                'q75': np.percentile(disparities, 75)
            }
            procrustes_stats_dict[trial].append(stats)

# Convert to a single DataFrame for plotting
plot_df_list = []
for trial, stats_list in procrustes_stats_dict.items():
    df_stats = pd.DataFrame(stats_list)
    df_stats['Condition'] = trial  # use trial name as condition
    plot_df_list.append(df_stats)

df_plot = pd.concat(plot_df_list, ignore_index=True)



In [ ]:
# Example plotting: median Procrustes distance vs number of filaments
all_n_points = df_plot['n_points'].values
n_bins = 50
bin_edges = np.linspace(all_n_points.min(), all_n_points.max(), n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

plt.figure(figsize=(10, 6))
# Use viridis palette
palette = sns.color_palette("viridis", n_colors=len(procrustes_stats_dict))

for i, cond in enumerate(df_plot['Condition'].unique()):
    df_cond = df_plot[df_plot['Condition'] == cond]
    median_pd_per_bin = []
    q25_per_bin = []
    q75_per_bin = []
    
    for start, end in zip(bin_edges[:-1], bin_edges[1:]):
        df_bin = df_cond[(df_cond['n_points'] >= start) & (df_cond['n_points'] < end)]
        if len(df_bin) > 0:
            median_pd_per_bin.append(df_bin['mean'].median())
            q25_per_bin.append(df_bin['q25'].median())
            q75_per_bin.append(df_bin['q75'].median())
        else:
            median_pd_per_bin.append(np.nan)
            q25_per_bin.append(np.nan)
            q75_per_bin.append(np.nan)
    
    plt.plot(bin_centers, median_pd_per_bin, marker='o', color=palette[i], label=cond, linewidth=2.5)
    plt.fill_between(bin_centers, q25_per_bin, q75_per_bin, color=palette[i], alpha=0.3)

plt.xscale('log')
plt.xlabel('Number of Filaments (binned)')
plt.ylabel('Median Procrustes Distance')
plt.title('Median Procrustes Distance vs Number of Filaments')
plt.legend(title='Condition')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


# Random Tests

In [ ]:
# Internalization velocity

In [ ]:
import numpy as np
import pandas as pd

# Dictionary to store horizontal velocity vectors
horizontal_velocity_vectors = {}

for trial in zero_trials:
    if trial not in branched_actin_bound_points:
        print(f"Skipping missing trial: {trial}")
        continue

    print(f"Processing trial: {trial}")
    df = branched_actin_bound_points[trial]

    # Check required columns
    required_cols = {'run', 'time', 'bud_xpos', 'bud_ypos'}
    if not required_cols.issubset(df.columns):
        print(f"Missing columns in {trial}: {required_cols - set(df.columns)}")
        continue

    # Group by run
    grouped = df.groupby('run')
    horizontal_velocity_vectors[trial] = {}

    for run_id, run_df in grouped:
        run_df = run_df.sort_values('time')

        # Compute time derivative (vx, vy)
        vx = np.gradient(run_df['bud_xpos'], run_df['time'])
        vy = np.gradient(run_df['bud_ypos'], run_df['time'])
        vz = np.gradient(run_df['bud_zpos'], run_df['time'])  # if needed

        # Store both the components and magnitude
        vel_df = pd.DataFrame({
            'time': run_df['time'],
            'vx': vx,
            'vy': vy,
            'vz': vz,
            'v_horizontal_mag': np.sqrt(vx**2 + vy**2),
            'bud_xpos': run_df['bud_xpos'],
            'bud_ypos': run_df['bud_ypos']
        }).set_index('time')

        horizontal_velocity_vectors[trial][run_id] = vel_df

        #print(f"  → computed (vx, vy) for run {run_id}")

print("✅ All velocity vectors computed.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Trial labels and discrete viridis colors
trial_labels = ['full', '3/4', '1/2', '1/4']
n_labels = len(trial_labels)
colors = cm.viridis(np.linspace(0.1, 0.9, n_labels))

# --- Compute per-run mean velocity vectors ---
run_points = []  # (trial, run_id, mean_vx, mean_vy)

for trial, runs_dict in horizontal_velocity_vectors.items():
    for run_id, run_df in runs_dict.items():
        mean_vx = np.nanmean(run_df['vx'].values)
        mean_vy = np.nanmean(run_df['vy'].values)
        run_points.append((trial, run_id, mean_vx, mean_vy))

# --- Plot all runs, color by trial ---
fig, ax = plt.subplots(figsize=(8, 8))

# assign colors per trial
trial_to_color = {t: colors[i % n_labels] for i, t in enumerate(horizontal_velocity_vectors.keys())}

for trial, run_id, vx, vy in run_points:
    color = trial_to_color[trial]
    label = trial_labels[list(horizontal_velocity_vectors.keys()).index(trial)] \
        if trial in horizontal_velocity_vectors else trial
    ax.scatter(vx, vy, color=color, s=50, edgecolor='k', alpha=0.8, label=label)

# --- Remove duplicate legend entries ---
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), title="Arp2/3 variant")

# --- Formatting ---
ax.axhline(0, color='gray', lw=0.6)
ax.axvline(0, color='gray', lw=0.6)
ax.set_xlabel("vx (mean per run)")
ax.set_ylabel("vy (mean per run)")
ax.set_title("Run-averaged horizontal velocity vectors (vx, vy)")
ax.set_aspect('equal', adjustable='box')
ax.grid(True, alpha=0.3)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Trial labels and discrete viridis colors
trial_labels = ['full', '3/4', '1/2', '1/4']
n_labels = len(trial_labels)
colors = cm.viridis(np.linspace(0.1, 0.9, n_labels))

# --- Compute per-run absolute velocities ---
run_points = []  # (trial_label, abs_vx, abs_vy)
for i, trial_key in enumerate(horizontal_velocity_vectors.keys()):
    runs_dict = horizontal_velocity_vectors[trial_key]
    trial_label = trial_labels[i]  # map to full, 3/4, 1/2, 1/4
    for run_id, run_df in runs_dict.items():
        abs_vx = np.nanmean(np.abs(run_df['vx'].values))
        abs_vy = np.nanmean(np.abs(run_df['vy'].values))
        run_points.append((trial_label, abs_vx, abs_vy))

# --- Convert to numpy arrays ---
trials, abs_vx, abs_vy = zip(*run_points)
trials = np.array(trials)
abs_vx = np.array(abs_vx)
abs_vy = np.array(abs_vy)

# Map trial label to color
trial_to_color = {label: colors[i] for i, label in enumerate(trial_labels)}

# --- Plot abs_vx vs abs_vy ---
fig, ax = plt.subplots(figsize=(8,8))
for label in trial_labels:
    mask = trials == label
    ax.scatter(abs_vx[mask], abs_vy[mask], color=trial_to_color[label],
               s=50, edgecolor='k', alpha=0.8, label=label)

ax.set_xlabel('|vx| per run')
ax.set_ylabel('|vy| per run')
ax.set_title('Run-averaged horizontal speed components (|vx| vs |vy|)')
ax.grid(True, alpha=0.3)
ax.legend(title="Arp2/3 variant")

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import ot

trial_results = {}

# --- Precompute uniform reference once ---
uniform_network = np.random.uniform(-np.pi, np.pi, 10000)
uniform_shift = (np.pi + uniform_network) / (2 * np.pi)

for trial in zero_trials:
    if trial not in branched_actin_bound_points:
        continue

    print(f"Processing trial: {trial}")
    trial_data = branched_actin_bound_points[trial].dropna(subset=['xpos_recalibrated', 'ypos_recalibrated', 'bud_internalization'])

    # --- Recalibrate positions and compute direction ---
    xpos_recal = trial_data['xpos_recalibrated'].values
    ypos_recal = trial_data['ypos_recalibrated'].values

    dir_angles = np.arctan2(ypos_recal, xpos_recal)
    dir_circle = (np.pi + dir_angles) / (2 * np.pi)  # Normalize to [0,1]
    trial_data['direction'] = dir_circle

    # --- Optionally, compute length per point if needed ---
    trial_data['length'] = 1  # Or other metric; here counting each point as 1

    # --- Group by run and time ---
    grouped = trial_data.groupby(['run', 'time'])

    rows = []
    for (run, time), group in grouped:
        # Collect directions safely
        dirs_list = []
        for d in group['direction']:
            if d is None or np.isnan(d):
                continue
            dirs_list.append(np.array([d], dtype=float))
        if not dirs_list:
            continue
        all_dirs = np.concatenate(dirs_list)

        # Internalization
        internal_vals = group['bud_internalization'].unique()
        internalization = internal_vals[0] if len(internal_vals) > 0 else np.nan

        # Bud positions: mean per group
        mean_bud_x = np.nanmean(group['bud_xpos'].values)
        mean_bud_y = np.nanmean(group['bud_ypos'].values)

        # Compute Wasserstein distance
        bl_directions = np.random.uniform(-np.pi, np.pi, all_dirs.size)
        bl_shift = (np.pi + bl_directions) / (2 * np.pi)
        directions_shift = (np.pi + all_dirs) / (2 * np.pi)
        wd = ot.wasserstein_circle(directions_shift, uniform_shift, p=1)
        wd_bl = ot.wasserstein_circle(bl_shift, uniform_shift, p=1)

        # Save row
        rows.append({
            'Run': run,
            'Time': time,
            'Internalization': internalization,
            'WD': wd,
            'WD_BL': wd_bl,
            'ActinAmount': all_dirs.size,
            'bud_xpos': mean_bud_x,
            'bud_ypos': mean_bud_y
        })

    # --- Save trial DataFrame ---
    df_trial = pd.DataFrame(rows)
    trial_results[trial] = df_trial

print("✅ Done")


In [ ]:


# Trial labels and colors
trial_labels = ['full', '3/4', '1/2', '1/4']
n_labels = len(trial_labels)
colors = cm.viridis(np.linspace(0.1, 0.9, n_labels))
trial_to_color = {trial: colors[i] for i, trial in enumerate(trial_labels)}

# --- Prepare points for plotting ---
plot_points = []  # (trial_label, avg_speed, WD)

for i, trial_key in enumerate(zero_trials):
    if trial_key not in trial_results:
        continue
    df = trial_results[trial_key].sort_values(['Run', 'Time'])
    trial_label = trial_labels[i]

    # Compute velocities for each run
    for run_id, run_df in df.groupby('Run'):
        run_df = run_df.sort_values('Time')
        dx = np.diff(run_df['bud_xpos'].values)
        dy = np.diff(run_df['bud_ypos'].values)
        dt = np.diff(run_df['Time'].values)

        # Avoid division by zero
        dt[dt == 0] = np.nan

        vx = dx / dt
        vy = dy / dt
        speed = np.sqrt(vx**2 + vy**2)
        avg_speed = np.nanmean(speed)

        # Take mean WD over run
        avg_wd = np.nanmean(run_df['WD'].values)

        plot_points.append((trial_label, avg_speed, avg_wd))

# --- Convert to arrays ---
trials, avg_speeds, avg_wds = zip(*plot_points)
trials = np.array(trials)
avg_speeds = np.array(avg_speeds)
avg_wds = np.array(avg_wds)

# --- Plot ---
fig, ax = plt.subplots(figsize=(8,6))
for trial_label in trial_labels:
    mask = trials == trial_label
    ax.scatter(avg_wds[mask], avg_speeds[mask],
               color=trial_to_color[trial_label],
               s=60, edgecolor='k', alpha=0.8,
               label=trial_label)

ax.set_xlabel('WD (Wasserstein distance)')
ax.set_ylabel('Average horizontal speed (sqrt(vx^2 + vy^2))')
ax.set_title('Average horizontal speed vs WD per run/trial')
ax.grid(True, alpha=0.3)
ax.legend(title='Arp2/3 variant')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Trial labels and colors
trial_labels = ['full', '3/4', '1/2', '1/4']
n_labels = len(trial_labels)
colors = cm.viridis(np.linspace(0.1, 0.9, n_labels))
trial_to_color = {trial: colors[i] for i, trial in enumerate(trial_labels)}

# --- Prepare points for plotting ---
plot_points = []  # (trial_label, avg_speed, avg_actin)

for i, trial_key in enumerate(zero_trials):
    if trial_key not in trial_results:
        continue
    df = trial_results[trial_key].sort_values(['Run', 'Time'])
    trial_label = trial_labels[i]

    # Compute velocities for each run
    for run_id, run_df in df.groupby('Run'):
        run_df = run_df.sort_values('Time')
        dx = np.diff(run_df['bud_xpos'].values)
        dy = np.diff(run_df['bud_ypos'].values)
        dt = np.diff(run_df['Time'].values)

        dt[dt == 0] = np.nan  # avoid division by zero

        vx = dx / dt
        vy = dy / dt
        speed = np.sqrt(vx**2 + vy**2)
        avg_speed = np.nanmean(speed)

        # Average actin amount
        avg_actin = np.nanmean(run_df['ActinAmount'].values)

        plot_points.append((trial_label, avg_speed, avg_actin))

# --- Convert to arrays ---
trials, avg_speeds, avg_actin = zip(*plot_points)
trials = np.array(trials)
avg_speeds = np.array(avg_speeds)
avg_actin = np.array(avg_actin)

# --- Plot ---
fig, ax = plt.subplots(figsize=(8,6))
for trial_label in trial_labels:
    mask = trials == trial_label
    ax.scatter(avg_actin[mask], avg_speeds[mask],
               color=trial_to_color[trial_label],
               s=60, edgecolor='k', alpha=0.8,
               label=trial_label)

ax.set_xlabel('Average Actin Amount per run')
ax.set_ylabel('Average horizontal speed (sqrt(vx^2 + vy^2))')
ax.set_title('Average speed vs average actin amount per run/trial')
ax.grid(True, alpha=0.3)
ax.legend(title='Arp2/3 variant')
plt.show()
